Qwen3.8-27B A/B on fixed NVARC inputs (docs/A100_PLAN.md) on L4x4: total token budget per sample, no forced final, conditions medium 32k, xhigh 32k, medium 64k, xhigh 64k.

In [ ]:
import os, time
NOTEBOOK_START = time.time()
for d in ['a100/fixed_nvarc', 'data']:  # %%writefile does not create directories
    os.makedirs(f'/kaggle/working/ab/{d}', exist_ok=True)
print('start', NOTEBOOK_START)

In [ ]:
%%writefile /kaggle/working/ab/a100/runner_parts.py
"""Verbatim parts of the hybrid5 Qwen runner (generated by scripts/build_a100_harness.py; do not edit).

source: kaggle/arc2-hybrid5-nvarc-qwen38-medium/arc2-hybrid5-nvarc-qwen38-medium.ipynb  sha256(runner)=9f5ab5a9e263aefe
"""
import hashlib, importlib.util, json, re, sys
from pathlib import Path

RUNTIME_CONFIG = {'schema': 'arc-qwen38-runtime-config-v1', 'model': {'served_name': 'Qwen/Qwen3.8-27B-FP8', 'source_hint': 'qwen3-8-27b-fp8-repacked-v1'}, 'vllm': {'version': '0.19.0', 'wheelhouse_hint': 'vllm019-cp311-cu128-wheelhouse', 'reasoning_parser': 'qwen3', 'tool_parser': 'qwen3_coder'}, 'hardware': {'gpu_count': 4, 'gpu_name_contains': 'L4', 'tensor_parallel_size': 4}, 'server': {'port': 1234, 'context_len': 65536, 'gpu_memory_utilization': 0.9}, 'scheduler': {'concurrency': 16, 'samples_per_query': 2, 'python_tool_max_concurrency': 4, 'progress_interval_s': 60}, 'sampling': {'temperature': 0.6, 'top_p': 0.95, 'top_k': 20, 'max_tokens': 57344, 'reasoning_effort': 'xhigh', 'enable_thinking': True}, 'agent': {'tools_enabled': True, 'max_turns': 10}, 'pass_policies': {'xhigh': {'reasoning_effort': 'xhigh', 'turn_max_tokens': 57344, 'direct_max_tokens': 57344, 'episode_budget_s': 3600, 'initial_predicted_s': 3600}, 'medium': {'reasoning_effort': 'medium', 'turn_max_tokens': 20480, 'direct_max_tokens': 20480, 'episode_budget_s': 1800, 'initial_predicted_s': 1800}}}

AGGREGATOR_SOURCE = '#!/usr/bin/env python3\n"""Aggregate immutable replica candidates into a complete ARC pass@2 file."""\n\nfrom __future__ import annotations\n\nimport argparse\nimport hashlib\nimport json\nimport math\nimport os\nimport re\nimport sys\nimport tempfile\nimport traceback\nimport uuid\nfrom collections import Counter\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nfrom typing import Any\n\nCOLLECTION_SCHEMA = "arc-qwen38-kaggle-collection-v1"\nCANDIDATE_SCHEMA = "arc-qwen38-kaggle-candidate-v1"\nSHA256_RE = re.compile(r"^[0-9a-f]{64}$")\n\n\ndef _stable_json(value: Any) -> str:\n    return json.dumps(value, sort_keys=True, separators=(",", ":"))\n\n\ndef _digest(value: Any) -> str:\n    return hashlib.sha256(_stable_json(value).encode()).hexdigest()\n\n\ndef _require_sha256(value: Any, *, context: str) -> str:\n    if not isinstance(value, str) or not SHA256_RE.fullmatch(value):\n        raise RuntimeError(f"{context} must be a lowercase SHA-256 digest")\n    return value\n\n\ndef _quarantine_blocking_directory(path: Path) -> Path:\n    """Move a literal destination directory aside without deleting its contents.\n\n    Kaggle work directories can survive interrupted notebook runs. If a stale\n    run leaves a *directory* at ``submission.json``, ``os.replace`` cannot\n    publish the guaranteed floor and the failure happens before the rich-path\n    degradation handler exists. Preserve the debris under a unique sibling and\n    free the required file path. Symlinks are intentionally not followed: an\n    atomic file replace safely replaces the link itself.\n    """\n    if not path.is_dir() or path.is_symlink():\n        raise NotADirectoryError(f"not a blocking literal directory: {path}")\n    for _ in range(32):\n        quarantine = path.with_name(\n            f"{path.name}.blocking-dir.{uuid.uuid4().hex}.quarantine"\n        )\n        try:\n            os.rename(path, quarantine)\n        except FileExistsError:\n            continue\n        return quarantine\n    raise RuntimeError(f"could not allocate quarantine path for {path}")\n\n\ndef _write_json(path: Path, value: Any) -> None:\n    """Atomically publish JSON via a UNIQUE same-directory temp + fsync.\n\n    A fixed ``<name>.tmp`` sibling can be stale work-directory debris (e.g. a\n    leftover *directory* at that exact path), which makes the FIRST\n    guaranteed-floor publication raise ``IsADirectoryError`` and leave no\n    ``submission.json`` at all. A unique ``mkstemp`` temp in the same directory,\n    flushed + fsynced before ``os.replace`` (and a best-effort directory fsync\n    of the rename), makes the publish crash-durable and immune to stale\n    fixed-name debris. If the destination itself is a literal directory, it is\n    quarantined non-destructively only after the replacement payload has been\n    flushed. ``os.replace`` is atomic within one filesystem.\n    """\n    path.parent.mkdir(parents=True, exist_ok=True)\n    data = json.dumps(value, indent=2, sort_keys=True) + "\\n"\n    fd, tmp_name = tempfile.mkstemp(\n        prefix=path.name + ".", suffix=".tmp", dir=str(path.parent))\n    tmp = Path(tmp_name)\n    try:\n        with os.fdopen(fd, "w") as handle:\n            handle.write(data)\n            handle.flush()\n            os.fsync(handle.fileno())\n        if path.is_dir() and not path.is_symlink():\n            _quarantine_blocking_directory(path)\n        os.replace(tmp, path)\n    except BaseException:\n        tmp.unlink(missing_ok=True)\n        raise\n    try:  # durability of the rename itself; failure here does not lose the file\n        dir_fd = os.open(str(path.parent), os.O_RDONLY)\n        try:\n            os.fsync(dir_fd)\n        finally:\n            os.close(dir_fd)\n    except OSError:\n        pass\n\n\ndef load_collection_contract(\n    output_root: Path,\n    challenges_path: Path,\n    *,\n    challenges: dict[str, Any] | None = None,\n    challenge_sha256: str | None = None,\n) -> dict[str, Any] | None:\n    """Load and verify the immutable sweep contract when candidates exist."""\n\n    path = output_root / "collection_contract.json"\n    candidate_paths = sorted((output_root / "candidates").glob("*/*.json"))\n    if not path.is_file():\n        if candidate_paths:\n            raise RuntimeError(\n                "candidate files exist without collection_contract.json; "\n                "refusing to mix unattributed or stale runs"\n            )\n        return None\n    try:\n        contract = json.loads(path.read_text())\n    except (OSError, json.JSONDecodeError) as exc:\n        raise RuntimeError(f"invalid collection contract: {exc}") from exc\n    if contract.get("schema") != COLLECTION_SCHEMA:\n        raise RuntimeError(f"unexpected collection schema: {contract.get(\'schema\')!r}")\n    collection_id = contract.get("collection_id")\n    payload = {key: value for key, value in contract.items() if key != "collection_id"}\n    if collection_id != _digest(payload):\n        raise RuntimeError("collection contract digest does not verify")\n    if challenges is None or challenge_sha256 is None:\n        challenges, challenge_sha256 = load_public_challenges_with_digest(\n            challenges_path\n        )\n    if contract.get("challenge_sha256") != challenge_sha256:\n        raise RuntimeError(\n            "collection contract challenge digest does not match aggregation input"\n        )\n    task_ids = sorted(str(task_id) for task_id in challenges)\n    if contract.get("task_count") != len(task_ids):\n        raise RuntimeError("collection contract task_count does not match challenges")\n    if contract.get("task_ids_sha256") != _digest(task_ids):\n        raise RuntimeError("collection contract task_ids_sha256 does not match challenges")\n    model = contract.get("model")\n    if not isinstance(model, dict):\n        raise RuntimeError("collection contract has no model identity")\n    if model.get("repo_id") != "Qwen/Qwen3.8-27B":\n        raise RuntimeError("collection contract binds the wrong model repository")\n    if model.get("revision") != "1d4bf0f2ff6012fd82039f2fa52739d0dd7c60c0":\n        raise RuntimeError("collection contract binds the wrong model revision")\n    _require_sha256(model.get("source_digest"), context="model.source_digest")\n    _require_sha256(\n        model.get("source_contract_sha256"),\n        context="model.source_contract_sha256",\n    )\n    _require_sha256(model.get("receipt_sha256"), context="model.receipt_sha256")\n    if model.get("weight_hashes_verified") is not True:\n        raise RuntimeError("collection contract lacks verified model weight hashes")\n    verified_file_count = model.get("verified_file_count")\n    if (\n        isinstance(verified_file_count, bool)\n        or not isinstance(verified_file_count, int)\n        or verified_file_count < 19\n    ):\n        raise RuntimeError("collection contract has an incomplete model file inventory")\n    runtime = contract.get("runtime")\n    if not isinstance(runtime, dict):\n        raise RuntimeError("collection contract has no runtime identity")\n    _require_sha256(\n        runtime.get("receipt_sha256"), context="runtime.receipt_sha256"\n    )\n    schedule = contract.get("schedule")\n    if not isinstance(schedule, dict):\n        raise RuntimeError("collection contract has no schedule")\n    if schedule.get("world_size") != 4:\n        raise RuntimeError("collection contract must use exactly four replicas")\n    for field in ("rollouts", "max_turns"):\n        value = schedule.get(field)\n        if isinstance(value, bool) or not isinstance(value, int) or value < 1:\n            raise RuntimeError(f"collection schedule {field} must be positive")\n    return contract\n\n\ndef validate_grid(grid: Any, *, context: str = "grid") -> None:\n    if not isinstance(grid, list) or not 1 <= len(grid) <= 30:\n        raise ValueError(f"{context}: invalid height")\n    if not all(isinstance(row, list) for row in grid):\n        raise ValueError(f"{context}: rows must be lists")\n    width = len(grid[0])\n    if not 1 <= width <= 30 or any(len(row) != width for row in grid):\n        raise ValueError(f"{context}: invalid rectangle")\n    if any(\n        isinstance(value, bool)\n        or not isinstance(value, int)\n        or not 0 <= value <= 9\n        for row in grid\n        for value in row\n    ):\n        raise ValueError(f"{context}: invalid color")\n\n\ndef _reject_duplicate_object_pairs(\n    pairs: list[tuple[str, Any]],\n) -> dict[str, Any]:\n    value: dict[str, Any] = {}\n    for key, item in pairs:\n        if key in value:\n            raise ValueError(f"duplicate JSON object key: {key!r}")\n        value[key] = item\n    return value\n\n\ndef validate_public_challenges(challenges: Any) -> dict[str, Any]:\n    """Accept only public ARC train labels and unlabeled test inputs."""\n\n    if not isinstance(challenges, dict) or not challenges:\n        raise ValueError("challenge JSON must be a non-empty task map")\n    for task_id, task in challenges.items():\n        if not isinstance(task_id, str) or not task_id:\n            raise ValueError("challenge task IDs must be non-empty strings")\n        context = f"challenge[{task_id!r}]"\n        if not isinstance(task, dict) or set(task) != {"train", "test"}:\n            raise ValueError(\n                f"{context}: task keys must be exactly train/test; "\n                "hidden labels or metadata are forbidden"\n            )\n        for split in ("train", "test"):\n            pairs = task[split]\n            if not isinstance(pairs, list) or not pairs:\n                raise ValueError(f"{context}.{split}: must be a non-empty list")\n            expected_keys = {"input", "output"} if split == "train" else {"input"}\n            for index, pair in enumerate(pairs):\n                pair_context = f"{context}.{split}[{index}]"\n                if not isinstance(pair, dict) or set(pair) != expected_keys:\n                    raise ValueError(\n                        f"{pair_context}: keys must be exactly "\n                        f"{sorted(expected_keys)!r}; hidden test labels are forbidden"\n                    )\n                validate_grid(pair["input"], context=f"{pair_context}.input")\n                if split == "train":\n                    validate_grid(pair["output"], context=f"{pair_context}.output")\n    return challenges\n\n\ndef load_public_challenges_with_digest(\n    path: Path,\n) -> tuple[dict[str, Any], str]:\n    try:\n        raw = path.read_bytes()\n        value = json.loads(\n            raw.decode("utf-8"), object_pairs_hook=_reject_duplicate_object_pairs\n        )\n    except (OSError, UnicodeDecodeError, json.JSONDecodeError, ValueError) as exc:\n        raise RuntimeError(f"invalid public challenge JSON: {exc}") from exc\n    try:\n        challenges = validate_public_challenges(value)\n    except ValueError as exc:\n        raise RuntimeError(f"unsafe public challenge JSON: {exc}") from exc\n    return challenges, hashlib.sha256(raw).hexdigest()\n\n\ndef load_public_challenges(path: Path) -> dict[str, Any]:\n    return load_public_challenges_with_digest(path)[0]\n\n\ndef _valid_outputs(task_id: str, task: dict[str, Any], outputs: Any) -> bool:\n    if not isinstance(outputs, list) or len(outputs) != len(task["test"]):\n        return False\n    try:\n        for index, grid in enumerate(outputs):\n            validate_grid(grid, context=f"{task_id}.candidate[{index}]")\n    except ValueError:\n        return False\n    return True\n\n\ndef _identity(task: dict[str, Any]) -> list[list[list[int]]]:\n    return [pair["input"] for pair in task["test"]]\n\n\ndef _background(task: dict[str, Any]) -> list[list[list[int]]]:\n    counts = Counter(\n        value\n        for pair in task["train"]\n        for row in pair["output"]\n        for value in row\n    )\n    color = counts.most_common(1)[0][0]\n    same_shape = all(\n        (len(pair["input"]), len(pair["input"][0]))\n        == (len(pair["output"]), len(pair["output"][0]))\n        for pair in task["train"]\n    )\n    shapes = {\n        (len(pair["output"]), len(pair["output"][0])) for pair in task["train"]\n    }\n    fixed = next(iter(shapes)) if len(shapes) == 1 else None\n    rows = []\n    for pair in task["test"]:\n        if same_shape or fixed is None:\n            height, width = len(pair["input"]), len(pair["input"][0])\n        else:\n            height, width = fixed\n        rows.append([[color] * width for _ in range(height)])\n    return rows\n\n\ndef _geometric_grid(source: str, grid: list[list[int]]) -> list[list[int]]:\n    if source == "fallback-flip-lr":\n        return [list(reversed(row)) for row in grid]\n    if source == "fallback-flip-ud":\n        return list(reversed([row[:] for row in grid]))\n    if source == "fallback-transpose":\n        return [list(row) for row in zip(*grid)]\n    if source == "fallback-rotate-cw":\n        return [list(row) for row in zip(*reversed(grid))]\n    if source == "fallback-rotate-ccw":\n        return [list(row) for row in reversed(list(zip(*grid)))]\n    if source == "fallback-rotate-180":\n        return [list(reversed(row)) for row in reversed(grid)]\n    raise ValueError(f"unknown geometric fallback: {source}")\n\n\ndef _constant_color(task: dict[str, Any], color: int) -> list[list[list[int]]]:\n    same_shape = all(\n        (len(pair["input"]), len(pair["input"][0]))\n        == (len(pair["output"]), len(pair["output"][0]))\n        for pair in task["train"]\n    )\n    shapes = {\n        (len(pair["output"]), len(pair["output"][0])) for pair in task["train"]\n    }\n    fixed = next(iter(shapes)) if len(shapes) == 1 else None\n    rows = []\n    for pair in task["test"]:\n        if same_shape or fixed is None:\n            height, width = len(pair["input"]), len(pair["input"][0])\n        else:\n            height, width = fixed\n        rows.append([[color] * width for _ in range(height)])\n    return rows\n\n\ndef _fallback_outputs(\n    task: dict[str, Any], source: str\n) -> list[list[list[int]]]:\n    if source == "fallback-identity":\n        return _identity(task)\n    if source == "fallback-background":\n        return _background(task)\n    if source.startswith("fallback-constant-"):\n        return _constant_color(task, int(source.rsplit("-", 1)[-1]))\n    return [_geometric_grid(source, pair["input"]) for pair in task["test"]]\n\n\ndef _fallback_demo_score(\n    task: dict[str, Any], source: str\n) -> tuple[bool, float]:\n    if source == "fallback-identity":\n        matches = [pair["input"] == pair["output"] for pair in task["train"]]\n    elif source == "fallback-background":\n        counts = Counter(\n            value\n            for pair in task["train"]\n            for row in pair["output"]\n            for value in row\n        )\n        color = counts.most_common(1)[0][0]\n        same_shape = all(\n            (len(pair["input"]), len(pair["input"][0]))\n            == (len(pair["output"]), len(pair["output"][0]))\n            for pair in task["train"]\n        )\n        shapes = {\n            (len(pair["output"]), len(pair["output"][0]))\n            for pair in task["train"]\n        }\n        fixed = next(iter(shapes)) if len(shapes) == 1 else None\n        matches = []\n        for pair in task["train"]:\n            if same_shape or fixed is None:\n                height, width = len(pair["input"]), len(pair["input"][0])\n            else:\n                height, width = fixed\n            matches.append([[color] * width for _ in range(height)] == pair["output"])\n    elif source.startswith("fallback-constant-"):\n        color = int(source.rsplit("-", 1)[-1])\n        same_shape = all(\n            (len(pair["input"]), len(pair["input"][0]))\n            == (len(pair["output"]), len(pair["output"][0]))\n            for pair in task["train"]\n        )\n        shapes = {\n            (len(pair["output"]), len(pair["output"][0]))\n            for pair in task["train"]\n        }\n        fixed = next(iter(shapes)) if len(shapes) == 1 else None\n        matches = []\n        for pair in task["train"]:\n            if same_shape or fixed is None:\n                height, width = len(pair["input"]), len(pair["input"][0])\n            else:\n                height, width = fixed\n            matches.append([[color] * width for _ in range(height)] == pair["output"])\n    else:\n        matches = [\n            _geometric_grid(source, pair["input"]) == pair["output"]\n            for pair in task["train"]\n        ]\n    return all(matches), sum(matches) / len(matches)\n\n\ndef _trajectory_id(row: dict[str, Any]) -> tuple[int, int] | None:\n    rank = row.get("rank")\n    rollout = row.get("rollout")\n    if not isinstance(rank, int) or isinstance(rank, bool):\n        return None\n    if not isinstance(rollout, int) or isinstance(rollout, bool):\n        return None\n    return rank, rollout\n\n\ndef _trajectory_candidate_key(\n    row: dict[str, Any],\n) -> tuple[bool, float, int, int, str]:\n    turn = row.get("turn")\n    turn_value = turn if isinstance(turn, int) and not isinstance(turn, bool) else -1\n    return (\n        row.get("visible_pass") is True,\n        float(row.get("demo_score") or 0.0),\n        turn_value,\n        -int(row.get("program_cost") or 0),\n        str(row.get("run_id") or ""),\n    )\n\n\ndef _validate_selection_candidates(\n    task_id: str, task: dict[str, Any], candidates: list[dict[str, Any]]\n) -> None:\n    """Reject malformed in-memory candidates before ranking or clustering."""\n\n    seen_run_ids: set[str] = set()\n    for index, row in enumerate(candidates):\n        context = f"{task_id}.candidates[{index}]"\n        if not isinstance(row, dict):\n            raise RuntimeError(f"{context}: candidate must be an object")\n        if _trajectory_id(row) is None:\n            raise RuntimeError(f"{context}: candidate has no valid rank/rollout identity")\n        turn = row.get("turn")\n        if turn is not None and (\n            isinstance(turn, bool) or not isinstance(turn, int) or turn < 0\n        ):\n            raise RuntimeError(f"{context}: turn must be a non-negative integer")\n        run_id = row.get("run_id")\n        if not isinstance(run_id, str) or not run_id:\n            raise RuntimeError(f"{context}: run_id must be non-empty")\n        if run_id in seen_run_ids:\n            raise RuntimeError(f"{context}: duplicate run_id {run_id!r}")\n        seen_run_ids.add(run_id)\n        if not isinstance(row.get("visible_pass"), bool):\n            raise RuntimeError(f"{context}: visible_pass must be boolean")\n        demo_score = row.get("demo_score")\n        if (\n            isinstance(demo_score, bool)\n            or not isinstance(demo_score, (int, float))\n            or not math.isfinite(float(demo_score))\n            or not 0.0 <= float(demo_score) <= 1.0\n        ):\n            raise RuntimeError(f"{context}: demo_score must be finite in 0..1")\n        program_cost = row.get("program_cost")\n        if (\n            isinstance(program_cost, bool)\n            or not isinstance(program_cost, int)\n            or program_cost < 0\n        ):\n            raise RuntimeError(f"{context}: program_cost must be non-negative")\n        if not _valid_outputs(task_id, task, row.get("test_outputs")):\n            raise RuntimeError(f"{context}: candidate test outputs are invalid")\n\n\ndef _trajectory_representatives(\n    candidates: list[dict[str, Any]],\n) -> list[dict[str, Any]]:\n    """Choose one vote-bearing candidate per independent trajectory."""\n\n    best: dict[tuple[int, int], dict[str, Any]] = {}\n    for row in candidates:\n        trajectory = _trajectory_id(row)\n        if trajectory is None:\n            continue\n        current = best.get(trajectory)\n        if current is None or _trajectory_candidate_key(row) > _trajectory_candidate_key(\n            current\n        ):\n            best[trajectory] = row\n    return [best[key] for key in sorted(best)]\n\n\ndef _selection_key(row: dict[str, Any]) -> tuple[Any, ...]:\n    """Order hypotheses by visible evidence without conflating repairs.\n\n    ``support`` contains only the one vote-bearing state selected from each\n    independent ``(rank, rollout)`` trajectory.  Older repair states remain\n    useful pass@2 hypotheses, but their raw provenance can break ties only\n    after independent support and can never manufacture another vote.\n    """\n\n    return (\n        row["visible_pass"],\n        row["demo_score"],\n        row["visible_pass_support"],\n        row["support"],\n        row["model_derived"],\n        row["raw_visible_pass_trajectory_support"],\n        row["raw_trajectory_support"],\n        row["replica_support"],\n        row["rollout_support"],\n        -row["program_cost"],\n        row["representative_id"],\n    )\n\n\ndef load_candidates(\n    output_root: Path,\n    task_id: str,\n    task: dict[str, Any],\n    *,\n    collection_contract: dict[str, Any] | None = None,\n) -> list[dict[str, Any]]:\n    rows = []\n    directory = output_root / "candidates" / task_id\n    if not directory.is_dir():\n        return rows\n    if collection_contract is None:\n        raise RuntimeError(\n            f"{task_id}: candidate directory exists without a verified collection contract"\n        )\n    collection_id = collection_contract["collection_id"]\n    contract_sha256 = _digest(collection_contract)\n    seen_run_ids: set[str] = set()\n    schedule = collection_contract["schedule"]\n    world_size = int(schedule["world_size"])\n    rollouts = int(schedule["rollouts"])\n    max_turns = int(schedule["max_turns"])\n    for path in sorted(directory.glob("*.json")):\n        try:\n            row = json.loads(path.read_text())\n        except (OSError, json.JSONDecodeError) as exc:\n            raise RuntimeError(f"{path}: invalid candidate JSON: {exc}") from exc\n        if not isinstance(row, dict):\n            raise RuntimeError(f"{path}: candidate must be an object")\n        if row.get("schema") != CANDIDATE_SCHEMA:\n            raise RuntimeError(f"{path}: unexpected candidate schema")\n        declared_record_sha256 = _require_sha256(\n            row.get("candidate_record_sha256"),\n            context=f"{path}: candidate_record_sha256",\n        )\n        record_payload = {\n            key: value\n            for key, value in row.items()\n            if key != "candidate_record_sha256"\n        }\n        if declared_record_sha256 != _digest(record_payload):\n            raise RuntimeError(f"{path}: candidate record digest does not verify")\n        if row.get("collection_id") != collection_id:\n            raise RuntimeError(\n                f"{path}: candidate belongs to a different or unattributed collection"\n            )\n        if row.get("collection_contract_sha256") != contract_sha256:\n            raise RuntimeError(f"{path}: candidate collection-contract digest mismatch")\n        if row.get("challenge_sha256") != collection_contract["challenge_sha256"]:\n            raise RuntimeError(f"{path}: candidate challenge digest mismatch")\n        if row.get("model") != collection_contract["model"]:\n            raise RuntimeError(f"{path}: candidate model identity mismatch")\n        if row.get("runtime_receipt_sha256") != collection_contract["runtime"].get(\n            "receipt_sha256"\n        ):\n            raise RuntimeError(f"{path}: candidate runtime identity mismatch")\n        if row.get("task_id") != task_id:\n            raise RuntimeError(f"{path}: candidate task ID does not match its directory")\n        trajectory = _trajectory_id(row)\n        if trajectory is None:\n            raise RuntimeError(f"{path}: candidate has no valid rank/rollout identity")\n        rank, rollout = trajectory\n        turn = row.get("turn")\n        if not 0 <= rank < world_size:\n            raise RuntimeError(f"{path}: rank is outside the collection world size")\n        if not 0 <= rollout < rollouts:\n            raise RuntimeError(f"{path}: rollout is outside the collection schedule")\n        if isinstance(turn, bool) or not isinstance(turn, int) or not 0 <= turn < max_turns:\n            raise RuntimeError(f"{path}: turn is outside the collection schedule")\n        expected_name = f"rank{rank}-rollout{rollout}-turn{turn}.json"\n        if path.name != expected_name:\n            raise RuntimeError(\n                f"{path}: candidate filename does not match trajectory identity"\n            )\n        expected_run_id = f"{task_id}-rank{rank}-rollout{rollout}-turn{turn}"\n        run_id = row.get("run_id")\n        if run_id != expected_run_id:\n            raise RuntimeError(f"{path}: candidate run_id does not match trajectory")\n        if run_id in seen_run_ids:\n            raise RuntimeError(f"{path}: duplicate candidate run_id {run_id!r}")\n        seen_run_ids.add(run_id)\n        if row.get("source") != "qwen38-nf4-replica":\n            raise RuntimeError(f"{path}: unexpected candidate source")\n        if not isinstance(row.get("visible_pass"), bool):\n            raise RuntimeError(f"{path}: visible_pass must be boolean")\n        demo_score = row.get("demo_score")\n        if (\n            isinstance(demo_score, bool)\n            or not isinstance(demo_score, (int, float))\n            or not math.isfinite(float(demo_score))\n            or not 0.0 <= float(demo_score) <= 1.0\n        ):\n            raise RuntimeError(f"{path}: demo_score must be finite in 0..1")\n        program = row.get("program")\n        if not isinstance(program, str) or not program.strip():\n            raise RuntimeError(f"{path}: program must be non-empty")\n        program_sha256 = _require_sha256(\n            row.get("program_sha256"), context=f"{path}: program_sha256"\n        )\n        if program_sha256 != hashlib.sha256(program.encode()).hexdigest():\n            raise RuntimeError(f"{path}: program_sha256 does not match program bytes")\n        program_cost = row.get("program_cost")\n        if (\n            isinstance(program_cost, bool)\n            or not isinstance(program_cost, int)\n            or program_cost != len(program)\n        ):\n            raise RuntimeError(f"{path}: program_cost does not match program length")\n        if not _valid_outputs(task_id, task, row.get("test_outputs")):\n            raise RuntimeError(f"{path}: candidate test outputs are invalid")\n        test_outputs_sha256 = _require_sha256(\n            row.get("test_outputs_sha256"),\n            context=f"{path}: test_outputs_sha256",\n        )\n        if test_outputs_sha256 != _digest(row["test_outputs"]):\n            raise RuntimeError(\n                f"{path}: test_outputs_sha256 does not match test outputs"\n            )\n        rows.append(row)\n    return rows\n\n\ndef validate_submission_completeness(\n    challenges: dict[str, Any], submission: Any\n) -> None:\n    """Assert a submission is structurally complete and grid-valid.\n\n    The single reusable definition of "a submittable file": exactly the\n    challenge task set, one attempt object per test query, each object exactly\n    {attempt_1, attempt_2}, every attempt a valid ARC grid (1..30 dims, colours\n    0..9), and the two attempts distinct. Reused by :func:`aggregate`, the\n    last-resort fallback path, and the offline robustness harness so the\n    guarantee has one source of truth.\n    """\n\n    if not isinstance(submission, dict):\n        raise ValueError("submission must be a task->rows object")\n    if set(submission) != set(challenges):\n        missing = sorted(set(challenges) - set(submission))\n        extra = sorted(set(submission) - set(challenges))\n        raise ValueError(\n            f"submission task set mismatch (missing={missing}, extra={extra})"\n        )\n    for task_id, task in challenges.items():\n        rows = submission[task_id]\n        if not isinstance(rows, list) or len(rows) != len(task["test"]):\n            raise ValueError(f"{task_id}: wrong submission output count")\n        for index, row in enumerate(rows):\n            if not isinstance(row, dict) or set(row) != {"attempt_1", "attempt_2"}:\n                raise ValueError(f"{task_id}[{index}]: malformed attempt object")\n            for name in ("attempt_1", "attempt_2"):\n                validate_grid(row[name], context=f"{task_id}[{index}].{name}")\n            if row["attempt_1"] == row["attempt_2"]:\n                raise ValueError(f"{task_id}[{index}]: duplicate pass@2 attempts")\n\n\ndef guaranteed_fallback_submission(\n    challenges: dict[str, Any],\n) -> dict[str, list[dict[str, Any]]]:\n    """Build a structurally-complete submission that cannot fail to be valid.\n\n    This is the robustness floor for the BASE: it depends only on the already\n    validated public test *inputs* (never on train outputs, the collection\n    contract, model candidates, or the rich selection machinery), so it can be\n    written to disk before the model even loads and can never itself raise for a\n    validated challenge set. attempt_1 is the test input (identity); attempt_2\n    is the first distinct dihedral transform. If the input is invariant under\n    every such transform, one cell is changed to keep pass@2 from wasting its\n    second slot on an exact duplicate. A better submission from\n    :func:`aggregate` overwrites it on success; if every richer path fails, this\n    remains and the run still submits a valid pass@2 file.\n    """\n\n    submission: dict[str, list[dict[str, Any]]] = {}\n    for task_id, task in challenges.items():\n        rows: list[dict[str, Any]] = []\n        for pair in task["test"]:\n            grid = pair["input"]\n            candidates = [\n                _geometric_grid("fallback-flip-lr", grid),\n                _geometric_grid("fallback-flip-ud", grid),\n                _geometric_grid("fallback-rotate-180", grid),\n                _geometric_grid("fallback-transpose", grid),\n            ]\n            second = next(\n                (candidate for candidate in candidates if candidate != grid),\n                None,\n            )\n            if second is None:\n                second = [row[:] for row in grid]\n                second[0][0] = (second[0][0] + 1) % 10\n            rows.append({"attempt_1": grid, "attempt_2": second})\n        submission[task_id] = rows\n    # Fail loud if this floor is ever not valid: it means the input contract\n    # was violated upstream, which must surface, not be silently shipped.\n    validate_submission_completeness(challenges, submission)\n    return submission\n\n\ndef select_task(\n    task_id: str, task: dict[str, Any], candidates: list[dict[str, Any]]\n) -> tuple[list[dict[str, Any]], dict[str, Any]]:\n    fallback_sources = (\n        "fallback-identity",\n        "fallback-background",\n        "fallback-flip-lr",\n        "fallback-flip-ud",\n        "fallback-transpose",\n        "fallback-rotate-cw",\n        "fallback-rotate-ccw",\n        "fallback-rotate-180",\n    ) + tuple(f"fallback-constant-{color}" for color in range(10))\n    fallback_outputs = {\n        source: _fallback_outputs(task, source) for source in fallback_sources\n    }\n    _validate_selection_candidates(task_id, task, candidates)\n    submission_rows = []\n    query_receipts = []\n    trajectory_candidates = _trajectory_representatives(candidates)\n    for query_index in range(len(task["test"])):\n        # One trajectory can revise its output over several turns. Select its\n        # strongest state before clustering so it cannot vote for competing\n        # hypotheses at the same cutoff.\n        groups: dict[str, list[dict[str, Any]]] = {}\n        for candidate in trajectory_candidates:\n            grid = candidate["test_outputs"][query_index]\n            groups.setdefault(_digest(grid), []).append(candidate)\n        raw_groups: dict[str, list[dict[str, Any]]] = {}\n        for candidate in candidates:\n            grid = candidate["test_outputs"][query_index]\n            raw_groups.setdefault(_digest(grid), []).append(candidate)\n        ranked = []\n        for output_sha256, raw_members in raw_groups.items():\n            members = groups.get(output_sha256, [])\n            # A vote-bearing representative must come from the trajectory\n            # representatives themselves.  Otherwise a superseded state with\n            # the same test output could silently inflate attempt-1\'s visible\n            # score.  Correlated-only clusters have no such member, so their\n            # best raw state is used for attempt-2 ranking only.\n            representative = max(\n                members or raw_members,\n                key=_trajectory_candidate_key,\n            )\n            replicas = sorted(\n                {\n                    int(row["rank"])\n                    for row in members\n                    if isinstance(row.get("rank"), int)\n                }\n            )\n            rollouts = sorted(\n                {\n                    int(row["rollout"])\n                    for row in members\n                    if isinstance(row.get("rollout"), int)\n                }\n            )\n            trajectories = {\n                (int(row.get("rank", -1)), int(row.get("rollout", -1)))\n                for row in members\n            }\n            visible_trajectories = {\n                (int(row.get("rank", -1)), int(row.get("rollout", -1)))\n                for row in members\n                if row.get("visible_pass") is True\n            }\n            raw_trajectories = {\n                (int(row.get("rank", -1)), int(row.get("rollout", -1)))\n                for row in raw_members\n            }\n            raw_visible_trajectories = {\n                (int(row.get("rank", -1)), int(row.get("rollout", -1)))\n                for row in raw_members\n                if row.get("visible_pass") is True\n            }\n            correlated_only = not members\n            ranked.append(\n                {\n                    "grid": representative["test_outputs"][query_index],\n                    "output_sha256": output_sha256,\n                    "source": str(\n                        representative.get("source") or "qwen38-nf4-replica"\n                    ),\n                    "representative_id": str(representative.get("run_id") or ""),\n                    "visible_pass": representative.get("visible_pass") is True,\n                    "demo_score": float(representative.get("demo_score") or 0.0),\n                    "program_cost": int(representative.get("program_cost") or 0),\n                    "model_derived": True,\n                    "correlated_only": correlated_only,\n                    "attempt_1_eligible": not correlated_only,\n                    "hypothesis_origin": (\n                        "correlated-repair-state"\n                        if correlated_only\n                        else "independent-trajectory-vote"\n                    ),\n                    # Multiple repair turns within one rollout are correlated\n                    # and count as one vote. Consensus means independent\n                    # rank/rollout trajectories, not raw candidate files.\n                    "support": len(trajectories),\n                    "independent_support": len(trajectories),\n                    "raw_candidate_support": len(raw_members),\n                    "raw_trajectory_support": len(raw_trajectories),\n                    "raw_visible_pass_trajectory_support": len(\n                        raw_visible_trajectories\n                    ),\n                    "visible_pass_support": len(visible_trajectories),\n                    "replica_support": len(replicas),\n                    "rollout_support": len(rollouts),\n                    "supporting_replicas": replicas,\n                    "supporting_rollouts": rollouts,\n                    "supporting_ids": sorted(\n                        str(row.get("run_id") or "") for row in members\n                    ),\n                    "raw_supporting_ids": sorted(\n                        str(row.get("run_id") or "") for row in raw_members\n                    ),\n                }\n            )\n        # Visible evidence is primary.  Among equally fitting programs, the\n        # output with the most independent support wins attempt 1.  Attempt 2\n        # is the strongest distinct cluster, maximizing pass@2 diversity.\n        ranked.sort(key=_selection_key, reverse=True)\n        for source in fallback_sources:\n            grid = fallback_outputs[source][query_index]\n            fallback_pass, fallback_score = _fallback_demo_score(task, source)\n            ranked.append(\n                {\n                    "grid": grid,\n                    "output_sha256": _digest(grid),\n                    "source": source,\n                    "representative_id": source,\n                    "visible_pass": fallback_pass,\n                    "demo_score": fallback_score,\n                    "program_cost": 0,\n                    "model_derived": False,\n                    "correlated_only": False,\n                    "attempt_1_eligible": True,\n                    "hypothesis_origin": "deterministic-fallback",\n                    "support": 0,\n                    "independent_support": 0,\n                    "raw_candidate_support": 0,\n                    "raw_trajectory_support": 0,\n                    "raw_visible_pass_trajectory_support": 0,\n                    "visible_pass_support": 0,\n                    "replica_support": 0,\n                    "rollout_support": 0,\n                    "supporting_replicas": [],\n                    "supporting_rollouts": [],\n                    "supporting_ids": [],\n                    "raw_supporting_ids": [],\n                }\n            )\n        ranked.sort(key=_selection_key, reverse=True)\n\n        # A superseded repair state is deliberately ineligible for attempt 1:\n        # its trajectory cast its sole independent vote for another state.\n        # It remains eligible for attempt 2, where preserving a strong,\n        # distinct model-derived hypothesis is preferable to erasing it and\n        # falling back to an unrelated geometric guess.\n        attempt_1_pool = [row for row in ranked if row["attempt_1_eligible"]]\n        if not attempt_1_pool:\n            raise RuntimeError(\n                f"{task_id}.test[{query_index}]: no attempt-1-eligible hypothesis"\n            )\n        selected = [attempt_1_pool[0]]\n        seen = {attempt_1_pool[0]["output_sha256"]}\n        for row in ranked:\n            if row["output_sha256"] in seen:\n                continue\n            selected.append(row)\n            seen.add(row["output_sha256"])\n            if len(selected) == 2:\n                break\n        if len(selected) != 2:\n            raise RuntimeError(\n                f"{task_id}.test[{query_index}]: fewer than two distinct "\n                "evidence-ranked output hypotheses survived selection"\n            )\n        submission_rows.append(\n            {"attempt_1": selected[0]["grid"], "attempt_2": selected[1]["grid"]}\n        )\n        query_receipts.append(\n            {\n                "query_index": query_index,\n                "candidate_clusters": len(groups),\n                "raw_candidate_clusters": len(raw_groups),\n                "correlated_only_candidate_clusters": len(\n                    set(raw_groups) - set(groups)\n                ),\n                "selected": [\n                    {key: row[key] for key in row if key != "grid"}\n                    for row in selected\n                ],\n            }\n        )\n    return submission_rows, {\n        "candidate_count": len(candidates),\n        "trajectory_candidate_count": len(trajectory_candidates),\n        "visible_pass_candidates": sum(\n            row.get("visible_pass") is True for row in candidates\n        ),\n        "queries": query_receipts,\n    }\n\n\ndef aggregate(\n    challenges_path: Path, output_root: Path, submission_path: Path\n) -> tuple[dict[str, Any], dict[str, Any]]:\n    challenges, challenge_sha256 = load_public_challenges_with_digest(\n        challenges_path\n    )\n    collection_contract = load_collection_contract(\n        output_root,\n        challenges_path,\n        challenges=challenges,\n        challenge_sha256=challenge_sha256,\n    )\n    submission = {}\n    tasks = {}\n    for task_id, task in challenges.items():\n        candidates = load_candidates(\n            output_root,\n            task_id,\n            task,\n            collection_contract=collection_contract,\n        )\n        rows, receipt = select_task(task_id, task, candidates)\n        submission[task_id] = rows\n        tasks[task_id] = receipt\n    validate_submission_completeness(challenges, submission)\n    _write_json(submission_path, submission)\n    selected_sources = Counter(\n        selected["source"]\n        for task in tasks.values()\n        for query in task["queries"]\n        for selected in query["selected"]\n    )\n    receipt = {\n        "schema": "arc-qwen38-kaggle-pass2-aggregation-v1",\n        "observed_utc": datetime.now(timezone.utc).isoformat(),\n        "selection_scope": "per-test-query",\n        "collection_id": (\n            collection_contract["collection_id"] if collection_contract else None\n        ),\n        "collection_contract_sha256": (\n            _digest(collection_contract) if collection_contract else None\n        ),\n        "selection_policy": (\n            "attempt-1 excludes superseded repair states; then visible-pass, "\n            "demo-score, independent visible-pass support, independent "\n            "trajectory support, model-derived provenance, raw trajectory "\n            "provenance, replica/rollout spread, shorter-program; attempt-2 "\n            "is the strongest distinct hypothesis including zero-vote "\n            "superseded repair states"\n        ),\n        "challenge_sha256": challenge_sha256,\n        "submission_sha256": hashlib.sha256(submission_path.read_bytes()).hexdigest(),\n        "task_count": len(challenges),\n        "query_count": sum(len(task["test"]) for task in challenges.values()),\n        "candidate_count": sum(row["candidate_count"] for row in tasks.values()),\n        "tasks_with_candidate": sum(row["candidate_count"] > 0 for row in tasks.values()),\n        "tasks_with_visible_pass_candidate": sum(\n            row["visible_pass_candidates"] > 0 for row in tasks.values()\n        ),\n        "selected_source_counts": dict(sorted(selected_sources.items())),\n        "tasks": tasks,\n        "status": "pass",\n    }\n    _write_json(output_root / "aggregation_receipt.json", receipt)\n    return submission, receipt\n\n\ndef _emit_summary(summary: dict[str, Any]) -> None:\n    print(\n        "QWEN38_PASS2_AGGREGATION=" + json.dumps(summary, sort_keys=True),\n        flush=True,\n    )\n\n\ndef main() -> int:\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--challenges", type=Path, required=True)\n    parser.add_argument("--output-root", type=Path, required=True)\n    parser.add_argument("--submission", type=Path, required=True)\n    parser.add_argument(\n        "--guarantee-only",\n        action="store_true",\n        help=(\n            "Write only the dependency-free guaranteed-fallback submission and "\n            "exit 0. The kernel calls this first so a valid, complete "\n            "submission exists on disk before the model loads or any candidate "\n            "is produced."\n        ),\n    )\n    args = parser.parse_args()\n\n    # ROBUSTNESS FLOOR: a valid, complete submission must exist on disk no\n    # matter what fails later.  Build it from the validated public inputs and\n    # write it before anything richer is attempted.  If the challenges\n    # themselves are unreadable there is genuinely nothing to submit, so that\n    # (and only that) is allowed to raise.\n    challenges = load_public_challenges(args.challenges)\n    _write_json(args.submission, guaranteed_fallback_submission(challenges))\n\n    base_summary = {\n        "schema": "arc-qwen38-kaggle-pass2-aggregation-v1",\n        "task_count": len(challenges),\n        "query_count": sum(len(task["test"]) for task in challenges.values()),\n    }\n    if args.guarantee_only:\n        _emit_summary({**base_summary, "status": "guaranteed-fallback"})\n        return 0\n\n    try:\n        _, receipt = aggregate(args.challenges, args.output_root, args.submission)\n        # Defensive: whatever landed on disk must be a complete, valid file.\n        validate_submission_completeness(\n            challenges, json.loads(args.submission.read_text())\n        )\n    except Exception as exc:  # never let aggregation abort the submit\n        # Fail LOUD but keep submitting: rewrite the guaranteed floor and record\n        # the failure so a degraded run is visible, not silently shipped.\n        degraded = {\n            "schema": "arc-qwen38-kaggle-aggregation-degraded-v1",\n            "observed_utc": datetime.now(timezone.utc).isoformat(),\n            "error": f"{type(exc).__name__}: {exc}",\n            "traceback": traceback.format_exc(),\n            "fallback": "guaranteed-fallback-submission-retained",\n        }\n        try:\n            _write_json(\n                args.output_root / "aggregation_degraded_receipt.json", degraded\n            )\n        except OSError:\n            pass\n        _write_json(args.submission, guaranteed_fallback_submission(challenges))\n        print(\n            "LOUD: aggregation failed; retained guaranteed-fallback submission. "\n            f"{type(exc).__name__}: {exc}",\n            file=sys.stderr,\n            flush=True,\n        )\n        _emit_summary(\n            {\n                **base_summary,\n                "status": "degraded-guaranteed-fallback",\n                "error": f"{type(exc).__name__}: {exc}",\n            }\n        )\n        return 0\n\n    _emit_summary(\n        {\n            key: receipt[key]\n            for key in (\n                "schema",\n                "observed_utc",\n                "collection_id",\n                "submission_sha256",\n                "task_count",\n                "query_count",\n                "candidate_count",\n                "tasks_with_candidate",\n                "tasks_with_visible_pass_candidate",\n                "selected_source_counts",\n                "status",\n            )\n        }\n    )\n    return 0\n\n\nif __name__ == "__main__":\n    raise SystemExit(main())\n'

SANDBOX_SOURCE = '#!/usr/bin/env python3\n"""Resource-bounded Python executor for the model\'s code-interpreter tool.\n\nThis is a resource sandbox, not a security boundary: model-written code retains\nnormal filesystem and network access. Each call runs in a fresh process session\nand temporary directory with bounded address space, CPU time, wall time,\nper-file size, open files, and captured output.\nThe parent kills the call\'s original process group after the leader exits, so\nordinary forked background work is cleaned up after both success and timeout.\nDeliberately evasive code can escape these controls because this is not a\nsecurity boundary.\n\nThe file-size rlimit is per file rather than an aggregate disk quota. Combined\nwith the throwaway working directory, process-group cleanup, and wall deadline,\nit limits the practical damage from accidental writes but should not be treated\nas filesystem isolation. RLIMIT_AS is per process rather than an aggregate\ncgroup memory cap, and same-UID processes are not isolated. Defaults are sized\nwell under a 125 GB host:\n4 GiB address space, 12 CPU-s, 20 s wall, 16 MiB per file, and 8,000 retained\noutput characters.\n\nRLIMIT_NPROC is deliberately not used. On Linux it counts tasks for the whole\nUID, not descendants of this tool call; a busy notebook/vLLM user can already\nbe above a small limit and then lose even benign subprocess support. Aggregate\ntool memory is instead bounded by the notebook-wide tool-call semaphore.\n"""\nfrom __future__ import annotations\n\nimport json\nimport os\nimport signal\nimport subprocess\nimport sys\nimport tempfile\nimport threading\nimport time\nfrom dataclasses import dataclass\n\nDEFAULT_LIMITS = {\n    "address_space_bytes": 4 * 1024**3,\n    "cpu_seconds": 12,\n    "wall_seconds": 20,\n    "file_size_bytes": 16 * 1024**2,\n    "max_output_chars": 8000,\n    "max_open_files": 64,\n}\n\n_INTEGER_LIMITS = {\n    "address_space_bytes",\n    "cpu_seconds",\n    "file_size_bytes",\n    "max_output_chars",\n    "max_open_files",\n}\n\n# The child bootstrap sets rlimits before loading task context. User stdout and\n# stderr are written into a bounded stream, not an unbounded StringIO later\n# sliced down. A dedicated inherited fd carries one authenticated-by-structure\n# result record; this is protocol separation, not protection from hostile code.\n_CHILD = r\'\'\'\nimport sys, json, io, os, resource, contextlib\n\nlimits = json.loads(sys.argv[1])\nresult_fd = int(sys.argv[2])\n\nrlimits = [\n    ("RLIMIT_AS", "address_space_bytes"),\n    ("RLIMIT_CPU", "cpu_seconds"),\n    ("RLIMIT_FSIZE", "file_size_bytes"),\n    ("RLIMIT_NOFILE", "max_open_files"),\n]\nfor resource_name, key in rlimits:\n    resource_id = getattr(resource, resource_name, None)\n    if resource_id is None:\n        continue\n    value = int(limits[key])\n    try:\n        resource.setrlimit(resource_id, (value, value))\n    except (ValueError, OSError):\n        pass\nif hasattr(resource, "RLIMIT_CORE"):\n    try:\n        resource.setrlimit(resource.RLIMIT_CORE, (0, 0))\n    except (ValueError, OSError):\n        pass\n\nclass CappedTextIO(io.TextIOBase):\n    encoding = "utf-8"\n\n    def __init__(self, cap):\n        self.cap = int(cap)\n        self.parts = []\n        self.retained = 0\n        self.truncated = False\n\n    def writable(self):\n        return True\n\n    def write(self, value):\n        if not isinstance(value, str):\n            value = str(value)\n        original_length = len(value)\n        remaining = self.cap - self.retained\n        if remaining > 0:\n            kept = value[:remaining]\n            self.parts.append(kept)\n            self.retained += len(kept)\n        if original_length > max(remaining, 0):\n            self.truncated = True\n        return original_length\n\n    def flush(self):\n        return None\n\n    def getvalue(self):\n        value = "".join(self.parts)\n        if self.truncated:\n            value += "\\n...[output truncated at %d chars]" % self.cap\n        return value\n\npayload = json.load(sys.stdin)\ng = {"__name__": "__main__"}\ng.update(payload.get("context") or {})\nbuf = CappedTextIO(limits["max_output_chars"])\nresult = {"ok": True, "error": None, "stdout": ""}\ntry:\n    with contextlib.redirect_stdout(buf), contextlib.redirect_stderr(buf):\n        exec(compile(payload["code"], "<model_code>", "exec"), g)\nexcept MemoryError:\n    result["ok"] = False\n    result["error"] = "MemoryError: hit the memory cap"\nexcept SystemExit as exc:\n    result["ok"] = False\n    result["error"] = "SystemExit(%r)" % (exc.code,)\nexcept BaseException as exc:\n    result["ok"] = False\n    result["error"] = "%s: %s" % (type(exc).__name__, str(exc)[:800])\nresult["stdout"] = buf.getvalue()\nrecord = ("RESULT:" + json.dumps(result, separators=(",", ":"))).encode()\nos.write(result_fd, record)\n\'\'\'\n\n\n@dataclass\nclass ExecResult:\n    ok: bool\n    stdout: str\n    error: str | None\n    timed_out: bool\n    killed_signal: int | None\n    wall_s: float\n\n\ndef _normalize_limits(overrides: dict | None) -> dict:\n    if overrides is not None and not isinstance(overrides, dict):\n        raise TypeError("limits must be a mapping or None")\n    unknown = sorted(set(overrides or {}) - set(DEFAULT_LIMITS))\n    if unknown:\n        raise ValueError(f"unknown resource limit(s): {unknown}")\n    limits = {**DEFAULT_LIMITS, **(overrides or {})}\n    for name in _INTEGER_LIMITS:\n        value = limits[name]\n        if isinstance(value, bool) or not isinstance(value, int) or value <= 0:\n            raise ValueError(f"{name} must be a positive integer, got {value!r}")\n    wall = limits["wall_seconds"]\n    if isinstance(wall, bool) or not isinstance(wall, (int, float)) or wall <= 0:\n        raise ValueError(f"wall_seconds must be positive, got {wall!r}")\n    limits["wall_seconds"] = float(wall)\n    return limits\n\n\ndef _kill_process_group(process_group_id: int) -> None:\n    """Best-effort cleanup for the leader and descendants still in its session."""\n\n    try:\n        os.killpg(process_group_id, signal.SIGKILL)\n    except ProcessLookupError:\n        pass\n    except PermissionError:\n        # The session is ours, but fail safely if a platform/container policy\n        # unexpectedly blocks group signaling.\n        pass\n\n\ndef _child_environment() -> dict[str, str]:\n    """Return a minimal deterministic environment with one native thread."""\n\n    environment = {\n        "PATH": os.environ.get("PATH", ""),\n        "PYTHONHASHSEED": "0",\n        "OMP_NUM_THREADS": "1",\n        "OPENBLAS_NUM_THREADS": "1",\n        "MKL_NUM_THREADS": "1",\n        "NUMEXPR_NUM_THREADS": "1",\n    }\n    # Some installations need the dynamic-loader path to import NumPy wheels.\n    if os.environ.get("LD_LIBRARY_PATH"):\n        environment["LD_LIBRARY_PATH"] = os.environ["LD_LIBRARY_PATH"]\n    return environment\n\n\ndef _drain_result_pipe(\n    result_fd: int,\n    *,\n    retain_bytes: int,\n    chunks: list[bytes],\n    overflow: list[bool],\n) -> None:\n    """Drain the child protocol pipe without letting it block on a full pipe.\n\n    The child emits its result only after model code finishes. Waiting for the\n    child to exit before reading can therefore deadlock when JSON escaping\n    expands retained Unicode output beyond the platform pipe capacity. Keep a\n    bounded prefix for decoding, but continue draining/discarding until EOF so\n    the child can always finish its write.\n    """\n\n    retained = 0\n    try:\n        while True:\n            chunk = os.read(result_fd, 65536)\n            if not chunk:\n                return\n            remaining = retain_bytes - retained\n            if remaining > 0:\n                kept = chunk[:remaining]\n                chunks.append(kept)\n                retained += len(kept)\n            if len(chunk) > max(remaining, 0):\n                overflow[0] = True\n    except OSError:\n        # Parent teardown may close the descriptor after process cleanup. Any\n        # incomplete record is handled by the normal "no result" path.\n        return\n\n\ndef run_python(\n    code: str,\n    context: dict | None = None,\n    limits: dict | None = None,\n) -> ExecResult:\n    """Execute ``code`` in a fresh resource-bounded child process.\n\n    ``context`` values are JSON-serialized and become globals in the child.\n    The function is designed for trusted orchestration around untrusted model\n    snippets; it bounds resources but does not prevent data or network access.\n    """\n\n    if not isinstance(code, str):\n        raise TypeError("code must be a string")\n    if context is not None and not isinstance(context, dict):\n        raise TypeError("context must be a mapping or None")\n    lim = _normalize_limits(limits)\n    payload = json.dumps(\n        {"code": code, "context": context or {}}, separators=(",", ":")\n    ).encode()\n    started = time.monotonic()\n    r_fd, w_fd = os.pipe()\n    proc = None\n    process_group_id = None\n    raw = b""\n    timed_out = False\n    killed = None\n    reader = None\n    result_chunks: list[bytes] = []\n    result_overflow = [False]\n\n    try:\n        with tempfile.TemporaryDirectory(prefix="pytool-") as cwd:\n            try:\n                proc = subprocess.Popen(\n                    [\n                        sys.executable,\n                        "-I",\n                        "-c",\n                        _CHILD,\n                        json.dumps(lim, separators=(",", ":")),\n                        str(w_fd),\n                    ],\n                    stdin=subprocess.PIPE,\n                    stdout=subprocess.DEVNULL,\n                    stderr=subprocess.DEVNULL,\n                    pass_fds=(w_fd,),\n                    cwd=cwd,\n                    start_new_session=True,\n                    env=_child_environment(),\n                )\n                process_group_id = proc.pid  # start_new_session => pgid == pid\n            finally:\n                os.close(w_fd)\n\n            # ``ensure_ascii=True`` can encode one non-BMP output character as\n            # a 12-byte surrogate-pair escape. Retain enough for the largest\n            # legitimate record and drain any excess without allocating it.\n            max_result_bytes = lim["max_output_chars"] * 12 + 65536\n            reader = threading.Thread(\n                target=_drain_result_pipe,\n                kwargs={\n                    "result_fd": r_fd,\n                    "retain_bytes": max_result_bytes,\n                    "chunks": result_chunks,\n                    "overflow": result_overflow,\n                },\n                name="safe-python-result-drain",\n                daemon=True,\n            )\n            reader.start()\n\n            try:\n                proc.communicate(input=payload, timeout=lim["wall_seconds"])\n            except subprocess.TimeoutExpired:\n                timed_out = True\n                _kill_process_group(process_group_id)\n                proc.wait()\n            finally:\n                # Also run after a normal leader exit: snippets may fork a child\n                # and return while that child keeps consuming resources.\n                if process_group_id is not None:\n                    _kill_process_group(process_group_id)\n\n            if proc.returncode and proc.returncode < 0:\n                killed = -proc.returncode\n\n            reader.join(timeout=2)\n            if reader.is_alive():\n                # All writers should be gone after process-group cleanup. Close\n                # our read descriptor as a final unblock and treat the record as\n                # incomplete rather than hanging the notebook process.\n                try:\n                    os.close(r_fd)\n                except OSError:\n                    pass\n                reader.join(timeout=1)\n            raw = b"".join(result_chunks)\n    finally:\n        try:\n            os.close(r_fd)\n        except OSError:\n            pass\n        if proc is not None and proc.poll() is None:\n            if process_group_id is not None:\n                _kill_process_group(process_group_id)\n            proc.wait()\n\n    wall = round(time.monotonic() - started, 2)\n    if timed_out:\n        return ExecResult(\n            False,\n            "",\n            f"timeout: exceeded {lim[\'wall_seconds\']:g}s wall limit",\n            True,\n            killed,\n            wall,\n        )\n\n    marker = raw.split(b"RESULT:", 1)\n    if len(marker) == 2:\n        try:\n            result = json.loads(marker[1].decode())\n            return ExecResult(\n                bool(result["ok"]),\n                result.get("stdout", ""),\n                result.get("error"),\n                False,\n                killed,\n                wall,\n            )\n        except (KeyError, TypeError, UnicodeDecodeError, json.JSONDecodeError):\n            pass\n\n    reason = f"killed by signal {killed}" if killed else "child produced no result"\n    if result_overflow[0]:\n        reason = "child result exceeded parent protocol bound"\n    if killed == signal.SIGKILL:\n        reason = "killed (CPU/memory limit)"\n    return ExecResult(False, "", reason, False, killed, wall)\n\n\n# --- OpenAI tool schema the model sees -------------------------------------\nTOOL_SCHEMA = {\n    "type": "function",\n    "function": {\n        "name": "run_python",\n        "description": (\n            "Execute Python 3 code in a fresh, resource-limited sandbox and get "\n            "its stdout back. The ARC task is preloaded as globals: `train` (a "\n            "list of {\\"input\\": grid, \\"output\\": grid} pairs) and `test_input` "\n            "(the grid to solve); grids are lists of lists of ints 0-9. Import "\n            "anything you need (e.g. numpy). PRINT what you want to inspect -- only "\n            "stdout is returned. State does NOT persist between calls; include all "\n            "needed code each time. Limits: ~4 GiB RAM, ~12 CPU-s, 20 s wall. "\n            "This is resource isolation, not filesystem/network isolation."\n        ),\n        "parameters": {\n            "type": "object",\n            "properties": {\n                "code": {\n                    "type": "string",\n                    "description": "Python 3 source to run.",\n                }\n            },\n            "required": ["code"],\n            "additionalProperties": False,\n        },\n    },\n}\n\n\ndef _selftest() -> int:\n    cases = [\n        (\n            "benign",\n            "print(\'rows:\', len(test_input)); print(sum(sum(r) for r in test_input))",\n            {"test_input": [[1, 2], [3, 4]]},\n            {},\n            lambda result: result.ok and "10" in result.stdout,\n        ),\n        (\n            "memory_bomb",\n            "x = [0] * (10**11)",\n            {},\n            {},\n            lambda result: not result.ok,\n        ),\n        (\n            "infinite_loop",\n            "while True:\\n    pass",\n            {},\n            {},\n            lambda result: not result.ok,\n        ),\n        (\n            "output_bomb",\n            "print(\'x\' * (10**8))",\n            {},\n            {},\n            lambda result: result.ok and "[output truncated" in result.stdout,\n        ),\n        (\n            "unicode_pipe",\n            "print(\'😀\' * 6000)",\n            {},\n            {"wall_seconds": 3, "cpu_seconds": 2},\n            lambda result: result.ok\n            and not result.timed_out\n            and result.stdout.count("😀") == 6000,\n        ),\n        (\n            "numpy",\n            "import numpy as np; print(np.array(test_input).T.tolist())",\n            {"test_input": [[1, 2], [3, 4]]},\n            {},\n            lambda result: result.ok and "[[1, 3], [2, 4]]" in result.stdout,\n        ),\n        (\n            "user_error",\n            "raise ValueError(\'boom\')",\n            {},\n            {},\n            lambda result: not result.ok and "ValueError" in (result.error or ""),\n        ),\n    ]\n    failures = 0\n    for name, source, context, limit_overrides, predicate in cases:\n        result = run_python(\n            source,\n            context,\n            limits={\n                "wall_seconds": 8,\n                "cpu_seconds": 5,\n                **limit_overrides,\n            },\n        )\n        passed = bool(predicate(result))\n        failures += not passed\n        preview = (result.stdout or result.error or "").replace("\\n", " ")[:80]\n        print(\n            f"{name:14s} pass={passed!s:5s} ok={result.ok!s:5s} "\n            f"timed_out={result.timed_out!s:5s} "\n            f"wall={result.wall_s:5.2f}s -> {preview}"\n        )\n    if failures:\n        print(f"SELFTEST FAILED: {failures} case(s)", file=sys.stderr)\n        return 1\n    print("SELFTEST PASS")\n    return 0\n\n\nif __name__ == "__main__":\n    sys.exit(_selftest())\n'

def load_module_from_source(name, path, source):
    """Write and import an embedded module with normal import semantics."""

    path.write_text(source)
    spec = importlib.util.spec_from_file_location(name, path)
    if spec is None or spec.loader is None:
        raise ImportError(f"could not create a loader for {name!r} at {path}")
    module = importlib.util.module_from_spec(spec)
    previous = sys.modules.get(name)
    sys.modules[name] = module
    try:
        spec.loader.exec_module(module)
    except BaseException:
        if previous is None:
            sys.modules.pop(name, None)
        else:
            sys.modules[name] = previous
        raise
    return module

def stable_sample_seed(task_id, query_index, slot):
    """Derive a deterministic, task-specific 31-bit sampling seed."""

    payload = f"{task_id}:{query_index}:{slot}".encode()
    return int.from_bytes(hashlib.sha256(payload).digest()[:4], "big") & 0x7FFFFFFF

def select_distinct_fallback(model_grid, fallback_row):
    """Choose a prevalidated fallback distinct from a first model hypothesis."""

    for name in ("attempt_2", "attempt_1"):
        candidate = fallback_row[name]
        if candidate != model_grid:
            return candidate
    raise ValueError("fallback pair unexpectedly contains no grid distinct from model output")

def parse_model_grid(text, *, require_closed_fence=False):
    """Return the last rectangular digit grid, preferring closed fences.

    A response stopped at a token limit can contain a rectangular prefix of an
    unfinished grid. Callers can require a closed fence in that case.
    """

    if not isinstance(text, str):
        return None

    def parse_block(block):
        rows = []
        for line in (block or "").strip().splitlines():
            tokens = line.strip().split()
            if tokens and all(token.isdigit() and len(token) == 1 for token in tokens):
                rows.append([int(token) for token in tokens])
            elif rows:
                break
        if not rows or any(len(row) != len(rows[0]) for row in rows):
            return None
        return rows

    fenced = re.findall(r"```(?:grid|text)?\s*\n(.*?)```", text or "", re.DOTALL)
    for candidate in reversed(fenced):
        grid = parse_block(candidate)
        if grid is not None:
            return grid

    if require_closed_fence:
        return None

    blocks = []
    current = []
    for line in (text or "").splitlines() + [""]:
        tokens = line.strip().split()
        if tokens and all(token.isdigit() and len(token) == 1 for token in tokens):
            current.append(line)
        elif current:
            blocks.append("\n".join(current))
            current = []
    for candidate in reversed(blocks):
        grid = parse_block(candidate)
        if grid is not None:
            return grid
    return None

def response_reasoning(message):
    """Read vLLM's current field while retaining older-server compatibility."""

    if not isinstance(message, dict):
        return ""
    value = message.get("reasoning") or message.get("reasoning_content") or ""
    return value if isinstance(value, str) else ""

def assistant_replay_message(message):
    """Normalize one assistant tool-call turn for vLLM history replay."""

    if not isinstance(message, dict):
        raise TypeError("assistant replay message must be an object")
    content = message.get("content") or ""
    if not isinstance(content, str):
        content = ""
    tool_calls = message.get("tool_calls")
    if tool_calls is None:
        tool_calls = []
    elif not isinstance(tool_calls, list):
        raise TypeError("assistant replay tool_calls must be a list")
    replay = {
        "role": "assistant",
        "content": content,
        "tool_calls": tool_calls,
    }
    reasoning = response_reasoning(message)
    if reasoning:
        replay["reasoning"] = reasoning
    return replay

def decode_tool_code(function):
    """Accept OpenAI JSON-string arguments and already-decoded mappings."""

    if not isinstance(function, dict):
        return ""
    arguments = function.get("arguments") or {}
    if isinstance(arguments, str):
        try:
            arguments = json.loads(arguments)
        except (TypeError, json.JSONDecodeError):
            return ""
    if not isinstance(arguments, dict):
        return ""
    code = arguments.get("code", "")
    return code if isinstance(code, str) else ""

def tool_wall_budget_seconds(deadline_s, now_s, default_wall_s=20.0):
    """Return a tool wall limit that cannot outlive its episode deadline."""

    default_wall_s = float(default_wall_s)
    if default_wall_s <= 0:
        raise ValueError("default tool wall limit must be positive")
    remaining_s = float(deadline_s) - float(now_s)
    if remaining_s <= 0:
        return None
    return min(default_wall_s, remaining_s)

def render(g): return "\n".join(" ".join(str(v) for v in row) for row in g)

def build_agent_prompt(task):
    parts = ["You are solving an abstract reasoning puzzle. Each training example maps "
             "an INPUT grid to an OUTPUT grid (digits 0-9 are colors; 0 is usually "
             "background). Infer the ONE rule that maps every input to its output, "
             "then apply it to the FINAL INPUT.\n",
             "You have a Python tool `run_python`. The task is preloaded there as "
             "`train` (a list of {\"input\": grid, \"output\": grid}) and `test_input` "
             "(the grid to solve). Use it however you find helpful -- to inspect grids, "
             "test a transformation against every example, or compute the answer -- or "
             "just reason directly. It is entirely your choice.\n"]
    for i, p in enumerate(task["train"]):
        parts.append("Example %d INPUT:\n%s\nExample %d OUTPUT:\n%s\n" %
                     (i+1, render(p["input"]), i+1, render(p["output"])))
    parts.append("FINAL INPUT:\n%s\n" % render(task["test_input"]))
    parts.append("When you are confident, give the final output grid as rows of "
                 "space-separated digits inside a fenced block:\n```grid\n<rows>\n```")
    return "\n".join(parts)

def build_prompt(task):
    parts = ["You are solving an abstract reasoning puzzle. Each example has an "
             "INPUT grid and its OUTPUT grid (digits 0-9 are colors; 0 is usually "
             "background). Infer the ONE rule that maps every input to its output, "
             "then apply it to the FINAL INPUT.\n"]
    for i, p in enumerate(task["train"]):
        parts.append("Example %d INPUT:\n%s\nExample %d OUTPUT:\n%s\n" %
                     (i+1, render(p["input"]), i+1, render(p["output"])))
    parts.append("FINAL INPUT:\n%s\n" % render(task["test_input"]))
    parts.append("Reason step by step however you like. Then give ONLY the final "
                 "output grid as rows of space-separated digits inside a fenced "
                 "block:\n```grid\n<rows>\n```\nOutput nothing after the block.")
    return "\n".join(parts)


In [ ]:
%%writefile /kaggle/working/ab/a100/qwen_ab.py
"""Qwen-only A/B runner for the A100 comparison (docs/A100_PLAN.md, steps 5-7).

Runs the production agent loop (same prompt, tool, parser and seeds as hybrid5; see runner_parts.py) on the fixed
NVARC-eligible outputs, with one change of rules: each sample gets a TOTAL completion-token budget summed over all
turns, and runs until it answers or the budget / context is used up. There is no wall-clock deadline, no forced-final
turn, no truncation recovery and no context-error halving; the turn limit is a high safety stop, not a budget.
Each request's max_tokens is min(budget left, context left), with the prompt counted by the server's own tokenizer
(/tokenize). Servers are started separately (serve.sh writes servers.json); this script only talks to them.

    python a100/qwen_ab.py --cohort explore --effort medium --budget 32768 --out runs/medium_32k \
        --servers server_logs/servers.json --concurrency 8

Files in --out: manifest.json (condition + input/server fingerprints, fixed at the first start and checked on every
resume), samples.jsonl (finished samples only), failures.jsonl (infrastructure failures; retried on the next start),
turns.jsonl (every request, including failed attempts), raw/<task>_<output>_<slot>.json (full conversation).
Score with a100/score.py.
"""
import argparse
import concurrent.futures as cf
import datetime
import glob
import hashlib
import json
import os
import pathlib
import subprocess
import sys
import threading
import time
import urllib.error
import urllib.request
import uuid

HERE = pathlib.Path(__file__).resolve().parent
sys.path.insert(0, str(HERE))
import runner_parts as rp  # noqa: E402

ROOT = HERE.parent
MODEL_NAME = rp.RUNTIME_CONFIG["model"]["served_name"]
SETTINGS = rp.RUNTIME_CONFIG["sampling"]

ap = argparse.ArgumentParser()
ap.add_argument("--cohort", required=True, help="explore | confirm | screen | tid:qi,tid:qi,...")
ap.add_argument("--effort", required=True, choices=["low", "medium", "high", "xhigh"])
ap.add_argument("--budget", type=int, required=True, help="total completion tokens per sample, all turns")
ap.add_argument("--samples", type=int, default=1, help="independent samples per output (slots 0..n-1)")
ap.add_argument("--tools", action=argparse.BooleanOptionalAction, default=True)
ap.add_argument("--max-turns", type=int, default=64, help="safety stop only; the token budget is the real limit")
ap.add_argument("--context-len", type=int, default=131072)
ap.add_argument("--servers", required=True, help="servers.json written by serve.sh")
ap.add_argument("--concurrency", type=int, default=8, help="in-flight samples in total (spread over servers)")
ap.add_argument("--tool-concurrency", type=int, default=8)
ap.add_argument("--fixed", default=str(HERE / "fixed_nvarc"))
ap.add_argument("--challenges", default=str(ROOT / "data" / "arc-agi_evaluation_challenges.json"))
ap.add_argument("--out", required=True)
ap.add_argument("--limit", type=int, default=0, help="run only the first N pending samples (smoke tests)")
ap.add_argument("--stop-launch-at", type=float, default=0, help="unix time after which no new sample starts (left pending)")
ap.add_argument("--allow-visible-solutions", action="store_true", help="smoke tests only")
args = ap.parse_args()

# the model's Python tool can read files: no answer file may be reachable from the inference side
visible = [p for d in {str(ROOT), str(pathlib.Path(args.challenges).resolve().parent), "/kaggle/input"}
           for p in glob.glob(os.path.join(d, "**", "*solution*.json"), recursive=True)]
if visible and not args.allow_visible_solutions:
    sys.exit(f"answer files are visible to the inference side, move them away first: {visible[:3]}")

OUT = pathlib.Path(args.out)
(OUT / "raw").mkdir(parents=True, exist_ok=True)
challenges = json.load(open(args.challenges))
fixed = pathlib.Path(args.fixed)
if args.cohort in ("explore", "confirm"):
    outputs = [tuple(x) for x in json.load(open(fixed / f"qwen_eligible_{args.cohort}.json"))]
    tasks = json.load(open(fixed / "cohort_manifest.json"))[args.cohort]
elif args.cohort == "screen":
    split = json.load(open(HERE / "screen8.json"))
    conf = json.load(open(fixed / "nvarc_confidence.json"))
    outputs = [(t, i) for t in split for i in range(len(challenges[t]["test"]))
               if conf[f"{t}_{i}"][0] < 2 or conf[f"{t}_{i}"][2] <= 2]
    tasks = split
else:
    outputs = [(x.split(":")[0], int(x.split(":")[1])) for x in args.cohort.split(",")]
    tasks = sorted({t for t, _ in outputs})
work = [(t, i, s) for s in range(args.samples) for t, i in outputs]
servers = json.load(open(args.servers))
URLS = [s["url"] for s in servers["servers"]]


def sha(b):
    return hashlib.sha256(b if isinstance(b, bytes) else json.dumps(b, sort_keys=True).encode()).hexdigest()[:16]


def post(url, path, body, timeout):
    req = urllib.request.Request(url + path, data=json.dumps(body).encode(), headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read().decode())


models = {}
for u in URLS:
    with urllib.request.urlopen(u + "/v1/models", timeout=30) as r:
        m = json.loads(r.read())
    assert any(d.get("id") == MODEL_NAME for d in m["data"]), f"{u} does not serve {MODEL_NAME}"
    models[u] = [{k: d.get(k) for k in ("id", "root", "max_model_len")} for d in m["data"]]
assert len({json.dumps(v, sort_keys=True) for v in models.values()}) == 1, f"servers differ: {models}"

# everything that defines the experiment; a resume must match it exactly
FIXED = {
    "condition": {k: getattr(args, k) for k in ["cohort", "effort", "budget", "samples", "tools", "max_turns", "context_len"]},
    "outputs": [list(o) for o in outputs], "tasks": sorted(tasks),
    "inputs": {"challenges": sha({t: challenges[t] for t in tasks}), "runner_parts": sha((HERE / "runner_parts.py").read_bytes()),
               "qwen_ab": sha(pathlib.Path(__file__).read_bytes()),
               "nvarc_submission": sha((fixed / "nvarc_submission.json").read_bytes()),
               "nvarc_confidence": sha((fixed / "nvarc_confidence.json").read_bytes())},
    "server": {k: servers[k] for k in servers if k != "servers"}, "model": next(iter(models.values())),
}
manifest_path = OUT / "manifest.json"
manifest = json.loads(manifest_path.read_text()) if manifest_path.exists() else {"fixed": FIXED, "starts": []}
if manifest["fixed"] != FIXED:
    diff = [k for k in FIXED if manifest["fixed"].get(k) != FIXED[k]]
    sys.exit(f"{OUT} was started with a different setup ({diff}); use a new --out")
manifest["condition"] = FIXED["condition"]

done = set()
if (OUT / "samples.jsonl").exists():
    for line in open(OUT / "samples.jsonl"):
        r = json.loads(line)
        done.add((r["task_id"], r["query_index"], r["slot"]))
todo = [w for w in work if w not in done]
if args.limit:
    todo = todo[:args.limit]

TMP = pathlib.Path(os.environ.get("TMPDIR", "/tmp"))
SANDBOX = rp.load_module_from_source("qwen38_safe_python_tool", TMP / "qwen38_safe_python_tool.py", rp.SANDBOX_SOURCE) \
    if args.tools else None
AGG = rp.load_module_from_source("qwen38_freeform_aggregator", TMP / "qwen38_freeform_aggregator.py", rp.AGGREGATOR_SOURCE)
TOOL_SLOTS = threading.BoundedSemaphore(args.tool_concurrency)
WRITE_LOCK = threading.Lock()


def chat_kwargs():
    return {"enable_thinking": SETTINGS["enable_thinking"], "reasoning_effort": args.effort}


def count_prompt(url, messages):
    """Exact prompt length from the server's tokenizer and chat template (tools included)."""
    body = {"model": MODEL_NAME, "messages": messages, "add_generation_prompt": True, "chat_template_kwargs": chat_kwargs()}
    if args.tools:
        body["tools"] = [SANDBOX.TOOL_SCHEMA]
    return int(post(url, "/tokenize", body, 120)["count"])


def valid_grid(grid):
    if grid is None:
        return None
    try:
        AGG.validate_grid(grid, context="a100 model grid")
        return grid
    except Exception:
        return None


def append(name, row):
    with WRITE_LOCK, open(OUT / name, "a") as f:
        f.write(json.dumps(row) + "\n")


class InfraError(Exception):
    pass


def solve(tid, qi, slot, url):
    test_input = challenges[tid]["test"][qi]["input"]
    task = {"train": challenges[tid]["train"], "test_input": test_input}
    prompt = rp.build_agent_prompt(task) if args.tools else rp.build_prompt(task)
    messages = [{"role": "user", "content": prompt}]
    context = {"train": task["train"], "test_input": test_input}
    seed = rp.stable_sample_seed(tid, qi, slot)
    key, attempt = f"{tid}_{qi}_{slot}", uuid.uuid4().hex[:8]
    used = turns = tool_calls = tool_errors = 0
    tool_s = 0.0
    finish, final, grid_any = "max_turns", "", None
    raw = []
    t0 = time.time()
    for turn in range(args.max_turns):
        try:
            prompt_tokens = count_prompt(url, messages)
        except Exception as exc:
            raise InfraError(f"tokenize: {type(exc).__name__}: {str(exc)[:200]}") from exc
        if args.budget - used <= 0:
            finish = "budget_exhausted"
            break
        max_tokens = min(args.budget - used, args.context_len - prompt_tokens)
        if max_tokens <= 0:
            finish = "context_exhausted"
            break
        body = {"model": MODEL_NAME, "messages": messages, "temperature": SETTINGS["temperature"],
                "top_p": SETTINGS["top_p"], "top_k": SETTINGS["top_k"], "max_tokens": max_tokens, "seed": seed,
                "chat_template_kwargs": chat_kwargs()}
        if args.tools:
            body["tools"] = [SANDBOX.TOOL_SCHEMA]
            body["tool_choice"] = "auto"
        ts = time.time()
        try:
            p = post(url, "/v1/chat/completions", body, 4 * 3600)
        except urllib.error.HTTPError as exc:
            raise InfraError("http%d: %s" % (exc.code, exc.read().decode(errors="replace")[:300])) from exc
        except Exception as exc:
            raise InfraError(f"chat: {type(exc).__name__}: {str(exc)[:200]}") from exc
        turns += 1
        ch = p["choices"][0]
        msg = ch["message"]
        usage = p.get("usage") or {}
        n_out = usage.get("completion_tokens") or 0
        used += n_out
        content = msg.get("content") if isinstance(msg.get("content"), str) else ""
        reasoning = rp.response_reasoning(msg)
        tcs = msg.get("tool_calls") or []
        turn_finish = ch.get("finish_reason")
        g = valid_grid(rp.parse_model_grid(content, require_closed_fence=turn_finish == "length"))
        grid_any = g or grid_any
        append("turns.jsonl", {"key": key, "attempt": attempt, "turn": turn, "prompt_tokens": prompt_tokens,
                               "usage_prompt_tokens": usage.get("prompt_tokens"), "max_tokens": max_tokens,
                               "completion_tokens": n_out, "finish_reason": turn_finish, "tool_calls": len(tcs),
                               "reasoning_chars": len(reasoning), "content_chars": len(content),
                               "wall_s": round(time.time() - ts, 1)})
        raw.append({"turn": turn, "finish_reason": turn_finish, "message": msg, "usage": usage})
        if not tcs and ("<tool_call>" in reasoning or "<tool_call>" in content):
            finish, final = "protocol_error:tool_call_markup", content
            break
        if tcs and args.tools and turn_finish != "length":
            messages.append(rp.assistant_replay_message(msg))
            for tc in tcs:
                fn = tc.get("function") or {}
                code = rp.decode_tool_code(fn) if fn.get("name") == "run_python" else ""
                tool_calls += 1
                if not code:
                    out = "ERROR: run_python arguments did not contain non-empty code" if fn.get("name") == "run_python" \
                        else "error: unknown tool %r" % fn.get("name")
                    tool_errors += 1
                else:
                    with TOOL_SLOTS:
                        res = SANDBOX.run_python(code, context)
                    tool_s += res.wall_s or 0
                    out = res.stdout if res.ok else "ERROR: " + (res.error or "")
                    tool_errors += 0 if res.ok else 1
                messages.append({"role": "tool", "tool_call_id": tc.get("id"), "name": fn.get("name") or "unknown_tool",
                                 "content": out or "(no output)"})
            continue
        final = content
        finish = turn_finish or "stop"
        break
    grid = valid_grid(rp.parse_model_grid(final, require_closed_fence=finish == "length"))
    row = {"task_id": tid, "query_index": qi, "slot": slot, "seed": seed, "attempt": attempt, "finish": finish,
           "grid": grid, "grid_any_turn": grid_any if grid is None else None,
           "completion_tokens": used, "turns": turns, "tool_calls": tool_calls, "tool_errors": tool_errors,
           "tool_s": round(tool_s, 1), "wall_s": round(time.time() - t0, 1), "server": url}
    (OUT / "raw" / f"{key}.json").write_text(json.dumps({"messages": messages, "responses": raw, "result": row}))
    append("samples.jsonl", row)
    return row


class NotStarted(Exception):
    pass


def run_one(t, i, s, url):
    if args.stop_launch_at and time.time() > args.stop_launch_at:
        raise NotStarted()
    try:
        return solve(t, i, s, url)
    except Exception as exc:  # not written to samples.jsonl, so the next start retries it
        append("failures.jsonl", {"task_id": t, "query_index": i, "slot": s, "server": url,
                                  "utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
                                  "error": f"{type(exc).__name__}: {str(exc)[:400]}"})
        raise


try:
    git = subprocess.run(["git", "-C", str(ROOT), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
except Exception:
    git = None
manifest["starts"].append({"utc": datetime.datetime.now(datetime.timezone.utc).isoformat(), "git": git, "urls": URLS,
                           "concurrency": args.concurrency, "todo": len(todo), "done_before": len(done)})
manifest_path.write_text(json.dumps(manifest, indent=1))

print(f"{len(work)} samples ({len(outputs)} outputs x {args.samples}); {len(done)} done; running {len(todo)}", flush=True)
t_start = time.time()
tokens, failed, skipped = [0], [0], [0]
with cf.ThreadPoolExecutor(args.concurrency) as ex:
    futs = {ex.submit(run_one, t, i, s, URLS[n % len(URLS)]): (t, i, s) for n, (t, i, s) in enumerate(todo)}
    for n, fut in enumerate(cf.as_completed(futs), 1):
        try:
            r = fut.result()
        except NotStarted:
            skipped[0] += 1
            continue
        except Exception as exc:
            failed[0] += 1
            print("FAILED", futs[fut], type(exc).__name__, str(exc)[:300], flush=True)
            continue
        tokens[0] += r["completion_tokens"]
        el = time.time() - t_start
        print(f"[{n}/{len(todo)}] {r['task_id']}:{r['query_index']}#{r['slot']} {r['finish']} grid={r['grid'] is not None} "
              f"tok={r['completion_tokens']} turns={r['turns']} tools={r['tool_calls']} {r['wall_s']}s | "
              f"agg {tokens[0] / el:.0f} tok/s", flush=True)
print(f"finished: {len(todo) - failed[0] - skipped[0]} ok, {failed[0]} failed, {skipped[0]} not started "
      "(rerun the same command to retry / continue)", flush=True)
sys.exit(1 if failed[0] or skipped[0] else 0)


In [ ]:
%%writefile /kaggle/working/ab/a100/serve.sh
#!/usr/bin/env bash
# Start one vLLM 0.19 server per group of TP GPUs for a100/qwen_ab.py (docs/A100_PLAN.md, steps 3-4).
#
#   MODEL=/data/qwen3-8-27b-fp8 VENV=/tmp/vllm_env TP=1 SEQS=8 bash a100/serve.sh
#
# GPUs   the ones in CUDA_VISIBLE_DEVICES (as assigned by the scheduler), else all GPUs nvidia-smi lists
# TP     tensor parallel size per server (1 on 80GB, 2 on 40GB); the GPU count must be a multiple of it
# SEQS   max concurrent sequences per server (= harness --concurrency / servers)
# MTP=1  add MTP speculative decoding (start without it first, per the plan)
# KVFP8=1  fp8 KV cache with the Triton backend (the L4 production setting; off on A100 by default)
# TOOLS=0  start without the tool-call parser (only for the tools-off condition)
# VENV   optional pip --target dir holding vLLM (the Kaggle cp311 wheelhouse install); launched with python -S
# Writes $LOGDIR/servers.json (pass it to qwen_ab.py --servers) and $LOGDIR/vllm_<port>.log. Default LOGDIR=server_logs.
set -euo pipefail
: "${MODEL:?set MODEL to the Qwen3.8-27B-FP8 directory}"
TP=${TP:-1}; SEQS=${SEQS:-8}; CTX=${CTX:-131072}; MEM=${MEM:-0.92}; PORT0=${PORT0:-8000}
NAME=${NAME:-Qwen/Qwen3.8-27B-FP8}; LOGDIR=${LOGDIR:-server_logs}; PY=${PY:-python}
mkdir -p "$LOGDIR"

if [ -n "${CUDA_VISIBLE_DEVICES:-}" ]; then IFS=, read -ra gpus <<< "$CUDA_VISIBLE_DEVICES"
else mapfile -t gpus < <(nvidia-smi --query-gpu=index --format=csv,noheader); fi
((${#gpus[@]} > 0 && ${#gpus[@]} % TP == 0)) || { echo "GPUs (${gpus[*]}) not a positive multiple of TP=$TP"; exit 1; }
nserv=$((${#gpus[@]} / TP))

flags=(--served-model-name "$NAME" --tensor-parallel-size "$TP" --max-model-len "$CTX" --gpu-memory-utilization "$MEM"
       --enable-prefix-caching --max-num-seqs "$SEQS" --reasoning-parser qwen3 --language-model-only --dtype bfloat16)
[ "${TOOLS:-1}" = 1 ] && flags+=(--enable-auto-tool-choice --tool-call-parser qwen3_coder)
[ "${MTP:-0}" = 1 ] && flags+=(--speculative-config '{"method":"mtp","num_speculative_tokens":3}')
if [ "${KVFP8:-0}" = 1 ]; then flags+=(--kv-cache-dtype fp8 --attention-backend TRITON_ATTN); export VLLM_FORCE_ATTN_BACKEND=TRITON_ATTN; fi
export VLLM_USE_FLASHINFER_SAMPLER=0
pyargs=(); if [ -n "${VENV:-}" ]; then pyargs=(-S); export PYTHONPATH="$VENV" PATH="$VENV/bin:$PATH"; fi
up() { python3 -c "import urllib.request; urllib.request.urlopen('http://127.0.0.1:$1/v1/models', timeout=5)" 2> /dev/null; }

pids=(); devs_list=()
for ((s = 0; s < nserv; s++)); do
  port=$((PORT0 + s)); devs=$(IFS=,; echo "${gpus[*]:$((s * TP)):$TP}")
  if up "$port"; then echo "port $port already serves something; stop it first"; exit 1; fi
  echo "server $s: GPUs $devs port $port flags: ${flags[*]}"
  CUDA_VISIBLE_DEVICES=$devs nohup "$PY" "${pyargs[@]}" -m vllm.entrypoints.openai.api_server --model "$MODEL" \
    --port "$port" "${flags[@]}" > "$LOGDIR/vllm_$port.log" 2>&1 &
  pids+=($!); devs_list+=("$devs")
done
for ((s = 0; s < nserv; s++)); do
  port=$((PORT0 + s)); t0=$SECONDS
  until up "$port"; do
    sleep 5
    if ! kill -0 "${pids[$s]}" 2> /dev/null; then echo "server on port $port died"; tail -30 "$LOGDIR/vllm_$port.log"; exit 1; fi
    if ((SECONDS - t0 > 3600)); then echo "port $port not ready after 1h; see $LOGDIR/vllm_$port.log"; exit 1; fi
  done
  kill -0 "${pids[$s]}" || { echo "server on port $port exited after answering"; exit 1; }
  echo "ready: port $port (pid ${pids[$s]}) after $((SECONDS - t0))s"
  grep -iE "quantiz|marlin|kv cache|maximum concurrency" "$LOGDIR/vllm_$port.log" | tail -5 || true
done

vllm_version=$("$PY" "${pyargs[@]}" -c "import vllm; print(vllm.__version__)" 2> /dev/null || echo unknown)
gpu_names=$(nvidia-smi --query-gpu=name,memory.total --format=csv,noheader | sort -u | paste -sd ";")
python3 - "$LOGDIR/servers.json" "$MODEL" "$vllm_version" "$gpu_names" "$PORT0" "${devs_list[@]}" -- "${flags[@]}" <<'EOF'
import json, sys
out, model, version, gpus, port0, *rest = sys.argv[1:]
cut = rest.index("--")
devs, flags = rest[:cut], rest[cut + 1:]
json.dump({"model_path": model, "vllm": version, "gpus": gpus, "flags": flags,
           "servers": [{"url": f"http://127.0.0.1:{int(port0) + s}", "gpus": d} for s, d in enumerate(devs)]},
          open(out, "w"), indent=1)
print("wrote", out)
EOF


In [ ]:
%%writefile /kaggle/working/ab/a100/screen8.json
["4c7dc4dd", "71e489b6", "7b3084d4", "8b9c3697", "b6f77b65", "c4d067a0", "dbff022c", "eee78d87"]

In [ ]:
%%writefile /kaggle/working/ab/a100/fixed_nvarc/cohort_manifest.json
{
 "seed": 20261010,
 "source_runs": [
  "t20_stock",
  "dev60_stock"
 ],
 "nvarc": "stock 16 views, score_kgmon",
 "explore": [
  "0934a4d8",
  "13e47133",
  "16b78196",
  "195c6913",
  "221dfab4",
  "269e22fb",
  "2b83f449",
  "2c181942",
  "35ab12c3",
  "3a25b0d8",
  "3dc255db",
  "446ef5d2",
  "4e34c42c",
  "62593bfd",
  "6ffbe589",
  "7666fa5d",
  "7b5033c1",
  "7ed72f31",
  "80a900e0",
  "88e364bc",
  "8b7bacbf",
  "8e5c0c38",
  "8f3a5a89",
  "9385bd28",
  "9bbf930d",
  "a47bf94d",
  "b5ca7ac4",
  "c7f57c3e",
  "d35bdbdc",
  "da515329",
  "db695cfb",
  "de809cff",
  "e12f9a14",
  "e87109e9",
  "edb79dae",
  "faa9f03d"
 ],
 "confirm": [
  "21897d95",
  "31f7f899",
  "332f06d7",
  "4c3d4a41",
  "581f7754",
  "58490d8a",
  "5961cc34",
  "89565ca0",
  "cbebaa4b",
  "db0c5428",
  "f560132c",
  "f931b4a8"
 ],
 "leftover": [
  "1818057f",
  "3e6067c3",
  "67e490f4",
  "78332cb0",
  "b9e38dc0",
  "fc7cae8d"
 ],
 "strata": {
  "13e47133": "Z",
  "446ef5d2": "Z",
  "6ffbe589": "Z",
  "4e34c42c": "Z",
  "8e5c0c38": "O",
  "c7f57c3e": "W",
  "8b7bacbf": "W",
  "221dfab4": "W",
  "35ab12c3": "W",
  "a47bf94d": "W",
  "db695cfb": "W",
  "e12f9a14": "W",
  "edb79dae": "W",
  "88e364bc": "W",
  "9385bd28": "W",
  "80a900e0": "W",
  "269e22fb": "W",
  "8f3a5a89": "W",
  "2c181942": "W",
  "7666fa5d": "W",
  "16b78196": "W",
  "7b5033c1": "W",
  "da515329": "W",
  "d35bdbdc": "W",
  "de809cff": "W",
  "3dc255db": "W",
  "e87109e9": "W",
  "faa9f03d": "W",
  "0934a4d8": "W",
  "2b83f449": "W",
  "195c6913": "W",
  "3a25b0d8": "W",
  "7ed72f31": "W",
  "9bbf930d": "W",
  "b5ca7ac4": "W",
  "62593bfd": "W",
  "f560132c": "Z",
  "21897d95": "Z",
  "cbebaa4b": "Z",
  "581f7754": "O",
  "5961cc34": "O",
  "db0c5428": "O",
  "89565ca0": "W",
  "f931b4a8": "W",
  "332f06d7": "W",
  "58490d8a": "W",
  "31f7f899": "W",
  "4c3d4a41": "W"
 },
 "explore_outputs": 44,
 "confirm_outputs": 16
}

In [ ]:
%%writefile /kaggle/working/ab/a100/fixed_nvarc/nvarc_confidence.json
{"0934a4d8_0": [10, 3, 2], "13e47133_0": [2, 2, 1], "13e47133_1": [0, 0, 0], "142ca369_0": [10, 4, 4], "142ca369_1": [6, 3, 4], "16b78196_0": [5, 5, 2], "1818057f_0": [2, 15, 2], "195c6913_0": [8, 2, 1], "195c6913_1": [11, 1, 2], "21897d95_0": [0, 0, 0], "21897d95_1": [6, 1, 1], "221dfab4_0": [3, 8, 6], "221dfab4_1": [11, 1, 1], "269e22fb_0": [8, 2, 3], "269e22fb_1": [5, 3, 2], "28a6681f_0": [5, 7, 4], "2b83f449_0": [12, 2, 1], "2c181942_0": [8, 4, 1], "2d0172a1_0": [6, 10, 5], "2d0172a1_1": [10, 6, 3], "31f7f899_0": [9, 6, 2], "332f06d7_0": [5, 14, 1], "35ab12c3_0": [9, 2, 1], "3a25b0d8_0": [14, 4, 2], "3a25b0d8_1": [4, 13, 1], "3dc255db_0": [12, 2, 1], "3e6067c3_0": [10, 2, 1], "3e6067c3_1": [16, 1, 1], "409aa875_0": [9, 6, 5], "446ef5d2_0": [0, 0, 0], "446ef5d2_1": [1, 1, 0], "45a5af55_0": [2, 16, 8], "4c3d4a41_0": [7, 8, 1], "4c3d4a41_1": [4, 10, 7], "4c7dc4dd_0": [17, 3, 2], "4c7dc4dd_1": [15, 5, 1], "4e34c42c_0": [0, 0, 0], "4e34c42c_1": [0, 0, 0], "53fb4810_0": [4, 11, 4], "581f7754_0": [1, 1, 0], "581f7754_1": [6, 2, 1], "58490d8a_0": [14, 3, 2], "5961cc34_0": [1, 16, 0], "62593bfd_0": [6, 10, 5], "62593bfd_1": [11, 2, 2], "67e490f4_0": [16, 1, 1], "6e453dd6_0": [8, 5, 4], "6ffbe589_0": [0, 0, 0], "71e489b6_0": [5, 11, 3], "71e489b6_1": [7, 9, 2], "7491f3cf_0": [9, 6, 5], "7666fa5d_0": [5, 12, 2], "78332cb0_0": [13, 5, 2], "78332cb0_1": [11, 1, 3], "7b3084d4_0": [12, 2, 2], "7b5033c1_0": [13, 2, 1], "7b80bb43_0": [5, 10, 8], "7ed72f31_0": [13, 3, 3], "7ed72f31_1": [8, 2, 1], "80a900e0_0": [4, 1, 1], "88e364bc_0": [12, 8, 1], "88e364bc_1": [10, 3, 4], "89565ca0_0": [14, 3, 2], "8b7bacbf_0": [9, 8, 3], "8b7bacbf_1": [15, 4, 1], "8b9c3697_0": [14, 6, 3], "8e5c0c38_0": [1, 1, 0], "8e5c0c38_1": [16, 1, 2], "8f215267_0": [11, 4, 3], "8f3a5a89_0": [3, 16, 2], "9385bd28_0": [6, 1, 1], "9385bd28_1": [9, 2, 1], "97d7923e_0": [11, 4, 3], "9bbf930d_0": [9, 1, 2], "a251c730_0": [8, 4, 5], "a47bf94d_0": [6, 3, 2], "a6f40cea_0": [10, 4, 4], "b5ca7ac4_0": [16, 2, 1], "b6f77b65_0": [10, 7, 2], "b6f77b65_1": [6, 13, 2], "b6f77b65_2": [6, 8, 1], "b9e38dc0_0": [7, 4, 2], "c4d067a0_0": [10, 2, 1], "c7f57c3e_0": [11, 2, 1], "cbebaa4b_0": [0, 0, 0], "cbebaa4b_1": [0, 0, 0], "d35bdbdc_0": [15, 1, 2], "d35bdbdc_1": [11, 8, 5], "d35bdbdc_2": [7, 14, 3], "d59b0160_0": [12, 4, 3], "d8e07eb2_0": [4, 14, 7], "d8e07eb2_1": [11, 9, 8], "da515329_0": [6, 2, 1], "db0c5428_0": [1, 16, 0], "db695cfb_0": [3, 2, 1], "dbff022c_0": [11, 5, 3], "de809cff_0": [11, 6, 2], "dfadab01_0": [4, 8, 6], "e12f9a14_0": [2, 1, 1], "e12f9a14_1": [3, 1, 1], "e376de54_0": [12, 4, 3], "e8686506_0": [5, 9, 4], "e87109e9_0": [11, 1, 2], "edb79dae_0": [7, 1, 1], "eee78d87_0": [3, 12, 6], "f560132c_0": [0, 0, 0], "f560132c_1": [5, 2, 1], "f931b4a8_0": [8, 4, 2], "f931b4a8_1": [5, 13, 3], "faa9f03d_0": [11, 3, 2], "faa9f03d_1": [8, 6, 4], "fc7cae8d_0": [5, 1, 1]}

In [ ]:
%%writefile /kaggle/working/ab/a100/fixed_nvarc/nvarc_submission.json
{"0934a4d8": [{"attempt_1": [[9, 9, 9], [9, 9, 9], [9, 9, 9], [9, 9, 9], [4, 4, 7], [4, 4, 7], [6, 6, 1], [6, 6, 6], [1, 6, 1]], "attempt_2": [[4, 4, 9], [4, 4, 9], [4, 4, 9], [4, 4, 9], [4, 4, 7], [4, 4, 7], [6, 6, 1], [6, 6, 6], [1, 6, 1]]}], "13e47133": [{"attempt_1": [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 8, 6, 6, 6, 6, 6, 6, 6, 6, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2], [1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 8, 6, 4, 4, 4, 4, 4, 4, 6, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2], [1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 8, 6, 4, 5, 5, 5, 5, 4, 6, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2], [1, 2, 1, 1, 1, 1, 1, 1, 1, 2, 1, 8, 6, 4, 5, 6, 6, 5, 4, 6, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2], [1, 2, 1, 2, 2, 2, 2, 2, 1, 2, 1, 8, 6, 4, 5, 5, 5, 5, 4, 6, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2], [1, 2, 1, 2, 1, 1, 1, 2, 1, 2, 1, 8, 6, 4, 4, 4, 4, 4, 4, 6, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2], [1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 8, 6, 6, 6, 6, 6, 6, 6, 6, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2], [1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 8, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2], [1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 8, 2, 2, 2, 2, 2, 8, 4, 4, 4, 4, 4, 8, 2, 2, 2, 2, 2, 2], [1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 8, 2, 2, 2, 2, 2, 8, 4, 1, 1, 1, 4, 8, 2, 2, 2, 2, 2, 2], [1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 8, 2, 2, 2, 2, 2, 8, 4, 1, 4, 1, 4, 8, 8, 8, 8, 8, 8, 8], [1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 8, 2, 2, 2, 2, 2, 8, 4, 1, 4, 1, 4, 8, 2, 2, 2, 2, 2, 2], [1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 8, 2, 2, 2, 2, 2, 8, 4, 1, 1, 1, 4, 8, 2, 2, 2, 2, 2, 2], [1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 8, 2, 2, 2, 2, 2, 8, 4, 4, 4, 4, 4, 8, 2, 2, 2, 2, 2, 2], [1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 8, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2], [1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [1, 2, 1, 2, 1, 1, 1, 2, 1, 2, 1, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [1, 2, 1, 2, 2, 2, 2, 2, 1, 2, 1, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [1, 2, 1, 1, 1, 1, 1, 1, 1, 2, 1, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [1, 2, 1, 1, 2, 2, 2, 2, 2, 2, 1, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [1, 2, 1, 1, 2, 1, 1, 1, 1, 1, 1, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [1, 2, 1, 1, 2, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [1, 2, 1, 1, 2, 1, 8, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [1, 2, 1, 1, 2, 1, 8, 6, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 6, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [1, 2, 2, 2, 2, 1, 8, 6, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 6, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [1, 1, 1, 1, 1, 1, 8, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]], "attempt_2": [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 8, 6, 6, 6, 6, 6, 6, 6, 6, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2], [1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 8, 6, 4, 4, 4, 4, 4, 4, 6, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2], [1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 8, 6, 4, 5, 5, 5, 5, 4, 6, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2], [1, 2, 1, 1, 1, 1, 1, 1, 1, 2, 1, 8, 6, 4, 5, 6, 6, 5, 4, 6, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2], [1, 2, 1, 2, 2, 2, 2, 2, 1, 2, 1, 8, 6, 4, 5, 5, 5, 5, 4, 6, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2], [1, 2, 1, 2, 1, 1, 1, 2, 1, 2, 1, 8, 6, 4, 4, 4, 4, 4, 4, 6, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2], [1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 8, 6, 6, 6, 6, 6, 6, 6, 6, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2], [1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 8, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2], [1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 8, 2, 2, 2, 2, 2, 8, 4, 4, 4, 4, 4, 8, 2, 2, 2, 2, 2, 2], [1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 8, 2, 2, 2, 2, 2, 8, 4, 1, 1, 1, 4, 8, 2, 2, 2, 2, 2, 2], [1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 8, 2, 2, 2, 2, 2, 8, 4, 1, 4, 1, 4, 8, 8, 8, 8, 8, 8, 8], [1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 8, 2, 2, 2, 2, 2, 8, 4, 1, 4, 1, 4, 8, 2, 2, 2, 2, 2, 2], [1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 8, 2, 2, 2, 2, 2, 8, 4, 1, 1, 1, 4, 8, 2, 2, 2, 2, 2, 2], [1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 8, 2, 2, 2, 2, 2, 8, 4, 4, 4, 4, 4, 8, 2, 2, 2, 2, 2, 2], [1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 8, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2], [1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [1, 2, 1, 2, 1, 2, 1, 2, 1, 2, 1, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [1, 2, 1, 2, 1, 1, 1, 2, 1, 2, 1, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [1, 2, 1, 2, 2, 2, 2, 2, 1, 2, 1, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [1, 2, 1, 1, 1, 1, 1, 1, 1, 2, 1, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [1, 2, 1, 1, 2, 2, 2, 2, 2, 2, 1, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [1, 2, 1, 1, 2, 1, 1, 1, 1, 1, 1, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [1, 2, 1, 1, 2, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [1, 2, 1, 1, 2, 1, 8, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [1, 2, 1, 1, 2, 1, 8, 6, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 6, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [1, 2, 2, 2, 2, 1, 8, 6, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 6, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [1, 1, 1, 1, 1, 1, 8, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]]}, {"attempt_1": [[0]], "attempt_2": [[0]]}], "142ca369": [{"attempt_1": [[0, 1, 0, 2, 0, 7, 0, 5, 0, 0, 2, 0, 8, 0, 6, 0, 4, 0], [1, 1, 2, 2, 7, 7, 5, 5, 0, 0, 2, 2, 8, 8, 6, 6, 4, 4], [0, 2, 1, 0, 2, 0, 7, 0, 5, 0, 0, 0, 0, 0, 0, 0, 6, 0], [2, 2, 0, 1, 0, 2, 0, 7, 0, 5, 0, 0, 0, 0, 0, 0, 6, 6], [0, 7, 2, 0, 1, 0, 2, 0, 7, 0, 5, 0, 0, 0, 0, 0, 8, 0], [7, 7, 0, 2, 0, 1, 0, 2, 0, 7, 0, 5, 0, 0, 0, 0, 8, 8], [0, 5, 7, 0, 2, 0, 1, 0, 2, 0, 7, 0, 5, 0, 0, 0, 2, 0], [5, 5, 0, 7, 0, 2, 0, 1, 0, 2, 0, 7, 0, 5, 0, 0, 2, 2], [0, 0, 5, 0, 7, 0, 2, 0, 1, 0, 2, 0, 7, 0, 5, 0, 0, 0], [0, 0, 0, 5, 0, 7, 0, 2, 0, 1, 0, 2, 0, 7, 0, 5, 0, 0], [0, 0, 0, 0, 5, 0, 7, 0, 2, 0, 1, 0, 2, 0, 7, 0, 5, 0], [0, 0, 0, 0, 0, 5, 0, 7, 0, 2, 0, 1, 0, 2, 0, 7, 0, 5], [0, 0, 0, 0, 0, 0, 5, 0, 7, 0, 2, 0, 1, 0, 2, 0, 7, 0], [0, 0, 0, 0, 0, 0, 0, 5, 0, 7, 0, 2, 0, 1, 0, 2, 0, 7], [0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 7, 0, 2, 0, 1, 0, 2, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 7, 0, 2, 0, 1, 0, 2], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 7, 0, 2, 0, 1, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 7, 0, 2, 0, 1]], "attempt_2": [[0, 1, 0, 2, 0, 7, 0, 5, 0, 0, 2, 0, 8, 0, 6, 0, 4, 0], [1, 1, 2, 2, 7, 7, 5, 5, 0, 0, 2, 2, 8, 8, 6, 6, 4, 4], [0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 0], [2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 6], [0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0], [7, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 8], [0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0], [5, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}, {"attempt_1": [[0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 0], [7, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 6], [0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 0, 0], [0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 0, 0, 0], [0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 0, 0, 0, 0], [0, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 6, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 7, 0, 7, 0, 6, 0, 6, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 7, 7, 0, 6, 6, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 7, 0, 6, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 6, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 6, 6, 0, 7, 7, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 6, 0, 6, 0, 7, 0, 7, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 6, 0, 0, 0, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0], [0, 0, 0, 0, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0, 0, 0, 0], [0, 0, 0, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0, 0, 0], [0, 0, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0, 0], [6, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 7], [0, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0]], "attempt_2": [[0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 0], [7, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 6], [0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 0, 0], [0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 0, 0, 0], [0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 0, 0, 0, 0], [0, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 6, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 7, 0, 7, 0, 6, 0, 6, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 7, 7, 6, 6, 6, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 7, 0, 6, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 6, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 6, 6, 7, 7, 7, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 6, 0, 6, 0, 7, 0, 7, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 6, 0, 0, 0, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0], [0, 0, 0, 0, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0, 0, 0, 0], [0, 0, 0, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0, 0, 0], [0, 0, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0, 0], [6, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 7], [0, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0]]}], "16b78196": [{"attempt_1": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], "attempt_2": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 8, 8, 8, 8, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 6, 8, 8, 8, 8, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 6, 6, 8, 8, 8, 8, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 6, 6, 6, 8, 4, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 6, 6, 6, 6, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}], "1818057f": [{"attempt_1": [[4, 2, 4, 2, 4, 4, 2, 4, 2, 4, 4, 2, 4, 2, 4, 2, 2, 4, 2, 4, 4, 2], [2, 2, 8, 2, 2, 2, 2, 4, 2, 2, 2, 2, 2, 2, 2, 2, 2, 4, 2, 2, 2, 2], [4, 8, 8, 8, 4, 4, 4, 2, 4, 4, 2, 4, 4, 4, 4, 2, 4, 4, 2, 2, 4, 2], [2, 2, 8, 2, 2, 2, 2, 4, 2, 2, 2, 2, 4, 2, 2, 2, 2, 4, 2, 2, 2, 2], [4, 2, 4, 2, 2, 4, 2, 2, 2, 4, 2, 2, 4, 2, 4, 2, 2, 4, 2, 4, 4, 2], [4, 2, 4, 2, 2, 4, 2, 2, 2, 4, 4, 2, 4, 2, 4, 4, 2, 4, 2, 2, 4, 2], [2, 2, 8, 2, 2, 2, 2, 4, 2, 2, 2, 2, 8, 2, 2, 2, 2, 4, 2, 2, 2, 2], [4, 8, 8, 8, 2, 4, 4, 2, 4, 2, 4, 8, 8, 8, 4, 4, 4, 2, 2, 4, 4, 4], [2, 2, 8, 2, 2, 2, 2, 4, 2, 2, 2, 2, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2], [4, 2, 4, 2, 4, 2, 2, 4, 2, 4, 4, 2, 4, 2, 4, 4, 2, 4, 2, 4, 4, 2], [4, 2, 4, 2, 2, 4, 2, 4, 2, 4, 4, 2, 2, 2, 2, 4, 2, 4, 2, 4, 4, 2], [2, 2, 4, 2, 2, 2, 2, 8, 2, 2, 2, 2, 4, 2, 2, 2, 2, 2, 2, 2, 2, 2], [4, 4, 2, 4, 4, 4, 8, 8, 8, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [2, 2, 4, 2, 2, 2, 2, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 4, 2, 2, 2, 2], [4, 2, 4, 2, 4, 4, 2, 4, 2, 4, 4, 2, 4, 2, 4, 2, 2, 4, 2, 4, 4, 2], [4, 2, 4, 2, 4, 2, 2, 4, 2, 4, 4, 2, 2, 2, 4, 4, 2, 4, 2, 4, 2, 2], [2, 2, 2, 2, 2, 2, 2, 8, 2, 2, 2, 2, 4, 2, 2, 2, 2, 8, 2, 2, 2, 2], [4, 4, 4, 4, 4, 4, 8, 8, 8, 2, 4, 2, 4, 4, 4, 4, 8, 8, 8, 2, 4, 2], [2, 2, 4, 2, 2, 2, 2, 8, 2, 2, 2, 2, 4, 2, 2, 2, 2, 8, 2, 2, 2, 2], [4, 2, 4, 2, 4, 4, 2, 4, 2, 4, 4, 2, 4, 2, 2, 4, 2, 2, 2, 4, 4, 2], [4, 2, 4, 2, 4, 4, 2, 4, 2, 4, 2, 2, 4, 2, 4, 4, 2, 2, 2, 4, 4, 2], [2, 2, 4, 2, 2, 2, 2, 2, 2, 2, 2, 2, 4, 2, 2, 2, 2, 4, 2, 2, 2, 2]], "attempt_2": [[4, 2, 4, 2, 4, 4, 2, 4, 2, 4, 4, 2, 4, 2, 4, 2, 2, 4, 2, 4, 4, 2], [2, 2, 8, 2, 2, 2, 2, 4, 2, 2, 2, 2, 2, 2, 2, 2, 2, 4, 2, 2, 2, 2], [4, 8, 8, 8, 4, 4, 4, 2, 4, 4, 2, 4, 4, 4, 4, 2, 4, 4, 2, 2, 4, 2], [2, 2, 8, 2, 2, 2, 2, 4, 2, 2, 2, 2, 4, 2, 2, 2, 2, 4, 2, 2, 2, 2], [4, 2, 4, 2, 2, 4, 2, 2, 2, 4, 2, 2, 4, 2, 4, 2, 2, 4, 2, 4, 4, 2], [4, 2, 4, 2, 2, 4, 2, 2, 2, 4, 4, 2, 4, 2, 4, 4, 2, 4, 2, 2, 4, 2], [2, 2, 8, 2, 2, 2, 2, 4, 2, 2, 2, 2, 8, 2, 2, 2, 2, 4, 2, 2, 2, 2], [4, 8, 8, 8, 2, 4, 4, 2, 4, 2, 4, 8, 8, 8, 4, 4, 4, 2, 2, 4, 4, 4], [2, 2, 8, 2, 2, 2, 2, 4, 2, 2, 2, 2, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2], [4, 2, 4, 2, 4, 2, 2, 4, 2, 4, 4, 2, 4, 2, 4, 4, 2, 4, 2, 4, 4, 2], [4, 2, 4, 2, 2, 4, 2, 4, 2, 4, 4, 2, 2, 2, 2, 4, 2, 4, 2, 4, 4, 2], [2, 2, 4, 2, 2, 2, 2, 4, 2, 2, 2, 2, 4, 2, 2, 2, 2, 2, 2, 2, 2, 2], [4, 4, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [2, 2, 4, 2, 2, 2, 2, 4, 2, 2, 2, 2, 2, 2, 2, 2, 2, 4, 2, 2, 2, 2], [4, 2, 4, 2, 4, 4, 2, 4, 2, 4, 4, 2, 4, 2, 4, 2, 2, 4, 2, 4, 4, 2], [4, 2, 4, 2, 4, 2, 2, 4, 2, 4, 4, 2, 2, 2, 4, 4, 2, 4, 2, 4, 2, 2], [2, 2, 2, 2, 2, 2, 2, 8, 2, 2, 2, 2, 4, 2, 2, 2, 2, 8, 2, 2, 2, 2], [4, 4, 4, 4, 4, 4, 8, 8, 8, 2, 4, 2, 4, 4, 4, 4, 8, 8, 8, 2, 4, 2], [2, 2, 4, 2, 2, 2, 2, 8, 2, 2, 2, 2, 4, 2, 2, 2, 2, 8, 2, 2, 2, 2], [4, 2, 4, 2, 4, 4, 2, 4, 2, 4, 4, 2, 4, 2, 2, 4, 2, 2, 2, 4, 4, 2], [4, 2, 4, 2, 4, 4, 2, 4, 2, 4, 2, 2, 4, 2, 4, 4, 2, 2, 2, 4, 4, 2], [2, 2, 4, 2, 2, 2, 2, 2, 2, 2, 2, 2, 4, 2, 2, 2, 2, 4, 2, 2, 2, 2]]}], "195c6913": [{"attempt_1": [[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4], [4, 4, 4, 4, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4], [4, 4, 7, 7, 7, 7, 7, 4, 4, 4, 4, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4], [7, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4], [7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4, 4, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4], [1, 2, 8, 3, 1, 2, 8, 3, 1, 2, 8, 3, 1, 2, 8, 3, 1, 2, 8, 3, 1, 2, 8, 3, 1, 2, 8, 3, 6, 4], [7, 7, 7, 4, 4, 4, 4, 4, 4, 4, 4, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4], [7, 7, 4, 4, 4, 4, 4, 4, 4, 4, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4], [7, 4, 4, 4, 4, 4, 4, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4], [7, 4, 4, 4, 4, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4, 4], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4, 4], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [1, 2, 8, 3, 1, 2, 8, 3, 1, 6, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [7, 7, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]], "attempt_2": [[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4], [4, 4, 4, 4, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4], [4, 4, 7, 7, 7, 7, 7, 4, 4, 4, 4, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4], [7, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4], [7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4, 4, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4], [1, 2, 8, 3, 1, 2, 8, 3, 1, 2, 8, 3, 1, 2, 8, 3, 1, 2, 8, 3, 1, 2, 8, 3, 1, 2, 8, 3, 1, 2], [7, 7, 7, 4, 4, 4, 4, 4, 4, 4, 4, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4], [7, 7, 4, 4, 4, 4, 4, 4, 4, 4, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4], [7, 4, 4, 4, 4, 4, 4, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4], [7, 4, 4, 4, 4, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4, 4], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4, 4], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [1, 2, 8, 3, 1, 2, 8, 3, 1, 2, 8, 3, 1, 2, 8, 3, 1, 2, 8, 3, 1, 2, 8, 3, 1, 2, 8, 3, 1, 2], [7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [7, 7, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]]}, {"attempt_1": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9], [3, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 8], [9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 8], [9, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8, 8], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [3, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]], "attempt_2": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9], [3, 3, 6, 3, 3, 6, 3, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 8], [9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 8], [9, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8, 8], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [3, 3, 6, 3, 3, 6, 3, 3, 6, 3, 3, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]]}], "21897d95": [{"attempt_1": [[0]], "attempt_2": [[0]]}, {"attempt_1": [[2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 5, 5, 5, 5], [2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 5, 5, 5, 5], [2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 5, 5, 5, 5], [2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 5, 5, 5, 5], [2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 5, 5, 5, 5], [2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 5, 5, 5, 5], [2, 2, 2, 2, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 5, 5, 5, 5], [2, 2, 2, 2, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 5, 5, 5, 5], [2, 2, 2, 2, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 5, 5, 5, 5], [2, 2, 2, 2, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 5, 5, 5, 5], [2, 2, 2, 2, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 5, 5, 5, 5], [2, 2, 2, 2, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 5, 5, 5, 5], [2, 2, 2, 2, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 5, 5, 5, 5], [2, 2, 2, 2, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 5, 5, 5, 5], [2, 2, 2, 2, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 5, 5, 5, 5], [2, 2, 2, 2, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 5, 5, 5, 5], [2, 2, 2, 2, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 5, 5, 5, 5], [2, 2, 2, 2, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 5, 5, 5, 5], [2, 2, 2, 2, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 5, 5, 5, 5], [2, 2, 2, 2, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 5, 5, 5, 5], [2, 2, 2, 2, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 5, 5, 5, 5], [2, 2, 2, 2, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 5, 5, 5, 5], [2, 2, 2, 2, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 5, 5, 5, 5], [2, 2, 2, 2, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 5, 5, 5, 5]], "attempt_2": [[3, 3, 3, 3, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 5, 5, 5, 5], [3, 3, 3, 3, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 5, 5, 5, 5], [3, 3, 3, 3, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 5, 5, 5, 5], [3, 3, 3, 3, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 5, 5, 5, 5], [3, 3, 3, 3, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 5, 5, 5, 5], [3, 3, 3, 3, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 5, 5, 5, 5], [3, 3, 3, 3, 4, 4, 4, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 5, 5, 5, 5], [3, 3, 3, 3, 4, 4, 4, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 5, 5, 5, 5], [3, 3, 3, 3, 4, 4, 4, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 5, 5, 5, 5], [3, 3, 3, 3, 4, 4, 4, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 5, 5, 5, 5], [3, 3, 3, 3, 4, 4, 4, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 5, 5, 5, 5], [3, 3, 3, 3, 4, 4, 4, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 5, 5, 5, 5], [3, 3, 3, 3, 4, 4, 4, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 5, 5, 5, 5], [3, 3, 3, 3, 4, 4, 4, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 5, 5, 5, 5], [3, 3, 3, 3, 4, 4, 4, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 5, 5, 5, 5], [3, 3, 3, 3, 4, 4, 4, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 5, 5, 5, 5], [3, 3, 3, 3, 4, 4, 4, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 5, 5, 5, 5], [3, 3, 3, 3, 4, 4, 4, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 5, 5, 5, 5], [3, 3, 3, 3, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 5, 5, 5, 5], [3, 3, 3, 3, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 5, 5, 5, 5], [3, 3, 3, 3, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 5, 5, 5, 5], [3, 3, 3, 3, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 5, 5, 5, 5], [3, 3, 3, 3, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 5, 5, 5, 5], [3, 3, 3, 3, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 5, 5, 5, 5]]}], "221dfab4": [{"attempt_1": [[8, 8, 8, 8, 8, 8, 8, 3, 3, 3, 3, 3, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 4, 4, 4, 4, 4, 4, 4, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 6, 6, 6, 6, 4, 4, 4, 4, 4, 4, 4, 4, 6, 6, 6, 6, 6, 8, 8, 8, 8, 8], [8, 8, 8, 8, 6, 6, 6, 8, 8, 8, 8, 8, 8, 8, 8, 6, 6, 6, 6, 6, 6, 6, 8, 8, 8], [8, 8, 8, 8, 8, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 6, 8, 6, 6, 6, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 4, 4, 4, 4, 4, 4, 4, 4, 6, 6, 6, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 6, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 6, 6, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 6, 6, 6, 6, 4, 4, 4, 4, 4, 4, 4, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 6, 6, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 3, 3, 3, 3, 3, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 6, 6, 6, 6, 6, 6, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 4, 4, 4, 4, 4, 4, 4, 4, 6, 6, 6, 6, 6, 6, 6, 6, 8, 8], [8, 8, 8, 8, 6, 6, 6, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 6, 8, 8, 8], [8, 8, 8, 6, 6, 6, 6, 4, 4, 4, 4, 4, 4, 4, 4, 8, 8, 8, 6, 6, 6, 6, 8, 8, 8], [8, 8, 6, 6, 6, 6, 6, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 6, 6, 6, 6, 6, 6, 6], [8, 8, 8, 8, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 8, 8, 3, 3, 3, 3, 3, 3, 3], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 6, 6, 6, 6, 6], [8, 8, 8, 8, 8, 6, 6, 4, 4, 4, 4, 4, 4, 4, 4, 6, 6, 8, 8, 6, 6, 6, 6, 6, 8], [8, 8, 6, 6, 6, 6, 6, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 6, 6, 4, 4, 4, 4, 4, 4, 4, 4, 6, 6, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 6, 6, 8, 8, 8, 8, 8, 8, 8, 8, 6, 6, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8]], "attempt_2": [[8, 8, 8, 8, 8, 8, 8, 4, 4, 4, 4, 4, 4, 4, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 3, 3, 3, 3, 3, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 6, 6, 6, 6, 4, 4, 4, 4, 4, 4, 4, 4, 6, 6, 6, 6, 6, 8, 8, 8, 8, 8], [8, 8, 8, 8, 6, 6, 6, 8, 8, 8, 8, 8, 8, 8, 8, 6, 6, 6, 6, 6, 6, 6, 8, 8, 8], [8, 8, 8, 8, 8, 6, 6, 4, 4, 4, 4, 4, 4, 4, 4, 6, 6, 6, 6, 6, 6, 6, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 6, 8, 6, 6, 6, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 6, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 6, 6, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 6, 6, 6, 6, 4, 4, 4, 4, 4, 4, 4, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 6, 6, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 4, 4, 4, 4, 4, 4, 4, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 6, 6, 6, 6, 6, 6, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 8], [8, 8, 8, 8, 6, 6, 6, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 6, 8, 8, 8], [8, 8, 8, 6, 6, 6, 6, 4, 4, 4, 4, 4, 4, 4, 4, 8, 8, 8, 6, 6, 6, 6, 8, 8, 8], [8, 8, 6, 6, 6, 6, 6, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 6, 6, 6, 6, 6, 6, 6], [8, 8, 8, 8, 6, 6, 6, 4, 4, 4, 4, 4, 4, 4, 4, 8, 8, 8, 6, 6, 6, 6, 6, 6, 6], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 6, 6, 6, 6, 6], [8, 8, 8, 8, 8, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 8, 3, 3, 3, 3, 3, 8], [8, 8, 6, 6, 6, 6, 6, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 6, 6, 4, 4, 4, 4, 4, 4, 4, 4, 6, 6, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 6, 6, 8, 8, 8, 8, 8, 8, 8, 8, 6, 6, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 4, 4, 4, 4, 4, 4, 4, 4, 6, 6, 8, 8, 8, 8, 8, 8, 8, 8]]}, {"attempt_1": [[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 2, 2], [2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 2], [2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 2], [2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1, 2], [4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 4, 2], [4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 4, 2], [4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 4, 2], [4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 4, 2], [4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 4, 2], [2, 2, 2, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2], [2, 2, 2, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2], [2, 2, 2, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2], [2, 2, 2, 1, 1, 2, 2, 2, 2, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 2, 2, 2, 2], [2, 1, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 2, 2, 2, 2], [2, 2, 1, 1, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, 2, 1, 2, 2, 2, 2, 2, 1, 1, 1, 2, 2, 2, 2], [2, 2, 1, 1, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 1, 1, 2, 2, 2, 2, 1, 1, 1, 1, 2, 2, 2, 2], [2, 2, 2, 2, 2, 1, 1, 2, 1, 1, 1, 1, 2, 2, 2, 2, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 2, 2, 2, 2, 1, 1, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2], [2, 2, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2], [2, 2, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2], [2, 2, 2, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 2, 2, 2], [2, 2, 2, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 2, 2, 2], [2, 2, 2, 1, 1, 1, 1, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 1, 1, 1, 2, 2, 2, 2], [2, 2, 1, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2], [2, 2, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2], [2, 2, 2, 1, 1, 1, 2, 2, 2, 2, 2, 2, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]], "attempt_2": [[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 2, 2], [2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 2], [2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 2], [2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1, 2], [3, 2, 4, 2, 3, 2, 4, 2, 3, 2, 4, 2, 3, 2, 4, 2, 3, 2, 4, 2, 3, 2, 4, 2, 3, 2, 4, 2, 3, 2], [3, 2, 4, 2, 3, 2, 4, 2, 3, 2, 4, 2, 3, 2, 4, 2, 3, 2, 4, 2, 3, 2, 4, 2, 3, 2, 4, 2, 3, 2], [3, 2, 4, 2, 3, 2, 4, 2, 3, 2, 4, 2, 3, 2, 4, 2, 3, 2, 4, 2, 3, 2, 4, 2, 3, 2, 4, 2, 3, 2], [3, 2, 4, 2, 3, 2, 4, 2, 3, 2, 4, 2, 3, 2, 4, 2, 3, 2, 4, 2, 3, 2, 4, 2, 3, 2, 4, 2, 3, 2], [3, 2, 4, 2, 3, 2, 4, 2, 3, 2, 4, 2, 3, 2, 4, 2, 3, 2, 4, 2, 3, 2, 4, 2, 3, 2, 4, 2, 3, 2], [2, 2, 2, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2], [2, 2, 2, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2], [2, 2, 2, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2], [2, 2, 2, 1, 1, 2, 2, 2, 2, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 2, 2, 2, 2], [2, 1, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 2, 2, 2, 2], [2, 2, 1, 1, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, 2, 1, 2, 2, 2, 2, 2, 1, 1, 1, 2, 2, 2, 2], [2, 2, 1, 1, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 1, 1, 2, 2, 2, 2, 1, 1, 1, 1, 2, 2, 2, 2], [2, 2, 2, 2, 2, 1, 1, 2, 1, 1, 1, 1, 2, 2, 2, 2, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 2, 2, 2, 2, 1, 1, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2], [2, 2, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2], [2, 2, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2], [2, 2, 2, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 2, 2, 2], [2, 2, 2, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 2, 2, 2], [2, 2, 2, 1, 1, 1, 1, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 1, 1, 1, 2, 2, 2, 2], [2, 2, 1, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2], [2, 2, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2], [2, 2, 2, 1, 1, 1, 2, 2, 2, 2, 2, 2, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]]}], "269e22fb": [{"attempt_1": [[8, 8, 8, 7, 7, 7, 8, 8, 7, 8, 8, 8, 7, 8, 8, 8, 8, 7, 7, 7], [8, 7, 7, 7, 8, 7, 8, 8, 8, 7, 8, 7, 8, 8, 8, 7, 7, 8, 7, 8], [7, 7, 8, 8, 8, 7, 7, 8, 8, 8, 7, 8, 8, 8, 8, 7, 8, 7, 7, 7], [8, 7, 7, 8, 8, 8, 7, 8, 8, 8, 7, 8, 8, 8, 8, 7, 7, 8, 7, 8], [8, 8, 7, 7, 8, 7, 7, 8, 8, 8, 7, 8, 8, 8, 8, 7, 8, 7, 7, 7], [8, 7, 7, 7, 7, 7, 8, 8, 8, 8, 7, 8, 8, 8, 8, 7, 7, 8, 7, 8], [8, 7, 7, 8, 8, 7, 8, 8, 8, 8, 7, 7, 7, 7, 7, 7, 8, 7, 7, 7], [8, 7, 7, 8, 8, 7, 7, 7, 8, 8, 7, 8, 8, 8, 8, 8, 7, 8, 7, 8], [8, 7, 7, 8, 8, 7, 8, 7, 8, 8, 7, 7, 7, 7, 7, 7, 7, 7, 7, 8], [8, 7, 7, 8, 8, 7, 8, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 7, 8], [8, 7, 7, 8, 8, 7, 8, 7, 7, 8, 7, 7, 7, 7, 7, 7, 7, 7, 7, 8], [8, 7, 7, 8, 8, 7, 8, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 7, 7, 8, 8, 7, 8, 7, 8, 8, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 7, 7, 8, 8, 7, 7, 7, 8, 8, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8], [8, 7, 7, 8, 8, 7, 8, 8, 8, 8, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8], [8, 7, 7, 7, 7, 7, 8, 8, 8, 8, 8, 7, 7, 7, 8, 8, 8, 8, 8, 8], [8, 8, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 7, 7, 7, 8, 8, 8, 7, 7], [8, 8, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 7, 7, 7, 7, 7, 7, 7], [8, 8, 8, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 7, 7, 7, 7, 7, 7], [8, 8, 8, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 7, 7]], "attempt_2": [[8, 8, 8, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 7, 7], [8, 8, 8, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 7, 7, 7, 7, 7, 7], [8, 8, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 7, 7, 7, 7, 7, 7, 7], [8, 8, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 7, 7, 7, 8, 8, 8, 7, 7], [8, 7, 7, 7, 7, 7, 8, 8, 8, 8, 8, 7, 7, 7, 8, 8, 8, 8, 8, 8], [8, 7, 7, 8, 8, 7, 8, 8, 8, 8, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8], [8, 7, 7, 8, 8, 7, 7, 7, 8, 8, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8], [8, 7, 7, 8, 8, 7, 8, 7, 8, 8, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 7, 7, 8, 8, 7, 8, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 7, 7, 8, 8, 7, 8, 7, 7, 8, 7, 7, 7, 7, 7, 7, 7, 7, 7, 8], [8, 7, 7, 8, 8, 7, 8, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 7, 8], [8, 7, 7, 8, 8, 7, 8, 7, 8, 8, 7, 7, 7, 7, 7, 7, 7, 7, 7, 8], [8, 7, 7, 8, 8, 7, 7, 7, 8, 8, 7, 8, 8, 8, 8, 8, 7, 8, 7, 8], [8, 7, 7, 8, 8, 7, 8, 8, 8, 8, 7, 7, 7, 7, 7, 7, 8, 7, 7, 7], [8, 7, 7, 7, 7, 7, 8, 8, 8, 8, 7, 8, 8, 8, 8, 7, 7, 8, 7, 8], [8, 8, 7, 7, 8, 7, 7, 8, 8, 8, 7, 8, 8, 8, 8, 7, 8, 7, 7, 7], [8, 7, 7, 8, 8, 8, 7, 8, 8, 8, 7, 8, 8, 8, 8, 7, 7, 8, 7, 8], [7, 7, 8, 8, 8, 7, 7, 8, 8, 8, 7, 8, 8, 8, 8, 7, 8, 7, 7, 7], [8, 7, 7, 7, 8, 7, 8, 8, 8, 7, 8, 7, 8, 8, 8, 7, 7, 8, 7, 8], [8, 8, 8, 7, 7, 7, 8, 8, 7, 8, 8, 8, 7, 8, 8, 8, 8, 7, 7, 7]]}, {"attempt_1": [[5, 5, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 5, 5, 6, 6, 6], [5, 5, 5, 5, 5, 5, 6, 6, 6, 6, 6, 6, 6, 6, 6, 5, 5, 6, 6, 6], [5, 5, 5, 5, 5, 5, 5, 6, 6, 6, 6, 6, 6, 6, 6, 5, 5, 5, 6, 6], [5, 5, 6, 6, 6, 5, 5, 5, 6, 6, 6, 6, 6, 6, 6, 5, 5, 5, 6, 6], [6, 6, 6, 6, 6, 6, 5, 5, 5, 6, 6, 6, 6, 6, 5, 5, 5, 5, 5, 6], [6, 6, 6, 6, 6, 6, 6, 5, 5, 5, 6, 6, 6, 6, 5, 6, 6, 5, 5, 6], [6, 6, 6, 6, 6, 6, 6, 6, 5, 5, 6, 6, 5, 5, 5, 6, 6, 5, 5, 6], [6, 6, 6, 6, 6, 6, 6, 6, 6, 5, 6, 6, 5, 6, 5, 6, 6, 5, 5, 6], [6, 6, 6, 6, 6, 6, 6, 6, 6, 5, 5, 5, 5, 6, 5, 6, 6, 5, 5, 6], [6, 5, 5, 5, 5, 5, 5, 5, 5, 5, 6, 5, 5, 6, 5, 6, 6, 5, 5, 6], [6, 5, 6, 6, 6, 6, 6, 6, 6, 5, 5, 5, 5, 6, 5, 6, 6, 5, 5, 6], [6, 5, 5, 5, 5, 5, 5, 5, 5, 5, 6, 6, 5, 6, 5, 6, 6, 5, 5, 6], [6, 5, 6, 5, 6, 6, 6, 6, 6, 5, 6, 6, 5, 5, 5, 6, 6, 5, 5, 6], [5, 5, 5, 6, 5, 5, 5, 5, 5, 5, 6, 6, 6, 6, 5, 6, 6, 5, 5, 6], [6, 5, 6, 5, 5, 6, 6, 6, 6, 5, 6, 6, 6, 6, 5, 5, 5, 5, 5, 6], [5, 5, 5, 6, 5, 6, 6, 6, 6, 5, 6, 6, 6, 5, 5, 6, 5, 5, 6, 6], [6, 5, 6, 5, 5, 6, 6, 6, 6, 5, 6, 6, 6, 5, 6, 6, 6, 5, 5, 6], [5, 5, 5, 6, 5, 6, 6, 6, 6, 5, 6, 6, 6, 5, 5, 6, 6, 6, 5, 5], [6, 5, 6, 5, 5, 6, 6, 6, 5, 6, 5, 6, 6, 6, 5, 6, 5, 5, 5, 6], [5, 5, 5, 6, 6, 6, 6, 5, 6, 6, 6, 5, 6, 6, 5, 5, 5, 6, 6, 6]], "attempt_2": [[6, 6, 6, 5, 5, 5, 6, 6, 5, 6, 6, 6, 5, 6, 6, 6, 6, 5, 5, 5], [6, 5, 5, 5, 6, 5, 6, 6, 6, 5, 6, 5, 6, 6, 6, 5, 5, 6, 5, 6], [5, 5, 6, 6, 6, 5, 5, 6, 6, 6, 5, 6, 6, 6, 6, 5, 6, 5, 5, 5], [6, 5, 5, 6, 6, 6, 5, 6, 6, 6, 5, 6, 6, 6, 6, 5, 5, 6, 5, 6], [6, 6, 5, 5, 6, 5, 5, 6, 6, 6, 5, 6, 6, 6, 6, 5, 6, 5, 5, 5], [6, 5, 5, 5, 5, 5, 6, 6, 6, 6, 5, 6, 6, 6, 6, 5, 5, 6, 5, 6], [6, 5, 5, 6, 6, 5, 6, 6, 6, 6, 5, 5, 5, 5, 5, 5, 6, 5, 5, 5], [6, 5, 5, 6, 6, 5, 5, 5, 6, 6, 5, 6, 6, 6, 6, 6, 5, 6, 5, 6], [6, 5, 5, 6, 6, 5, 6, 5, 6, 6, 5, 5, 5, 5, 5, 5, 5, 5, 5, 6], [6, 5, 5, 6, 6, 5, 6, 5, 5, 5, 5, 6, 6, 6, 6, 6, 6, 6, 5, 6], [6, 5, 5, 6, 6, 5, 6, 5, 5, 6, 5, 5, 5, 5, 5, 5, 5, 5, 5, 6], [6, 5, 5, 6, 6, 5, 6, 5, 5, 5, 5, 6, 6, 6, 6, 6, 6, 6, 6, 6], [6, 5, 5, 6, 6, 5, 6, 5, 6, 6, 5, 6, 6, 6, 6, 6, 6, 6, 6, 6], [6, 5, 5, 6, 6, 5, 5, 5, 6, 6, 5, 5, 6, 6, 6, 6, 6, 6, 6, 6], [6, 5, 5, 6, 6, 5, 6, 6, 6, 6, 5, 5, 5, 6, 6, 6, 6, 6, 6, 6], [6, 5, 5, 5, 5, 5, 6, 6, 6, 6, 6, 5, 5, 5, 6, 6, 6, 6, 6, 6], [6, 6, 5, 5, 5, 6, 6, 6, 6, 6, 6, 6, 5, 5, 5, 6, 6, 6, 5, 5], [6, 6, 5, 5, 5, 6, 6, 6, 6, 6, 6, 6, 6, 5, 5, 5, 5, 5, 5, 5], [6, 6, 6, 5, 5, 6, 6, 6, 6, 6, 6, 6, 6, 6, 5, 5, 5, 5, 5, 5], [6, 6, 6, 5, 5, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 5, 5]]}], "28a6681f": [{"attempt_1": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [9, 9, 0, 0, 0, 0, 0, 0, 0, 0], [0, 9, 9, 0, 0, 0, 0, 0, 0, 0], [0, 0, 9, 9, 9, 0, 0, 0, 0, 5], [0, 0, 0, 0, 9, 9, 9, 0, 0, 5], [0, 0, 0, 0, 0, 0, 0, 5, 0, 5], [3, 0, 0, 0, 0, 0, 0, 0, 0, 5], [3, 3, 0, 0, 0, 0, 5, 5, 1, 5], [3, 3, 3, 0, 0, 5, 5, 5, 1, 5], [3, 3, 3, 3, 5, 5, 5, 5, 1, 5]], "attempt_2": [[1, 1, 0, 0, 0, 0, 0, 0, 0, 0], [9, 9, 1, 1, 0, 0, 0, 0, 0, 0], [0, 9, 9, 1, 0, 0, 0, 0, 0, 0], [0, 0, 9, 9, 9, 0, 0, 0, 0, 5], [0, 0, 0, 0, 9, 9, 9, 0, 0, 5], [0, 0, 0, 0, 0, 0, 0, 5, 0, 5], [3, 0, 0, 0, 0, 0, 0, 0, 0, 5], [3, 3, 0, 0, 0, 0, 5, 5, 0, 5], [3, 3, 3, 0, 0, 5, 5, 5, 0, 5], [3, 3, 3, 3, 5, 5, 5, 5, 0, 5]]}], "2b83f449": [{"attempt_1": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 8, 8, 8, 8, 8, 8], [0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 6, 8, 0, 0, 0, 0, 0], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 8, 8, 6, 8, 8, 8], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 6, 8, 0, 0], [3, 8, 8, 8, 8, 8, 8, 0, 3, 8, 8, 8, 8, 6, 8, 3, 3], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [8, 8, 8, 8, 8, 8, 8, 6, 8, 8, 8, 8, 8, 8, 8, 8, 8], [0, 0, 0, 0, 0, 0, 8, 6, 8, 0, 0, 0, 0, 0, 0, 0, 0], [8, 8, 8, 8, 6, 8, 8, 6, 8, 8, 8, 8, 8, 8, 8, 8, 8], [0, 0, 0, 8, 6, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [3, 3, 8, 8, 6, 8, 8, 8, 8, 3, 0, 8, 8, 8, 8, 8, 3], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 8, 8, 8, 8, 8, 8], [0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 6, 8, 0, 0, 0, 0, 0], [8, 8, 8, 8, 8, 8, 8, 6, 8, 8, 6, 8, 8, 8, 8, 3, 3], [0, 0, 0, 0, 0, 0, 8, 6, 8, 0, 0, 0, 0, 0, 0, 0, 0], [8, 8, 8, 6, 8, 8, 8, 6, 8, 8, 8, 8, 6, 8, 8, 8, 8], [0, 0, 8, 6, 8, 0, 0, 0, 0, 0, 0, 8, 6, 8, 0, 0, 0], [3, 8, 8, 6, 8, 8, 8, 0, 3, 8, 8, 8, 6, 8, 8, 8, 3]], "attempt_2": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 8, 8, 8, 8, 8, 8], [0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 6, 8, 0, 0, 0, 0, 0], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 8, 8, 6, 8, 8, 8], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 6, 8, 0, 0], [3, 8, 8, 8, 8, 8, 8, 0, 3, 8, 8, 8, 8, 6, 8, 3, 3], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [8, 8, 8, 8, 8, 8, 8, 6, 8, 8, 8, 8, 8, 8, 8, 8, 8], [0, 0, 0, 0, 0, 0, 8, 6, 8, 0, 0, 0, 0, 0, 0, 0, 0], [8, 8, 8, 8, 6, 8, 8, 6, 8, 8, 8, 8, 8, 8, 8, 8, 8], [0, 0, 0, 8, 6, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [3, 3, 8, 8, 6, 8, 8, 8, 8, 8, 0, 3, 8, 8, 8, 8, 3], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 8, 8, 8, 8, 8, 8], [0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 6, 8, 0, 0, 0, 0, 0], [8, 8, 8, 8, 8, 8, 8, 6, 8, 8, 6, 8, 8, 8, 8, 3, 3], [0, 0, 0, 0, 0, 0, 8, 6, 8, 0, 0, 0, 0, 0, 0, 0, 0], [8, 8, 8, 6, 8, 8, 8, 6, 8, 8, 8, 8, 6, 8, 8, 8, 8], [0, 0, 8, 6, 8, 0, 0, 0, 0, 0, 0, 8, 6, 8, 0, 0, 0], [3, 8, 8, 6, 8, 8, 8, 0, 3, 8, 8, 8, 6, 8, 8, 8, 3]]}], "2c181942": [{"attempt_1": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 4, 8, 8, 1, 1, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 4, 4, 8, 8, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 4, 4, 8, 8, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 4, 8, 8, 6, 6, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]], "attempt_2": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 4, 8, 8, 1, 1, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 4, 4, 8, 8, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 4, 4, 8, 8, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 4, 8, 8, 6, 6, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 6, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 6, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 6, 6, 6, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]]}], "2d0172a1": [{"attempt_1": [[9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 7, 7, 7, 7, 7, 7, 7, 7, 7, 9], [9, 7, 9, 9, 9, 9, 9, 7, 7, 7, 9], [9, 7, 9, 7, 7, 7, 9, 7, 7, 7, 9], [9, 7, 9, 7, 9, 7, 9, 7, 9, 7, 9], [9, 7, 9, 7, 7, 7, 9, 7, 7, 7, 9], [9, 7, 9, 9, 9, 9, 9, 7, 7, 7, 9], [9, 7, 7, 7, 7, 7, 7, 7, 7, 7, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9]], "attempt_2": [[9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 7, 7, 7, 7, 7, 7, 7, 7, 7, 9], [9, 7, 9, 9, 9, 9, 9, 7, 7, 7, 9], [9, 7, 9, 7, 7, 7, 9, 7, 7, 7, 9], [9, 7, 9, 7, 9, 7, 9, 7, 9, 7, 9], [9, 7, 9, 7, 7, 7, 9, 7, 7, 7, 9], [9, 7, 9, 9, 9, 9, 9, 7, 7, 7, 9], [9, 7, 7, 7, 7, 7, 7, 7, 7, 7, 9], [9, 7, 7, 7, 9, 7, 7, 7, 7, 7, 9], [9, 7, 7, 7, 7, 7, 7, 7, 7, 7, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9]]}, {"attempt_1": [[6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6], [6, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6], [6, 8, 8, 8, 6, 6, 6, 6, 6, 8, 6], [6, 8, 8, 8, 6, 8, 8, 8, 6, 8, 6], [6, 8, 6, 8, 6, 8, 6, 8, 6, 8, 6], [6, 8, 8, 8, 6, 8, 8, 8, 6, 8, 6], [6, 8, 8, 8, 6, 6, 6, 6, 6, 8, 6], [6, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6], [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 6, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]], "attempt_2": [[6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6], [6, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6], [6, 8, 8, 8, 8, 8, 6, 8, 8, 8, 6], [6, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6], [6, 8, 8, 8, 6, 6, 6, 6, 6, 8, 6], [6, 8, 8, 8, 6, 8, 8, 8, 6, 8, 6], [6, 8, 6, 8, 6, 8, 6, 8, 6, 8, 6], [6, 8, 8, 8, 6, 8, 8, 8, 6, 8, 6], [6, 8, 8, 8, 6, 6, 6, 6, 6, 8, 6], [6, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6], [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 6, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]]}], "31f7f899": [{"attempt_1": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 7], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 7], [8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 8, 8, 8, 8, 8, 8, 7], [8, 8, 5, 5, 5, 8, 8, 2, 2, 2, 8, 8, 8, 8, 8, 8, 7], [9, 8, 5, 5, 5, 8, 8, 2, 2, 2, 8, 8, 8, 8, 8, 8, 7], [9, 6, 5, 5, 5, 6, 6, 2, 2, 2, 6, 6, 6, 6, 6, 6, 7], [9, 8, 5, 5, 5, 8, 8, 2, 2, 2, 8, 8, 8, 8, 8, 8, 7], [8, 8, 5, 5, 5, 8, 8, 2, 2, 2, 8, 8, 8, 8, 8, 8, 7], [8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 8, 8, 8, 8, 8, 8, 7], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 7], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]], "attempt_2": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 7], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 7], [8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 8, 8, 8, 8, 8, 8, 7], [9, 8, 5, 5, 5, 8, 8, 2, 2, 2, 8, 8, 8, 8, 8, 8, 7], [9, 8, 5, 5, 5, 8, 8, 2, 2, 2, 8, 8, 8, 8, 8, 8, 7], [9, 6, 5, 5, 5, 6, 6, 2, 2, 2, 6, 6, 6, 6, 6, 6, 7], [9, 8, 5, 5, 5, 8, 8, 2, 2, 2, 8, 8, 8, 8, 8, 8, 7], [9, 8, 5, 5, 5, 8, 8, 2, 2, 2, 8, 8, 8, 8, 8, 8, 7], [8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 8, 8, 8, 8, 8, 8, 7], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 7], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]]}], "332f06d7": [{"attempt_1": [[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 1, 1, 1, 1, 1, 1, 1, 3, 3, 3, 3, 3, 3, 1, 1, 1, 1, 1, 3], [3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3], [3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3], [3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3], [3, 1, 1, 1, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 1, 1, 1, 1, 3], [3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 1, 1, 1, 1, 3], [3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 3, 1, 1, 1, 3], [3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 3, 3, 1, 1, 3], [3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 3, 3, 1, 1, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 1, 1, 1, 3, 3, 3, 1, 1, 1, 3], [3, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 3, 3, 1, 1, 1, 1, 3], [3, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 3, 3, 1, 1, 1, 1, 3], [3, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 3], [3, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 3, 1, 2, 2, 2, 2], [3, 3, 3, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 2, 2, 2, 2], [3, 3, 3, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 1, 2, 2, 2, 2], [3, 3, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 1, 2, 2, 2, 2], [3, 3, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]], "attempt_2": [[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 1, 1, 1, 1, 1, 1, 1, 3, 3, 3, 3, 3, 3, 1, 1, 1, 1, 1, 3], [3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3], [3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3], [3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3], [3, 1, 1, 1, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 1, 1, 1, 1, 3], [3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 1, 1, 1, 1, 3], [3, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 3, 3, 3, 1, 1, 1, 3], [3, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 3, 3, 3, 3, 1, 1, 3], [3, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 3, 3, 3, 3, 1, 1, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 3, 3, 3, 1, 1, 1, 3], [3, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 3, 3, 1, 1, 1, 1, 3], [3, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 3, 3, 1, 1, 1, 1, 3], [3, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 3], [3, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 3, 1, 2, 2, 2, 2], [3, 3, 3, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 2, 2, 2, 2], [3, 3, 3, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 1, 2, 2, 2, 2], [3, 3, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 1, 2, 2, 2, 2], [3, 3, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]]}], "35ab12c3": [{"attempt_1": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0], [0, 0, 0, 6, 0, 0, 7, 1, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0], [0, 0, 4, 6, 0, 0, 7, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0], [0, 0, 4, 6, 0, 0, 7, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0], [0, 0, 4, 6, 0, 0, 7, 1, 0, 3, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0], [0, 0, 4, 6, 0, 0, 7, 1, 0, 0, 3, 3, 3, 3, 0, 0, 0, 3, 0, 0, 0], [0, 0, 4, 6, 0, 0, 7, 1, 0, 0, 0, 0, 0, 0, 3, 0, 0, 3, 0, 0, 0], [0, 0, 4, 6, 0, 0, 7, 1, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 0, 0, 0], [0, 0, 4, 6, 0, 0, 7, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 4, 6, 0, 0, 7, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 4, 6, 0, 0, 7, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 4, 6, 0, 0, 7, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 4, 6, 0, 0, 7, 1, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 4, 6, 0, 0, 7, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 4, 6, 0, 0, 7, 1, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0], [0, 0, 4, 6, 0, 0, 7, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 6, 0, 0, 0, 1, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], "attempt_2": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0], [0, 0, 0, 6, 0, 0, 7, 1, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0], [0, 0, 4, 6, 0, 0, 7, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0], [0, 0, 0, 6, 0, 0, 7, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0], [0, 0, 0, 6, 0, 0, 7, 1, 0, 3, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0], [0, 0, 0, 6, 0, 0, 7, 1, 0, 0, 3, 3, 3, 3, 0, 0, 0, 3, 0, 0, 0], [0, 0, 0, 6, 0, 0, 7, 1, 0, 0, 0, 0, 0, 0, 3, 0, 0, 3, 0, 0, 0], [0, 0, 0, 6, 0, 0, 7, 1, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 0, 0, 0], [0, 0, 0, 6, 0, 0, 7, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 6, 0, 0, 7, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 6, 0, 0, 7, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 6, 0, 0, 7, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 6, 0, 0, 7, 1, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 6, 0, 0, 7, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 6, 0, 0, 7, 1, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0], [0, 0, 0, 6, 0, 0, 7, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 6, 0, 0, 0, 1, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}], "3a25b0d8": [{"attempt_1": [[0, 0, 0, 4, 4, 0, 0, 0, 4, 4, 0, 0, 0], [0, 0, 4, 3, 3, 4, 4, 4, 8, 8, 4, 0, 0], [0, 4, 3, 3, 3, 3, 4, 8, 8, 8, 8, 4, 0], [0, 4, 3, 3, 3, 3, 4, 8, 8, 8, 8, 4, 0], [0, 4, 4, 4, 3, 3, 4, 8, 8, 4, 4, 4, 0], [0, 0, 4, 4, 4, 4, 4, 4, 4, 4, 4, 0, 0], [0, 4, 5, 5, 5, 4, 4, 4, 5, 5, 5, 4, 0], [0, 4, 5, 5, 5, 4, 2, 4, 5, 5, 5, 4, 0], [0, 4, 4, 4, 4, 2, 2, 2, 4, 4, 4, 4, 0], [0, 4, 1, 4, 4, 4, 2, 4, 4, 4, 1, 4, 0], [0, 4, 4, 4, 7, 4, 4, 4, 7, 4, 4, 4, 0], [0, 0, 4, 7, 7, 7, 4, 7, 7, 7, 4, 0, 0], [4, 4, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4], [0, 4, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 0], [0, 4, 7, 7, 7, 7, 4, 7, 7, 7, 7, 4, 0], [0, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 0], [0, 0, 0, 0, 4, 6, 6, 6, 4, 0, 0, 0, 0], [0, 0, 0, 0, 4, 4, 4, 4, 4, 0, 0, 0, 0]], "attempt_2": [[0, 0, 0, 4, 4, 0, 0, 0, 4, 4, 0, 0, 0], [0, 0, 4, 3, 3, 4, 4, 4, 3, 3, 4, 0, 0], [0, 4, 3, 3, 3, 3, 4, 3, 3, 3, 3, 4, 0], [0, 4, 3, 3, 3, 3, 4, 3, 3, 3, 3, 4, 0], [0, 4, 4, 4, 3, 3, 4, 3, 3, 4, 4, 4, 0], [0, 0, 4, 4, 4, 4, 4, 4, 4, 4, 4, 0, 0], [0, 4, 5, 5, 5, 4, 4, 4, 5, 5, 5, 4, 0], [0, 4, 5, 5, 5, 4, 2, 4, 5, 5, 5, 4, 0], [0, 4, 4, 4, 4, 2, 2, 2, 4, 4, 4, 4, 0], [0, 4, 1, 4, 4, 4, 2, 4, 4, 4, 1, 4, 0], [0, 4, 4, 4, 7, 4, 4, 4, 7, 4, 4, 4, 0], [0, 0, 4, 7, 7, 7, 4, 7, 7, 7, 4, 0, 0], [4, 4, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4], [0, 4, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 0], [0, 4, 7, 7, 7, 7, 4, 7, 7, 7, 7, 4, 0], [0, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 0], [0, 0, 0, 0, 4, 6, 6, 6, 4, 0, 0, 0, 0], [0, 0, 0, 0, 4, 4, 4, 4, 4, 0, 0, 0, 0]]}, {"attempt_1": [[4, 4, 4, 4, 2, 2, 2, 4, 4, 4, 4], [4, 4, 4, 2, 2, 8, 2, 2, 4, 4, 4], [4, 2, 2, 2, 8, 8, 8, 2, 2, 2, 4], [2, 1, 1, 2, 2, 8, 2, 2, 9, 9, 2], [2, 1, 1, 2, 7, 2, 7, 2, 9, 9, 2], [2, 2, 2, 2, 7, 7, 7, 2, 2, 2, 2], [4, 2, 2, 7, 7, 7, 7, 7, 2, 2, 4], [4, 2, 2, 2, 2, 2, 2, 2, 2, 2, 4], [4, 2, 8, 8, 8, 2, 8, 8, 8, 2, 4], [4, 2, 8, 8, 8, 2, 8, 8, 8, 2, 4], [2, 5, 2, 2, 2, 3, 2, 2, 2, 5, 2], [2, 2, 2, 2, 3, 3, 3, 2, 2, 2, 2], [4, 4, 4, 2, 2, 3, 2, 2, 4, 4, 4], [4, 4, 4, 4, 2, 2, 2, 4, 4, 4, 4]], "attempt_2": [[4, 4, 4, 4, 2, 2, 2, 4, 4, 4, 4], [4, 4, 4, 2, 2, 8, 2, 2, 4, 4, 4], [4, 2, 2, 2, 8, 8, 8, 2, 2, 2, 4], [2, 1, 1, 2, 2, 8, 2, 2, 1, 1, 2], [2, 1, 1, 2, 7, 2, 7, 2, 1, 1, 2], [2, 2, 2, 2, 7, 7, 7, 2, 2, 2, 2], [4, 2, 2, 7, 7, 7, 7, 7, 2, 2, 4], [4, 2, 2, 2, 2, 2, 2, 2, 2, 2, 4], [4, 2, 8, 8, 8, 2, 8, 8, 8, 2, 4], [4, 2, 8, 8, 8, 2, 8, 8, 8, 2, 4], [2, 5, 2, 2, 2, 3, 2, 2, 2, 5, 2], [2, 2, 2, 2, 3, 3, 3, 2, 2, 2, 2], [4, 4, 4, 2, 2, 3, 2, 2, 4, 4, 4], [4, 4, 4, 4, 2, 2, 2, 4, 4, 4, 4]]}], "3dc255db": [{"attempt_1": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 2, 2, 2, 2, 2, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 4, 4], [0, 0, 0, 7, 0, 2, 2, 2, 2, 2, 0, 0, 0], [0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 9, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 9, 9, 0, 0, 0, 0, 0, 8, 0, 0, 0], [0, 9, 0, 0, 9, 0, 0, 0, 0, 0, 8, 0, 0], [0, 9, 0, 0, 9, 0, 0, 2, 0, 0, 0, 8, 2], [0, 9, 0, 0, 9, 0, 0, 0, 0, 0, 8, 0, 0], [0, 9, 0, 0, 9, 0, 0, 0, 0, 8, 0, 0, 0]], "attempt_2": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 2, 2, 2, 2, 2, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 4, 4], [0, 0, 0, 7, 0, 2, 2, 2, 2, 2, 0, 0, 0], [0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 9, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 9, 9, 0, 0, 0, 0, 0, 8, 0, 0, 0], [0, 9, 0, 0, 9, 0, 0, 0, 0, 0, 8, 0, 0], [0, 9, 0, 0, 9, 0, 2, 2, 0, 0, 0, 8, 2], [0, 9, 0, 0, 9, 0, 0, 0, 0, 0, 8, 0, 0], [0, 9, 0, 0, 9, 0, 0, 0, 0, 8, 0, 0, 0]]}], "3e6067c3": [{"attempt_1": [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 2, 2, 2, 2, 2, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1], [1, 2, 2, 2, 2, 2, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1], [1, 2, 2, 4, 2, 2, 4, 4, 2, 2, 8, 2, 2, 8, 8, 8, 8, 8, 8, 8, 2, 2, 9, 2, 2, 1, 1, 1, 1], [1, 2, 2, 2, 2, 2, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1], [1, 2, 2, 2, 2, 2, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 9, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 9, 1, 1, 1, 1, 1, 1], [1, 2, 2, 2, 2, 2, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1], [1, 2, 2, 2, 2, 2, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1], [1, 2, 2, 6, 2, 2, 1, 1, 2, 2, 7, 2, 2, 4, 4, 4, 4, 4, 4, 4, 2, 2, 4, 2, 2, 1, 1, 1, 1], [1, 2, 2, 2, 2, 2, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1], [1, 2, 2, 2, 2, 2, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1], [1, 1, 1, 6, 1, 1, 1, 1, 1, 1, 7, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 6, 1, 1, 1, 1, 1, 1, 7, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 2, 2, 2, 2, 2, 1, 1, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1], [1, 2, 2, 2, 2, 2, 1, 1, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1], [1, 2, 2, 9, 2, 2, 1, 1, 2, 2, 4, 2, 2, 5, 2, 2, 5, 2, 2, 1, 2, 2, 3, 2, 2, 1, 1, 1, 1], [1, 2, 2, 2, 2, 2, 1, 1, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1], [1, 2, 2, 2, 2, 2, 1, 1, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 7, 2, 2, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 4, 1, 8, 1, 9, 1, 4, 1, 7, 1, 6, 1, 9, 1, 4, 1, 5, 1, 3, 1, 7, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], "attempt_2": [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 2, 2, 2, 2, 2, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1], [1, 2, 2, 2, 2, 2, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1], [1, 2, 2, 4, 2, 2, 4, 4, 2, 2, 8, 2, 2, 8, 8, 8, 8, 8, 8, 8, 2, 2, 9, 2, 2, 1, 1, 1, 1], [1, 2, 2, 2, 2, 2, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1], [1, 2, 2, 2, 2, 2, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 9, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 9, 1, 1, 1, 1, 1, 1], [1, 2, 2, 2, 2, 2, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1], [1, 2, 2, 2, 2, 2, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1], [1, 2, 2, 6, 2, 2, 1, 1, 2, 2, 7, 2, 2, 4, 4, 4, 4, 4, 4, 4, 2, 2, 4, 2, 2, 1, 1, 1, 1], [1, 2, 2, 2, 2, 2, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1], [1, 2, 2, 2, 2, 2, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1], [1, 1, 1, 6, 1, 1, 1, 1, 1, 1, 7, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 6, 1, 1, 1, 1, 1, 1, 7, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 2, 2, 2, 2, 2, 1, 1, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1], [1, 2, 2, 2, 2, 2, 1, 1, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1], [1, 2, 2, 9, 2, 2, 9, 9, 2, 2, 4, 2, 2, 1, 2, 2, 5, 2, 2, 5, 2, 2, 3, 2, 2, 1, 1, 1, 1], [1, 2, 2, 2, 2, 2, 1, 1, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1], [1, 2, 2, 2, 2, 2, 1, 1, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 7, 2, 2, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 4, 1, 8, 1, 9, 1, 4, 1, 7, 1, 6, 1, 9, 1, 4, 1, 5, 1, 3, 1, 7, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]}, {"attempt_1": [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1], [1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1], [1, 8, 8, 2, 2, 8, 8, 2, 2, 2, 8, 8, 4, 4, 8, 8, 4, 4, 4, 4, 8, 8, 3, 3, 8, 8, 1, 1, 1, 1], [1, 8, 8, 2, 2, 8, 8, 2, 2, 2, 8, 8, 4, 4, 8, 8, 4, 4, 4, 4, 8, 8, 3, 3, 8, 8, 1, 1, 1, 1], [1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1], [1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1], [1, 1, 1, 3, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 1, 1, 1, 1, 1, 1], [1, 1, 1, 3, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 1, 1, 1, 1, 1, 1], [1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1], [1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1], [1, 8, 8, 3, 3, 8, 8, 1, 1, 1, 8, 8, 6, 6, 8, 8, 9, 9, 9, 9, 8, 8, 9, 9, 8, 8, 1, 1, 1, 1], [1, 8, 8, 3, 3, 8, 8, 1, 1, 1, 8, 8, 6, 6, 8, 8, 9, 9, 9, 9, 8, 8, 9, 9, 8, 8, 1, 1, 1, 1], [1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1], [1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1], [1, 1, 1, 9, 9, 1, 1, 1, 1, 1, 1, 1, 6, 6, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1], [1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1], [1, 8, 8, 9, 9, 8, 8, 4, 4, 4, 8, 8, 4, 4, 8, 8, 1, 1, 1, 1, 8, 8, 5, 5, 8, 8, 1, 1, 1, 1], [1, 8, 8, 9, 9, 8, 8, 4, 4, 4, 8, 8, 4, 4, 8, 8, 1, 1, 1, 1, 8, 8, 5, 5, 8, 8, 1, 1, 1, 1], [1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1], [1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 2, 1, 4, 1, 3, 1, 9, 1, 5, 1, 4, 1, 6, 1, 3, 1, 9, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], "attempt_2": [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1], [1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1], [1, 8, 8, 2, 2, 8, 8, 2, 2, 2, 8, 8, 4, 4, 8, 8, 4, 4, 4, 4, 8, 8, 3, 3, 8, 8, 1, 1, 1, 1], [1, 8, 8, 2, 2, 8, 8, 2, 2, 2, 8, 8, 4, 4, 8, 8, 4, 4, 4, 4, 8, 8, 3, 3, 8, 8, 1, 1, 1, 1], [1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1], [1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 1, 1, 1, 1, 1, 1], [1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1], [1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1], [1, 8, 8, 3, 3, 8, 8, 1, 1, 1, 8, 8, 6, 6, 8, 8, 9, 9, 9, 9, 8, 8, 9, 9, 8, 8, 1, 1, 1, 1], [1, 8, 8, 3, 3, 8, 8, 1, 1, 1, 8, 8, 6, 6, 8, 8, 9, 9, 9, 9, 8, 8, 9, 9, 8, 8, 1, 1, 1, 1], [1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1], [1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1], [1, 1, 1, 3, 3, 1, 1, 1, 1, 1, 1, 1, 6, 6, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1], [1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1], [1, 8, 8, 9, 9, 8, 8, 9, 9, 9, 8, 8, 4, 4, 8, 8, 1, 1, 1, 1, 8, 8, 5, 5, 8, 8, 1, 1, 1, 1], [1, 8, 8, 9, 9, 8, 8, 9, 9, 9, 8, 8, 4, 4, 8, 8, 1, 1, 1, 1, 8, 8, 5, 5, 8, 8, 1, 1, 1, 1], [1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1], [1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 2, 1, 4, 1, 3, 1, 9, 1, 5, 1, 4, 1, 6, 1, 3, 1, 9, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]}], "409aa875": [{"attempt_1": [[7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [7, 7, 6, 7, 7, 7, 7, 7, 6, 7, 6, 7, 7, 7, 7, 7], [7, 6, 6, 7, 7, 7, 7, 7, 7, 6, 7, 7, 7, 7, 6, 7], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 6, 7, 7], [7, 7, 7, 7, 7, 7, 7, 7, 7, 6, 7, 7, 7, 7, 6, 7], [7, 7, 7, 7, 7, 7, 7, 7, 6, 7, 6, 7, 7, 7, 7, 7], [7, 7, 6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [7, 6, 7, 6, 9, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 9, 7], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [7, 7, 7, 7, 6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [7, 7, 7, 6, 7, 6, 7, 7, 6, 6, 7, 7, 7, 7, 6, 7], [7, 7, 7, 7, 7, 7, 7, 7, 7, 6, 7, 7, 7, 6, 7, 6]], "attempt_2": [[7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [7, 7, 6, 7, 7, 7, 7, 7, 6, 7, 6, 7, 7, 7, 7, 7], [7, 6, 6, 7, 7, 7, 7, 7, 7, 6, 7, 7, 7, 7, 6, 7], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 6, 7, 7], [7, 7, 7, 7, 7, 7, 7, 7, 7, 6, 7, 7, 7, 7, 6, 7], [7, 7, 7, 7, 7, 7, 7, 7, 6, 7, 6, 7, 7, 7, 7, 7], [7, 7, 6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [7, 6, 7, 6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [7, 7, 7, 7, 7, 7, 7, 7, 7, 9, 7, 7, 7, 7, 9, 7], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [7, 7, 7, 7, 6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [7, 7, 7, 6, 7, 6, 7, 7, 6, 6, 7, 7, 7, 7, 6, 7], [7, 7, 7, 7, 7, 7, 7, 7, 7, 6, 7, 7, 7, 6, 7, 6]]}], "446ef5d2": [{"attempt_1": [[0]], "attempt_2": [[0]]}, {"attempt_1": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [8, 8, 8, 8, 9, 3, 3, 3, 3, 9, 9, 9, 9, 9, 9, 3, 9, 9, 3, 9], [8, 8, 8, 8, 9, 3, 9, 9, 3, 9, 9, 9, 3, 3, 3, 3, 3, 3, 3, 9], [8, 8, 8, 8, 9, 3, 3, 3, 3, 3, 3, 3, 3, 9, 9, 3, 3, 3, 3, 9], [8, 8, 8, 8, 9, 3, 9, 9, 3, 3, 3, 3, 3, 9, 9, 3, 3, 3, 3, 9], [8, 8, 8, 8, 9, 3, 3, 3, 3, 9, 9, 9, 3, 3, 3, 3, 3, 3, 3, 9], [8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]], "attempt_2": [[0]]}], "45a5af55": [{"attempt_1": [[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3], [3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3], [3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3], [3, 3, 8, 8, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 8, 8, 8, 3, 3], [3, 3, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 3, 3], [3, 3, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 3, 3], [3, 3, 8, 8, 8, 2, 8, 8, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 8, 2, 8, 8, 8, 3, 3], [3, 3, 8, 8, 8, 2, 8, 8, 3, 7, 7, 7, 7, 7, 7, 7, 7, 3, 8, 8, 2, 8, 8, 8, 3, 3], [3, 3, 8, 8, 8, 2, 8, 8, 3, 7, 7, 7, 7, 7, 7, 7, 7, 3, 8, 8, 2, 8, 8, 8, 3, 3], [3, 3, 8, 8, 8, 2, 8, 8, 3, 7, 7, 5, 5, 5, 5, 7, 7, 3, 8, 8, 2, 8, 8, 8, 3, 3], [3, 3, 8, 8, 8, 2, 8, 8, 3, 7, 7, 5, 8, 8, 5, 7, 7, 3, 8, 8, 2, 8, 8, 8, 3, 3], [3, 3, 8, 8, 8, 2, 8, 8, 3, 7, 7, 5, 8, 8, 5, 7, 7, 3, 8, 8, 2, 8, 8, 8, 3, 3], [3, 3, 8, 8, 8, 2, 8, 8, 3, 7, 7, 5, 5, 5, 5, 7, 7, 3, 8, 8, 2, 8, 8, 8, 3, 3], [3, 3, 8, 8, 8, 2, 8, 8, 3, 7, 7, 7, 7, 7, 7, 7, 7, 3, 8, 8, 2, 8, 8, 8, 3, 3], [3, 3, 8, 8, 8, 2, 8, 8, 3, 7, 7, 7, 7, 7, 7, 7, 7, 3, 8, 8, 2, 8, 8, 8, 3, 3], [3, 3, 8, 8, 8, 2, 8, 8, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 8, 2, 8, 8, 8, 3, 3], [3, 3, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 3, 3], [3, 3, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 3, 3], [3, 3, 8, 8, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 8, 8, 8, 3, 3], [3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3], [3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3], [3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]], "attempt_2": [[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3], [3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3], [3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3], [3, 3, 8, 8, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 8, 8, 8, 3, 3], [3, 3, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 3, 3], [3, 3, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 3, 3], [3, 3, 8, 8, 8, 2, 8, 8, 3, 3, 3, 3, 3, 3, 8, 8, 2, 8, 8, 8, 3, 3], [3, 3, 8, 8, 8, 2, 8, 8, 3, 7, 7, 7, 7, 3, 8, 8, 2, 8, 8, 8, 3, 3], [3, 3, 8, 8, 8, 2, 8, 8, 3, 7, 7, 7, 7, 3, 8, 8, 2, 8, 8, 8, 3, 3], [3, 3, 8, 8, 8, 2, 8, 8, 3, 7, 7, 7, 7, 3, 8, 8, 2, 8, 8, 8, 3, 3], [3, 3, 8, 8, 8, 2, 8, 8, 3, 7, 7, 7, 7, 3, 8, 8, 2, 8, 8, 8, 3, 3], [3, 3, 8, 8, 8, 2, 8, 8, 3, 3, 3, 3, 3, 3, 8, 8, 2, 8, 8, 8, 3, 3], [3, 3, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 3, 3], [3, 3, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 3, 3], [3, 3, 8, 8, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 8, 8, 8, 3, 3], [3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3], [3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3], [3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]]}], "4c3d4a41": [{"attempt_1": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 5, 0, 6, 0, 3, 0, 8, 0, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 9, 0, 5, 0, 5, 0, 8, 0, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 5, 0, 5, 0, 5, 0, 8, 0, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 5, 0, 5, 0, 5, 0, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 5, 5, 5, 0, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5]], "attempt_2": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 5, 0, 5, 0, 5, 0, 8, 0, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 9, 0, 5, 0, 5, 0, 8, 0, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 5, 0, 5, 0, 5, 0, 8, 0, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 5, 0, 5, 0, 5, 0, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 5, 5, 5, 0, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5]]}, {"attempt_1": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 3, 0, 2, 0, 4, 0, 7, 0, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 3, 0, 2, 0, 4, 0, 7, 0, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 3, 0, 2, 0, 4, 0, 7, 0, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 5, 0, 5, 0, 5, 0, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 5, 5, 5, 0, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5]], "attempt_2": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 3, 0, 2, 0, 4, 0, 5, 0, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 3, 0, 2, 0, 4, 0, 7, 0, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 3, 0, 2, 0, 4, 0, 7, 0, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 5, 0, 5, 0, 5, 0, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 5, 5, 5, 0, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5]]}], "4c7dc4dd": [{"attempt_1": [[0, 0, 0, 0, 0], [0, 0, 4, 0, 0], [0, 4, 2, 4, 0], [0, 0, 4, 0, 0], [0, 0, 0, 0, 0]], "attempt_2": [[2, 4, 2, 4, 2], [4, 2, 0, 2, 4], [2, 0, 0, 0, 2], [4, 2, 0, 2, 4], [2, 4, 2, 4, 2]]}, {"attempt_1": [[0, 0, 0, 0, 8, 0], [0, 0, 0, 0, 6, 8], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0]], "attempt_2": [[0, 8, 0, 0, 8, 0], [8, 0, 0, 0, 0, 8], [0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0], [8, 0, 0, 0, 0, 8], [0, 8, 0, 0, 8, 0]]}], "4e34c42c": [{"attempt_1": [[0]], "attempt_2": [[0]]}, {"attempt_1": [[0]], "attempt_2": [[0]]}], "53fb4810": [{"attempt_1": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 7, 3, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 5, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 7, 3, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 5, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 1, 1, 8, 8, 8, 8, 7, 3, 8, 8, 8, 8, 8, 8, 8], [8, 8, 1, 1, 1, 1, 8, 8, 8, 2, 5, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 1, 1, 8, 8, 8, 8, 7, 3, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 4, 3, 8, 8, 8, 8, 2, 5, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 6, 2, 8, 8, 8, 8, 7, 3, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 4, 3, 8, 8, 8, 8, 2, 5, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 6, 2, 8, 8, 8, 8, 7, 3, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 4, 3, 8, 8, 8, 8, 2, 5, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 6, 2, 8, 8, 8, 8, 7, 3, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 4, 3, 8, 8, 8, 8, 2, 5, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 6, 2, 8, 8, 8, 8, 7, 3, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 4, 3, 8, 8, 8, 8, 1, 1, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 6, 2, 8, 8, 8, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8], [8, 8, 8, 4, 3, 8, 8, 8, 8, 1, 1, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 6, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 4, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 8, 8], [8, 8, 8, 6, 2, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 8], [8, 8, 8, 4, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 8, 8], [8, 8, 8, 6, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 4, 8, 8], [8, 8, 8, 6, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 5, 8, 8], [8, 8, 8, 4, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 4, 8, 8]], "attempt_2": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 7, 3, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 5, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 7, 3, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 5, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 1, 1, 8, 8, 8, 8, 7, 3, 8, 8, 8, 8, 8, 8, 8], [8, 8, 1, 1, 1, 1, 8, 8, 8, 2, 5, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 1, 1, 8, 8, 8, 8, 7, 3, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 4, 3, 8, 8, 8, 8, 2, 5, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 6, 2, 8, 8, 8, 8, 7, 3, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 4, 3, 8, 8, 8, 8, 2, 5, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 6, 2, 8, 8, 8, 8, 7, 3, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 4, 3, 8, 8, 8, 8, 2, 5, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 6, 2, 8, 8, 8, 8, 7, 3, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 4, 3, 8, 8, 8, 8, 2, 5, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 6, 2, 8, 8, 8, 8, 7, 3, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 4, 3, 8, 8, 8, 8, 1, 1, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 6, 2, 8, 8, 8, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8], [8, 8, 8, 4, 3, 8, 8, 8, 8, 1, 1, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 6, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 4, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 8, 8], [8, 8, 8, 6, 2, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 8], [8, 8, 8, 4, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 8, 8], [8, 8, 8, 6, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 4, 8, 8], [8, 8, 8, 6, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 5, 8, 8], [8, 8, 8, 4, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]]}], "581f7754": [{"attempt_1": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 6, 6, 6, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 3], [6, 8, 8, 6, 4, 6, 2, 8, 8, 8, 8, 8, 8, 4, 6, 4, 8, 3, 8, 3], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 8, 8, 4, 4, 4, 8, 3, 8, 3], [8, 8, 8, 8, 8, 8, 8, 2, 4, 4, 4, 8, 8, 8, 4, 8, 8, 3, 6, 3], [8, 8, 8, 8, 4, 8, 8, 8, 4, 4, 6, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 2, 8, 4, 4, 4, 8, 8, 8, 8, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 4, 8, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 4, 4, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [4, 8, 8, 8, 6, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]], "attempt_2": [[0]]}, {"attempt_1": [[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 1, 1, 1, 4, 4, 4, 4, 4, 4, 3, 1, 4, 4, 4, 1, 4, 4, 4, 4], [4, 1, 1, 1, 4, 1, 4, 4, 4, 1, 2, 1, 3, 1, 4, 1, 1, 1, 4, 4, 4, 2], [4, 4, 4, 1, 4, 1, 1, 1, 4, 1, 4, 1, 4, 1, 4, 1, 4, 1, 4, 4, 4, 4], [4, 4, 4, 1, 2, 1, 4, 4, 4, 1, 1, 1, 4, 1, 4, 2, 4, 1, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 8, 4, 4, 4, 4, 4, 4, 4, 8], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]], "attempt_2": [[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 2, 1, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 4, 1, 3, 1, 4, 4, 4, 1, 4, 4, 4, 2], [4, 4, 4, 1, 1, 1, 4, 4, 4, 1, 1, 1, 3, 1, 4, 1, 1, 1, 4, 4, 4, 4], [4, 1, 1, 1, 4, 1, 4, 4, 4, 4, 4, 4, 4, 1, 4, 1, 4, 1, 4, 4, 4, 4], [4, 4, 4, 1, 4, 1, 1, 1, 4, 4, 4, 4, 4, 1, 4, 2, 4, 1, 4, 4, 4, 4], [4, 4, 4, 1, 2, 1, 4, 4, 4, 4, 4, 4, 4, 1, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 4, 4, 4, 4, 4, 4, 4, 8], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 8, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]]}], "58490d8a": [{"attempt_1": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 2, 0, 2, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 8, 0, 8, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 3, 0, 3, 0, 3, 0, 3, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], "attempt_2": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 2, 0, 2, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 8, 0, 8, 0, 8, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 3, 0, 3, 0, 3, 0, 3, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}], "5961cc34": [{"attempt_1": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 2, 2, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8], [8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8], [8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8], [8, 2, 2, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]], "attempt_2": [[0]]}], "62593bfd": [{"attempt_1": [[1, 1, 1, 1, 3, 1, 1, 1, 3, 1, 1, 1, 4, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 3, 3, 3, 1, 1, 1, 1, 4, 4, 4, 4, 1, 1, 3, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 3, 1, 3, 1, 1, 1, 1, 1, 4, 1, 1, 1, 1, 3, 3, 3, 1, 1, 1, 1], [1, 1, 1, 1, 1, 3, 3, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1], [1, 1, 1, 1, 3, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 5, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 5, 5, 5, 1, 1], [1, 1, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 5, 1, 5, 1, 1], [1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 5, 5, 5, 1, 1], [1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 5, 1, 1, 1, 5, 1]], "attempt_2": [[1, 1, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 4, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 4, 4, 4, 4, 1, 1, 3, 1, 1, 1, 1, 1, 1], [1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 4, 1, 1, 1, 1, 3, 3, 3, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 3, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 5, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 3, 3, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 5, 5, 5, 1, 1], [1, 1, 1, 1, 1, 3, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 5, 1, 5, 1, 1], [1, 1, 1, 1, 1, 3, 3, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 5, 5, 5, 1, 1], [1, 1, 1, 1, 3, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 5, 1, 1, 1, 5, 1]]}, {"attempt_1": [[9, 9, 8, 9, 9, 9, 9, 9, 9, 9, 9, 7, 7, 9, 9, 4, 4, 4, 9, 9, 9, 9, 6, 9, 9, 9, 9, 9], [9, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 7, 9, 9, 9, 9, 4, 9, 9, 9, 9, 9, 6, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 4, 9, 9, 9, 9, 6, 6, 6, 6, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 6, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 1, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 1, 1, 1, 1, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 3, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 1, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 3, 9, 3, 9, 9, 9], [9, 9, 9, 9, 9, 9, 1, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 3, 9, 9, 9, 9]], "attempt_2": [[9, 9, 9, 9, 9, 9, 1, 9, 9, 9, 9, 7, 7, 9, 9, 4, 4, 4, 9, 9, 9, 9, 6, 9, 9, 9, 9, 9], [9, 9, 9, 9, 1, 1, 1, 1, 9, 9, 9, 7, 9, 9, 9, 9, 4, 9, 9, 9, 9, 9, 6, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 1, 9, 9, 9, 9, 9, 9, 9, 9, 9, 4, 9, 9, 9, 9, 6, 6, 6, 6, 9, 9, 9], [9, 9, 9, 9, 9, 9, 1, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 6, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 3, 9, 9, 9, 9], [9, 9, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 3, 9, 3, 9, 9, 9], [9, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 3, 9, 9, 9, 9]]}], "67e490f4": [{"attempt_1": [[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 9, 9, 4, 4, 4, 9, 9, 9, 4, 9, 4, 9, 9, 9, 9, 9, 9, 4, 9, 4, 9, 9, 9, 4, 4, 4, 9, 9, 4], [4, 9, 4, 4, 8, 4, 4, 9, 4, 4, 9, 4, 9, 4, 4, 4, 4, 9, 4, 9, 4, 4, 9, 4, 4, 8, 4, 4, 9, 4], [4, 4, 4, 8, 8, 8, 4, 4, 4, 4, 9, 4, 4, 4, 9, 9, 4, 4, 4, 9, 4, 4, 4, 4, 8, 8, 8, 4, 4, 4], [4, 9, 4, 4, 8, 4, 4, 9, 4, 4, 9, 4, 9, 4, 4, 4, 4, 9, 4, 9, 4, 4, 9, 4, 4, 8, 4, 4, 9, 4], [4, 9, 9, 4, 4, 4, 9, 9, 9, 4, 9, 4, 9, 9, 9, 9, 9, 9, 4, 9, 4, 9, 9, 9, 4, 4, 4, 9, 9, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]], "attempt_2": [[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 2, 2, 4, 4, 4, 2, 2, 2, 4, 2, 4, 2, 2, 2, 2, 2, 2, 4, 2, 4, 2, 2, 2, 4, 4, 4, 2, 2, 4], [4, 2, 4, 4, 8, 4, 4, 2, 4, 4, 2, 4, 2, 4, 4, 4, 4, 2, 4, 2, 4, 4, 2, 4, 4, 8, 4, 4, 2, 4], [4, 4, 4, 8, 8, 8, 4, 4, 4, 4, 2, 4, 4, 4, 2, 2, 4, 4, 4, 2, 4, 4, 4, 4, 8, 8, 8, 4, 4, 4], [4, 2, 4, 4, 8, 4, 4, 2, 4, 4, 2, 4, 2, 4, 4, 4, 4, 2, 4, 2, 4, 4, 2, 4, 4, 8, 4, 4, 2, 4], [4, 2, 2, 4, 4, 4, 2, 2, 2, 4, 2, 4, 2, 2, 2, 2, 2, 2, 4, 2, 4, 2, 2, 2, 4, 4, 4, 2, 2, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]]}], "6e453dd6": [{"attempt_1": [[6, 6, 6, 6, 6, 6, 6, 6, 5, 6, 6, 6], [6, 6, 6, 6, 6, 6, 6, 6, 5, 6, 6, 6], [6, 6, 6, 0, 0, 0, 0, 6, 5, 6, 6, 6], [6, 6, 6, 0, 6, 0, 6, 6, 5, 6, 6, 6], [6, 6, 6, 0, 0, 0, 0, 0, 5, 6, 6, 6], [6, 6, 6, 6, 6, 6, 6, 0, 5, 6, 6, 6], [6, 6, 6, 6, 6, 6, 6, 0, 5, 6, 6, 6], [6, 6, 6, 6, 6, 6, 6, 6, 5, 6, 6, 6], [6, 6, 6, 6, 6, 0, 0, 0, 5, 6, 6, 6], [6, 6, 6, 6, 6, 6, 6, 0, 5, 6, 6, 6], [6, 6, 6, 6, 6, 0, 0, 0, 5, 6, 6, 6], [6, 6, 6, 6, 6, 6, 6, 6, 5, 6, 6, 6], [6, 6, 6, 6, 0, 0, 0, 0, 5, 6, 6, 6], [6, 6, 6, 6, 0, 6, 6, 0, 5, 2, 2, 2], [6, 6, 6, 6, 0, 0, 0, 0, 5, 6, 6, 6], [6, 6, 6, 6, 6, 6, 6, 6, 5, 6, 6, 6], [6, 6, 6, 6, 6, 0, 0, 0, 5, 6, 6, 6], [6, 6, 6, 6, 6, 0, 6, 0, 5, 2, 2, 2], [6, 6, 6, 6, 0, 0, 0, 0, 5, 6, 6, 6], [6, 6, 6, 6, 6, 0, 6, 0, 5, 2, 2, 2], [6, 6, 6, 6, 6, 0, 0, 0, 5, 6, 6, 6], [6, 6, 6, 6, 6, 6, 6, 6, 5, 6, 6, 6], [6, 6, 6, 6, 6, 0, 0, 0, 5, 6, 6, 6], [6, 6, 6, 6, 6, 0, 6, 0, 5, 2, 2, 2], [6, 6, 6, 6, 6, 0, 0, 0, 5, 6, 6, 6]], "attempt_2": [[6, 6, 6, 6, 6, 6, 6, 6, 5, 6, 6, 6], [6, 6, 6, 6, 6, 6, 6, 6, 5, 6, 6, 6], [6, 6, 6, 0, 0, 0, 0, 6, 5, 6, 6, 6], [6, 6, 6, 0, 6, 0, 6, 6, 5, 6, 6, 6], [6, 6, 6, 0, 0, 0, 0, 0, 5, 6, 6, 6], [6, 6, 6, 6, 6, 6, 6, 0, 5, 6, 6, 6], [6, 6, 6, 6, 6, 6, 6, 0, 5, 6, 6, 6], [6, 6, 6, 6, 6, 6, 6, 6, 5, 6, 6, 6], [6, 6, 6, 6, 6, 0, 0, 0, 5, 6, 6, 6], [6, 6, 6, 6, 6, 6, 6, 0, 5, 2, 2, 2], [6, 6, 6, 6, 6, 0, 0, 0, 5, 6, 6, 6], [6, 6, 6, 6, 6, 6, 6, 6, 5, 6, 6, 6], [6, 6, 6, 6, 0, 0, 0, 0, 5, 6, 6, 6], [6, 6, 6, 6, 0, 6, 6, 0, 5, 2, 2, 2], [6, 6, 6, 6, 0, 0, 0, 0, 5, 6, 6, 6], [6, 6, 6, 6, 6, 6, 6, 6, 5, 6, 6, 6], [6, 6, 6, 6, 6, 0, 0, 0, 5, 6, 6, 6], [6, 6, 6, 6, 6, 0, 6, 0, 5, 2, 2, 2], [6, 6, 6, 6, 0, 0, 0, 0, 5, 6, 6, 6], [6, 6, 6, 6, 6, 0, 6, 0, 5, 2, 2, 2], [6, 6, 6, 6, 6, 0, 0, 0, 5, 6, 6, 6], [6, 6, 6, 6, 6, 6, 6, 6, 5, 6, 6, 6], [6, 6, 6, 6, 6, 0, 0, 0, 5, 6, 6, 6], [6, 6, 6, 6, 6, 0, 6, 0, 5, 2, 2, 2], [6, 6, 6, 6, 6, 0, 0, 0, 5, 6, 6, 6]]}], "6ffbe589": [{"attempt_1": [[0]], "attempt_2": [[0]]}], "71e489b6": [{"attempt_1": [[1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 7, 7, 7, 0, 0, 0], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 7, 0, 7, 1, 0, 0], [1, 1, 1, 1, 7, 7, 7, 1, 1, 1, 7, 7, 7, 1, 0, 0], [1, 1, 1, 7, 7, 0, 7, 1, 1, 1, 1, 1, 1, 1, 0, 0], [1, 1, 7, 7, 0, 7, 7, 1, 1, 1, 1, 1, 1, 1, 0, 0], [1, 1, 7, 0, 7, 7, 1, 1, 1, 1, 7, 7, 7, 1, 0, 0], [1, 1, 7, 7, 7, 1, 1, 1, 1, 1, 7, 0, 7, 1, 0, 0], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 7, 0, 7, 1, 0, 0], [1, 1, 1, 1, 1, 7, 7, 7, 1, 1, 7, 7, 7, 1, 0, 0], [1, 1, 1, 1, 1, 7, 0, 7, 1, 1, 1, 1, 1, 1, 0, 0], [1, 1, 1, 1, 1, 7, 7, 7, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], "attempt_2": [[1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 7, 7, 7, 0, 0, 0], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 7, 0, 7, 1, 0, 0], [1, 1, 1, 1, 7, 7, 7, 1, 1, 1, 7, 7, 7, 1, 0, 0], [1, 1, 1, 7, 7, 0, 7, 1, 1, 1, 1, 1, 1, 1, 0, 0], [1, 1, 7, 7, 0, 7, 7, 1, 1, 1, 1, 1, 1, 1, 0, 0], [1, 1, 7, 0, 7, 7, 1, 1, 1, 1, 7, 7, 7, 1, 0, 0], [1, 1, 7, 7, 7, 1, 1, 1, 1, 1, 7, 0, 7, 1, 0, 0], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 7, 0, 7, 1, 0, 0], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 7, 7, 7, 1, 0, 0], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0], [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}, {"attempt_1": [[0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [0, 0, 1, 7, 7, 7, 1, 1, 1, 1, 1, 1, 1, 1, 1, 7, 7, 7, 1], [0, 0, 1, 7, 0, 7, 1, 1, 1, 1, 1, 1, 1, 1, 1, 7, 0, 7, 1], [0, 0, 1, 7, 7, 7, 7, 7, 1, 0, 0, 0, 0, 0, 0, 7, 7, 7, 0], [0, 0, 1, 1, 1, 7, 0, 7, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 1, 1, 1, 7, 7, 7, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1], [0, 0, 1, 1, 7, 7, 7, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1], [0, 0, 1, 7, 7, 0, 7, 7, 1, 0, 0, 1, 1, 7, 7, 7, 1, 1, 1], [0, 0, 1, 7, 0, 7, 0, 7, 1, 0, 0, 1, 1, 7, 0, 7, 1, 1, 1], [0, 0, 1, 7, 7, 7, 7, 7, 1, 0, 0, 1, 1, 7, 0, 7, 1, 1, 1], [0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 7, 7, 7, 1, 1, 1], [0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 7, 7], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 7, 7, 1, 1, 1, 7, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0, 7, 7, 1, 1, 7, 7], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 7, 0, 7, 1, 1, 1, 1]], "attempt_2": [[7, 0, 7, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [7, 7, 7, 7, 7, 7, 1, 1, 1, 1, 1, 1, 1, 1, 1, 7, 7, 7, 1], [0, 0, 1, 7, 0, 7, 1, 1, 1, 1, 1, 1, 1, 1, 1, 7, 0, 7, 1], [0, 0, 1, 7, 7, 7, 7, 7, 1, 0, 0, 0, 0, 0, 0, 7, 7, 7, 0], [0, 0, 1, 1, 1, 7, 0, 7, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 1, 1, 1, 7, 7, 7, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1], [0, 0, 1, 1, 7, 7, 7, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1], [0, 0, 1, 7, 7, 0, 7, 7, 1, 0, 0, 1, 1, 7, 7, 7, 1, 1, 1], [0, 0, 1, 7, 0, 7, 0, 7, 1, 0, 0, 1, 1, 7, 0, 7, 1, 1, 1], [0, 0, 1, 7, 7, 7, 7, 7, 1, 0, 0, 1, 1, 7, 0, 7, 1, 1, 1], [0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 7, 7, 7, 1, 1, 1], [0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 7, 7], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 7, 7, 1, 1, 1, 7, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0, 7, 7, 1, 1, 7, 7], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 7, 0, 7, 1, 1, 1, 1]]}], "7491f3cf": [{"attempt_1": [[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [2, 5, 5, 8, 5, 8, 2, 5, 3, 5, 3, 5, 2, 7, 7, 5, 7, 7, 2, 5, 3, 5, 7, 7, 2], [2, 5, 5, 8, 5, 5, 2, 3, 3, 5, 3, 3, 2, 7, 5, 5, 5, 7, 2, 3, 3, 5, 5, 7, 2], [2, 5, 5, 8, 8, 8, 2, 5, 5, 5, 5, 5, 2, 5, 5, 5, 5, 5, 2, 5, 5, 5, 5, 5, 2], [2, 5, 5, 5, 5, 5, 2, 3, 3, 5, 3, 3, 2, 7, 5, 5, 5, 7, 2, 3, 3, 5, 5, 7, 2], [2, 5, 5, 5, 5, 5, 2, 5, 3, 5, 3, 5, 2, 7, 7, 5, 7, 7, 2, 5, 3, 5, 7, 7, 2], [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]], "attempt_2": [[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [2, 5, 5, 8, 5, 8, 2, 5, 3, 5, 3, 5, 2, 7, 7, 5, 7, 7, 2, 5, 3, 5, 3, 5, 2], [2, 5, 5, 8, 5, 5, 2, 3, 3, 5, 3, 3, 2, 7, 5, 5, 5, 7, 2, 3, 3, 5, 3, 3, 2], [2, 5, 5, 8, 8, 8, 2, 5, 5, 5, 5, 5, 2, 5, 5, 5, 5, 5, 2, 5, 5, 5, 5, 5, 2], [2, 5, 5, 5, 5, 5, 2, 3, 3, 5, 3, 3, 2, 7, 5, 5, 5, 7, 2, 7, 5, 5, 5, 7, 2], [2, 5, 5, 5, 5, 5, 2, 5, 3, 5, 3, 5, 2, 7, 7, 5, 7, 7, 2, 7, 7, 5, 7, 7, 2], [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]]}], "7666fa5d": [{"attempt_1": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 9, 8, 8, 8, 8, 9, 8, 8, 8, 8, 8, 8], [8, 8, 8, 9, 2, 2, 8, 8, 9, 2, 2, 8, 8, 9, 8, 8], [8, 8, 8, 8, 2, 2, 2, 9, 2, 2, 2, 2, 9, 2, 2, 8], [8, 8, 8, 8, 8, 2, 9, 2, 2, 2, 2, 9, 2, 2, 2, 9], [8, 8, 8, 8, 8, 9, 2, 2, 2, 2, 9, 2, 2, 2, 9, 8], [8, 8, 8, 8, 9, 2, 2, 2, 2, 9, 2, 2, 2, 9, 8, 8], [8, 8, 8, 8, 8, 2, 2, 2, 9, 2, 2, 2, 9, 8, 8, 8], [8, 8, 8, 8, 8, 8, 2, 9, 2, 2, 2, 9, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 9, 2, 2, 2, 9, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 9, 2, 2, 2, 9, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 2, 2, 9, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 9, 8, 8, 8, 8, 8, 8, 8, 8]], "attempt_2": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 9, 8, 8, 8, 8, 9, 8, 8, 8, 8, 8, 8], [8, 8, 8, 9, 2, 2, 8, 8, 9, 2, 2, 8, 8, 9, 8, 8], [8, 8, 8, 8, 2, 2, 2, 9, 2, 2, 2, 2, 9, 8, 8, 8], [8, 8, 8, 8, 8, 2, 9, 2, 2, 2, 2, 9, 2, 2, 8, 9], [8, 8, 8, 8, 8, 9, 2, 2, 2, 2, 9, 2, 2, 2, 9, 8], [8, 8, 8, 8, 9, 2, 2, 2, 2, 9, 2, 2, 2, 9, 8, 8], [8, 8, 8, 8, 8, 2, 2, 2, 9, 2, 2, 2, 9, 8, 8, 8], [8, 8, 8, 8, 8, 8, 2, 9, 2, 2, 2, 9, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 9, 2, 2, 2, 9, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 9, 2, 2, 2, 9, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 2, 2, 9, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 9, 8, 8, 8, 8, 8, 8, 8, 8]]}], "78332cb0": [{"attempt_1": [[7, 7, 7, 7, 7, 6, 7, 7, 7, 7, 7, 6, 7, 7, 7, 7, 7, 6, 7, 7, 7, 7, 7], [7, 4, 4, 4, 7, 6, 4, 4, 4, 4, 7, 6, 7, 4, 4, 4, 7, 6, 7, 4, 7, 4, 4], [7, 4, 7, 4, 7, 6, 7, 7, 7, 4, 4, 6, 4, 4, 7, 4, 7, 6, 7, 4, 7, 4, 7], [7, 4, 7, 4, 4, 6, 7, 4, 4, 4, 7, 6, 7, 4, 4, 4, 7, 6, 4, 4, 4, 4, 7], [7, 7, 7, 7, 7, 6, 7, 7, 7, 7, 7, 6, 7, 7, 7, 7, 7, 6, 7, 7, 7, 7, 7]], "attempt_2": [[7, 7, 7, 7, 7, 6, 7, 7, 7, 7, 7, 6, 7, 7, 7, 7, 7, 6, 7, 7, 7, 7, 7], [7, 4, 4, 4, 7, 6, 7, 4, 7, 4, 4, 6, 7, 4, 4, 4, 7, 6, 4, 4, 4, 4, 7], [4, 4, 7, 4, 7, 6, 7, 4, 7, 4, 7, 6, 7, 4, 7, 4, 7, 6, 7, 7, 7, 4, 4], [7, 4, 4, 4, 7, 6, 4, 4, 4, 4, 7, 6, 7, 4, 7, 4, 4, 6, 7, 4, 4, 4, 7], [7, 7, 7, 7, 7, 6, 7, 7, 7, 7, 7, 6, 7, 7, 7, 7, 7, 6, 7, 7, 7, 7, 7]]}, {"attempt_1": [[7, 7, 7, 7, 7], [7, 7, 7, 9, 7], [7, 7, 9, 9, 7], [7, 9, 9, 7, 7], [7, 9, 7, 7, 7], [6, 6, 6, 6, 6], [7, 2, 7, 7, 7], [7, 2, 7, 7, 7], [7, 2, 2, 7, 7], [7, 7, 2, 7, 7], [7, 7, 2, 7, 7], [6, 6, 6, 6, 6], [7, 7, 2, 7, 7], [7, 2, 2, 2, 7], [7, 7, 7, 2, 7], [7, 7, 7, 2, 7], [7, 7, 7, 2, 7], [6, 6, 6, 6, 6], [7, 7, 7, 9, 7], [7, 7, 9, 9, 7], [7, 9, 9, 7, 7], [7, 9, 7, 7, 7], [7, 7, 7, 7, 7]], "attempt_2": [[7, 7, 7, 9, 7, 6, 7, 7, 2, 7, 7, 6, 7, 2, 7, 7, 7, 6, 7, 7, 7, 7, 7], [7, 7, 9, 9, 7, 6, 7, 2, 2, 2, 7, 6, 7, 2, 7, 7, 7, 6, 7, 7, 7, 9, 7], [7, 9, 9, 7, 7, 6, 7, 7, 7, 2, 7, 6, 7, 2, 2, 7, 7, 6, 7, 7, 9, 9, 7], [7, 9, 7, 7, 7, 6, 7, 7, 7, 2, 7, 6, 7, 7, 2, 7, 7, 6, 7, 9, 9, 7, 7], [7, 7, 7, 7, 7, 6, 7, 7, 7, 2, 7, 6, 7, 7, 2, 7, 7, 6, 7, 9, 7, 7, 7]]}], "7b3084d4": [{"attempt_1": [[5, 2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3], [2, 2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3], [2, 2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3], [2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [2, 2, 2, 7, 2, 2, 2, 3, 3, 3, 8, 3, 3, 3], [2, 2, 2, 7, 2, 2, 2, 3, 3, 3, 8, 3, 3, 3], [2, 2, 2, 7, 2, 2, 2, 3, 3, 3, 8, 3, 3, 3], [7, 7, 7, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8], [7, 7, 7, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8], [7, 7, 7, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8], [7, 7, 7, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8], [7, 7, 7, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8], [7, 7, 7, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8], [7, 7, 7, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8]], "attempt_2": [[5, 2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3], [2, 2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3], [2, 2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3], [2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [2, 2, 2, 7, 2, 2, 2, 3, 3, 3, 8, 3, 3, 3], [2, 2, 2, 7, 2, 2, 2, 3, 3, 3, 8, 3, 3, 3], [2, 2, 2, 7, 2, 2, 2, 3, 3, 3, 8, 3, 3, 3], [7, 7, 7, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8], [7, 7, 7, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8], [7, 7, 7, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8], [8, 8, 7, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8], [7, 7, 7, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8], [7, 7, 7, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8], [7, 7, 7, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8]]}], "7b5033c1": [{"attempt_1": [[1], [1], [1], [1], [1], [3], [3], [3], [3], [8], [8], [8], [8], [8], [8], [8], [8], [4], [4], [4], [4], [4], [4], [4]], "attempt_2": [[1], [1], [1], [1], [1], [3], [3], [3], [3], [8], [8], [8], [8], [8], [4], [4], [4], [4], [4], [1], [1], [1]]}], "7b80bb43": [{"attempt_1": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 9, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 9, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 9, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 9, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 9, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 9, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 9, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 9, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 9, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 9, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 9, 9, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8]], "attempt_2": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 9, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 9, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 9, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 9, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 9, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 9, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 9, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 9, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 9, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 9, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8]]}], "7ed72f31": [{"attempt_1": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 4, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 4, 4, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 4, 4, 8, 8, 8], [8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 1, 8, 8, 8, 4, 4, 4, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 1, 1, 1, 8, 4, 4, 4, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 2, 8, 8, 8, 8, 8, 4, 4, 8, 8, 8, 8, 8, 8, 8], [1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 3, 8, 8, 8, 8, 8, 8], [8, 8, 1, 8, 8, 8, 8, 8, 8, 3, 3, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 8, 3, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 3, 3, 3, 3, 3, 3, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 3, 3, 3, 3, 3, 3, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 8, 3, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 8, 8, 8, 8, 8, 8]], "attempt_2": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 4, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 4, 4, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 4, 4, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 4, 4, 4, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 4, 4, 4, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 2, 8, 8, 8, 8, 8, 4, 4, 8, 8, 8, 8, 8, 8, 8], [1, 1, 1, 8, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 1, 8, 1, 8, 8, 8, 8, 8, 8, 3, 8, 8, 8, 8, 8, 8], [8, 8, 1, 8, 1, 8, 8, 8, 8, 3, 3, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 8, 3, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 3, 3, 3, 3, 3, 3, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 3, 3, 3, 3, 3, 3, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 8, 3, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 8, 8, 8, 8, 8, 8]]}, {"attempt_1": [[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 1, 1, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 3, 2, 3, 1, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 1, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 4, 4, 4, 3, 4, 3, 3, 3, 3, 3, 3, 3, 3, 1, 1, 1, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 4, 3, 3, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 4, 3, 3, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 4, 4, 4, 3, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 3, 3, 8, 3, 3, 3, 3, 8, 2, 8, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 8, 8, 3, 3, 8, 8, 2, 8, 8, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 8, 3, 3, 8, 3, 8, 2, 8, 3, 8, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 3, 3, 3, 3, 8, 8, 8, 2, 8, 8, 8, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 8, 3, 3, 3, 3, 3, 8, 3, 3, 2, 3, 3, 8, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 8, 8, 8, 3, 3, 3, 3, 3, 3, 8, 8, 2, 8, 8, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 1, 3, 2, 3, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 6, 3, 3, 3, 3, 3, 3], [3, 3, 3, 1, 2, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 6, 3, 6, 3, 3, 3, 3, 3], [3, 3, 3, 1, 2, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 6, 6, 3, 3, 3, 3, 3], [3, 3, 1, 1, 2, 1, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 3, 3, 3, 3], [3, 3, 1, 3, 3, 3, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 6, 6, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 6, 3, 6, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 3, 3, 3, 3, 3, 6, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]], "attempt_2": [[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 1, 1, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 3, 2, 3, 1, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 1, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 4, 4, 4, 3, 4, 3, 3, 3, 3, 3, 3, 3, 3, 1, 1, 1, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 4, 3, 3, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 4, 3, 3, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 4, 4, 4, 3, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 3, 3, 8, 3, 3, 3, 3, 8, 2, 8, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 8, 8, 3, 3, 8, 8, 2, 8, 8, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 8, 3, 3, 8, 3, 8, 2, 8, 3, 8, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 3, 3, 3, 3, 8, 8, 8, 2, 8, 8, 8, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 8, 3, 3, 3, 3, 3, 8, 3, 3, 2, 3, 3, 8, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 8, 8, 8, 3, 3, 3, 3, 3, 3, 8, 8, 2, 8, 8, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 1, 3, 2, 3, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 6, 3, 3, 3, 6, 3, 3], [3, 3, 3, 1, 2, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 6, 3, 6, 3, 6, 3, 6, 3], [3, 3, 3, 1, 2, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 6, 6, 3, 6, 6, 3, 3], [3, 3, 1, 1, 2, 1, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 3, 3, 3, 3], [3, 3, 1, 3, 3, 3, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]]}], "80a900e0": [{"attempt_1": [[2, 0, 1, 0, 2, 0, 1, 0, 1, 0, 4, 0, 1, 0, 4, 0, 1, 0, 1, 0, 1], [0, 2, 0, 1, 0, 2, 0, 1, 0, 4, 0, 1, 0, 4, 0, 1, 0, 1, 0, 1, 0], [1, 0, 2, 0, 1, 0, 2, 0, 4, 0, 1, 0, 4, 0, 1, 0, 1, 0, 1, 0, 1], [0, 1, 0, 2, 0, 2, 0, 6, 0, 4, 0, 4, 0, 1, 0, 1, 0, 1, 0, 1, 0], [1, 0, 1, 0, 2, 0, 6, 0, 6, 0, 4, 0, 1, 0, 1, 0, 1, 0, 1, 0, 6], [0, 1, 0, 1, 0, 6, 0, 1, 0, 6, 0, 1, 0, 1, 0, 1, 0, 1, 0, 6, 0], [1, 0, 1, 0, 9, 0, 6, 0, 6, 0, 1, 0, 1, 0, 1, 0, 1, 0, 6, 0, 1], [0, 1, 0, 9, 0, 9, 0, 6, 0, 1, 0, 1, 0, 1, 0, 1, 0, 6, 0, 1, 0], [1, 0, 9, 0, 1, 0, 9, 0, 1, 0, 1, 0, 1, 0, 1, 0, 6, 0, 1, 0, 6], [0, 9, 0, 1, 0, 9, 0, 1, 0, 1, 0, 1, 0, 1, 0, 6, 0, 1, 0, 6, 0], [9, 0, 1, 0, 9, 0, 1, 0, 1, 0, 1, 0, 1, 0, 6, 0, 1, 0, 6, 0, 1], [0, 1, 0, 9, 0, 1, 0, 1, 0, 1, 0, 1, 0, 6, 0, 1, 0, 6, 0, 1, 0], [1, 0, 9, 0, 1, 0, 1, 0, 1, 0, 1, 0, 3, 0, 6, 0, 6, 0, 1, 0, 1], [0, 9, 0, 1, 0, 1, 0, 1, 0, 1, 0, 3, 0, 3, 0, 6, 0, 1, 0, 1, 0], [9, 0, 1, 0, 1, 0, 1, 0, 1, 0, 3, 0, 1, 0, 3, 0, 1, 0, 1, 0, 1], [0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 3, 0, 3, 0, 7, 0, 1, 0, 1, 0], [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 3, 0, 7, 0, 7, 0, 1, 0, 1], [0, 1, 0, 1, 0, 2, 0, 1, 0, 1, 0, 1, 0, 7, 0, 1, 0, 7, 0, 1, 0], [1, 0, 1, 0, 2, 0, 2, 0, 1, 0, 1, 0, 1, 0, 7, 0, 1, 0, 7, 0, 1], [0, 1, 0, 2, 0, 1, 0, 2, 0, 1, 0, 1, 0, 1, 0, 7, 0, 1, 0, 7, 0], [1, 0, 6, 0, 2, 0, 2, 0, 6, 0, 1, 0, 1, 0, 1, 0, 7, 0, 1, 0, 7], [0, 6, 0, 6, 0, 2, 0, 6, 0, 6, 0, 1, 0, 1, 0, 1, 0, 7, 0, 1, 0]], "attempt_2": [[2, 0, 1, 0, 2, 0, 1, 0, 1, 0, 4, 0, 1, 0, 4, 0, 1, 0, 1, 0, 1], [0, 2, 0, 1, 0, 2, 0, 1, 0, 4, 0, 1, 0, 4, 0, 1, 0, 1, 0, 1, 0], [1, 0, 2, 0, 1, 0, 2, 0, 4, 0, 1, 0, 4, 0, 1, 0, 1, 0, 1, 0, 1], [0, 1, 0, 2, 0, 2, 0, 6, 0, 4, 0, 4, 0, 1, 0, 1, 0, 1, 0, 1, 0], [1, 0, 1, 0, 2, 0, 6, 0, 6, 0, 4, 0, 1, 0, 1, 0, 1, 0, 1, 0, 6], [0, 1, 0, 1, 0, 6, 0, 1, 0, 6, 0, 1, 0, 1, 0, 1, 0, 1, 0, 6, 0], [1, 0, 1, 0, 9, 0, 6, 0, 6, 0, 1, 0, 1, 0, 1, 0, 1, 0, 6, 0, 1], [0, 1, 0, 9, 0, 9, 0, 6, 0, 1, 0, 1, 0, 1, 0, 1, 0, 6, 0, 1, 0], [1, 0, 9, 0, 1, 0, 9, 0, 1, 0, 1, 0, 1, 0, 1, 0, 6, 0, 1, 0, 6], [0, 9, 0, 1, 0, 9, 0, 1, 0, 1, 0, 1, 0, 1, 0, 6, 0, 1, 0, 6, 0], [9, 0, 1, 0, 9, 0, 1, 0, 1, 0, 1, 0, 1, 0, 6, 0, 1, 0, 6, 0, 1], [0, 1, 0, 9, 0, 1, 0, 1, 0, 1, 0, 1, 0, 6, 0, 1, 0, 6, 0, 1, 0], [1, 0, 9, 0, 1, 0, 1, 0, 1, 0, 1, 0, 3, 0, 6, 0, 6, 0, 1, 0, 1], [0, 9, 0, 1, 0, 1, 0, 1, 0, 1, 0, 3, 0, 3, 0, 6, 0, 1, 0, 1, 0], [9, 0, 1, 0, 1, 0, 1, 0, 1, 0, 3, 0, 1, 0, 3, 0, 1, 0, 1, 0, 1], [0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 3, 0, 3, 0, 7, 0, 1, 0, 1, 0], [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 3, 0, 7, 0, 7, 0, 1, 0, 1], [0, 1, 0, 1, 0, 2, 0, 1, 0, 1, 0, 1, 0, 7, 0, 1, 0, 7, 0, 1, 0], [1, 0, 1, 0, 2, 0, 2, 0, 1, 0, 1, 0, 1, 0, 7, 0, 1, 0, 7, 0, 1], [0, 1, 0, 2, 0, 1, 0, 2, 0, 1, 0, 1, 0, 1, 0, 7, 0, 1, 0, 7, 0], [1, 0, 6, 0, 2, 0, 2, 0, 6, 0, 1, 0, 1, 0, 1, 0, 7, 0, 1, 0, 7], [0, 1, 0, 6, 0, 2, 0, 6, 0, 1, 0, 1, 0, 1, 0, 1, 0, 7, 0, 1, 0]]}], "88e364bc": [{"attempt_1": [[5, 5, 5, 5, 5, 5, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [5, 5, 5, 1, 5, 5, 5, 0, 0, 5, 5, 5, 0, 0, 0, 0, 0, 0, 0, 0], [5, 1, 5, 5, 1, 5, 5, 0, 0, 5, 0, 5, 5, 5, 0, 0, 0, 0, 0, 0], [5, 5, 1, 5, 5, 2, 5, 0, 0, 5, 0, 0, 0, 5, 0, 0, 5, 5, 0, 0], [5, 5, 5, 2, 5, 5, 5, 0, 0, 5, 0, 0, 0, 5, 5, 5, 5, 5, 5, 0], [5, 5, 5, 5, 5, 5, 5, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 0, 0, 0, 0, 0, 0, 0, 5, 0], [0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 5, 4, 0, 0, 0, 0, 0, 0, 5, 0], [0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 5, 5, 5, 0, 0, 5, 5, 5, 5, 0], [0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 5, 5, 5, 5, 0, 0, 0, 0], [0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0], [1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0], [1, 2, 1, 2, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0], [1, 5, 1, 5, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0], [1, 5, 1, 5, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0], [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0]], "attempt_2": [[5, 5, 5, 5, 5, 5, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [5, 5, 5, 1, 5, 5, 5, 0, 0, 5, 5, 5, 0, 0, 0, 0, 0, 0, 0, 0], [5, 1, 5, 5, 1, 5, 5, 0, 0, 5, 0, 5, 5, 5, 0, 0, 0, 0, 0, 0], [5, 5, 1, 5, 5, 2, 5, 0, 0, 5, 0, 0, 0, 5, 0, 0, 5, 5, 0, 0], [5, 5, 5, 2, 5, 5, 5, 0, 0, 5, 0, 0, 0, 5, 5, 5, 5, 5, 5, 0], [5, 5, 5, 5, 5, 5, 5, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 0, 0, 0, 0, 0, 0, 0, 5, 0], [0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 5, 4, 0, 0, 0, 0, 0, 0, 5, 0], [0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 5, 5, 5, 0, 0, 5, 5, 5, 5, 0], [0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 5, 5, 5, 5, 0, 0, 0, 0], [0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 4, 1, 1, 0], [1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0], [1, 2, 1, 2, 1, 0, 0, 0, 0, 1, 0, 4, 0, 0, 0, 0, 0, 0, 1, 0], [1, 5, 1, 5, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0], [1, 5, 1, 5, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0], [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0]]}, {"attempt_1": [[6, 6, 6, 6, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [6, 2, 6, 2, 6, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 6, 6, 6, 6], [6, 1, 6, 1, 6, 0, 0, 3, 0, 0, 0, 0, 0, 3, 0, 0, 6, 0, 0, 6], [6, 1, 6, 1, 6, 0, 3, 3, 0, 0, 0, 0, 0, 3, 6, 6, 6, 6, 0, 6], [6, 6, 6, 6, 6, 0, 3, 0, 0, 0, 0, 0, 3, 3, 6, 0, 0, 0, 0, 6], [0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 3, 0, 6, 0, 0, 0, 0, 6], [0, 0, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 3, 3, 6, 6, 6, 0, 0, 6], [0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 0, 6, 0, 0, 6], [0, 0, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 0, 6, 0, 0, 6], [0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 0, 0, 6, 6, 0, 6], [0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 6, 6, 6], [0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 3, 3, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 3, 3, 0, 0, 0, 0, 0, 0, 3, 0, 0, 3, 3, 3, 3, 3, 3, 3], [0, 0, 3, 0, 0, 0, 0, 0, 0, 3, 3, 0, 0, 3, 3, 3, 2, 3, 3, 3], [0, 0, 3, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 3, 2, 3, 3, 1, 3, 3], [0, 0, 3, 3, 0, 0, 0, 0, 0, 3, 0, 0, 0, 3, 3, 1, 3, 3, 1, 3], [0, 0, 0, 3, 3, 3, 0, 0, 3, 3, 0, 0, 0, 3, 3, 3, 1, 3, 3, 3], [0, 0, 0, 0, 0, 3, 3, 3, 3, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3]], "attempt_2": [[6, 6, 6, 6, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [6, 2, 6, 2, 6, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 6, 6, 6, 6], [6, 1, 6, 1, 6, 0, 0, 3, 0, 4, 0, 4, 0, 3, 0, 0, 6, 0, 4, 6], [6, 1, 6, 1, 6, 0, 3, 3, 0, 0, 0, 0, 0, 3, 6, 6, 6, 6, 0, 6], [6, 6, 6, 6, 6, 0, 3, 0, 0, 0, 0, 0, 3, 3, 6, 0, 0, 0, 0, 6], [0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 3, 0, 6, 0, 0, 0, 0, 6], [0, 0, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 3, 3, 6, 6, 6, 0, 0, 6], [0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 0, 6, 0, 0, 6], [0, 0, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 0, 6, 0, 0, 6], [0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 0, 0, 6, 6, 0, 6], [0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 6, 6, 6], [0, 0, 0, 0, 3, 0, 4, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 3, 3, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 3, 3, 0, 0, 0, 0, 0, 0, 3, 0, 0, 3, 3, 3, 3, 3, 3, 3], [0, 0, 3, 0, 0, 0, 0, 0, 0, 3, 3, 0, 0, 3, 3, 3, 2, 3, 3, 3], [0, 0, 3, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 3, 2, 3, 3, 1, 3, 3], [0, 0, 3, 3, 0, 0, 0, 0, 0, 3, 0, 0, 0, 3, 3, 1, 3, 3, 1, 3], [0, 0, 0, 3, 3, 3, 0, 0, 3, 3, 0, 0, 0, 3, 3, 3, 1, 3, 3, 3], [0, 0, 0, 0, 0, 3, 3, 3, 3, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3]]}], "89565ca0": [{"attempt_1": [[1, 9, 9, 9], [2, 2, 9, 9], [8, 8, 8, 9], [3, 3, 3, 9], [4, 4, 4, 4]], "attempt_2": [[1, 9, 9, 9], [2, 2, 9, 9], [3, 3, 3, 9], [8, 8, 8, 9], [4, 4, 4, 4]]}], "8b7bacbf": [{"attempt_1": [[6, 1, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 3, 8], [1, 1, 1, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 3, 3], [5, 5, 5, 1, 9, 9, 1, 1, 1, 1, 1, 1, 1, 9, 9, 9, 9, 9, 9, 9, 5, 5, 5, 5, 5, 5, 3, 3, 5, 5], [5, 5, 5, 9, 8, 8, 9, 5, 5, 5, 5, 5, 9, 8, 8, 8, 8, 8, 8, 8, 9, 5, 5, 5, 5, 5, 3, 5, 5, 5], [5, 5, 5, 9, 8, 8, 9, 5, 5, 5, 5, 5, 9, 8, 8, 8, 8, 8, 8, 8, 9, 5, 5, 5, 5, 3, 3, 5, 5, 5], [5, 5, 5, 5, 9, 9, 5, 5, 5, 5, 5, 5, 5, 9, 9, 9, 9, 9, 9, 9, 1, 1, 1, 1, 5, 3, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 1, 3, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 3, 1, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 3, 5, 5, 1, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 9, 9, 3, 5, 5, 1, 5, 5], [5, 5, 5, 5, 9, 9, 9, 9, 9, 9, 9, 5, 5, 5, 5, 5, 9, 3, 3, 5, 5, 9, 8, 8, 9, 5, 5, 1, 5, 5], [5, 5, 5, 9, 8, 8, 8, 8, 8, 8, 8, 9, 5, 5, 5, 9, 9, 9, 3, 3, 3, 9, 8, 8, 9, 5, 5, 1, 5, 5], [5, 5, 5, 9, 8, 8, 8, 8, 8, 8, 8, 9, 5, 5, 5, 9, 8, 9, 5, 5, 3, 3, 9, 9, 5, 5, 5, 1, 1, 5], [5, 5, 3, 3, 9, 9, 9, 9, 9, 9, 9, 5, 5, 5, 5, 9, 8, 9, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 1, 5], [5, 0, 3, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 9, 8, 9, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 1, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 9, 8, 9, 5, 5, 5, 5, 5, 5, 5, 5, 5, 1, 1, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 9, 8, 9, 5, 5, 5, 5, 5, 5, 5, 5, 5, 1, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 9, 8, 9, 5, 5, 5, 5, 5, 5, 5, 5, 5, 1, 5, 5], [5, 5, 0, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 9, 8, 9, 5, 5, 5, 5, 5, 5, 5, 5, 1, 1, 5, 5], [5, 5, 3, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 9, 8, 9, 5, 5, 5, 5, 5, 5, 5, 5, 1, 5, 5, 5], [5, 5, 3, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 3, 3, 3, 9, 5, 5, 5, 5, 5, 5, 5, 5, 1, 1, 5, 5, 5], [5, 5, 3, 3, 5, 5, 5, 5, 5, 5, 3, 3, 3, 3, 5, 5, 5, 5, 5, 5, 9, 9, 9, 1, 1, 1, 5, 5, 5, 5], [5, 5, 5, 3, 9, 9, 3, 3, 3, 3, 3, 5, 5, 5, 5, 5, 5, 5, 5, 9, 8, 8, 9, 1, 1, 5, 5, 5, 5, 5], [5, 5, 5, 9, 8, 8, 9, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 9, 8, 8, 8, 9, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 9, 8, 8, 9, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 9, 8, 8, 8, 9, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 9, 9, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 9, 8, 8, 8, 9, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 9, 8, 8, 8, 9, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 9, 9, 9, 9, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5]], "attempt_2": [[6, 1, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 3, 8], [1, 1, 1, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 3, 3], [5, 5, 5, 1, 9, 9, 1, 1, 1, 1, 1, 1, 1, 9, 9, 9, 9, 9, 9, 9, 5, 5, 5, 5, 5, 5, 3, 3, 5, 5], [5, 5, 5, 9, 6, 6, 9, 5, 5, 5, 5, 5, 9, 6, 6, 6, 6, 6, 6, 6, 9, 5, 5, 5, 5, 5, 3, 5, 5, 5], [5, 5, 5, 9, 6, 6, 9, 5, 5, 5, 5, 5, 9, 6, 6, 6, 6, 6, 6, 6, 9, 5, 5, 5, 5, 3, 3, 5, 5, 5], [5, 5, 5, 5, 9, 9, 5, 5, 5, 5, 5, 5, 5, 9, 9, 9, 9, 9, 9, 9, 1, 1, 1, 1, 5, 3, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 1, 3, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 3, 1, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 3, 5, 5, 1, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 9, 9, 3, 5, 5, 1, 5, 5], [5, 5, 5, 5, 9, 9, 9, 9, 9, 9, 9, 5, 5, 5, 5, 5, 9, 3, 3, 5, 5, 9, 6, 6, 9, 5, 5, 1, 5, 5], [5, 5, 5, 9, 6, 6, 6, 6, 6, 6, 6, 9, 5, 5, 5, 9, 9, 9, 3, 3, 3, 9, 6, 6, 9, 5, 5, 1, 5, 5], [5, 5, 5, 9, 6, 6, 6, 6, 6, 6, 6, 9, 5, 5, 5, 9, 6, 9, 5, 5, 3, 3, 9, 9, 5, 5, 5, 1, 1, 5], [5, 5, 3, 3, 9, 9, 9, 9, 9, 9, 9, 5, 5, 5, 5, 9, 6, 9, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 1, 5], [5, 0, 3, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 9, 6, 9, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 1, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 9, 6, 9, 5, 5, 5, 5, 5, 5, 5, 5, 5, 1, 1, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 9, 6, 9, 5, 5, 5, 5, 5, 5, 5, 5, 5, 1, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 9, 6, 9, 5, 5, 5, 5, 5, 5, 5, 5, 5, 1, 5, 5], [5, 5, 0, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 9, 6, 9, 5, 5, 5, 5, 5, 5, 5, 5, 1, 1, 5, 5], [5, 5, 3, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 9, 6, 9, 5, 5, 5, 5, 5, 5, 5, 5, 1, 5, 5, 5], [5, 5, 3, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 3, 3, 3, 9, 5, 5, 5, 5, 5, 5, 5, 5, 1, 1, 5, 5, 5], [5, 5, 3, 3, 5, 5, 5, 5, 5, 5, 3, 3, 3, 3, 5, 5, 5, 5, 5, 5, 9, 9, 9, 1, 1, 1, 5, 5, 5, 5], [5, 5, 5, 3, 9, 9, 3, 3, 3, 3, 3, 5, 5, 5, 5, 5, 5, 5, 5, 9, 6, 6, 9, 1, 1, 5, 5, 5, 5, 5], [5, 5, 5, 9, 6, 6, 9, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 9, 6, 6, 6, 9, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 9, 6, 6, 9, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 9, 6, 6, 6, 9, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 9, 9, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 9, 6, 6, 6, 9, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 9, 6, 6, 6, 9, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 9, 9, 9, 9, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5]]}, {"attempt_1": [[0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 8, 8, 0], [0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 8, 3, 3, 8], [0, 0, 0, 0, 2, 2, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 8, 8, 0, 0, 8, 3, 3, 8], [0, 0, 0, 2, 3, 3, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 3, 3, 8, 0, 0, 8, 8, 0], [0, 0, 0, 2, 3, 3, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 3, 3, 8, 0, 0, 0, 0, 0], [0, 0, 0, 1, 2, 2, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 8, 1, 0, 0, 0, 0, 0], [0, 0, 1, 1, 0, 0, 1, 1, 6, 6, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0], [0, 0, 1, 0, 0, 0, 0, 6, 3, 3, 3, 6, 0, 0, 0, 8, 8, 0, 0, 0, 0, 0, 0, 0, 1, 0], [0, 0, 1, 0, 0, 0, 0, 6, 3, 3, 3, 6, 0, 0, 8, 3, 3, 8, 0, 0, 0, 5, 5, 5, 1, 0], [0, 0, 1, 0, 0, 0, 0, 6, 3, 3, 3, 6, 0, 0, 8, 3, 3, 8, 0, 0, 5, 3, 3, 3, 5, 0], [0, 0, 1, 0, 0, 0, 0, 0, 6, 6, 6, 0, 0, 9, 9, 8, 8, 9, 0, 0, 5, 3, 3, 3, 5, 0], [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 9, 0, 0, 0, 0, 9, 9, 0, 0, 5, 5, 5, 0, 0], [0, 0, 1, 0, 0, 0, 0, 0, 8, 8, 9, 9, 0, 0, 0, 0, 0, 0, 9, 0, 0, 0, 0, 0, 0, 0], [0, 0, 1, 0, 0, 0, 0, 8, 3, 3, 8, 0, 0, 0, 0, 0, 0, 0, 9, 0, 0, 0, 0, 0, 0, 0], [0, 0, 1, 0, 0, 0, 0, 8, 3, 3, 8, 0, 0, 0, 0, 0, 0, 0, 9, 0, 0, 0, 0, 0, 0, 0], [0, 0, 1, 0, 0, 0, 0, 0, 8, 8, 0, 0, 0, 0, 0, 0, 0, 0, 9, 0, 0, 0, 8, 8, 0, 0], [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 8, 9, 0, 0, 8, 3, 3, 8, 0], [0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 3, 3, 8, 0, 0, 8, 3, 3, 8, 0], [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 3, 3, 8, 0, 0, 9, 8, 8, 0, 0], [0, 1, 0, 0, 0, 0, 0, 9, 9, 9, 9, 0, 0, 0, 0, 9, 8, 8, 9, 9, 9, 0, 0, 0, 0, 0], [1, 1, 0, 0, 9, 9, 9, 0, 0, 0, 9, 9, 9, 9, 9, 9, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [4, 1, 0, 0, 9, 3, 9, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], "attempt_2": [[0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 8, 8, 0], [0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 8, 3, 3, 8], [0, 0, 0, 0, 2, 2, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 8, 8, 0, 0, 8, 3, 3, 8], [0, 0, 0, 2, 4, 4, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 3, 3, 8, 0, 0, 8, 8, 0], [0, 0, 0, 2, 4, 4, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 3, 3, 8, 0, 0, 0, 0, 0], [0, 0, 0, 1, 2, 2, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 8, 1, 0, 0, 0, 0, 0], [0, 0, 1, 1, 0, 0, 1, 1, 6, 6, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0], [0, 0, 1, 0, 0, 0, 0, 6, 4, 4, 4, 6, 0, 0, 0, 8, 8, 0, 0, 0, 0, 0, 0, 0, 1, 0], [0, 0, 1, 0, 0, 0, 0, 6, 4, 4, 4, 6, 0, 0, 8, 3, 3, 8, 0, 0, 0, 5, 5, 5, 1, 0], [0, 0, 1, 0, 0, 0, 0, 6, 4, 4, 4, 6, 0, 0, 8, 3, 3, 8, 0, 0, 5, 4, 4, 4, 5, 0], [0, 0, 1, 0, 0, 0, 0, 0, 6, 6, 6, 0, 0, 9, 9, 8, 8, 9, 0, 0, 5, 4, 4, 4, 5, 0], [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 9, 0, 0, 0, 0, 9, 9, 0, 0, 5, 5, 5, 0, 0], [0, 0, 1, 0, 0, 0, 0, 0, 8, 8, 9, 9, 0, 0, 0, 0, 0, 0, 9, 0, 0, 0, 0, 0, 0, 0], [0, 0, 1, 0, 0, 0, 0, 8, 3, 3, 8, 0, 0, 0, 0, 0, 0, 0, 9, 0, 0, 0, 0, 0, 0, 0], [0, 0, 1, 0, 0, 0, 0, 8, 3, 3, 8, 0, 0, 0, 0, 0, 0, 0, 9, 0, 0, 0, 0, 0, 0, 0], [0, 0, 1, 0, 0, 0, 0, 0, 8, 8, 0, 0, 0, 0, 0, 0, 0, 0, 9, 0, 0, 0, 8, 8, 0, 0], [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 8, 9, 0, 0, 8, 3, 3, 8, 0], [0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 3, 3, 8, 0, 0, 8, 3, 3, 8, 0], [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 3, 3, 8, 0, 0, 9, 8, 8, 0, 0], [0, 1, 0, 0, 0, 0, 0, 9, 9, 9, 9, 0, 0, 0, 0, 9, 8, 8, 9, 9, 9, 0, 0, 0, 0, 0], [1, 1, 0, 0, 9, 9, 9, 0, 0, 0, 9, 9, 9, 9, 9, 9, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [4, 1, 0, 0, 9, 3, 9, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}], "8b9c3697": [{"attempt_1": [[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 4, 3, 4, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 2, 3, 4, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 3, 3, 3, 4, 3, 0, 3, 4, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 3, 3, 3, 3, 3, 3, 0, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 3, 3, 3, 3, 3, 0, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 0, 0, 0, 0, 0, 2, 2, 4, 4, 3, 3, 3, 3, 3, 3, 0, 3, 3, 3, 3, 0, 0, 3, 3, 3], [3, 3, 0, 0, 0, 0, 0, 2, 2, 4, 4, 3, 3, 3, 3, 3, 3, 0, 3, 3, 3, 3, 0, 0, 3, 3, 3], [3, 3, 0, 0, 0, 0, 0, 2, 2, 4, 4, 3, 3, 3, 3, 3, 3, 0, 3, 3, 3, 3, 0, 0, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 0, 0, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 0, 0, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 0, 0, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 0, 0, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 0, 0, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 2, 2, 3, 4, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 2, 2, 3, 4, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 4, 4, 3, 4, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]], "attempt_2": [[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 4, 3, 4, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 2, 3, 4, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 3, 3, 3, 4, 3, 0, 3, 4, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 3, 3, 3, 3, 3, 3, 0, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 3, 3, 3, 3, 3, 0, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 0, 0, 0, 0, 0, 2, 2, 4, 4, 3, 3, 3, 3, 3, 3, 0, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 0, 0, 0, 0, 0, 2, 2, 4, 4, 3, 3, 3, 3, 3, 3, 0, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 0, 0, 0, 0, 0, 2, 2, 4, 4, 3, 3, 3, 3, 3, 3, 0, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 3, 3, 3, 4, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 3, 3, 3, 4, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 4, 4, 3, 4, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]]}], "8e5c0c38": [{"attempt_1": [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 6, 6, 6, 6, 1, 1, 6, 6, 6, 6, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 6, 1, 1, 1, 1, 1, 1, 6, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 6, 1, 1, 1, 1, 1, 1, 6, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 6, 6, 6, 1, 1, 1, 1, 1, 1, 6, 6, 6, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 6, 6, 6, 1, 6, 6, 6, 6, 1, 6, 6, 6, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 6, 6, 6, 1, 1, 6, 6, 6, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 6, 1, 1, 1, 1, 6, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 6, 6, 6, 6, 6, 6, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 6, 1, 1, 1, 1, 6, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 3, 1, 1], [1, 1, 1, 1, 1, 1, 4, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 3, 1, 1], [1, 1, 1, 1, 1, 4, 4, 4, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1], [1, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 4, 4, 4, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 4, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], "attempt_2": [[0]]}, {"attempt_1": [[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 2, 2, 2, 4, 4, 4, 2, 2, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 2, 4, 4, 2, 2, 2, 4, 4, 2, 4, 4, 4, 4, 3, 3, 3, 3, 3, 4, 4], [4, 4, 2, 2, 4, 4, 4, 4, 4, 2, 2, 4, 4, 4, 4, 4, 3, 4, 3, 4, 4, 4], [4, 4, 4, 2, 4, 2, 2, 2, 4, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 2, 2, 2, 4, 2, 2, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 1, 4, 4, 1, 1, 1, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 4, 1, 1, 1, 1, 4, 1, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 4, 4, 4, 4, 4, 4, 1, 1, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 4, 4, 4, 4, 4, 4, 1, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 1, 1, 1, 1, 1, 1, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]], "attempt_2": [[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 2, 2, 2, 4, 4, 4, 2, 2, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 2, 4, 4, 2, 2, 2, 4, 4, 2, 4, 4, 4, 4, 3, 3, 3, 3, 3, 3, 4], [4, 4, 2, 2, 4, 4, 4, 4, 4, 2, 2, 4, 4, 4, 4, 4, 4, 4, 3, 4, 4, 4], [4, 4, 4, 2, 4, 2, 2, 2, 4, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 2, 2, 2, 4, 2, 2, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 1, 4, 4, 1, 1, 1, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 4, 1, 1, 1, 1, 4, 1, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 4, 4, 4, 4, 4, 4, 1, 1, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 4, 4, 4, 4, 4, 4, 1, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 1, 1, 1, 1, 1, 1, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]]}], "8f215267": [{"attempt_1": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 2, 8, 8, 8, 8, 8, 2, 8, 2, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 4, 8, 8, 8, 4, 8, 4, 8, 4, 8, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 3, 8, 8, 8, 8, 8, 3, 8, 3, 8, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 1, 8, 8, 8, 8, 8, 8, 8, 1, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]], "attempt_2": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 2, 8, 8, 8, 8, 8, 2, 8, 2, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 4, 8, 8, 8, 8, 8, 4, 8, 4, 8, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 3, 8, 8, 8, 8, 8, 3, 8, 3, 8, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 1, 8, 8, 8, 8, 8, 8, 8, 1, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]]}], "8f3a5a89": [{"attempt_1": [[7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 1, 7], [7, 8, 8, 8, 8, 8, 8, 8, 8, 7, 1, 7], [7, 8, 8, 8, 8, 8, 8, 8, 8, 7, 1, 7], [7, 8, 8, 8, 8, 8, 7, 7, 7, 7, 1, 7], [7, 8, 8, 8, 8, 8, 7, 1, 1, 1, 1, 7], [7, 8, 8, 8, 8, 8, 7, 1, 1, 1, 1, 7], [7, 8, 8, 8, 8, 8, 7, 7, 1, 1, 1, 7], [7, 8, 8, 8, 8, 8, 8, 7, 7, 7, 1, 7], [7, 8, 8, 8, 8, 8, 8, 8, 7, 7, 1, 7], [7, 8, 8, 8, 8, 8, 8, 8, 7, 1, 1, 1], [7, 7, 7, 7, 7, 7, 7, 8, 7, 1, 1, 1], [6, 7, 1, 1, 1, 1, 7, 7, 7, 1, 1, 1]], "attempt_2": [[7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 1, 8], [7, 8, 8, 8, 8, 8, 8, 8, 8, 7, 1, 8], [7, 8, 8, 8, 8, 8, 8, 8, 8, 7, 1, 8], [7, 8, 8, 8, 8, 8, 7, 7, 7, 7, 1, 8], [7, 8, 8, 8, 8, 8, 7, 1, 1, 1, 1, 8], [7, 8, 8, 8, 8, 8, 7, 1, 1, 1, 1, 8], [7, 8, 8, 8, 8, 8, 7, 7, 1, 1, 1, 8], [7, 8, 8, 8, 8, 8, 8, 7, 7, 7, 1, 8], [7, 8, 8, 8, 8, 8, 8, 8, 7, 7, 1, 8], [7, 8, 8, 8, 8, 8, 8, 8, 7, 1, 1, 8], [7, 7, 7, 7, 7, 7, 7, 8, 7, 1, 1, 8], [6, 7, 1, 1, 1, 1, 7, 7, 7, 1, 1, 8]]}], "9385bd28": [{"attempt_1": [[9, 9, 9, 9, 4, 4, 9, 2, 2, 9, 9, 9, 9], [9, 9, 9, 9, 4, 9, 9, 2, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 1, 1, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1], [9, 1, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 5, 5, 9, 9, 9, 9, 9, 9, 5], [9, 9, 9, 9, 5, 9, 9, 9, 9, 9, 9, 5, 5], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 3, 3, 9, 9, 9, 9, 9, 9, 3], [9, 9, 9, 9, 3, 9, 9, 9, 9, 9, 9, 3, 3], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 2, 2, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 5, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 3, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 1, 4, 9, 9, 4, 9, 9, 2, 9, 9, 9, 9], [9, 4, 6, 9, 4, 4, 9, 2, 2, 9, 9, 9, 9]], "attempt_2": [[9, 9, 9, 9, 4, 4, 6, 2, 2, 9, 9, 9, 9], [9, 9, 9, 9, 4, 6, 6, 2, 6, 9, 9, 9, 9], [9, 9, 9, 9, 6, 6, 6, 6, 6, 9, 9, 9, 9], [9, 1, 1, 4, 6, 6, 6, 6, 6, 4, 4, 4, 1], [9, 1, 4, 4, 6, 6, 6, 6, 6, 4, 4, 1, 1], [9, 9, 9, 9, 6, 6, 6, 6, 6, 9, 9, 9, 9], [9, 9, 9, 9, 5, 5, 6, 6, 6, 9, 9, 9, 5], [9, 9, 9, 9, 5, 6, 6, 6, 6, 9, 9, 5, 5], [9, 9, 9, 9, 6, 6, 6, 6, 6, 9, 9, 9, 9], [9, 9, 9, 9, 3, 3, 6, 6, 6, 9, 9, 9, 3], [9, 9, 9, 9, 3, 6, 6, 6, 6, 9, 9, 3, 3], [9, 9, 9, 9, 6, 6, 6, 6, 6, 9, 9, 9, 9], [9, 9, 9, 9, 6, 6, 6, 6, 6, 9, 9, 9, 9], [9, 9, 9, 9, 6, 6, 6, 6, 6, 9, 9, 9, 9], [9, 2, 2, 9, 6, 6, 6, 6, 6, 9, 9, 9, 9], [9, 5, 9, 9, 6, 6, 6, 6, 6, 9, 9, 9, 9], [9, 3, 9, 9, 6, 6, 6, 6, 6, 9, 9, 9, 9], [9, 1, 4, 9, 6, 4, 6, 6, 2, 9, 9, 9, 9], [9, 4, 6, 9, 4, 4, 6, 2, 2, 9, 9, 9, 9]]}, {"attempt_1": [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 2, 2, 1, 1, 1, 7, 7, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 2, 2, 1, 1, 1, 7, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 6, 6, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 6, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 6, 4, 2, 2, 1, 1], [1, 1, 3, 4, 1, 1, 8, 8, 8, 8, 8, 8, 6, 6, 4, 4, 2, 1, 1], [1, 1, 6, 8, 1, 1, 1, 1, 1, 3, 3, 1, 1, 4, 4, 4, 4, 1, 1], [1, 1, 7, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 4, 4, 4, 1, 1], [1, 1, 8, 2, 1, 1, 1, 1, 7, 1, 1, 1, 1, 2, 2, 4, 4, 1, 1], [1, 1, 2, 4, 1, 1, 1, 7, 7, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], "attempt_2": [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 2, 2, 1, 1, 1, 7, 7, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 2, 2, 1, 1, 1, 7, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 6, 6, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 6, 8, 3, 3, 8, 8, 8, 8, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 8, 8, 3, 8, 8, 8, 8, 6, 4, 2, 2, 1, 1], [1, 1, 3, 4, 1, 1, 8, 8, 8, 8, 3, 8, 6, 6, 4, 4, 2, 1, 1], [1, 1, 6, 8, 1, 1, 1, 1, 1, 3, 3, 1, 1, 4, 4, 4, 4, 1, 1], [1, 1, 7, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 4, 4, 4, 1, 1], [1, 1, 8, 2, 1, 1, 1, 1, 7, 1, 1, 1, 1, 2, 2, 4, 4, 1, 1], [1, 1, 2, 4, 1, 1, 1, 7, 7, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]}], "97d7923e": [{"attempt_1": [[1, 2, 3, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0], [0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0], [0, 2, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 5, 0, 1, 0], [0, 2, 0, 4, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 5, 0, 1, 0], [0, 2, 0, 5, 0, 0, 0, 0, 5, 0, 3, 0, 0, 0, 0, 0, 5, 0, 1, 1], [0, 2, 0, 5, 0, 0, 3, 0, 5, 0, 5, 0, 0, 2, 0, 4, 5, 0, 1, 5], [0, 2, 0, 5, 0, 0, 5, 0, 5, 0, 5, 0, 0, 5, 0, 5, 5, 0, 1, 5], [0, 2, 0, 4, 0, 0, 3, 0, 3, 0, 3, 0, 0, 2, 0, 4, 1, 0, 1, 1]], "attempt_2": [[1, 2, 3, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0], [0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0], [0, 2, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 5, 0, 1, 0], [0, 2, 0, 4, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 5, 0, 1, 0], [0, 2, 0, 5, 0, 0, 0, 0, 3, 0, 3, 0, 0, 0, 0, 0, 5, 0, 1, 1], [0, 2, 0, 5, 0, 0, 3, 0, 3, 0, 5, 0, 0, 2, 0, 4, 5, 0, 1, 5], [0, 2, 0, 5, 0, 0, 5, 0, 3, 0, 5, 0, 0, 5, 0, 5, 5, 0, 1, 5], [0, 2, 0, 4, 0, 0, 3, 0, 3, 0, 3, 0, 0, 2, 0, 4, 1, 0, 1, 1]]}], "9bbf930d": [{"attempt_1": [[6, 7, 9, 9, 9, 9, 9, 9, 9, 7, 7, 7, 7, 7, 6, 7, 9, 9, 9, 9, 9], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 9], [6, 7, 9, 9, 9, 9, 9, 9, 9, 7, 7, 7, 7, 0, 7, 0, 9, 9, 9, 7, 9], [6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 0, 7, 0, 7, 7, 9, 7, 9], [6, 7, 0, 0, 0, 0, 0, 0, 0, 7, 7, 7, 0, 0, 7, 0, 7, 7, 9, 7, 9], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 0, 7, 7, 9, 7, 9], [6, 7, 0, 0, 0, 0, 0, 0, 0, 7, 7, 7, 0, 0, 0, 0, 7, 7, 9, 7, 9], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 0, 7, 7, 7, 7, 7, 6], [6, 7, 0, 0, 0, 0, 0, 0, 0, 7, 7, 7, 0, 7, 0, 8, 8, 8, 8, 7, 7], [7, 7, 7, 7, 7, 7, 7, 7, 1, 7, 1, 7, 0, 7, 0, 7, 7, 7, 7, 7, 6], [6, 7, 8, 8, 8, 8, 8, 7, 1, 7, 1, 7, 0, 7, 0, 8, 8, 8, 8, 7, 7], [6, 7, 7, 7, 7, 7, 7, 7, 1, 7, 1, 7, 0, 7, 0, 7, 7, 7, 7, 7, 7], [6, 7, 1, 1, 1, 1, 1, 1, 1, 7, 1, 7, 0, 7, 0, 7, 7, 7, 7, 7, 7], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 1, 7, 0, 7, 0, 0, 0, 7, 7, 7, 6], [6, 7, 1, 1, 1, 1, 1, 1, 1, 1, 1, 7, 0, 7, 7, 7, 7, 7, 7, 7, 7], [6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 0, 0, 0, 0, 0, 7, 7, 7, 7], [6, 7, 4, 4, 4, 4, 4, 4, 4, 4, 7, 7, 7, 4, 4, 4, 4, 7, 7, 7, 7], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 7, 7, 7, 6], [6, 7, 4, 4, 4, 4, 4, 4, 4, 4, 2, 7, 2, 4, 4, 7, 4, 7, 7, 7, 7], [6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 2, 7, 2, 7, 4, 7, 4, 7, 9, 7, 9], [6, 7, 2, 2, 2, 2, 2, 2, 2, 2, 2, 7, 2, 7, 4, 7, 4, 9, 9, 7, 9], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 2, 7, 7, 7, 7, 7, 7, 7, 9], [6, 7, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 7, 7, 6, 7, 9, 9, 9, 9]], "attempt_2": [[6, 7, 9, 9, 9, 9, 9, 9, 9, 7, 7, 7, 7, 7, 6, 7, 9, 9, 9, 9, 9], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 9], [6, 7, 9, 9, 9, 9, 9, 9, 9, 7, 7, 7, 7, 0, 7, 0, 9, 9, 9, 7, 9], [6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 0, 7, 0, 7, 7, 9, 7, 9], [6, 7, 0, 0, 0, 0, 0, 0, 0, 7, 7, 7, 0, 0, 7, 0, 7, 7, 9, 7, 9], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 0, 7, 7, 9, 7, 9], [6, 7, 0, 0, 0, 0, 0, 0, 0, 7, 7, 7, 0, 0, 0, 0, 7, 7, 9, 7, 9], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 0, 7, 7, 7, 7, 7, 6], [6, 7, 0, 0, 0, 0, 0, 0, 0, 7, 7, 7, 0, 7, 0, 8, 8, 8, 8, 7, 7], [7, 7, 7, 7, 7, 7, 7, 7, 1, 7, 1, 7, 0, 7, 0, 7, 7, 7, 7, 7, 6], [6, 7, 8, 8, 8, 8, 8, 7, 1, 7, 1, 7, 0, 7, 0, 8, 8, 8, 8, 7, 7], [6, 7, 7, 7, 7, 7, 7, 7, 1, 7, 1, 7, 0, 7, 0, 7, 7, 7, 7, 7, 7], [6, 7, 1, 1, 1, 1, 1, 1, 1, 7, 1, 7, 0, 7, 0, 7, 7, 7, 7, 7, 7], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 1, 7, 0, 7, 0, 0, 0, 7, 7, 7, 6], [6, 7, 1, 1, 1, 1, 1, 1, 1, 1, 1, 7, 0, 7, 7, 7, 7, 7, 7, 7, 6], [6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 0, 0, 0, 0, 0, 7, 7, 7, 7], [6, 7, 4, 4, 4, 4, 4, 4, 4, 4, 7, 7, 7, 4, 4, 4, 4, 7, 7, 7, 7], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 7, 7, 7, 6], [6, 7, 4, 4, 4, 4, 4, 4, 4, 4, 2, 7, 2, 4, 4, 7, 4, 7, 7, 7, 7], [6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 2, 7, 2, 7, 4, 7, 4, 7, 9, 7, 9], [6, 7, 2, 2, 2, 2, 2, 2, 2, 2, 2, 7, 2, 7, 4, 7, 4, 9, 9, 7, 9], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 2, 7, 7, 7, 7, 7, 7, 7, 9], [6, 7, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 7, 7, 6, 7, 9, 9, 9, 9]]}], "a251c730": [{"attempt_1": [[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2], [2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2], [2, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 2], [2, 8, 8, 8, 8, 1, 6, 1, 8, 5, 5, 5, 8, 2], [2, 8, 1, 8, 8, 1, 1, 1, 8, 5, 7, 5, 8, 2], [2, 1, 6, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2], [2, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2], [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]], "attempt_2": [[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4], [4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4], [4, 3, 3, 3, 3, 3, 1, 3, 3, 3, 3, 5, 5, 5, 3, 3, 3, 4], [4, 3, 3, 3, 3, 1, 6, 1, 3, 3, 3, 5, 7, 5, 3, 3, 3, 4], [4, 3, 3, 3, 3, 1, 1, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4], [4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4], [4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 3, 3, 3, 3, 4], [4, 3, 3, 3, 5, 5, 5, 3, 3, 3, 3, 1, 6, 1, 3, 3, 3, 4], [4, 3, 3, 3, 5, 7, 5, 3, 3, 3, 3, 1, 1, 1, 3, 3, 3, 4], [4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]]}], "a47bf94d": [{"attempt_1": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 1, 0, 0, 0, 0, 4, 0, 0, 0, 0, 2, 0, 0, 0, 0, 3, 0, 0, 0, 0, 6, 0, 0, 0], [0, 1, 0, 1, 0, 0, 4, 0, 4, 0, 0, 2, 0, 2, 0, 0, 3, 0, 3, 0, 0, 6, 0, 6, 0, 0], [0, 0, 1, 0, 0, 0, 0, 4, 0, 0, 0, 0, 2, 0, 0, 0, 0, 3, 0, 0, 0, 0, 6, 0, 0, 0], [0, 0, 8, 0, 0, 0, 0, 8, 0, 0, 0, 0, 8, 0, 0, 0, 0, 8, 0, 0, 0, 0, 8, 0, 0, 0], [0, 0, 8, 0, 0, 0, 0, 8, 0, 0, 0, 0, 8, 8, 8, 8, 0, 8, 0, 0, 8, 8, 8, 0, 0, 0], [0, 0, 8, 0, 0, 0, 0, 8, 0, 9, 9, 0, 0, 0, 0, 5, 0, 5, 0, 0, 5, 0, 0, 0, 0, 0], [0, 0, 8, 0, 0, 0, 0, 8, 8, 9, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 0, 0], [0, 0, 8, 0, 0, 0, 0, 0, 0, 9, 9, 0, 0, 0, 0, 5, 0, 5, 0, 0, 5, 0, 8, 0, 0, 0], [0, 0, 8, 8, 8, 8, 8, 8, 8, 9, 9, 8, 8, 0, 0, 8, 0, 8, 0, 0, 8, 0, 8, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 9, 9, 0, 8, 0, 0, 8, 0, 8, 0, 0, 8, 0, 8, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 5, 0, 8, 0, 9, 9, 9, 9, 9, 0, 0], [0, 0, 0, 0, 0, 0, 0, 8, 8, 8, 8, 5, 8, 5, 8, 8, 8, 8, 0, 9, 9, 9, 9, 9, 0, 0], [0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 8, 0, 0, 5, 0, 0, 0, 9, 9, 9, 9, 9, 0, 0], [0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 8, 0, 0, 8, 8, 8, 0, 0, 8, 0, 8, 0, 0, 0], [0, 9, 9, 9, 9, 0, 0, 8, 0, 0, 0, 0, 9, 9, 0, 0, 0, 5, 0, 0, 8, 0, 8, 0, 0, 0], [0, 9, 9, 9, 9, 8, 5, 8, 5, 8, 8, 5, 9, 9, 8, 8, 8, 8, 8, 8, 8, 0, 8, 0, 0, 0], [0, 9, 9, 9, 9, 0, 0, 8, 0, 0, 0, 0, 8, 0, 0, 0, 0, 5, 0, 0, 0, 0, 8, 0, 0, 0], [0, 0, 8, 0, 0, 0, 0, 8, 0, 0, 0, 0, 8, 0, 0, 0, 0, 8, 0, 0, 0, 0, 8, 0, 0, 0], [0, 0, 8, 0, 0, 0, 0, 8, 0, 0, 0, 0, 8, 0, 0, 0, 0, 8, 0, 0, 0, 0, 8, 0, 0, 0], [0, 6, 0, 6, 0, 0, 1, 0, 1, 0, 0, 2, 0, 2, 0, 0, 3, 0, 3, 0, 0, 4, 0, 4, 0, 0], [0, 0, 6, 0, 0, 0, 0, 1, 0, 0, 0, 0, 2, 0, 0, 0, 0, 3, 0, 0, 0, 0, 4, 0, 0, 0], [0, 6, 0, 6, 0, 0, 1, 0, 1, 0, 0, 2, 0, 2, 0, 0, 3, 0, 3, 0, 0, 4, 0, 4, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], "attempt_2": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 1, 0, 0, 0, 0, 4, 0, 0, 0, 0, 2, 0, 0, 0, 0, 3, 0, 0, 0, 0, 6, 0, 0, 0], [0, 1, 0, 1, 0, 0, 4, 0, 4, 0, 0, 2, 0, 2, 0, 0, 3, 0, 3, 0, 0, 6, 0, 6, 0, 0], [0, 0, 1, 0, 0, 0, 0, 4, 0, 0, 0, 0, 2, 0, 0, 0, 0, 3, 0, 0, 0, 0, 6, 0, 0, 0], [0, 0, 8, 0, 0, 0, 0, 8, 0, 0, 0, 0, 8, 0, 0, 0, 0, 8, 0, 0, 0, 0, 8, 0, 0, 0], [0, 0, 8, 0, 0, 0, 0, 8, 0, 0, 0, 0, 8, 8, 8, 8, 0, 8, 0, 0, 8, 8, 8, 0, 0, 0], [0, 0, 8, 0, 0, 0, 0, 8, 0, 9, 9, 0, 0, 0, 0, 5, 0, 5, 0, 0, 5, 0, 0, 0, 0, 0], [0, 0, 8, 0, 0, 0, 0, 8, 8, 9, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 0, 0], [0, 0, 8, 0, 0, 0, 0, 0, 0, 9, 9, 0, 0, 0, 0, 5, 0, 5, 0, 0, 5, 0, 8, 0, 0, 0], [0, 0, 8, 8, 8, 8, 8, 8, 8, 9, 9, 8, 8, 0, 0, 8, 0, 8, 0, 0, 8, 0, 8, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 9, 9, 0, 8, 0, 0, 8, 0, 8, 0, 0, 8, 0, 8, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 5, 0, 8, 0, 9, 9, 9, 9, 9, 0, 0], [0, 0, 0, 0, 0, 0, 0, 8, 8, 8, 8, 5, 8, 5, 8, 8, 8, 8, 0, 9, 9, 9, 9, 9, 0, 0], [0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 8, 0, 0, 5, 0, 0, 0, 9, 9, 9, 9, 9, 0, 0], [0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 8, 0, 0, 8, 8, 8, 0, 0, 8, 0, 8, 0, 0, 0], [0, 9, 9, 9, 9, 0, 0, 8, 0, 0, 0, 0, 9, 9, 0, 0, 0, 5, 0, 0, 8, 0, 8, 0, 0, 0], [0, 9, 9, 9, 9, 8, 5, 8, 5, 8, 8, 5, 9, 9, 8, 8, 8, 8, 8, 8, 8, 0, 8, 0, 0, 0], [0, 9, 9, 9, 9, 0, 0, 8, 0, 0, 0, 0, 8, 0, 0, 0, 0, 5, 0, 0, 0, 0, 8, 0, 0, 0], [0, 0, 8, 0, 0, 0, 0, 8, 0, 0, 0, 0, 8, 0, 0, 0, 0, 8, 0, 0, 0, 0, 8, 0, 0, 0], [0, 0, 8, 0, 0, 0, 0, 8, 0, 0, 0, 0, 8, 0, 0, 0, 0, 8, 0, 0, 0, 0, 8, 0, 0, 0], [0, 2, 0, 2, 0, 0, 6, 0, 6, 0, 0, 1, 0, 1, 0, 0, 3, 0, 3, 0, 0, 4, 0, 4, 0, 0], [0, 0, 2, 0, 0, 0, 0, 6, 0, 0, 0, 0, 1, 0, 0, 0, 0, 3, 0, 0, 0, 0, 4, 0, 0, 0], [0, 2, 0, 2, 0, 0, 6, 0, 6, 0, 0, 1, 0, 1, 0, 0, 3, 0, 3, 0, 0, 4, 0, 4, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}], "a6f40cea": [{"attempt_1": [[7, 7, 7, 7, 7, 8, 7, 7, 7, 7], [0, 0, 0, 0, 7, 8, 7, 7, 7, 7], [7, 7, 7, 0, 7, 8, 8, 8, 8, 8], [7, 7, 7, 0, 7, 7, 7, 7, 7, 7], [7, 7, 7, 0, 7, 7, 7, 7, 7, 7], [7, 7, 7, 0, 7, 7, 7, 4, 9, 4], [7, 7, 7, 0, 7, 7, 7, 9, 7, 7], [7, 7, 7, 0, 7, 7, 7, 4, 7, 7]], "attempt_2": [[7, 7, 7, 7, 7, 8, 7, 7, 7, 7], [0, 0, 0, 0, 7, 8, 7, 7, 7, 7], [7, 7, 7, 0, 7, 8, 8, 8, 8, 8], [7, 7, 7, 0, 7, 7, 7, 7, 7, 7], [7, 7, 7, 0, 7, 7, 7, 7, 7, 7], [7, 7, 7, 0, 7, 7, 7, 9, 4, 9], [7, 7, 7, 0, 7, 7, 7, 4, 7, 7], [7, 7, 7, 0, 7, 7, 7, 9, 7, 7]]}], "b5ca7ac4": [{"attempt_1": [[9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 2, 2, 2, 2, 2, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 2, 4, 4, 4, 2, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 2, 4, 4, 4, 2, 2, 2, 2, 2, 2], [9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 9, 9, 2, 4, 4, 4, 2, 2, 3, 3, 3, 2], [9, 9, 9, 9, 9, 8, 4, 4, 4, 8, 9, 9, 2, 2, 2, 2, 2, 2, 3, 3, 3, 2], [9, 9, 9, 9, 9, 8, 4, 4, 4, 8, 9, 9, 9, 9, 9, 9, 9, 2, 3, 3, 3, 2], [9, 9, 9, 9, 9, 8, 4, 4, 4, 8, 9, 9, 9, 9, 9, 9, 9, 2, 2, 2, 2, 2], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [8, 1, 1, 1, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [8, 1, 1, 1, 8, 8, 7, 7, 7, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [8, 1, 1, 1, 8, 8, 7, 7, 7, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [8, 8, 8, 8, 8, 8, 7, 7, 7, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 2, 2, 2, 2, 2, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 2, 6, 6, 6, 2, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 2, 6, 6, 6, 2, 2, 2, 2, 2, 2], [8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 2, 6, 6, 6, 2, 2, 3, 3, 3, 2], [8, 3, 3, 3, 8, 9, 9, 9, 9, 9, 9, 9, 2, 2, 2, 2, 2, 2, 3, 3, 3, 2], [8, 3, 3, 3, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 2, 3, 3, 3, 2], [8, 3, 3, 3, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 2, 2, 2, 2, 2], [8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9]], "attempt_2": [[9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 2, 2, 2, 2, 2, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 2, 4, 4, 4, 2, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 2, 4, 4, 4, 2, 2, 2, 2, 2, 2], [9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 9, 9, 2, 4, 4, 4, 2, 2, 3, 3, 3, 2], [9, 9, 9, 9, 9, 8, 4, 4, 4, 8, 9, 9, 2, 2, 2, 2, 2, 2, 3, 3, 3, 2], [9, 9, 9, 9, 9, 8, 4, 4, 4, 8, 9, 9, 9, 9, 9, 9, 9, 2, 3, 3, 3, 2], [9, 9, 9, 9, 9, 8, 4, 4, 4, 8, 9, 9, 9, 9, 9, 9, 9, 2, 2, 2, 2, 2], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [8, 1, 1, 1, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [8, 1, 1, 1, 8, 8, 7, 7, 7, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [8, 1, 1, 1, 8, 8, 7, 7, 7, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [8, 8, 8, 8, 8, 8, 7, 7, 7, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 2, 2, 2, 2, 2], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 2, 6, 6, 6, 2], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 2, 2, 2, 2, 2, 2, 6, 6, 6, 2], [8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 2, 3, 3, 3, 2, 2, 6, 6, 6, 2], [8, 3, 3, 3, 8, 9, 9, 9, 9, 9, 9, 9, 2, 3, 3, 3, 2, 2, 2, 2, 2, 2], [8, 3, 3, 3, 8, 9, 9, 9, 9, 9, 9, 9, 2, 3, 3, 3, 2, 9, 9, 9, 9, 9], [8, 3, 3, 3, 8, 9, 9, 9, 9, 9, 9, 9, 2, 2, 2, 2, 2, 9, 9, 9, 9, 9], [8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9]]}], "b6f77b65": [{"attempt_1": [[3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 4, 0, 0, 0, 8, 7, 7, 7, 0, 0], [0, 0, 4, 0, 0, 0, 8, 0, 0, 7, 0, 0], [0, 0, 4, 0, 0, 0, 8, 0, 0, 7, 0, 0], [0, 0, 6, 5, 5, 5, 5, 5, 0, 7, 0, 0], [0, 0, 6, 0, 0, 0, 0, 5, 0, 7, 0, 0], [0, 0, 6, 0, 0, 0, 0, 5, 0, 7, 0, 0], [0, 0, 1, 1, 1, 0, 2, 2, 2, 2, 9, 0], [0, 0, 0, 0, 1, 0, 2, 0, 0, 0, 9, 0], [0, 0, 0, 0, 1, 0, 2, 0, 0, 0, 9, 0]], "attempt_2": [[3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 4, 0, 0, 0, 8, 7, 7, 7, 0, 0], [0, 0, 4, 0, 0, 0, 8, 0, 0, 7, 0, 0], [0, 0, 4, 0, 0, 0, 8, 0, 0, 7, 0, 0], [0, 0, 6, 5, 5, 5, 5, 5, 0, 7, 0, 0], [0, 0, 6, 0, 1, 0, 2, 5, 0, 7, 9, 0], [0, 0, 6, 0, 1, 0, 2, 5, 0, 7, 9, 0], [0, 0, 1, 1, 1, 0, 2, 2, 2, 2, 9, 0]]}, {"attempt_1": [[7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 4, 3, 3, 0, 0, 0, 0, 0, 0, 0], [0, 0, 4, 0, 3, 0, 8, 0, 0, 0, 0, 0], [0, 0, 4, 0, 3, 0, 8, 0, 0, 0, 0, 0], [0, 0, 4, 0, 3, 0, 8, 0, 0, 0, 0, 0], [0, 0, 6, 5, 5, 5, 5, 5, 0, 0, 0, 0], [0, 0, 6, 0, 0, 0, 0, 5, 0, 0, 0, 0], [0, 0, 6, 0, 0, 0, 0, 5, 0, 0, 0, 0], [0, 3, 1, 1, 1, 0, 2, 2, 2, 2, 9, 0], [0, 3, 0, 0, 1, 0, 2, 0, 0, 0, 9, 0], [0, 3, 0, 0, 1, 0, 2, 0, 0, 0, 9, 0]], "attempt_2": [[7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 4, 3, 3, 0, 0, 0, 0, 0, 0, 0], [0, 0, 4, 0, 3, 0, 8, 0, 0, 0, 0, 0], [0, 0, 4, 0, 3, 0, 8, 0, 0, 0, 0, 0], [0, 0, 4, 0, 3, 0, 8, 0, 0, 0, 0, 0], [0, 0, 6, 5, 5, 5, 5, 5, 0, 0, 0, 0], [0, 3, 6, 0, 1, 0, 2, 5, 0, 0, 9, 0], [0, 3, 6, 0, 1, 0, 2, 5, 0, 0, 9, 0], [0, 3, 1, 1, 1, 0, 2, 2, 2, 2, 9, 0]]}, {"attempt_1": [[2, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 4, 3, 3, 0, 0, 0, 0, 0, 0, 0], [0, 0, 4, 0, 3, 0, 8, 7, 7, 7, 0, 0], [0, 0, 4, 0, 3, 0, 8, 0, 0, 7, 0, 0], [0, 0, 4, 0, 3, 0, 8, 0, 0, 7, 0, 0], [0, 0, 6, 5, 5, 5, 5, 5, 0, 7, 0, 0], [0, 0, 6, 0, 0, 0, 0, 5, 0, 7, 0, 0], [0, 0, 6, 0, 0, 0, 0, 5, 0, 7, 0, 0], [0, 3, 1, 1, 1, 0, 0, 0, 0, 0, 9, 0], [0, 3, 0, 0, 1, 0, 0, 0, 0, 0, 9, 0], [0, 3, 0, 0, 1, 0, 0, 0, 0, 0, 9, 0]], "attempt_2": [[2, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 4, 3, 3, 0, 0, 0, 0, 0, 0, 0], [0, 0, 4, 0, 3, 0, 8, 7, 7, 7, 0, 0], [0, 0, 4, 0, 3, 0, 8, 0, 0, 7, 0, 0], [0, 0, 4, 0, 3, 0, 8, 0, 0, 7, 0, 0], [0, 3, 6, 5, 5, 5, 5, 5, 0, 7, 9, 0], [0, 3, 6, 0, 0, 0, 0, 5, 0, 7, 9, 0], [0, 3, 6, 0, 0, 0, 0, 5, 0, 7, 9, 0]]}], "b9e38dc0": [{"attempt_1": [[2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [2, 2, 2, 2, 2, 2, 2, 2, 4, 4, 1, 1, 1, 1, 4, 4, 4, 1, 1], [1, 1, 1, 1, 3, 2, 2, 2, 2, 2, 4, 4, 1, 4, 2, 2, 4, 1, 1], [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 4, 2, 2, 2, 4, 1, 1], [1, 1, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 4, 1, 1], [1, 1, 1, 1, 1, 1, 3, 2, 2, 2, 2, 2, 4, 4, 2, 2, 4, 1, 1], [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 4, 4, 1, 4, 4, 4, 1, 1, 1], [2, 2, 2, 2, 2, 2, 2, 2, 4, 4, 1, 1, 1, 1, 1, 1, 1, 1, 1], [2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], "attempt_2": [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 4, 4, 1, 1, 1, 1, 4, 4, 4, 1, 1], [1, 1, 1, 1, 3, 1, 1, 1, 2, 2, 4, 4, 1, 4, 2, 2, 4, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 4, 2, 2, 2, 4, 1, 1], [1, 1, 8, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 4, 1, 1], [1, 1, 1, 1, 1, 1, 3, 1, 2, 2, 2, 2, 4, 4, 2, 2, 4, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 4, 4, 1, 4, 4, 4, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 4, 4, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]}], "c4d067a0": [{"attempt_1": [[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 1, 3, 1, 3, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 4, 3, 4, 3, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 1, 3, 4, 3, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 8, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 3, 2, 2, 2, 2, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 3, 2, 2, 2, 2, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 3, 2, 2, 2, 2, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 3, 2, 2, 2, 2, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]], "attempt_2": [[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 1, 3, 1, 3, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 4, 3, 4, 3, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 1, 3, 4, 3, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 8, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 3, 2, 2, 2, 2, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 3, 2, 2, 2, 2, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 3, 2, 2, 2, 2, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 3, 2, 2, 2, 2, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 3, 1, 1, 1, 1, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 3, 1, 1, 1, 1, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 3, 1, 1, 1, 1, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 3, 1, 1, 1, 1, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]]}], "c7f57c3e": [{"attempt_1": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 1, 3, 3, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 3, 3, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 8, 8, 8, 8, 8, 8], [8, 8, 1, 1, 1, 3, 3, 3, 3, 3, 3, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 8, 8, 8, 8, 8, 8], [8, 8, 1, 1, 1, 3, 3, 3, 3, 3, 3, 1, 1, 1, 8, 8, 8, 8, 8, 1, 3, 3, 1, 8, 8, 8, 8, 8], [8, 8, 1, 1, 1, 3, 3, 3, 3, 3, 3, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]], "attempt_2": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 1, 3, 3, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 3, 3, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 8, 8, 8, 8, 8, 8], [8, 8, 1, 1, 1, 6, 6, 6, 6, 6, 6, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 8, 8, 8, 8, 8, 8], [8, 8, 1, 1, 1, 6, 6, 6, 6, 6, 6, 1, 1, 1, 8, 8, 8, 8, 8, 1, 3, 3, 1, 8, 8, 8, 8, 8], [8, 8, 1, 1, 1, 6, 6, 6, 6, 6, 6, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 4, 4, 4, 4, 4, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 4, 4, 4, 4, 4, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 4, 4, 4, 4, 4, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]]}], "cbebaa4b": [{"attempt_1": [[0]], "attempt_2": [[0]]}, {"attempt_1": [[0]], "attempt_2": [[0]]}], "d35bdbdc": [{"attempt_1": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 4, 0, 0, 0, 0, 0, 0, 0], [0, 4, 3, 4, 0, 0, 0, 0, 0, 0], [0, 0, 4, 0, 0, 0, 0, 5, 0, 0], [0, 0, 0, 0, 0, 0, 5, 0, 0, 0], [0, 0, 0, 0, 0, 0, 5, 0, 3, 0], [0, 7, 0, 0, 0, 0, 5, 3, 3, 3], [7, 9, 7, 0, 0, 0, 5, 0, 3, 0], [0, 7, 0, 5, 0, 5, 0, 0, 0, 0], [0, 5, 5, 0, 5, 0, 0, 0, 0, 0]], "attempt_2": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 4, 0, 0, 0, 0, 0, 0, 0], [0, 4, 3, 4, 0, 0, 0, 0, 0, 0], [0, 0, 4, 0, 0, 0, 0, 5, 0, 0], [0, 0, 0, 0, 0, 0, 5, 0, 0, 0], [0, 0, 0, 0, 0, 0, 5, 0, 0, 0], [0, 7, 0, 0, 0, 0, 5, 0, 0, 0], [7, 9, 7, 0, 0, 0, 5, 0, 0, 0], [0, 7, 0, 5, 0, 5, 0, 0, 0, 0], [0, 5, 5, 0, 5, 0, 0, 0, 0, 0]]}, {"attempt_1": [[0, 0, 0, 4, 4, 4, 0, 0, 0, 0], [0, 0, 5, 4, 3, 4, 5, 0, 0, 0], [0, 0, 5, 4, 4, 4, 5, 0, 0, 0], [0, 0, 0, 5, 0, 0, 5, 5, 5, 5], [0, 0, 5, 1, 1, 1, 0, 0, 0, 5], [0, 5, 0, 1, 6, 1, 5, 5, 5, 0], [0, 0, 5, 1, 1, 1, 5, 0, 0, 0], [0, 0, 0, 5, 0, 0, 5, 0, 0, 0], [0, 0, 0, 5, 5, 5, 5, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], "attempt_2": [[0, 0, 0, 4, 4, 4, 0, 2, 2, 2], [0, 0, 5, 4, 3, 4, 5, 2, 6, 2], [0, 0, 5, 4, 4, 4, 5, 2, 2, 2], [0, 0, 0, 5, 0, 0, 5, 5, 5, 5], [0, 0, 5, 1, 1, 1, 0, 0, 0, 5], [0, 5, 0, 1, 6, 1, 5, 5, 5, 0], [0, 0, 5, 1, 1, 1, 5, 0, 0, 0], [0, 0, 0, 5, 0, 0, 5, 0, 0, 0], [0, 0, 0, 5, 5, 5, 5, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}, {"attempt_1": [[0, 0, 0, 0, 0, 0, 0, 6, 6, 6], [0, 0, 0, 0, 0, 0, 0, 6, 3, 6], [0, 0, 0, 0, 0, 0, 0, 6, 6, 6], [5, 5, 5, 5, 5, 0, 0, 0, 5, 0], [5, 0, 0, 0, 5, 0, 0, 0, 5, 0], [5, 0, 0, 0, 5, 0, 0, 0, 5, 0], [5, 0, 0, 0, 5, 0, 0, 0, 0, 5], [0, 5, 5, 0, 5, 0, 0, 0, 0, 5], [0, 7, 7, 7, 0, 5, 5, 0, 0, 5], [0, 7, 4, 7, 0, 0, 0, 5, 5, 0]], "attempt_2": [[0, 0, 0, 1, 1, 1, 0, 6, 6, 6], [0, 0, 0, 1, 9, 1, 0, 6, 3, 6], [0, 0, 0, 1, 1, 1, 0, 6, 6, 6], [5, 5, 5, 5, 5, 0, 0, 0, 5, 0], [5, 0, 0, 0, 5, 0, 0, 0, 5, 0], [5, 0, 0, 0, 5, 0, 0, 0, 5, 0], [5, 0, 0, 0, 5, 0, 0, 0, 0, 5], [0, 5, 5, 0, 5, 0, 0, 0, 0, 5], [0, 7, 7, 7, 0, 5, 5, 0, 0, 5], [0, 7, 4, 7, 0, 0, 0, 5, 5, 0]]}], "d59b0160": [{"attempt_1": [[7, 2, 7, 3, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [7, 7, 5, 3, 7, 7, 7, 7, 7, 7, 0, 0, 0, 0, 0, 7], [7, 4, 7, 3, 7, 7, 7, 7, 7, 7, 4, 0, 5, 0, 0, 7], [3, 3, 3, 3, 7, 7, 7, 7, 7, 7, 4, 0, 8, 0, 0, 7], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [9, 0, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [0, 8, 7, 0, 0, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [0, 0, 7, 0, 0, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [0, 0, 7, 0, 0, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [0, 0, 7, 0, 5, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [6, 0, 7, 0, 5, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [0, 0, 7, 5, 0, 7, 7, 7, 7, 7, 7, 0, 0, 0, 0, 0], [5, 0, 7, 5, 0, 7, 7, 7, 7, 7, 7, 0, 0, 9, 0, 1], [0, 0, 7, 0, 5, 7, 7, 7, 7, 7, 7, 1, 0, 4, 0, 0], [5, 0, 7, 0, 4, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [0, 0, 7, 0, 2, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7]], "attempt_2": [[7, 2, 7, 3, 7, 7, 0, 0, 0, 7, 7, 7, 7, 7, 7, 7], [7, 7, 5, 3, 7, 7, 0, 0, 6, 7, 7, 7, 7, 7, 7, 7], [7, 4, 7, 3, 7, 7, 0, 0, 1, 7, 7, 7, 7, 7, 7, 7], [3, 3, 3, 3, 7, 7, 0, 0, 0, 7, 7, 7, 7, 7, 7, 7], [7, 7, 7, 7, 7, 7, 0, 0, 0, 7, 7, 7, 7, 7, 7, 7], [7, 7, 7, 7, 7, 7, 0, 0, 0, 7, 9, 0, 0, 0, 0, 0], [7, 7, 7, 0, 0, 7, 0, 5, 0, 7, 0, 0, 0, 6, 5, 0], [7, 7, 7, 0, 0, 7, 2, 0, 4, 7, 0, 2, 0, 0, 0, 0], [7, 7, 7, 0, 0, 7, 0, 0, 0, 7, 0, 0, 4, 0, 8, 0], [7, 7, 7, 0, 5, 7, 0, 1, 0, 7, 0, 9, 4, 0, 5, 0], [7, 7, 7, 0, 5, 7, 0, 0, 9, 7, 7, 7, 7, 7, 7, 7], [7, 7, 7, 5, 0, 7, 0, 0, 1, 7, 7, 7, 7, 7, 7, 7], [7, 7, 7, 5, 0, 7, 0, 9, 0, 7, 7, 7, 7, 7, 7, 7], [7, 7, 7, 0, 5, 7, 0, 9, 8, 7, 7, 7, 7, 7, 7, 7], [7, 7, 7, 0, 4, 7, 0, 0, 0, 7, 7, 7, 7, 7, 7, 7], [7, 7, 7, 0, 2, 7, 0, 0, 5, 7, 7, 7, 7, 7, 7, 7]]}], "d8e07eb2": [{"attempt_1": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 0, 0, 0, 8, 8, 1, 8, 1, 8, 8, 9, 8, 9, 8, 8, 2, 2, 2, 8, 8], [8, 8, 0, 0, 0, 8, 8, 1, 1, 8, 8, 8, 9, 9, 8, 8, 8, 2, 8, 2, 8, 8], [8, 8, 0, 0, 0, 8, 8, 1, 8, 1, 8, 8, 9, 9, 9, 8, 8, 2, 2, 2, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 3, 3, 3, 8], [8, 8, 2, 8, 2, 8, 8, 0, 8, 8, 8, 8, 8, 8, 7, 8, 3, 9, 3, 9, 3, 8], [8, 8, 8, 2, 8, 8, 8, 0, 0, 0, 8, 8, 8, 8, 7, 8, 3, 9, 9, 3, 3, 8], [8, 8, 2, 8, 2, 8, 8, 0, 8, 0, 8, 8, 7, 7, 7, 8, 3, 9, 9, 9, 3, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 3, 3, 3, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 3, 3, 3, 8], [8, 8, 7, 8, 8, 8, 8, 1, 1, 1, 8, 8, 8, 6, 6, 8, 3, 0, 0, 0, 3, 8], [8, 8, 7, 7, 7, 8, 8, 8, 1, 8, 8, 8, 8, 8, 6, 8, 3, 0, 0, 0, 3, 8], [8, 8, 7, 8, 8, 8, 8, 1, 1, 1, 8, 8, 8, 6, 6, 8, 3, 0, 0, 0, 3, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 3, 3, 3, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 3, 3, 3, 8], [8, 8, 4, 8, 4, 8, 8, 2, 2, 8, 8, 8, 5, 5, 8, 8, 3, 1, 3, 1, 3, 8], [8, 8, 4, 8, 4, 8, 8, 2, 2, 2, 8, 8, 5, 8, 8, 8, 3, 1, 1, 3, 3, 8], [8, 8, 4, 4, 4, 8, 8, 8, 2, 8, 8, 8, 5, 5, 8, 8, 3, 1, 3, 1, 3, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 3, 3, 3, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 3, 3, 3, 8], [8, 8, 9, 8, 8, 8, 8, 6, 6, 6, 8, 8, 8, 4, 8, 8, 3, 2, 2, 2, 3, 8], [8, 8, 9, 9, 9, 8, 8, 6, 8, 6, 8, 8, 4, 4, 4, 8, 3, 2, 3, 2, 3, 8], [8, 8, 8, 8, 9, 8, 8, 6, 8, 6, 8, 8, 8, 4, 8, 8, 3, 2, 2, 2, 3, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 3, 3, 3, 8], [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6], [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]], "attempt_2": [[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 0, 0, 0, 3, 3, 1, 3, 1, 3, 3, 9, 3, 9, 3, 3, 2, 2, 2, 3, 3], [3, 3, 0, 0, 0, 3, 3, 1, 1, 3, 3, 3, 9, 9, 3, 3, 3, 2, 3, 2, 3, 3], [3, 3, 0, 0, 0, 3, 3, 1, 3, 1, 3, 3, 9, 9, 9, 3, 3, 2, 2, 2, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 3, 3, 3, 8], [8, 8, 2, 8, 2, 8, 8, 0, 8, 8, 8, 8, 8, 8, 7, 8, 3, 9, 3, 9, 3, 8], [8, 8, 8, 2, 8, 8, 8, 0, 0, 0, 8, 8, 8, 8, 7, 8, 3, 9, 9, 3, 3, 8], [8, 8, 2, 8, 2, 8, 8, 0, 8, 0, 8, 8, 7, 7, 7, 8, 3, 9, 9, 9, 3, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 3, 3, 3, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 3, 3, 3, 8], [8, 8, 7, 8, 8, 8, 8, 1, 1, 1, 8, 8, 8, 6, 6, 8, 3, 0, 0, 0, 3, 8], [8, 8, 7, 7, 7, 8, 8, 8, 1, 8, 8, 8, 8, 8, 6, 8, 3, 0, 0, 0, 3, 8], [8, 8, 7, 8, 8, 8, 8, 1, 1, 1, 8, 8, 8, 6, 6, 8, 3, 0, 0, 0, 3, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 3, 3, 3, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 3, 3, 3, 8], [8, 8, 4, 8, 4, 8, 8, 2, 2, 8, 8, 8, 5, 5, 8, 8, 3, 1, 3, 1, 3, 8], [8, 8, 4, 8, 4, 8, 8, 2, 2, 2, 8, 8, 5, 8, 8, 8, 3, 1, 1, 3, 3, 8], [8, 8, 4, 4, 4, 8, 8, 8, 2, 8, 8, 8, 5, 5, 8, 8, 3, 1, 3, 1, 3, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 3, 3, 3, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 3, 3, 3, 8], [8, 8, 9, 8, 8, 8, 8, 6, 6, 6, 8, 8, 8, 4, 8, 8, 3, 2, 2, 2, 3, 8], [8, 8, 9, 9, 9, 8, 8, 6, 8, 6, 8, 8, 4, 4, 4, 8, 3, 2, 3, 2, 3, 8], [8, 8, 8, 8, 9, 8, 8, 6, 8, 6, 8, 8, 8, 4, 8, 8, 3, 2, 2, 2, 3, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 3, 3, 3, 8], [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]]}, {"attempt_1": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 2, 8, 2, 8, 8, 7, 8, 8, 8, 8, 2, 2, 8, 8, 8, 5, 5, 8, 8, 8], [8, 8, 8, 2, 8, 8, 8, 7, 7, 7, 8, 8, 2, 2, 2, 8, 8, 5, 8, 8, 8, 8], [8, 8, 2, 8, 2, 8, 8, 7, 8, 8, 8, 8, 8, 2, 8, 8, 8, 5, 5, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 3, 3, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 3, 2, 3, 2, 3, 8, 0, 8, 8, 8, 8, 8, 8, 7, 8, 8, 9, 8, 9, 8, 8], [8, 3, 3, 2, 3, 3, 8, 0, 0, 0, 8, 8, 8, 8, 7, 8, 8, 9, 9, 8, 8, 8], [8, 3, 2, 3, 2, 3, 8, 0, 8, 0, 8, 8, 7, 7, 7, 8, 8, 9, 9, 9, 8, 8], [8, 3, 3, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 3, 3, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 3, 7, 3, 3, 3, 8, 1, 1, 1, 8, 8, 8, 6, 6, 8, 8, 0, 0, 0, 8, 8], [8, 3, 7, 7, 7, 3, 8, 8, 1, 8, 8, 8, 8, 8, 6, 8, 8, 0, 0, 0, 8, 8], [8, 3, 7, 3, 3, 3, 8, 1, 1, 1, 8, 8, 8, 6, 6, 8, 8, 0, 0, 0, 8, 8], [8, 3, 3, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 8, 8, 8, 8, 8], [8, 8, 4, 8, 4, 8, 3, 2, 2, 3, 3, 3, 5, 5, 3, 3, 8, 1, 8, 1, 8, 8], [8, 8, 4, 8, 4, 8, 3, 2, 2, 2, 3, 3, 5, 3, 3, 3, 8, 1, 1, 8, 8, 8], [8, 8, 4, 4, 4, 8, 3, 3, 2, 3, 3, 3, 5, 5, 3, 3, 8, 1, 8, 1, 8, 8], [8, 8, 8, 8, 8, 8, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 9, 8, 8, 8, 8, 6, 6, 6, 8, 8, 8, 4, 8, 8, 8, 2, 2, 2, 8, 8], [8, 8, 9, 9, 9, 8, 8, 6, 8, 6, 8, 8, 4, 4, 4, 8, 8, 2, 8, 2, 8, 8], [8, 8, 8, 8, 9, 8, 8, 6, 8, 6, 8, 8, 8, 4, 8, 8, 8, 2, 2, 2, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6], [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]], "attempt_2": [[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 2, 3, 2, 3, 3, 7, 3, 3, 3, 3, 2, 2, 3, 3, 3, 5, 5, 3, 3, 3], [3, 3, 3, 2, 3, 3, 3, 7, 7, 7, 3, 3, 2, 2, 2, 3, 3, 5, 3, 3, 3, 3], [3, 3, 2, 3, 2, 3, 3, 7, 3, 3, 3, 3, 3, 2, 3, 3, 3, 5, 5, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 3, 3, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 3, 2, 3, 2, 3, 8, 0, 8, 8, 8, 8, 8, 8, 7, 8, 8, 9, 8, 9, 8, 8], [8, 3, 3, 2, 3, 3, 8, 0, 0, 0, 8, 8, 8, 8, 7, 8, 8, 9, 9, 8, 8, 8], [8, 3, 2, 3, 2, 3, 8, 0, 8, 0, 8, 8, 7, 7, 7, 8, 8, 9, 9, 9, 8, 8], [8, 3, 3, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 3, 3, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 3, 7, 3, 3, 3, 8, 1, 1, 1, 8, 8, 8, 6, 6, 8, 8, 0, 0, 0, 8, 8], [8, 3, 7, 7, 7, 3, 8, 8, 1, 8, 8, 8, 8, 8, 6, 8, 8, 0, 0, 0, 8, 8], [8, 3, 7, 3, 3, 3, 8, 1, 1, 1, 8, 8, 8, 6, 6, 8, 8, 0, 0, 0, 8, 8], [8, 3, 3, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 8, 8, 8, 8, 8], [8, 8, 4, 8, 4, 8, 3, 2, 2, 3, 3, 3, 5, 5, 3, 3, 8, 1, 8, 1, 8, 8], [8, 8, 4, 8, 4, 8, 3, 2, 2, 2, 3, 3, 5, 3, 3, 3, 8, 1, 1, 8, 8, 8], [8, 8, 4, 4, 4, 8, 3, 3, 2, 3, 3, 3, 5, 5, 3, 3, 8, 1, 8, 1, 8, 8], [8, 8, 8, 8, 8, 8, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 9, 8, 8, 8, 8, 6, 6, 6, 8, 8, 8, 4, 8, 8, 8, 2, 2, 2, 8, 8], [8, 8, 9, 9, 9, 8, 8, 6, 8, 6, 8, 8, 4, 4, 4, 8, 8, 2, 8, 2, 8, 8], [8, 8, 8, 8, 9, 8, 8, 6, 8, 6, 8, 8, 8, 4, 8, 8, 8, 2, 2, 2, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]]}], "da515329": [{"attempt_1": [[8, 0, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [8, 0, 8, 0, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0], [0, 0, 8, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0], [8, 8, 8, 0, 8, 0, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 8, 0], [0, 0, 0, 0, 8, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 8, 0], [8, 8, 8, 8, 8, 0, 8, 0, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 8, 0, 8, 0], [8, 0, 0, 0, 0, 0, 8, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 8, 0, 8, 0], [8, 0, 8, 8, 8, 8, 8, 0, 8, 0, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 8, 0, 8, 0, 8, 0], [8, 0, 8, 0, 0, 0, 0, 0, 8, 0, 8, 0, 0, 0, 0, 0, 0, 0, 8, 0, 8, 0, 8, 0, 8, 0], [8, 0, 8, 0, 8, 8, 8, 8, 8, 0, 8, 8, 8, 8, 8, 8, 8, 0, 8, 0, 8, 0, 8, 0, 8, 0], [8, 0, 8, 0, 8, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 8, 0, 8, 0, 8, 0, 8, 0, 8, 0], [8, 0, 8, 0, 8, 0, 8, 8, 8, 8, 0, 8, 8, 8, 8, 0, 8, 0, 8, 0, 8, 0, 8, 0, 8, 0], [8, 0, 8, 0, 8, 0, 8, 0, 8, 0, 8, 0, 8, 0, 8, 0, 8, 0, 8, 0, 8, 0, 8, 0, 8, 0], [8, 0, 8, 0, 8, 0, 8, 0, 8, 0, 8, 8, 8, 0, 8, 0, 8, 0, 8, 0, 8, 0, 8, 0, 8, 0], [8, 0, 8, 0, 8, 0, 8, 0, 8, 0, 8, 0, 0, 0, 8, 0, 8, 0, 8, 0, 8, 0, 8, 0, 8, 0], [8, 0, 8, 0, 8, 0, 8, 0, 8, 0, 8, 8, 8, 8, 8, 0, 8, 0, 8, 0, 8, 0, 8, 0, 8, 0], [8, 0, 8, 0, 8, 0, 8, 0, 8, 0, 0, 0, 0, 0, 0, 0, 8, 0, 8, 0, 8, 0, 8, 0, 8, 0], [8, 0, 8, 0, 8, 0, 8, 0, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 8, 0, 8, 0, 8, 0, 8, 0], [8, 0, 8, 0, 8, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 8, 0, 8, 0, 8, 0], [8, 0, 8, 0, 8, 0, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 8, 0, 8, 0, 8, 0], [8, 0, 8, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 8, 0, 8, 0], [8, 0, 8, 0, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 8, 0, 8, 0], [8, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 8, 0], [8, 0, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 8, 0], [8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0]], "attempt_2": [[8, 0, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 0, 0, 0, 0], [8, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 8, 8, 8, 8, 0], [8, 0, 8, 0, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 0, 0, 0, 0, 8, 0], [8, 0, 8, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 8, 8, 8, 8, 0, 8, 0], [8, 0, 8, 0, 8, 0, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 0, 0, 0, 0, 8, 0, 8, 0], [8, 0, 8, 0, 8, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 8, 8, 8, 8, 0, 8, 0, 8, 0], [8, 0, 8, 0, 8, 0, 8, 0, 8, 8, 8, 8, 8, 8, 8, 0, 0, 0, 0, 0, 8, 0, 8, 0, 8, 0], [8, 0, 8, 0, 8, 0, 8, 0, 8, 0, 0, 0, 0, 0, 8, 8, 8, 8, 8, 0, 8, 0, 8, 0, 8, 0], [8, 0, 8, 0, 8, 0, 8, 0, 8, 0, 8, 8, 8, 0, 0, 0, 0, 0, 8, 0, 8, 0, 8, 0, 8, 0], [8, 0, 8, 0, 8, 0, 8, 0, 8, 0, 8, 0, 8, 8, 8, 8, 8, 0, 8, 0, 8, 0, 8, 0, 8, 0], [8, 0, 8, 0, 8, 0, 8, 0, 8, 0, 8, 0, 0, 0, 0, 0, 8, 0, 8, 0, 8, 0, 8, 0, 8, 0], [8, 0, 8, 0, 8, 0, 8, 0, 8, 0, 8, 8, 8, 8, 8, 0, 8, 0, 8, 0, 8, 0, 8, 0, 8, 0], [8, 0, 8, 0, 8, 0, 8, 8, 8, 8, 0, 8, 8, 8, 8, 0, 8, 0, 8, 0, 8, 0, 8, 0, 8, 0], [8, 0, 8, 0, 8, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 8, 0, 8, 0, 8, 0, 8, 0, 8, 0], [8, 0, 8, 0, 8, 8, 8, 8, 8, 0, 8, 8, 8, 8, 8, 8, 8, 0, 8, 0, 8, 0, 8, 0, 8, 0], [8, 0, 8, 0, 0, 0, 0, 0, 8, 0, 8, 0, 0, 0, 0, 0, 0, 0, 8, 0, 8, 0, 8, 0, 8, 0], [8, 0, 8, 8, 8, 8, 8, 0, 8, 0, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 8, 0, 8, 0, 8, 0], [8, 0, 0, 0, 0, 0, 8, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 8, 0, 8, 0], [8, 8, 8, 8, 8, 0, 8, 0, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 8, 0, 8, 0], [0, 0, 0, 0, 8, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 8, 0], [8, 8, 8, 0, 8, 0, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 8, 0], [0, 0, 8, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0], [8, 0, 8, 0, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0], [8, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [8, 0, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}], "db0c5428": [{"attempt_1": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 4, 4, 7, 8, 8, 8, 7, 1, 7, 8, 8, 8, 7, 4, 4, 8], [8, 4, 7, 7, 8, 8, 8, 7, 7, 7, 8, 8, 8, 7, 7, 4, 8], [8, 7, 7, 3, 8, 8, 8, 7, 7, 7, 8, 8, 8, 3, 7, 7, 8], [8, 8, 8, 8, 3, 7, 7, 7, 7, 7, 7, 7, 3, 8, 8, 8, 8], [8, 8, 8, 8, 7, 7, 4, 7, 7, 7, 4, 7, 7, 8, 8, 8, 8], [8, 8, 8, 8, 7, 4, 4, 7, 1, 7, 4, 4, 7, 8, 8, 8, 8], [8, 7, 7, 7, 7, 7, 7, 4, 1, 4, 7, 7, 7, 7, 7, 7, 8], [8, 1, 7, 7, 7, 7, 1, 1, 7, 1, 1, 7, 7, 7, 7, 1, 8], [8, 7, 7, 7, 7, 7, 7, 4, 1, 4, 7, 7, 7, 7, 7, 7, 8], [8, 8, 8, 8, 7, 4, 4, 7, 1, 7, 4, 4, 7, 8, 8, 8, 8], [8, 8, 8, 8, 7, 7, 4, 7, 7, 7, 4, 7, 7, 8, 8, 8, 8], [8, 8, 8, 8, 3, 7, 7, 7, 7, 7, 7, 7, 3, 8, 8, 8, 8], [8, 7, 7, 3, 8, 8, 8, 7, 7, 7, 8, 8, 8, 3, 7, 7, 8], [8, 4, 7, 7, 8, 8, 8, 7, 7, 7, 8, 8, 8, 7, 7, 4, 8], [8, 4, 4, 7, 8, 8, 8, 7, 1, 7, 8, 8, 8, 7, 4, 4, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]], "attempt_2": [[0]]}], "db695cfb": [{"attempt_1": [[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 3, 3, 3, 3, 3, 3, 3, 1, 3, 3, 3], [3, 3, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 6, 3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 3, 3, 3, 3, 3, 3], [3, 3, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 6, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 6, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [6, 3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 6, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 6, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 6, 3], [3, 3, 3, 6, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 6, 3, 3, 3, 3, 3, 3, 6, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 6, 3, 3, 6, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 6, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 6, 3, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 6, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 1, 3, 6, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 1, 3, 3, 3, 6, 3, 6, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 1, 3, 3, 3, 3, 3, 6, 3, 3, 3, 3, 3, 3, 1, 3, 3], [3, 3, 3, 3, 1, 3, 3, 3, 3, 3, 3, 3, 6, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 1, 3, 3, 3, 3, 3, 3, 3, 1, 3, 6, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 6, 3, 3, 3, 3, 3, 3]], "attempt_2": [[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 3, 3, 3, 3, 3, 3, 3, 1, 3, 3, 6], [3, 3, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 6, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 6, 3, 3], [3, 3, 3, 3, 6, 3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 3, 3, 6, 3, 3, 3], [3, 3, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 6, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 6, 3, 3, 3, 3, 3, 3, 3, 3, 6, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 6, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 6, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 6, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 6, 3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 3, 3, 3, 3, 1, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 6, 3, 3, 3, 3, 3, 3, 1, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 6, 3, 3, 3, 3, 1, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 1, 3, 3, 3, 3, 3, 3, 3, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]]}], "dbff022c": [{"attempt_1": [[0, 1, 1, 1, 1, 1, 0, 0, 3, 3, 3, 3, 0], [0, 1, 0, 0, 0, 1, 0, 0, 3, 8, 8, 3, 0], [0, 1, 1, 0, 0, 1, 0, 0, 3, 8, 8, 3, 0], [0, 0, 1, 1, 1, 1, 0, 0, 3, 3, 3, 3, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 0, 0], [0, 0, 2, 0, 0, 0, 0, 5, 5, 5, 5, 5, 0], [0, 2, 0, 2, 0, 0, 0, 5, 5, 5, 5, 5, 5], [0, 0, 2, 0, 0, 0, 0, 0, 5, 5, 5, 5, 5], [0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0], [5, 4, 6, 7, 8, 0, 0, 0, 0, 2, 0, 2, 0], [9, 8, 3, 1, 2, 0, 0, 0, 0, 0, 2, 0, 0]], "attempt_2": [[0, 1, 1, 1, 1, 1, 0, 0, 3, 3, 3, 3, 0], [0, 1, 0, 0, 0, 1, 0, 0, 3, 0, 0, 3, 0], [0, 1, 1, 0, 0, 1, 0, 0, 3, 0, 0, 3, 0], [0, 0, 1, 1, 1, 1, 0, 0, 3, 3, 3, 3, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 0, 0], [0, 0, 2, 0, 0, 0, 0, 5, 5, 5, 5, 5, 0], [0, 2, 0, 2, 0, 0, 0, 5, 5, 5, 5, 5, 5], [0, 0, 2, 0, 0, 0, 0, 0, 5, 5, 5, 5, 5], [0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0], [5, 4, 6, 7, 8, 0, 0, 0, 0, 2, 0, 2, 0], [9, 8, 3, 1, 2, 0, 0, 0, 0, 0, 2, 0, 0]]}], "de809cff": [{"attempt_1": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 6, 1, 1, 1, 6, 6, 6, 6, 6, 1, 1, 1, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6], [0, 0, 0, 0, 0, 0, 0, 0, 6, 1, 8, 1, 6, 6, 6, 6, 6, 1, 8, 1, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6], [0, 0, 0, 0, 0, 0, 0, 0, 6, 1, 1, 1, 6, 6, 6, 6, 6, 1, 1, 1, 6, 6, 6, 6, 1, 1, 1, 6, 6, 6], [0, 0, 0, 0, 0, 0, 0, 0, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 1, 8, 1, 1, 6, 6], [0, 1, 1, 6, 6, 6, 1, 1, 1, 1, 1, 1, 6, 6, 6, 1, 1, 1, 1, 1, 1, 6, 6, 6, 1, 1, 8, 1, 6, 6], [0, 1, 1, 6, 8, 6, 1, 1, 1, 1, 1, 1, 6, 8, 6, 1, 1, 1, 1, 1, 1, 6, 6, 6, 6, 1, 1, 1, 1, 1], [0, 1, 1, 6, 6, 6, 1, 1, 1, 1, 1, 1, 6, 6, 6, 1, 1, 1, 1, 1, 1, 6, 6, 6, 6, 6, 6, 1, 8, 1], [0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 6, 6, 6, 6, 6, 6, 1, 1, 1], [0, 1, 1, 1, 1, 6, 6, 6, 1, 1, 1, 1, 1, 1, 1, 6, 6, 6, 1, 1, 1, 6, 6, 6, 6, 6, 6, 6, 6, 6], [0, 1, 1, 1, 1, 6, 8, 6, 1, 1, 1, 1, 1, 1, 1, 6, 8, 6, 1, 1, 1, 6, 6, 6, 6, 6, 6, 6, 6, 6], [0, 1, 1, 1, 1, 6, 6, 6, 1, 1, 1, 1, 1, 1, 1, 6, 8, 6, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0], [6, 6, 6, 1, 1, 1, 1, 1, 6, 6, 6, 1, 1, 1, 1, 6, 6, 6, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0], [6, 8, 6, 1, 1, 1, 1, 1, 6, 8, 6, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0], [6, 6, 6, 1, 1, 1, 1, 1, 6, 6, 6, 1, 1, 1, 1, 1, 1, 1, 1, 6, 6, 6, 0, 0, 0, 0, 0, 0, 0, 0], [0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 6, 8, 6, 0, 0, 0, 0, 0, 0, 0, 0], [0, 1, 6, 6, 6, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 6, 6, 6, 0, 0, 0, 0, 0, 0, 0, 0], [0, 1, 6, 8, 6, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 6, 6, 6, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 1, 6, 6, 6, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 6, 8, 6, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 1, 1, 1, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 0, 0, 0, 0], [0, 0, 0, 0, 6, 6, 6, 6, 6, 6, 6, 6, 6, 1, 1, 1, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 0, 0, 0, 0], [0, 0, 0, 0, 6, 6, 6, 6, 6, 6, 6, 6, 6, 1, 8, 1, 6, 6, 6, 6, 6, 1, 1, 1, 6, 6, 0, 0, 0, 0], [0, 0, 0, 0, 6, 1, 1, 1, 6, 6, 6, 6, 6, 1, 1, 1, 6, 6, 6, 6, 6, 1, 8, 1, 6, 6, 0, 0, 0, 0], [0, 0, 0, 0, 6, 1, 8, 1, 6, 6, 6, 6, 6, 6, 6, 6, 1, 1, 1, 6, 6, 1, 1, 1, 6, 6, 0, 0, 0, 0], [0, 0, 0, 0, 6, 1, 1, 1, 6, 6, 6, 6, 6, 6, 6, 6, 1, 8, 1, 6, 6, 1, 8, 1, 6, 6, 0, 0, 0, 0], [0, 0, 0, 1, 1, 1, 6, 6, 6, 6, 1, 1, 1, 6, 6, 6, 1, 1, 1, 6, 6, 1, 1, 1, 6, 6, 0, 0, 0, 0], [0, 0, 0, 1, 8, 1, 6, 6, 6, 6, 1, 8, 1, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 0, 0, 0, 0], [0, 0, 0, 1, 1, 1, 6, 6, 6, 6, 1, 1, 1, 6, 6, 6, 6, 6, 6, 6, 1, 1, 1, 1, 6, 6, 0, 0, 0, 0], [0, 0, 0, 0, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 1, 8, 8, 1, 6, 6, 0, 0, 0, 0], [0, 0, 0, 0, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 1, 1, 1, 1, 6, 6, 0, 0, 0, 0]], "attempt_2": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 6, 1, 1, 1, 6, 6, 6, 6, 6, 1, 1, 1, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6], [0, 0, 0, 0, 0, 0, 0, 0, 6, 1, 8, 1, 6, 6, 6, 6, 6, 1, 8, 1, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6], [0, 0, 0, 0, 0, 0, 0, 0, 6, 1, 1, 1, 6, 6, 6, 6, 6, 1, 1, 1, 6, 6, 6, 6, 1, 1, 1, 6, 6, 6], [0, 0, 0, 0, 0, 0, 0, 0, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 1, 8, 1, 1, 6, 6], [0, 1, 1, 6, 6, 6, 1, 1, 1, 1, 1, 1, 6, 6, 6, 1, 1, 1, 1, 1, 1, 6, 6, 6, 1, 1, 8, 1, 6, 6], [0, 1, 1, 6, 8, 6, 1, 1, 1, 1, 1, 1, 6, 8, 6, 1, 1, 1, 1, 1, 1, 6, 6, 6, 6, 1, 1, 1, 1, 1], [0, 1, 1, 6, 6, 6, 1, 1, 1, 1, 1, 1, 6, 6, 6, 1, 1, 1, 1, 1, 1, 6, 6, 6, 6, 6, 6, 1, 8, 1], [0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 6, 6, 6, 6, 6, 6, 1, 1, 1], [0, 1, 1, 1, 1, 6, 6, 6, 1, 1, 1, 1, 1, 1, 1, 6, 6, 6, 1, 1, 1, 6, 6, 6, 6, 6, 6, 6, 6, 6], [0, 1, 1, 1, 1, 6, 8, 6, 1, 1, 1, 1, 1, 1, 1, 6, 8, 6, 1, 1, 1, 6, 6, 6, 6, 6, 6, 6, 6, 6], [0, 1, 1, 1, 1, 6, 6, 6, 1, 1, 1, 1, 1, 1, 1, 6, 8, 6, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0], [6, 6, 6, 1, 1, 1, 1, 1, 6, 6, 6, 1, 1, 1, 1, 6, 6, 6, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0], [6, 8, 6, 1, 1, 1, 1, 1, 6, 8, 6, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0], [6, 6, 6, 1, 1, 1, 1, 1, 6, 6, 6, 1, 1, 1, 1, 1, 1, 1, 1, 6, 6, 6, 0, 0, 0, 0, 0, 0, 0, 0], [0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 6, 8, 6, 0, 0, 0, 0, 0, 0, 0, 0], [0, 1, 6, 6, 6, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 6, 6, 6, 0, 0, 0, 0, 0, 0, 0, 0], [0, 1, 6, 8, 6, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 6, 6, 6, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 1, 6, 6, 6, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 6, 8, 6, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 1, 1, 1, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 0, 0, 0, 0], [0, 0, 0, 0, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 0, 0, 0, 0], [0, 0, 0, 0, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 8, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 0, 0, 0, 0], [0, 0, 0, 0, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 8, 6, 6, 6, 0, 0, 0, 0], [0, 0, 0, 0, 6, 6, 8, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 0, 0, 0, 0], [0, 0, 0, 0, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 8, 6, 6, 6, 6, 8, 6, 6, 6, 0, 0, 0, 0], [0, 0, 0, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 0, 0, 0, 0], [0, 0, 0, 6, 8, 6, 6, 6, 6, 6, 6, 8, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 0, 0, 0, 0], [0, 0, 0, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 0, 0, 0, 0], [0, 0, 0, 0, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 8, 8, 6, 6, 6, 0, 0, 0, 0], [0, 0, 0, 0, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 0, 0, 0, 0]]}], "dfadab01": [{"attempt_1": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [7, 0, 0, 7, 7, 0, 0, 7, 7, 0, 0, 7, 7, 0, 0, 7, 7, 0, 0, 7], [0, 7, 7, 0, 0, 7, 7, 0, 0, 7, 7, 0, 0, 7, 7, 0, 0, 7, 7, 0], [0, 7, 7, 0, 0, 7, 7, 0, 0, 7, 7, 0, 0, 7, 7, 0, 0, 7, 7, 0], [7, 0, 0, 7, 7, 0, 0, 7, 7, 0, 0, 7, 7, 0, 0, 7, 7, 0, 0, 7], [7, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0, 0, 7], [0, 7, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 7, 0], [0, 7, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 7, 0], [7, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0, 0, 7], [7, 0, 0, 7, 0, 0, 0, 0, 7, 0, 0, 7, 0, 0, 0, 0, 7, 0, 0, 7], [0, 7, 7, 0, 0, 0, 0, 0, 0, 7, 7, 0, 0, 0, 0, 0, 0, 7, 7, 0], [0, 7, 7, 0, 0, 0, 0, 0, 0, 7, 7, 0, 0, 0, 0, 0, 0, 7, 7, 0], [7, 0, 0, 7, 0, 0, 0, 0, 7, 0, 0, 7, 0, 0, 0, 0, 7, 0, 0, 7], [7, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0, 0, 7, 7, 0, 0, 7], [0, 7, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 7, 0, 0, 7, 7, 0], [0, 7, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 7, 0, 0, 7, 7, 0], [7, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0, 0, 7, 7, 0, 0, 7], [7, 0, 0, 7, 7, 0, 0, 7, 7, 0, 0, 7, 7, 0, 0, 7, 7, 0, 0, 7], [0, 7, 7, 0, 0, 7, 7, 0, 0, 7, 7, 0, 0, 7, 7, 0, 0, 7, 7, 0], [0, 7, 7, 0, 0, 7, 7, 0, 0, 7, 7, 0, 0, 7, 7, 0, 0, 7, 7, 0]], "attempt_2": [[7, 0, 0, 7, 7, 0, 0, 7, 7, 0, 0, 7, 7, 0, 0, 7, 7, 0, 0, 7], [0, 7, 7, 0, 0, 7, 7, 0, 0, 7, 7, 0, 0, 7, 7, 0, 0, 7, 7, 0], [0, 7, 7, 0, 0, 7, 7, 0, 0, 7, 7, 0, 0, 7, 7, 0, 0, 7, 7, 0], [7, 0, 0, 7, 7, 0, 0, 7, 7, 0, 0, 7, 7, 0, 0, 7, 7, 0, 0, 7], [7, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0, 0, 7], [0, 7, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 7, 0], [0, 7, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 7, 0], [7, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0, 0, 7], [7, 0, 0, 7, 0, 0, 0, 0, 7, 0, 0, 7, 0, 0, 0, 0, 7, 0, 0, 7], [0, 7, 7, 0, 0, 0, 0, 0, 0, 7, 7, 0, 0, 0, 0, 0, 0, 7, 7, 0], [0, 7, 7, 0, 0, 0, 0, 0, 0, 7, 7, 0, 0, 0, 0, 0, 0, 7, 7, 0], [7, 0, 0, 7, 0, 0, 0, 0, 7, 0, 0, 7, 0, 0, 0, 0, 7, 0, 0, 7], [7, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0, 0, 7, 7, 0, 0, 7], [0, 7, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 7, 0, 0, 7, 7, 0], [0, 7, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 7, 0, 0, 7, 7, 0], [7, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0, 0, 7, 7, 0, 0, 7], [7, 0, 0, 7, 7, 0, 0, 7, 7, 0, 0, 7, 7, 0, 0, 7, 7, 0, 0, 7], [0, 7, 7, 0, 0, 7, 7, 0, 0, 7, 7, 0, 0, 7, 7, 0, 0, 7, 7, 0], [0, 7, 7, 0, 0, 7, 7, 0, 0, 7, 7, 0, 0, 7, 7, 0, 0, 7, 7, 0], [7, 0, 0, 7, 7, 0, 0, 7, 7, 0, 0, 7, 7, 0, 0, 7, 7, 0, 0, 7]]}], "e12f9a14": [{"attempt_1": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 8, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 2, 2, 8, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 2, 2, 8, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 8, 8, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 8, 8, 0, 0, 0, 0, 8, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 8, 3, 3, 8, 0, 0, 8, 4, 4, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 8, 3, 3, 8, 0, 0, 8, 4, 4, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 8, 0, 8, 0, 0, 0, 0, 8, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 8, 8, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 7, 7, 8, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 7, 7, 7, 7, 7, 7], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 7, 8, 8, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0]], "attempt_2": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 8, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 2, 2, 8, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 2, 2, 8, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 8, 8, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 8, 8, 0, 0, 0, 0, 8, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 8, 3, 3, 8, 0, 0, 8, 4, 4, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 8, 3, 3, 8, 0, 0, 8, 4, 4, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 8, 0, 8, 0, 0, 0, 0, 8, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 8, 8, 7, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 7, 7, 8, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 7, 7, 7, 7, 7, 7], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 7, 8, 8, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0]]}, {"attempt_1": [[8, 8, 9, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 8, 6], [8, 8, 9, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 8, 6, 8], [8, 8, 9, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 8, 6, 8, 8], [8, 8, 9, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 8, 6, 8, 8, 8], [8, 8, 9, 3, 2, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 3, 8, 8, 8, 8, 8, 4, 8, 6, 8, 8, 8, 8], [8, 8, 9, 2, 2, 2, 3, 8, 8, 8, 8, 8, 8, 8, 3, 4, 4, 4, 4, 4, 4, 4, 4, 8, 6, 8, 8, 8, 8, 8], [8, 8, 9, 3, 2, 2, 3, 8, 8, 8, 8, 8, 8, 8, 3, 4, 4, 3, 8, 8, 8, 8, 8, 6, 8, 8, 8, 8, 8, 8], [8, 8, 9, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 3, 8, 3, 3, 8, 8, 8, 8, 8, 6, 8, 8, 8, 8, 8, 8], [8, 8, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 8, 8, 8, 8, 8, 8], [8, 8, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 8, 8, 8, 8, 8, 6], [8, 8, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 8, 8, 8, 8, 6, 8], [8, 8, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 8, 8, 8, 6, 8, 8], [8, 8, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 8, 8, 6, 8, 8, 8], [8, 8, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 6, 3, 6, 8, 8, 8, 8], [8, 8, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 6, 6, 3, 8, 8, 8, 8], [8, 8, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 6, 6, 3, 8, 8, 8, 8], [8, 8, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 3, 8, 8, 8, 8, 8], [8, 8, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 3, 3, 8, 8, 8, 8, 8], [8, 8, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 7, 7, 7, 7, 7, 7, 7, 7], [8, 8, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 7, 7, 3, 8, 8, 8, 8, 8], [8, 8, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 3, 8, 8, 8, 8, 8, 8], [8, 8, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [3, 3, 9, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [3, 9, 9, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [3, 9, 9, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [3, 3, 3, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]], "attempt_2": [[8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 8], [8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 8, 6], [8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 8, 6, 8], [8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 8, 6, 8, 8], [8, 8, 8, 3, 2, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 3, 8, 8, 8, 8, 8, 8, 4, 8, 6, 8, 8, 8], [2, 2, 2, 2, 2, 2, 3, 8, 8, 8, 8, 8, 8, 8, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 8, 6, 8, 8, 8, 8], [8, 8, 8, 3, 2, 2, 3, 8, 8, 8, 8, 8, 8, 8, 3, 4, 4, 3, 8, 8, 8, 8, 8, 8, 6, 8, 8, 8, 8, 8], [8, 8, 8, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 3, 4, 3, 3, 8, 8, 8, 8, 8, 6, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 8, 8, 8, 8, 8, 8, 8, 6, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 8, 8, 8, 8, 8, 8, 8, 6, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 8, 8, 8, 8, 8, 8, 8, 6, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 8, 8, 8, 8, 8, 8, 8, 6, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 8, 8, 8, 8, 8, 8, 8, 6, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 8, 8, 8, 8, 8, 8, 3, 6, 3, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 8, 8, 8, 8, 8, 8, 3, 6, 6, 3, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 8, 8, 8, 8, 8, 8, 3, 6, 6, 3, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 8, 8, 8, 8, 8, 8, 3, 3, 3, 6, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 8, 8, 8, 8, 8, 3, 3, 3, 3, 8, 8, 8, 6, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 8, 8, 8, 8, 8, 3, 7, 7, 8, 8, 8, 8, 8, 6], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 8, 8, 8, 8, 8, 3, 7, 7, 3, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 8, 8, 8, 8, 8, 3, 3, 3, 7, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 7, 8, 8, 8, 8], [3, 3, 8, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 7, 8, 8, 8], [3, 9, 9, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 7, 8, 8], [3, 9, 9, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 7, 8], [3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 7], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]]}], "e376de54": [{"attempt_1": [[7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [7, 7, 7, 7, 7, 8, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [7, 7, 7, 7, 8, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [7, 7, 7, 8, 7, 7, 7, 8, 7, 7, 7, 7, 7, 7, 7, 7], [7, 7, 8, 7, 7, 7, 8, 7, 7, 7, 7, 7, 7, 7, 7, 7], [7, 7, 7, 7, 7, 8, 7, 7, 7, 6, 7, 7, 7, 7, 7, 7], [7, 7, 7, 7, 8, 7, 7, 7, 6, 7, 7, 7, 7, 7, 7, 7], [7, 7, 7, 7, 7, 7, 7, 6, 7, 7, 7, 6, 7, 7, 7, 7], [7, 7, 7, 7, 7, 7, 6, 7, 7, 7, 6, 7, 7, 7, 7, 7], [7, 7, 7, 7, 7, 7, 7, 7, 7, 6, 7, 7, 7, 6, 7, 7], [7, 7, 7, 7, 7, 7, 7, 7, 6, 7, 7, 7, 6, 7, 7, 7], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 6, 7, 7, 7, 6], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 6, 7, 7, 7, 6, 7], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 6, 7, 7], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 6, 7, 7, 7]], "attempt_2": [[7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [7, 7, 7, 7, 7, 8, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [7, 7, 7, 7, 8, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [7, 7, 7, 8, 7, 7, 7, 8, 7, 7, 7, 7, 7, 7, 7, 7], [7, 7, 8, 7, 7, 7, 8, 7, 7, 7, 7, 7, 7, 7, 7, 7], [7, 7, 7, 7, 7, 8, 7, 7, 7, 6, 7, 7, 7, 7, 7, 7], [7, 7, 7, 7, 8, 7, 7, 7, 6, 7, 7, 7, 7, 7, 7, 7], [7, 7, 7, 7, 7, 7, 7, 6, 7, 7, 7, 7, 7, 6, 7, 7], [7, 7, 7, 7, 7, 7, 6, 7, 7, 7, 7, 7, 6, 7, 7, 7], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 6, 7, 7, 7, 6], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 6, 7, 7, 7, 6, 7], [7, 7, 7, 7, 7, 7, 7, 7, 7, 6, 7, 7, 7, 6, 7, 7], [7, 7, 7, 7, 7, 7, 7, 7, 6, 7, 7, 7, 6, 7, 7, 7], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 6, 7, 7, 7, 7], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 6, 7, 7, 7, 7, 7]]}], "e8686506": [{"attempt_1": [[1, 8, 8, 8, 1], [8, 6, 6, 6, 8], [3, 8, 6, 8, 3], [3, 8, 9, 8, 3], [3, 8, 9, 8, 3], [3, 8, 6, 8, 3], [8, 6, 6, 6, 8], [1, 8, 8, 8, 1]], "attempt_2": [[1, 8, 8, 8, 1], [8, 6, 6, 6, 8], [3, 8, 6, 8, 3], [3, 8, 3, 8, 3], [3, 8, 3, 8, 3], [3, 8, 6, 8, 3], [8, 6, 6, 6, 8], [1, 8, 8, 8, 1]]}], "e87109e9": [{"attempt_1": [[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3], [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3], [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 3, 3, 3], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 3, 3, 3], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 3, 3, 3], [1, 1, 1, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 3, 3, 3], [1, 1, 1, 9, 9, 9, 9, 9, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 3, 3, 3], [1, 1, 1, 9, 9, 9, 9, 9, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 3, 3, 3], [1, 1, 1, 9, 9, 9, 9, 9, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 3, 3, 3], [1, 1, 1, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 3, 3, 3], [1, 1, 1, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 3, 3, 3], [1, 1, 1, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 3, 3, 3], [1, 1, 1, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [1, 1, 1, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [1, 1, 1, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 4, 4, 4, 4, 4, 4, 4, 4, 4, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 4, 4, 4, 4, 4, 4, 4, 4, 4, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 4, 4, 4, 4, 4, 4, 4, 4, 4, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9]], "attempt_2": [[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3], [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3], [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3], [9, 9, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 3], [9, 9, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 3], [9, 9, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 3], [1, 1, 1, 8, 8, 8, 9, 9, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 3, 3, 3], [1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 3], [1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 3], [1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 3], [1, 1, 1, 8, 8, 8, 9, 9, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 3, 3, 3], [1, 1, 1, 8, 8, 8, 9, 9, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 3, 3, 3], [1, 1, 1, 8, 8, 8, 9, 9, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 3, 3, 3], [1, 1, 1, 8, 8, 8, 9, 9, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 9, 9, 9], [1, 1, 1, 8, 8, 8, 9, 9, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 9, 9, 9], [1, 1, 1, 8, 8, 8, 9, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [9, 9, 9, 8, 8, 8, 9, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [9, 9, 9, 8, 8, 8, 9, 4, 4, 4, 4, 4, 4, 4, 4, 4, 9, 9, 9, 8, 8, 9, 9, 9], [9, 9, 9, 8, 8, 8, 9, 4, 4, 4, 4, 4, 4, 4, 4, 4, 9, 9, 9, 8, 8, 9, 9, 9], [9, 9, 9, 8, 8, 8, 9, 4, 4, 4, 4, 4, 4, 4, 4, 4, 9, 9, 9, 8, 8, 9, 9, 9], [9, 9, 9, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 9, 9, 9], [9, 9, 9, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 9, 9, 9], [9, 9, 9, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 9, 9, 9], [9, 9, 9, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 9, 9, 9]]}], "edb79dae": [{"attempt_1": [[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 5], [5, 4, 9, 9, 4, 4, 9, 4, 4, 4, 4, 4, 4, 4, 6, 6, 6, 6, 6, 4, 5], [5, 4, 9, 4, 4, 4, 9, 4, 4, 4, 4, 4, 4, 4, 6, 4, 4, 4, 4, 4, 5], [5, 4, 9, 4, 9, 9, 9, 4, 4, 4, 4, 4, 4, 4, 6, 6, 6, 6, 6, 4, 5], [5, 4, 9, 9, 9, 9, 9, 4, 4, 4, 4, 4, 4, 4, 4, 4, 6, 6, 4, 4, 5], [5, 4, 9, 9, 4, 4, 9, 4, 4, 4, 4, 4, 4, 4, 6, 6, 6, 6, 6, 4, 5], [5, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 5], [5, 4, 9, 9, 4, 4, 9, 4, 1, 1, 1, 1, 1, 4, 6, 4, 4, 4, 4, 4, 5], [5, 4, 9, 4, 4, 4, 9, 4, 1, 1, 1, 1, 1, 4, 6, 6, 6, 6, 6, 4, 5], [5, 4, 9, 4, 9, 9, 9, 4, 1, 4, 4, 4, 1, 4, 6, 6, 4, 4, 6, 4, 5], [5, 4, 9, 9, 9, 9, 9, 4, 1, 4, 4, 4, 1, 4, 6, 6, 6, 6, 6, 4, 5], [5, 4, 9, 9, 4, 4, 9, 4, 1, 4, 1, 1, 1, 4, 6, 4, 4, 4, 4, 4, 5], [5, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 5], [5, 4, 4, 4, 4, 4, 4, 4, 7, 7, 4, 4, 7, 4, 1, 1, 1, 1, 1, 4, 5], [5, 4, 4, 4, 4, 4, 4, 4, 7, 7, 4, 4, 7, 4, 1, 1, 1, 1, 1, 4, 5], [5, 4, 4, 4, 4, 4, 4, 4, 7, 4, 4, 4, 4, 4, 1, 4, 4, 4, 1, 4, 5], [5, 4, 4, 4, 4, 4, 4, 4, 7, 7, 4, 4, 7, 4, 1, 4, 4, 4, 1, 4, 5], [5, 4, 4, 4, 4, 4, 4, 4, 7, 7, 7, 7, 7, 4, 1, 4, 1, 1, 1, 4, 5], [5, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5]], "attempt_2": [[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 5], [5, 4, 9, 9, 4, 4, 9, 4, 4, 4, 4, 4, 4, 4, 2, 2, 2, 2, 2, 4, 5], [5, 4, 9, 4, 4, 4, 9, 4, 4, 4, 4, 4, 4, 4, 2, 4, 4, 4, 4, 4, 5], [5, 4, 9, 4, 9, 9, 9, 4, 4, 4, 4, 4, 4, 4, 2, 2, 2, 2, 2, 4, 5], [5, 4, 9, 9, 9, 9, 9, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 2, 4, 4, 5], [5, 4, 9, 9, 4, 4, 9, 4, 4, 4, 4, 4, 4, 4, 2, 2, 2, 2, 2, 4, 5], [5, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 5], [5, 4, 9, 9, 4, 4, 9, 4, 1, 1, 1, 1, 1, 4, 6, 6, 6, 6, 6, 4, 5], [5, 4, 9, 4, 4, 4, 9, 4, 1, 1, 1, 1, 1, 4, 6, 4, 4, 4, 4, 4, 5], [5, 4, 9, 4, 9, 9, 9, 4, 1, 4, 4, 4, 1, 4, 6, 6, 6, 6, 6, 4, 5], [5, 4, 9, 9, 9, 9, 9, 4, 1, 4, 4, 4, 1, 4, 4, 4, 6, 6, 4, 4, 5], [5, 4, 9, 9, 4, 4, 9, 4, 1, 4, 1, 1, 1, 4, 6, 6, 6, 6, 6, 4, 5], [5, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 5], [5, 4, 4, 4, 4, 4, 4, 4, 7, 7, 4, 4, 7, 4, 1, 1, 1, 1, 1, 4, 5], [5, 4, 4, 4, 4, 4, 4, 4, 7, 7, 4, 4, 7, 4, 1, 1, 1, 1, 1, 4, 5], [5, 4, 4, 4, 4, 4, 4, 4, 7, 4, 4, 4, 4, 4, 1, 4, 4, 4, 1, 4, 5], [5, 4, 4, 4, 4, 4, 4, 4, 7, 7, 4, 4, 7, 4, 1, 4, 4, 4, 1, 4, 5], [5, 4, 4, 4, 4, 4, 4, 4, 7, 7, 7, 7, 7, 4, 1, 4, 1, 1, 1, 4, 5], [5, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5]]}], "eee78d87": [{"attempt_1": [[0, 7, 7, 0, 7, 7, 0, 7, 7, 0, 7, 7, 0, 7, 7, 0], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [0, 7, 7, 0, 7, 7, 0, 7, 7, 0, 7, 7, 0, 7, 7, 0], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [7, 7, 7, 7, 7, 9, 7, 9, 9, 7, 9, 7, 7, 7, 7, 7], [0, 7, 7, 0, 7, 7, 7, 7, 7, 7, 7, 7, 0, 7, 7, 0], [7, 7, 7, 7, 7, 9, 7, 9, 9, 7, 9, 7, 7, 7, 7, 7], [7, 7, 7, 7, 7, 9, 7, 9, 9, 7, 9, 7, 7, 7, 7, 7], [0, 7, 7, 0, 7, 7, 7, 7, 7, 7, 7, 7, 0, 7, 7, 0], [7, 7, 7, 7, 7, 9, 7, 9, 9, 7, 9, 7, 7, 7, 7, 7], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [0, 7, 7, 0, 7, 7, 0, 7, 7, 0, 7, 7, 0, 7, 7, 0], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [0, 7, 7, 0, 7, 7, 0, 7, 7, 0, 7, 7, 0, 7, 7, 0]], "attempt_2": [[7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [7, 0, 0, 7, 0, 0, 7, 0, 0, 7, 0, 0, 7, 0, 0, 7], [7, 0, 0, 7, 0, 0, 7, 0, 0, 7, 0, 0, 7, 0, 0, 7], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [7, 0, 0, 7, 0, 0, 7, 0, 0, 7, 0, 0, 7, 0, 0, 7], [7, 0, 0, 7, 0, 9, 7, 9, 9, 7, 9, 0, 7, 0, 0, 7], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [7, 0, 0, 7, 0, 9, 7, 9, 9, 7, 9, 0, 7, 0, 0, 7], [7, 0, 0, 7, 0, 9, 7, 9, 9, 7, 9, 0, 7, 0, 0, 7], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [7, 0, 0, 7, 0, 9, 7, 9, 9, 7, 9, 0, 7, 0, 0, 7], [7, 0, 0, 7, 0, 0, 7, 0, 0, 7, 0, 0, 7, 0, 0, 7], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [7, 0, 0, 7, 0, 0, 7, 0, 0, 7, 0, 0, 7, 0, 0, 7], [7, 0, 0, 7, 0, 0, 7, 0, 0, 7, 0, 0, 7, 0, 0, 7], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7]]}], "f560132c": [{"attempt_1": [[0]], "attempt_2": [[0]]}, {"attempt_1": [[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 6, 6], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 6, 6], [3, 3, 3, 3, 8, 8, 8, 8, 3, 3, 6, 6], [3, 3, 3, 3, 8, 8, 8, 8, 3, 3, 6, 6], [3, 3, 8, 8, 8, 8, 8, 8, 3, 3, 6, 6], [3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [3, 3, 3, 3, 3, 3, 3, 8, 8, 8, 8, 8], [3, 3, 3, 3, 3, 3, 3, 8, 8, 8, 8, 8], [4, 4, 4, 4, 4, 4, 8, 8, 8, 8, 8, 8], [4, 4, 4, 4, 4, 4, 8, 8, 8, 8, 8, 8], [4, 4, 4, 4, 4, 4, 8, 8, 8, 8, 8, 8], [4, 4, 4, 4, 4, 4, 8, 8, 8, 8, 8, 8]], "attempt_2": [[3, 3, 3, 6, 6, 6, 6, 6, 6, 6], [3, 3, 3, 6, 6, 6, 6, 6, 6, 6], [3, 3, 3, 6, 6, 6, 6, 6, 6, 6], [3, 6, 6, 6, 6, 6, 6, 6, 6, 6], [3, 4, 4, 4, 4, 8, 8, 8, 8, 8], [3, 4, 4, 4, 4, 8, 8, 8, 8, 8], [4, 4, 4, 4, 4, 8, 8, 8, 8, 8], [4, 4, 4, 4, 4, 8, 8, 8, 8, 8], [4, 4, 4, 4, 4, 8, 8, 8, 8, 8], [4, 4, 4, 4, 4, 8, 8, 8, 8, 8]]}], "f931b4a8": [{"attempt_1": [[1, 2, 1, 2], [2, 1, 3, 1], [1, 2, 1, 2], [2, 2, 3, 2], [2, 3, 3, 3], [3, 2, 3, 2], [3, 2, 2, 5], [3, 3, 1, 2]], "attempt_2": [[1, 2, 1, 2], [2, 1, 3, 1], [1, 2, 1, 2]]}, {"attempt_1": [[7, 3], [6, 8]], "attempt_2": [[7, 3, 7, 0], [6, 8, 0, 7]]}], "faa9f03d": [{"attempt_1": [[0, 0, 0, 6, 0, 0, 0, 3, 0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0], [3, 3, 3, 6, 3, 3, 3, 3, 0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0], [3, 0, 0, 6, 0, 0, 0, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0], [3, 0, 0, 6, 0, 0, 0, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0], [3, 0, 0, 6, 0, 7, 7, 7, 7, 7, 7, 7, 0, 0, 0, 0, 0, 0, 0, 0], [3, 0, 0, 6, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [3, 0, 0, 6, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [3, 0, 0, 6, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [3, 0, 0, 6, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [3, 0, 0, 6, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [3, 0, 0, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 0, 0, 0, 0, 0, 0, 0], [3, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 6, 0, 0, 0, 0, 0, 0, 0], [3, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 6, 0, 0, 0, 0, 0, 0, 0], [3, 3, 3, 3, 0, 7, 0, 0, 0, 0, 0, 0, 6, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 3, 0, 7, 0, 0, 0, 0, 0, 0, 6, 6, 6, 6, 6, 6, 0, 0], [0, 0, 0, 3, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 0, 0], [0, 0, 0, 3, 0, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 6, 7, 7], [0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 0, 0], [3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 0, 0]], "attempt_2": [[0, 0, 0, 6, 0, 0, 0, 3, 0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0], [3, 3, 3, 6, 3, 3, 3, 3, 0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0], [3, 0, 0, 6, 0, 0, 0, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0], [3, 0, 0, 6, 0, 0, 0, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0], [3, 0, 0, 6, 0, 7, 7, 7, 7, 7, 7, 7, 0, 0, 0, 0, 0, 0, 0, 0], [3, 0, 0, 6, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [3, 0, 0, 6, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [3, 0, 0, 6, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [3, 0, 0, 6, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [3, 0, 0, 6, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [3, 0, 0, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 0, 0, 0, 0, 0, 0, 0], [3, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 6, 0, 0, 0, 0, 0, 0, 0], [3, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 6, 0, 0, 0, 0, 0, 0, 0], [3, 3, 3, 3, 0, 7, 0, 0, 0, 0, 0, 0, 6, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 3, 0, 7, 0, 0, 0, 0, 0, 0, 6, 6, 6, 6, 6, 6, 0, 0], [0, 0, 0, 3, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 0, 0], [0, 0, 0, 3, 0, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 0, 0], [3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 0, 0]]}, {"attempt_1": [[0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 7, 0], [0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 7, 0], [0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 7, 0], [7, 7, 7, 7, 7, 0, 0, 7, 7, 7, 7, 0], [0, 7, 0, 0, 0, 0, 0, 7, 0, 0, 7, 0], [0, 7, 0, 0, 0, 0, 0, 7, 0, 0, 0, 0], [0, 7, 0, 0, 0, 0, 0, 7, 0, 0, 0, 0], [6, 7, 6, 6, 6, 6, 6, 7, 6, 6, 6, 6], [0, 7, 0, 0, 0, 0, 0, 7, 0, 0, 0, 0], [0, 7, 7, 7, 7, 7, 7, 7, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], "attempt_2": [[0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 7, 0], [0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 7, 0], [0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 7, 0], [7, 7, 7, 7, 7, 0, 0, 7, 7, 7, 7, 0], [0, 7, 0, 0, 0, 0, 0, 7, 0, 0, 7, 0], [0, 7, 0, 0, 0, 0, 0, 7, 0, 0, 7, 0], [0, 7, 0, 0, 0, 0, 0, 7, 0, 0, 7, 0], [6, 7, 6, 6, 6, 6, 6, 7, 6, 6, 7, 6], [0, 7, 0, 0, 0, 0, 0, 7, 0, 0, 7, 0], [0, 7, 7, 7, 7, 7, 7, 7, 0, 0, 7, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0]]}], "fc7cae8d": [{"attempt_1": [[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [7, 7, 7, 7, 7, 7, 7, 7, 4, 8, 8, 4], [0, 4, 4, 4, 4, 4, 4, 7, 4, 8, 8, 4], [0, 0, 4, 4, 6, 6, 4, 7, 4, 9, 9, 4], [0, 0, 4, 4, 4, 6, 4, 7, 4, 4, 4, 4], [0, 0, 4, 4, 6, 4, 4, 7, 4, 9, 9, 4], [0, 4, 4, 4, 6, 4, 4, 7, 4, 4, 4, 4], [4, 4, 4, 4, 6, 4, 4, 7, 4, 9, 9, 4], [4, 4, 4, 4, 6, 4, 4, 7, 4, 4, 4, 4], [6, 6, 6, 6, 6, 4, 4, 7, 4, 9, 9, 4], [4, 4, 4, 4, 4, 4, 4, 7, 4, 4, 4, 4], [4, 6, 6, 4, 4, 4, 4, 7, 4, 9, 9, 4], [4, 6, 4, 4, 4, 4, 4, 7, 4, 4, 4, 4], [4, 6, 4, 4, 4, 4, 4, 7, 4, 9, 9, 4], [4, 6, 4, 4, 4, 6, 4, 7, 4, 4, 4, 4], [4, 6, 6, 6, 6, 6, 4, 7, 4, 9, 9, 4], [0, 4, 4, 4, 4, 4, 4, 7, 4, 4, 4, 4], [0, 7, 7, 7, 7, 7, 7, 7, 4, 9, 9, 4], [0, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 0, 0, 4, 4, 0, 4, 4, 4, 4, 4]], "attempt_2": [[4, 4, 0, 4, 4, 0, 4, 4, 4, 4, 4, 4], [0, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [0, 7, 7, 7, 7, 7, 7, 7, 4, 9, 9, 4], [0, 4, 4, 4, 4, 4, 4, 7, 4, 4, 4, 4], [4, 6, 6, 6, 6, 6, 4, 7, 4, 9, 9, 4], [4, 6, 4, 4, 4, 6, 4, 7, 4, 4, 4, 4], [4, 6, 4, 4, 4, 4, 4, 7, 4, 9, 9, 4], [4, 6, 4, 4, 4, 4, 4, 7, 4, 4, 4, 4], [4, 6, 6, 4, 4, 4, 4, 7, 4, 9, 9, 4], [4, 4, 4, 4, 4, 4, 4, 7, 4, 4, 4, 4], [6, 6, 6, 6, 6, 4, 4, 7, 4, 9, 9, 4], [4, 4, 4, 4, 6, 4, 4, 7, 4, 4, 4, 4], [4, 4, 4, 4, 6, 4, 4, 7, 4, 9, 9, 4], [0, 4, 4, 4, 6, 4, 4, 7, 4, 4, 4, 4], [0, 0, 4, 4, 6, 4, 4, 7, 4, 9, 9, 4], [0, 0, 4, 4, 4, 6, 4, 7, 4, 4, 4, 4], [0, 0, 4, 4, 6, 6, 4, 7, 4, 9, 9, 4], [0, 4, 4, 4, 4, 4, 4, 7, 4, 8, 8, 4], [7, 7, 7, 7, 7, 7, 7, 7, 4, 8, 8, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]]}]}

In [ ]:
%%writefile /kaggle/working/ab/a100/fixed_nvarc/qwen_eligible_confirm.json
[["f560132c", 0], ["f560132c", 1], ["21897d95", 0], ["21897d95", 1], ["cbebaa4b", 0], ["cbebaa4b", 1], ["581f7754", 0], ["581f7754", 1], ["5961cc34", 0], ["db0c5428", 0], ["89565ca0", 0], ["f931b4a8", 0], ["332f06d7", 0], ["58490d8a", 0], ["31f7f899", 0], ["4c3d4a41", 0]]

In [ ]:
%%writefile /kaggle/working/ab/a100/fixed_nvarc/qwen_eligible_explore.json
[["13e47133", 0], ["13e47133", 1], ["446ef5d2", 0], ["446ef5d2", 1], ["6ffbe589", 0], ["4e34c42c", 0], ["4e34c42c", 1], ["8e5c0c38", 0], ["8e5c0c38", 1], ["c7f57c3e", 0], ["8b7bacbf", 1], ["221dfab4", 1], ["35ab12c3", 0], ["a47bf94d", 0], ["db695cfb", 0], ["e12f9a14", 0], ["e12f9a14", 1], ["edb79dae", 0], ["88e364bc", 0], ["9385bd28", 0], ["9385bd28", 1], ["80a900e0", 0], ["269e22fb", 1], ["8f3a5a89", 0], ["2c181942", 0], ["7666fa5d", 0], ["16b78196", 0], ["7b5033c1", 0], ["da515329", 0], ["d35bdbdc", 0], ["de809cff", 0], ["3dc255db", 0], ["e87109e9", 0], ["faa9f03d", 0], ["0934a4d8", 0], ["2b83f449", 0], ["195c6913", 0], ["195c6913", 1], ["3a25b0d8", 0], ["3a25b0d8", 1], ["7ed72f31", 1], ["9bbf930d", 0], ["b5ca7ac4", 0], ["62593bfd", 1]]

In [ ]:
%%writefile /kaggle/working/ab/data/arc-agi_evaluation_challenges.json
{"0934a4d8": {"train": [{"input": [[3, 5, 3, 3, 6, 6, 5, 4, 1, 4, 9, 9, 4, 3, 9, 9, 9, 9, 3, 4, 9, 9, 4, 1, 4, 5, 6, 6, 3, 3], [5, 3, 3, 3, 6, 6, 4, 5, 4, 1, 9, 9, 3, 4, 9, 1, 1, 9, 4, 3, 9, 9, 1, 4, 5, 4, 6, 6, 3, 3], [1, 1, 3, 5, 5, 4, 6, 6, 9, 1, 1, 4, 9, 9, 4, 5, 5, 4, 9, 9, 4, 1, 1, 9, 6, 6, 4, 5, 5, 3], [1, 1, 5, 3, 4, 5, 6, 6, 1, 9, 4, 1, 9, 1, 4, 4, 4, 4, 1, 9, 1, 4, 9, 1, 6, 6, 5, 4, 3, 5], [6, 9, 9, 9, 3, 5, 3, 3, 4, 3, 9, 9, 9, 2, 6, 9, 9, 6, 2, 9, 9, 9, 3, 4, 3, 3, 5, 3, 9, 9], [9, 6, 9, 9, 5, 3, 3, 3, 3, 4, 9, 1, 9, 9, 9, 6, 6, 9, 9, 9, 1, 9, 4, 3, 3, 3, 3, 5, 9, 9], [9, 9, 6, 9, 1, 1, 3, 5, 9, 9, 4, 4, 6, 9, 9, 2, 2, 9, 9, 6, 4, 4, 9, 9, 5, 3, 1, 1, 9, 6], [9, 9, 9, 6, 1, 1, 5, 3, 9, 1, 5, 4, 9, 6, 9, 9, 9, 9, 6, 9, 4, 5, 1, 9, 3, 5, 1, 1, 6, 9], [1, 4, 9, 1, 4, 3, 9, 9, 5, 5, 7, 2, 4, 3, 2, 4, 4, 2, 3, 4, 2, 7, 5, 5, 9, 9, 3, 4, 1, 9], [4, 1, 1, 9, 3, 4, 9, 1, 4, 5, 2, 7, 3, 4, 4, 2, 2, 4, 4, 3, 7, 2, 5, 4, 1, 9, 4, 3, 9, 1], [9, 9, 1, 4, 9, 9, 4, 5, 6, 4, 5, 5, 2, 4, 4, 3, 3, 4, 4, 2, 5, 5, 4, 6, 5, 4, 9, 9, 4, 1], [9, 9, 4, 1, 9, 1, 4, 4, 4, 5, 4, 5, 4, 2, 3, 4, 4, 3, 2, 4, 5, 4, 5, 4, 4, 4, 1, 9, 1, 4], [4, 3, 9, 9, 9, 9, 6, 9, 5, 9, 7, 7, 5, 5, 7, 2, 2, 7, 5, 5, 7, 7, 9, 5, 9, 6, 9, 9, 9, 9], [3, 4, 9, 1, 2, 9, 9, 6, 9, 5, 7, 7, 4, 5, 2, 7, 7, 2, 5, 4, 7, 7, 5, 9, 6, 9, 9, 2, 1, 9], [9, 9, 4, 4, 6, 9, 9, 9, 7, 7, 5, 9, 5, 4, 5, 5, 5, 5, 4, 5, 9, 5, 7, 7, 9, 8, 8, 8, 8, 4], [9, 1, 5, 4, 9, 6, 2, 9, 7, 7, 9, 5, 4, 6, 4, 5, 5, 4, 6, 4, 5, 9, 7, 7, 9, 8, 8, 8, 8, 5], [9, 1, 5, 4, 9, 6, 2, 9, 7, 7, 9, 5, 4, 6, 4, 5, 5, 4, 6, 4, 5, 9, 7, 7, 9, 8, 8, 8, 8, 5], [9, 9, 4, 4, 6, 9, 9, 9, 7, 7, 5, 9, 5, 4, 5, 5, 5, 5, 4, 5, 9, 5, 7, 7, 9, 8, 8, 8, 8, 4], [3, 4, 9, 1, 2, 9, 9, 6, 9, 5, 7, 7, 4, 5, 2, 7, 7, 2, 5, 4, 7, 7, 5, 9, 6, 8, 8, 8, 8, 9], [4, 3, 9, 9, 9, 9, 6, 9, 5, 9, 7, 7, 5, 5, 7, 2, 2, 7, 5, 5, 7, 7, 9, 5, 9, 8, 8, 8, 8, 9], [9, 9, 4, 1, 9, 1, 4, 4, 4, 5, 4, 5, 4, 2, 3, 4, 4, 3, 2, 4, 5, 4, 5, 4, 4, 8, 8, 8, 8, 4], [9, 9, 1, 4, 9, 9, 4, 5, 6, 4, 5, 5, 2, 4, 4, 3, 3, 4, 4, 2, 5, 5, 4, 6, 5, 8, 8, 8, 8, 1], [4, 1, 1, 9, 3, 4, 9, 1, 4, 5, 2, 7, 3, 4, 4, 2, 2, 4, 4, 3, 7, 2, 5, 4, 1, 8, 8, 8, 8, 1], [1, 4, 9, 1, 4, 3, 9, 9, 5, 5, 7, 2, 4, 3, 2, 4, 4, 2, 3, 4, 2, 7, 5, 5, 9, 9, 3, 4, 1, 9], [9, 9, 9, 6, 1, 1, 5, 3, 9, 1, 5, 4, 9, 6, 9, 9, 9, 9, 6, 9, 4, 5, 1, 9, 3, 5, 1, 1, 6, 9], [9, 9, 6, 9, 1, 1, 3, 5, 9, 9, 4, 4, 6, 9, 9, 2, 2, 9, 9, 6, 4, 4, 9, 9, 5, 3, 1, 1, 9, 6], [9, 6, 9, 9, 5, 3, 3, 3, 3, 4, 9, 1, 9, 9, 9, 6, 6, 9, 9, 9, 1, 9, 4, 3, 3, 3, 3, 5, 9, 9], [6, 9, 9, 9, 3, 5, 3, 3, 4, 3, 9, 9, 9, 2, 6, 9, 9, 6, 2, 9, 9, 9, 3, 4, 3, 3, 5, 3, 9, 9], [1, 1, 5, 3, 4, 5, 6, 6, 1, 9, 4, 1, 9, 1, 4, 4, 4, 4, 1, 9, 1, 4, 9, 1, 6, 6, 5, 4, 3, 5], [1, 1, 3, 5, 5, 4, 6, 6, 9, 1, 1, 4, 9, 9, 4, 5, 5, 4, 9, 9, 4, 1, 1, 9, 6, 6, 4, 5, 5, 3]], "output": [[9, 9, 6, 4], [2, 6, 9, 4], [2, 6, 9, 4], [9, 9, 6, 4], [9, 9, 2, 1], [6, 9, 9, 9], [4, 1, 9, 1], [4, 9, 9, 4], [9, 4, 3, 9]]}, {"input": [[9, 9, 2, 3, 4, 4, 7, 5, 3, 3, 6, 6, 3, 5, 6, 4, 4, 6, 5, 3, 6, 6, 3, 3, 5, 7, 4, 4, 3, 2], [7, 9, 3, 5, 4, 4, 5, 7, 3, 3, 6, 6, 6, 3, 4, 6, 6, 4, 3, 6, 6, 6, 3, 3, 7, 5, 4, 4, 5, 3], [3, 2, 9, 9, 7, 5, 4, 4, 4, 1, 3, 3, 6, 4, 4, 7, 7, 4, 4, 6, 3, 8, 8, 8, 8, 8, 5, 7, 9, 9], [2, 3, 7, 9, 5, 7, 4, 4, 1, 4, 3, 3, 4, 6, 7, 4, 4, 7, 6, 4, 3, 8, 8, 8, 8, 8, 7, 5, 9, 7], [7, 7, 9, 3, 9, 9, 5, 3, 3, 6, 6, 4, 6, 7, 9, 9, 9, 9, 7, 6, 4, 8, 8, 8, 8, 8, 9, 9, 3, 9], [7, 7, 3, 9, 7, 9, 3, 2, 5, 3, 4, 6, 2, 6, 9, 9, 9, 9, 6, 2, 6, 8, 8, 8, 8, 8, 9, 7, 9, 3], [9, 3, 7, 7, 3, 2, 9, 9, 6, 4, 4, 7, 9, 2, 6, 7, 7, 6, 2, 9, 7, 4, 4, 6, 9, 9, 2, 3, 7, 7], [3, 9, 7, 7, 2, 3, 7, 9, 4, 6, 7, 4, 2, 9, 2, 6, 6, 2, 9, 2, 4, 7, 6, 4, 9, 7, 3, 2, 7, 7], [3, 3, 4, 1, 3, 5, 6, 4, 2, 4, 7, 7, 1, 6, 7, 2, 2, 7, 6, 1, 7, 7, 4, 2, 4, 6, 5, 3, 1, 4], [3, 3, 1, 4, 6, 3, 4, 6, 2, 2, 7, 1, 6, 1, 2, 7, 7, 2, 1, 6, 1, 7, 2, 2, 6, 4, 3, 6, 4, 1], [6, 6, 3, 3, 6, 4, 4, 7, 1, 1, 2, 4, 7, 2, 1, 6, 6, 1, 2, 7, 4, 2, 1, 1, 7, 4, 4, 6, 3, 3], [6, 6, 3, 3, 4, 6, 7, 4, 1, 3, 2, 2, 2, 7, 6, 1, 1, 6, 7, 2, 2, 2, 3, 1, 4, 7, 6, 4, 3, 3], [3, 6, 6, 4, 6, 2, 9, 2, 9, 9, 9, 7, 2, 4, 1, 7, 7, 1, 4, 2, 7, 9, 9, 9, 2, 9, 2, 6, 4, 6], [5, 3, 4, 6, 7, 6, 2, 9, 9, 9, 7, 9, 2, 2, 7, 7, 7, 7, 2, 2, 9, 7, 9, 9, 9, 2, 6, 7, 6, 4], [6, 4, 4, 7, 9, 9, 6, 2, 9, 7, 9, 9, 3, 1, 2, 4, 4, 2, 1, 3, 9, 9, 7, 9, 2, 6, 9, 9, 7, 4], [4, 6, 7, 4, 9, 9, 7, 6, 7, 9, 9, 9, 1, 1, 2, 2, 2, 2, 1, 1, 9, 9, 9, 7, 6, 7, 9, 9, 4, 7], [4, 6, 7, 4, 9, 9, 7, 6, 7, 9, 9, 9, 1, 1, 2, 2, 2, 2, 1, 1, 9, 9, 9, 7, 6, 7, 9, 9, 4, 7], [6, 4, 4, 7, 9, 9, 6, 2, 9, 7, 9, 9, 3, 1, 2, 4, 4, 2, 1, 3, 9, 9, 7, 9, 2, 6, 9, 9, 7, 4], [5, 3, 4, 6, 7, 6, 2, 9, 9, 9, 7, 9, 2, 2, 7, 7, 7, 7, 2, 2, 9, 7, 9, 9, 9, 2, 6, 7, 6, 4], [3, 6, 6, 4, 6, 2, 9, 2, 9, 9, 9, 7, 2, 4, 1, 7, 7, 1, 4, 2, 7, 9, 9, 9, 2, 9, 2, 6, 4, 6], [6, 6, 3, 3, 4, 6, 7, 4, 1, 3, 2, 2, 2, 7, 6, 1, 1, 6, 7, 2, 2, 2, 3, 1, 4, 7, 6, 4, 3, 3], [6, 6, 3, 3, 6, 4, 4, 7, 1, 1, 2, 4, 7, 2, 1, 6, 6, 1, 2, 7, 4, 2, 1, 1, 7, 4, 4, 6, 3, 3], [3, 3, 1, 4, 6, 3, 4, 6, 2, 2, 7, 1, 6, 1, 2, 7, 7, 2, 1, 6, 1, 7, 2, 2, 6, 4, 3, 6, 4, 1], [3, 3, 4, 1, 3, 5, 6, 4, 2, 4, 7, 7, 1, 6, 7, 2, 2, 7, 6, 1, 7, 7, 4, 2, 4, 6, 5, 3, 1, 4], [3, 9, 7, 7, 2, 3, 7, 9, 4, 6, 7, 4, 2, 9, 2, 6, 6, 2, 9, 2, 4, 7, 6, 4, 9, 7, 3, 2, 7, 7], [9, 3, 7, 7, 3, 2, 9, 9, 6, 4, 4, 7, 9, 2, 6, 7, 7, 6, 2, 9, 7, 4, 4, 6, 9, 9, 2, 3, 7, 7], [7, 7, 3, 9, 7, 9, 3, 2, 5, 3, 4, 6, 2, 6, 9, 9, 9, 9, 6, 2, 6, 4, 3, 5, 2, 3, 9, 7, 9, 3], [7, 7, 9, 3, 9, 9, 5, 3, 3, 6, 6, 4, 6, 7, 9, 9, 9, 9, 7, 6, 4, 6, 6, 3, 3, 5, 9, 9, 3, 9], [2, 3, 7, 9, 5, 7, 4, 4, 1, 4, 3, 3, 4, 6, 7, 4, 4, 7, 6, 4, 3, 3, 4, 1, 4, 4, 7, 5, 9, 7], [3, 2, 9, 9, 7, 5, 4, 4, 4, 1, 3, 3, 6, 4, 4, 7, 7, 4, 4, 6, 3, 3, 1, 4, 4, 4, 5, 7, 9, 9]], "output": [[3, 1, 4, 4, 4], [3, 4, 1, 4, 4], [6, 6, 3, 3, 5], [4, 3, 5, 2, 3]]}, {"input": [[1, 9, 4, 4, 9, 9, 2, 7, 6, 6, 9, 9, 7, 6, 7, 2, 2, 7, 6, 7, 9, 9, 6, 6, 7, 2, 9, 9, 4, 4], [7, 1, 4, 4, 9, 9, 7, 2, 6, 6, 9, 9, 6, 7, 2, 7, 7, 2, 7, 6, 9, 9, 6, 6, 2, 7, 9, 9, 4, 4], [2, 7, 1, 9, 2, 7, 9, 9, 4, 4, 6, 6, 7, 2, 5, 1, 1, 5, 2, 7, 6, 6, 4, 4, 9, 9, 7, 2, 9, 1], [7, 2, 7, 1, 7, 2, 9, 9, 4, 4, 6, 6, 2, 7, 5, 5, 5, 5, 7, 2, 6, 6, 4, 4, 9, 9, 2, 7, 1, 7], [9, 6, 7, 2, 1, 9, 4, 4, 7, 6, 7, 2, 9, 2, 6, 4, 4, 6, 2, 9, 2, 7, 6, 7, 4, 4, 9, 1, 2, 7], [6, 9, 2, 7, 7, 1, 4, 4, 6, 7, 2, 7, 9, 9, 4, 6, 6, 4, 9, 9, 7, 2, 7, 6, 4, 4, 1, 7, 7, 2], [7, 2, 9, 6, 2, 7, 1, 9, 7, 2, 5, 5, 4, 5, 9, 2, 2, 9, 5, 4, 5, 5, 2, 7, 9, 1, 7, 2, 6, 9], [2, 7, 6, 9, 7, 2, 7, 1, 2, 7, 1, 5, 5, 4, 9, 9, 9, 9, 4, 5, 5, 1, 7, 2, 1, 7, 2, 7, 9, 6], [6, 6, 4, 4, 7, 6, 7, 2, 3, 7, 1, 4, 9, 7, 7, 6, 6, 7, 7, 9, 4, 1, 7, 3, 2, 7, 6, 7, 4, 4], [6, 6, 4, 4, 6, 7, 2, 7, 4, 3, 4, 4, 7, 9, 6, 7, 7, 6, 9, 7, 4, 4, 3, 4, 7, 2, 7, 6, 4, 4], [9, 9, 6, 6, 7, 2, 5, 1, 3, 7, 3, 7, 7, 6, 9, 7, 7, 9, 6, 7, 7, 3, 7, 3, 1, 5, 2, 7, 6, 6], [9, 9, 6, 6, 2, 7, 5, 5, 7, 7, 4, 3, 6, 7, 7, 9, 9, 7, 7, 6, 3, 4, 7, 7, 5, 5, 7, 2, 6, 6], [7, 6, 7, 2, 9, 9, 4, 5, 6, 6, 5, 9, 3, 7, 4, 4, 4, 4, 7, 3, 9, 5, 6, 6, 5, 4, 9, 9, 2, 7], [6, 7, 2, 7, 2, 9, 5, 4, 6, 6, 9, 5, 4, 3, 4, 1, 1, 4, 3, 4, 5, 9, 6, 6, 4, 5, 9, 2, 7, 2], [7, 2, 5, 5, 6, 4, 9, 9, 5, 9, 6, 6, 7, 7, 3, 7, 7, 3, 7, 7, 6, 6, 9, 5, 9, 9, 4, 6, 5, 5], [2, 7, 1, 5, 4, 6, 2, 9, 9, 5, 6, 6, 7, 3, 4, 3, 3, 4, 3, 7, 6, 6, 5, 9, 9, 2, 6, 4, 5, 1], [2, 7, 1, 5, 4, 6, 2, 9, 9, 5, 6, 6, 7, 3, 4, 3, 3, 4, 3, 7, 6, 6, 5, 9, 9, 2, 6, 4, 5, 1], [7, 2, 5, 5, 6, 4, 9, 9, 5, 9, 6, 6, 7, 7, 3, 7, 7, 3, 7, 7, 6, 6, 9, 5, 9, 9, 4, 6, 5, 5], [6, 7, 2, 7, 2, 9, 5, 4, 6, 6, 9, 5, 4, 3, 4, 1, 1, 4, 3, 4, 5, 9, 6, 6, 4, 5, 9, 2, 7, 2], [7, 6, 7, 2, 9, 9, 4, 5, 6, 6, 5, 9, 8, 8, 8, 8, 8, 8, 8, 3, 9, 5, 6, 6, 5, 4, 9, 9, 2, 7], [9, 9, 6, 6, 2, 7, 5, 5, 7, 7, 4, 3, 8, 8, 8, 8, 8, 8, 8, 6, 3, 4, 7, 7, 5, 5, 7, 2, 6, 6], [9, 9, 6, 6, 7, 2, 5, 1, 3, 7, 3, 7, 8, 8, 8, 8, 8, 8, 8, 7, 7, 3, 7, 3, 1, 5, 2, 7, 6, 6], [6, 6, 4, 4, 6, 7, 2, 7, 4, 3, 4, 4, 7, 9, 6, 7, 7, 6, 9, 7, 4, 4, 3, 4, 7, 2, 7, 6, 4, 4], [6, 6, 4, 4, 7, 6, 7, 2, 3, 7, 1, 4, 9, 7, 7, 6, 6, 7, 7, 9, 4, 1, 7, 3, 2, 7, 6, 7, 4, 4], [2, 7, 6, 9, 7, 2, 7, 1, 2, 7, 1, 5, 5, 4, 9, 9, 9, 9, 4, 5, 5, 1, 7, 2, 1, 7, 2, 7, 9, 6], [7, 2, 9, 6, 2, 7, 1, 9, 7, 2, 5, 5, 4, 5, 9, 2, 2, 9, 5, 4, 5, 5, 2, 7, 9, 1, 7, 2, 6, 9], [6, 9, 2, 7, 7, 1, 4, 4, 6, 7, 2, 7, 9, 9, 4, 6, 6, 4, 9, 9, 7, 2, 7, 6, 4, 4, 1, 7, 7, 2], [9, 6, 7, 2, 1, 9, 4, 4, 7, 6, 7, 2, 9, 2, 6, 4, 4, 6, 2, 9, 2, 7, 6, 7, 4, 4, 9, 1, 2, 7], [7, 2, 7, 1, 7, 2, 9, 9, 4, 4, 6, 6, 2, 7, 5, 5, 5, 5, 7, 2, 6, 6, 4, 4, 9, 9, 2, 7, 1, 7], [2, 7, 1, 9, 2, 7, 9, 9, 4, 4, 6, 6, 7, 2, 5, 1, 1, 5, 2, 7, 6, 6, 4, 4, 9, 9, 7, 2, 9, 1]], "output": [[3, 7, 4, 4, 4, 4, 7], [6, 7, 7, 9, 9, 7, 7], [7, 6, 9, 7, 7, 9, 6]]}, {"input": [[3, 1, 1, 9, 5, 6, 7, 1, 1, 4, 5, 7, 3, 9, 9, 1, 1, 9, 9, 3, 7, 5, 4, 1, 1, 7, 6, 5, 9, 1], [1, 3, 9, 5, 6, 5, 1, 7, 4, 1, 7, 5, 4, 3, 1, 3, 3, 1, 3, 4, 5, 7, 1, 4, 7, 1, 5, 6, 5, 9], [6, 9, 3, 1, 7, 1, 5, 6, 9, 9, 1, 4, 9, 1, 1, 4, 4, 1, 1, 9, 4, 1, 9, 9, 6, 5, 1, 7, 1, 3], [9, 1, 1, 3, 1, 7, 6, 5, 9, 9, 4, 1, 1, 3, 4, 1, 1, 4, 3, 1, 1, 4, 9, 9, 5, 6, 7, 1, 3, 1], [6, 6, 6, 7, 3, 1, 5, 9, 3, 4, 9, 1, 6, 7, 2, 5, 5, 2, 7, 6, 1, 9, 4, 3, 9, 5, 1, 3, 7, 6], [6, 6, 7, 6, 1, 3, 9, 1, 9, 3, 1, 3, 7, 6, 5, 2, 2, 5, 6, 7, 3, 1, 3, 9, 1, 9, 3, 1, 6, 7], [6, 7, 6, 6, 1, 9, 3, 1, 9, 1, 1, 4, 6, 9, 6, 7, 7, 6, 9, 6, 4, 1, 1, 9, 1, 3, 9, 1, 6, 6], [7, 6, 6, 6, 9, 6, 1, 3, 1, 3, 4, 1, 9, 6, 7, 6, 6, 7, 6, 9, 1, 4, 3, 1, 3, 1, 8, 8, 8, 8], [1, 4, 9, 9, 3, 9, 9, 1, 1, 1, 6, 1, 5, 2, 5, 5, 5, 5, 2, 5, 1, 6, 1, 1, 1, 9, 8, 8, 8, 8], [4, 1, 9, 9, 4, 3, 1, 3, 1, 1, 1, 6, 2, 5, 5, 5, 5, 5, 5, 2, 6, 1, 1, 1, 3, 1, 8, 8, 8, 8], [5, 7, 1, 4, 9, 1, 1, 4, 2, 2, 1, 1, 5, 5, 5, 2, 2, 5, 5, 5, 1, 1, 2, 2, 4, 1, 8, 8, 8, 8], [7, 5, 4, 1, 1, 3, 4, 1, 2, 1, 1, 1, 5, 5, 2, 5, 5, 2, 5, 5, 1, 1, 1, 2, 1, 4, 3, 1, 1, 4], [3, 4, 9, 1, 6, 7, 6, 9, 7, 6, 3, 3, 1, 1, 6, 1, 1, 6, 1, 1, 3, 3, 6, 7, 9, 6, 7, 6, 1, 9], [9, 3, 1, 3, 7, 6, 9, 6, 6, 7, 3, 3, 1, 1, 1, 6, 6, 1, 1, 1, 3, 3, 7, 6, 6, 9, 6, 7, 3, 1], [9, 1, 1, 4, 2, 5, 6, 7, 3, 3, 7, 6, 1, 2, 1, 1, 1, 1, 2, 1, 6, 7, 3, 3, 7, 6, 5, 2, 4, 1], [1, 3, 4, 1, 5, 2, 7, 6, 3, 3, 6, 7, 2, 2, 1, 1, 1, 1, 2, 2, 7, 6, 3, 3, 6, 7, 2, 5, 1, 4], [1, 3, 4, 1, 5, 2, 7, 6, 3, 3, 6, 7, 2, 2, 1, 1, 1, 1, 2, 2, 7, 6, 3, 3, 6, 7, 2, 5, 1, 4], [9, 1, 1, 4, 2, 5, 6, 7, 3, 3, 7, 6, 1, 2, 1, 1, 1, 1, 2, 1, 6, 7, 3, 3, 7, 6, 5, 2, 4, 1], [9, 3, 1, 3, 7, 6, 9, 6, 6, 7, 3, 3, 1, 1, 1, 6, 6, 1, 1, 1, 3, 3, 7, 6, 6, 9, 6, 7, 3, 1], [3, 4, 9, 1, 6, 7, 6, 9, 7, 6, 3, 3, 1, 1, 6, 1, 1, 6, 1, 1, 3, 3, 6, 7, 9, 6, 7, 6, 1, 9], [7, 5, 4, 1, 1, 3, 4, 1, 2, 1, 1, 1, 5, 5, 2, 5, 5, 2, 5, 5, 1, 1, 1, 2, 1, 4, 3, 1, 1, 4], [5, 7, 1, 4, 9, 1, 1, 4, 2, 2, 1, 1, 5, 5, 5, 2, 2, 5, 5, 5, 1, 1, 2, 2, 4, 1, 1, 9, 4, 1], [4, 1, 9, 9, 4, 3, 1, 3, 1, 1, 1, 6, 2, 5, 5, 5, 5, 5, 5, 2, 6, 1, 1, 1, 3, 1, 3, 4, 9, 9], [1, 4, 9, 9, 3, 9, 9, 1, 1, 1, 6, 1, 5, 2, 5, 5, 5, 5, 2, 5, 1, 6, 1, 1, 1, 9, 9, 3, 9, 9], [7, 6, 6, 6, 9, 6, 1, 3, 1, 3, 4, 1, 9, 6, 7, 6, 6, 7, 6, 9, 1, 4, 3, 1, 3, 1, 6, 9, 6, 6], [6, 7, 6, 6, 1, 9, 3, 1, 9, 1, 1, 4, 6, 9, 6, 7, 7, 6, 9, 6, 4, 1, 1, 9, 1, 3, 9, 1, 6, 6], [6, 6, 7, 6, 1, 3, 9, 1, 9, 3, 1, 3, 7, 6, 5, 2, 2, 5, 6, 7, 3, 1, 3, 9, 1, 9, 3, 1, 6, 7], [6, 6, 6, 7, 3, 1, 5, 9, 3, 4, 9, 1, 6, 7, 2, 5, 5, 2, 7, 6, 1, 9, 4, 3, 9, 5, 1, 3, 7, 6], [9, 1, 1, 3, 1, 7, 6, 5, 9, 9, 4, 1, 1, 3, 4, 1, 1, 4, 3, 1, 1, 4, 9, 9, 5, 6, 7, 1, 3, 1], [6, 9, 3, 1, 7, 1, 5, 6, 9, 9, 1, 4, 9, 1, 1, 4, 4, 1, 1, 9, 4, 1, 9, 9, 6, 5, 1, 7, 1, 3]], "output": [[6, 9, 6, 6], [9, 3, 9, 9], [3, 4, 9, 9], [1, 9, 4, 1]]}], "test": [{"input": [[4, 4, 1, 3, 5, 7, 7, 9, 6, 1, 6, 6, 4, 4, 7, 7, 7, 7, 4, 4, 6, 6, 1, 6, 9, 7, 7, 5, 3, 1], [4, 4, 3, 3, 7, 5, 9, 7, 6, 6, 6, 6, 4, 4, 7, 2, 2, 7, 4, 4, 6, 6, 6, 6, 7, 9, 5, 7, 3, 3], [3, 4, 4, 4, 7, 9, 5, 7, 5, 1, 6, 1, 7, 7, 9, 9, 9, 9, 7, 7, 1, 6, 1, 5, 7, 5, 9, 7, 4, 4], [4, 3, 4, 4, 9, 7, 7, 5, 1, 5, 6, 6, 7, 2, 1, 9, 9, 1, 2, 7, 6, 6, 5, 1, 5, 7, 7, 9, 4, 4], [9, 7, 7, 4, 4, 4, 3, 3, 4, 4, 7, 7, 9, 7, 3, 2, 2, 3, 7, 9, 7, 7, 4, 4, 3, 3, 4, 4, 4, 7], [7, 9, 4, 7, 4, 4, 3, 1, 4, 4, 7, 2, 7, 9, 2, 3, 3, 2, 9, 7, 2, 7, 4, 4, 1, 3, 4, 4, 7, 4], [7, 4, 9, 7, 3, 4, 4, 4, 7, 7, 9, 1, 7, 4, 9, 7, 7, 9, 4, 7, 1, 9, 7, 7, 4, 4, 4, 3, 7, 9], [4, 7, 7, 9, 4, 3, 4, 4, 7, 2, 9, 9, 4, 7, 7, 9, 9, 7, 7, 4, 9, 9, 2, 7, 4, 4, 3, 4, 9, 7], [6, 6, 5, 1, 4, 4, 7, 7, 7, 2, 2, 6, 4, 6, 2, 2, 2, 2, 6, 4, 6, 2, 2, 7, 7, 7, 4, 4, 1, 5], [1, 6, 1, 5, 4, 4, 7, 2, 3, 7, 6, 6, 6, 4, 2, 2, 2, 2, 4, 6, 6, 6, 7, 3, 2, 7, 4, 4, 5, 1], [6, 6, 6, 6, 7, 7, 9, 9, 9, 1, 7, 2, 2, 2, 4, 6, 6, 4, 2, 2, 2, 7, 1, 9, 9, 9, 7, 7, 6, 6], [6, 6, 1, 6, 7, 2, 1, 9, 1, 5, 3, 7, 2, 2, 6, 4, 4, 6, 2, 2, 7, 3, 5, 1, 9, 1, 2, 7, 6, 1], [4, 4, 7, 7, 9, 7, 7, 4, 9, 9, 1, 6, 7, 2, 6, 6, 6, 6, 2, 7, 6, 1, 9, 9, 4, 7, 7, 9, 7, 7], [4, 4, 7, 2, 7, 9, 4, 7, 9, 9, 6, 1, 3, 7, 6, 2, 2, 6, 7, 3, 1, 6, 9, 9, 7, 4, 9, 7, 2, 7], [8, 8, 8, 1, 3, 2, 9, 7, 1, 6, 9, 9, 5, 1, 7, 2, 2, 7, 1, 5, 9, 9, 6, 1, 7, 9, 2, 3, 1, 9], [8, 8, 8, 9, 2, 3, 7, 9, 6, 1, 9, 9, 1, 9, 3, 7, 7, 3, 9, 1, 9, 9, 1, 6, 9, 7, 3, 2, 9, 9], [8, 8, 8, 9, 2, 3, 7, 9, 6, 1, 9, 9, 1, 9, 3, 7, 7, 3, 9, 1, 9, 9, 1, 6, 9, 7, 3, 2, 9, 9], [8, 8, 8, 1, 3, 2, 9, 7, 1, 6, 9, 9, 5, 1, 7, 2, 2, 7, 1, 5, 9, 9, 6, 1, 7, 9, 2, 3, 1, 9], [8, 8, 8, 2, 7, 9, 4, 7, 9, 9, 6, 1, 3, 7, 6, 2, 2, 6, 7, 3, 1, 6, 9, 9, 7, 4, 9, 7, 2, 7], [8, 8, 8, 7, 9, 7, 7, 4, 9, 9, 1, 6, 7, 2, 6, 6, 6, 6, 2, 7, 6, 1, 9, 9, 4, 7, 7, 9, 7, 7], [8, 8, 8, 6, 7, 2, 1, 9, 1, 5, 3, 7, 2, 2, 6, 4, 4, 6, 2, 2, 7, 3, 5, 1, 9, 1, 2, 7, 6, 1], [8, 8, 8, 6, 7, 7, 9, 9, 9, 1, 7, 2, 2, 2, 4, 6, 6, 4, 2, 2, 2, 7, 1, 9, 9, 9, 7, 7, 6, 6], [8, 8, 8, 5, 4, 4, 7, 2, 3, 7, 6, 6, 6, 4, 2, 2, 2, 2, 4, 6, 6, 6, 7, 3, 2, 7, 4, 4, 5, 1], [6, 6, 5, 1, 4, 4, 7, 7, 7, 2, 2, 6, 4, 6, 2, 2, 2, 2, 6, 4, 6, 2, 2, 7, 7, 7, 4, 4, 1, 5], [4, 7, 7, 9, 4, 3, 4, 4, 7, 2, 9, 9, 4, 7, 7, 9, 9, 7, 7, 4, 9, 9, 2, 7, 4, 4, 3, 4, 9, 7], [7, 4, 9, 7, 3, 4, 4, 4, 7, 7, 9, 1, 7, 4, 9, 7, 7, 9, 4, 7, 1, 9, 7, 7, 4, 4, 4, 3, 7, 9], [7, 9, 4, 7, 4, 4, 3, 1, 4, 4, 7, 2, 7, 9, 2, 3, 3, 2, 9, 7, 2, 7, 4, 4, 1, 3, 4, 4, 7, 4], [9, 7, 7, 4, 4, 4, 3, 3, 4, 4, 7, 7, 9, 7, 3, 2, 2, 3, 7, 9, 7, 7, 4, 4, 3, 3, 4, 4, 4, 7], [4, 3, 4, 4, 9, 7, 7, 5, 1, 5, 6, 6, 7, 2, 1, 9, 9, 1, 2, 7, 6, 6, 5, 1, 5, 7, 7, 9, 4, 4], [3, 4, 4, 4, 7, 9, 5, 7, 5, 1, 6, 1, 7, 7, 9, 9, 9, 9, 7, 7, 1, 6, 1, 5, 7, 5, 9, 7, 4, 4]]}]}, "13e47133": {"train": [{"input": [[7, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 8], [1, 8, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 1], [1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 3, 2, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], "output": [[7, 7, 7, 7, 7, 7, 7, 7, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [7, 8, 8, 8, 8, 8, 8, 7, 2, 8, 4, 4, 4, 4, 4, 4, 4, 4, 4, 8], [7, 8, 7, 7, 7, 7, 8, 7, 2, 8, 4, 8, 8, 8, 8, 8, 8, 8, 4, 8], [7, 8, 7, 8, 8, 7, 8, 7, 2, 8, 4, 8, 4, 4, 4, 4, 4, 8, 4, 8], [7, 8, 7, 8, 8, 7, 8, 7, 2, 8, 4, 8, 4, 8, 8, 8, 4, 8, 4, 8], [7, 8, 7, 8, 8, 7, 8, 7, 2, 8, 4, 8, 4, 4, 4, 4, 4, 8, 4, 8], [7, 8, 7, 8, 8, 7, 8, 7, 2, 8, 4, 8, 8, 8, 8, 8, 8, 8, 4, 8], [7, 8, 7, 8, 8, 7, 8, 7, 2, 8, 4, 4, 4, 4, 4, 4, 4, 4, 4, 8], [7, 8, 7, 8, 8, 7, 8, 7, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 8], [7, 8, 7, 8, 8, 7, 8, 7, 2, 2, 2, 2, 2, 2, 2, 2, 2, 8, 4, 8], [7, 8, 7, 8, 8, 7, 8, 7, 2, 3, 3, 3, 3, 3, 3, 3, 2, 8, 4, 8], [7, 8, 7, 8, 8, 7, 8, 7, 2, 3, 3, 3, 3, 3, 3, 3, 2, 8, 4, 8], [7, 8, 7, 8, 8, 7, 8, 7, 2, 3, 3, 3, 3, 3, 3, 3, 2, 8, 4, 8], [7, 8, 7, 8, 8, 7, 8, 7, 2, 3, 3, 3, 3, 3, 3, 3, 2, 8, 4, 8], [7, 8, 7, 8, 8, 7, 8, 7, 2, 3, 3, 3, 3, 3, 3, 3, 2, 8, 4, 8], [7, 8, 7, 8, 8, 7, 8, 7, 2, 2, 2, 2, 2, 2, 2, 2, 2, 8, 4, 8], [7, 8, 7, 8, 8, 7, 8, 7, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 8], [7, 8, 7, 7, 7, 7, 8, 7, 2, 8, 4, 4, 4, 4, 4, 4, 4, 4, 4, 8], [7, 8, 8, 8, 8, 8, 8, 7, 2, 8, 4, 4, 4, 4, 4, 4, 4, 4, 4, 8], [7, 7, 7, 7, 7, 7, 7, 7, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]]}, {"input": [[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 8, 4, 4, 4, 4, 4, 4, 4, 4], [4, 3, 4, 4, 4, 4, 4, 4, 4, 4, 2, 4, 1, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 5, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 4, 0, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 4, 4, 1, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4]], "output": [[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8], [4, 3, 3, 3, 3, 3, 3, 3, 3, 4, 2, 8, 1, 1, 1, 1, 1, 1, 1, 8], [4, 3, 4, 4, 4, 4, 4, 4, 3, 4, 2, 8, 1, 8, 8, 8, 8, 8, 1, 8], [4, 3, 4, 3, 3, 3, 3, 4, 3, 4, 2, 8, 1, 8, 1, 1, 1, 8, 1, 8], [4, 3, 4, 3, 4, 4, 3, 4, 3, 4, 2, 8, 1, 8, 1, 8, 1, 8, 1, 8], [4, 3, 4, 3, 4, 4, 3, 4, 3, 4, 2, 8, 1, 8, 1, 8, 1, 8, 1, 8], [4, 3, 4, 3, 4, 4, 3, 4, 3, 4, 2, 8, 1, 8, 1, 8, 1, 8, 1, 8], [4, 3, 4, 3, 4, 4, 3, 4, 3, 4, 2, 8, 1, 8, 1, 8, 1, 8, 1, 8], [4, 3, 4, 3, 4, 4, 3, 4, 3, 4, 2, 8, 1, 8, 1, 1, 1, 8, 1, 8], [4, 3, 4, 3, 4, 4, 3, 4, 3, 4, 2, 8, 1, 8, 8, 8, 8, 8, 1, 8], [4, 3, 4, 3, 4, 4, 3, 4, 3, 4, 2, 8, 1, 1, 1, 1, 1, 1, 1, 8], [4, 3, 4, 3, 4, 4, 3, 4, 3, 4, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8], [4, 3, 4, 3, 4, 4, 3, 4, 3, 4, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [4, 3, 4, 3, 4, 4, 3, 4, 3, 4, 2, 5, 5, 5, 5, 5, 5, 5, 5, 5], [4, 3, 4, 3, 4, 4, 3, 4, 3, 4, 2, 5, 0, 0, 0, 0, 0, 0, 0, 5], [4, 3, 4, 3, 4, 4, 3, 4, 3, 4, 2, 5, 0, 1, 1, 1, 1, 1, 0, 5], [4, 3, 4, 3, 3, 3, 3, 4, 3, 4, 2, 5, 0, 1, 5, 5, 5, 1, 0, 5], [4, 3, 4, 4, 4, 4, 4, 4, 3, 4, 2, 5, 0, 1, 1, 1, 1, 1, 0, 5], [4, 3, 3, 3, 3, 3, 3, 3, 3, 4, 2, 5, 0, 0, 0, 0, 0, 0, 0, 5], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 5, 5, 5, 5, 5, 5, 5, 5, 5]]}, {"input": [[9, 8, 8, 8, 8, 8, 6, 8, 8, 8, 8, 8, 7], [8, 1, 8, 8, 8, 8, 6, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 6, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 6, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 6, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 6, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 6, 6, 6, 6, 6, 6, 6], [8, 8, 8, 8, 8, 8, 6, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 6, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 6, 8, 8, 8, 8, 8, 8]], "output": [[9, 9, 9, 9, 9, 9, 6, 7, 7, 7, 7, 7, 7], [9, 1, 1, 1, 1, 9, 6, 7, 7, 7, 7, 7, 7], [9, 1, 9, 9, 1, 9, 6, 7, 7, 7, 7, 7, 7], [9, 1, 9, 9, 1, 9, 6, 7, 7, 7, 7, 7, 7], [9, 1, 9, 9, 1, 9, 6, 7, 7, 7, 7, 7, 7], [9, 1, 9, 9, 1, 9, 6, 7, 7, 7, 7, 7, 7], [9, 1, 9, 9, 1, 9, 6, 6, 6, 6, 6, 6, 6], [9, 1, 9, 9, 1, 9, 6, 8, 8, 8, 8, 8, 8], [9, 1, 1, 1, 1, 9, 6, 8, 8, 8, 8, 8, 8], [9, 9, 9, 9, 9, 9, 6, 8, 8, 8, 8, 8, 8]]}], "test": [{"input": [[1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 6, 3, 3, 3, 3, 3, 3, 3, 8, 2, 3, 3, 3, 3, 3, 3, 3, 3], [3, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 3, 4, 3, 3, 3, 3, 3, 3, 8, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 3, 3, 5, 3, 3, 3, 3, 3, 8, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 3, 3, 3, 3, 3, 3, 3, 3, 8, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 3, 3, 3, 3, 3, 3, 3, 3, 8, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 3, 3, 3, 3, 3, 3, 3, 3, 8, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 3, 3, 3, 3, 3, 3, 3, 3, 8, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 3, 3, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 3, 3, 3, 3, 3, 8, 4, 3, 3, 3, 3, 8, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 3, 3, 3, 3, 3, 8, 3, 1, 3, 3, 3, 8, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 3, 3, 3, 3, 3, 8, 3, 3, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 3, 3, 3, 3, 3, 8, 3, 3, 3, 3, 3, 8, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 3, 3, 3, 3, 3, 8, 3, 3, 3, 3, 3, 8, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 3, 3, 3, 3, 3, 8, 3, 3, 3, 3, 3, 8, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 3, 3, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 8, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 8, 3, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 8, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 8, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]]}, {"input": [[8, 8, 8, 8, 8, 8, 4, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 3, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 1, 8, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 1, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [8, 8, 8, 8, 8, 8, 6, 1, 8, 8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 5, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 4, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 7, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 2, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 1, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 1, 8, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 1, 1, 1, 1, 1], [8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 9, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 8, 2, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8]]}]}, "16b78196": {"train": [{"input": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 4, 0, 0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 4, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 6, 6, 6, 6, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 6, 6, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [8, 8, 8, 8, 8, 0, 0, 0, 8, 8, 8, 8, 8, 8, 8, 0, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 8, 8, 8], [8, 8, 8, 8, 8, 8, 0, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 0, 8, 8, 8, 8, 8, 8, 8, 8, 0, 0, 8, 8, 8, 8, 8, 0, 0, 8, 8, 8, 8], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0], [0, 0, 0, 2, 0, 2, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 3, 3, 0, 0, 0], [0, 0, 0, 2, 2, 2, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 3, 3, 3, 0, 0, 0], [0, 0, 0, 0, 2, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 3, 3, 3, 3, 0, 0, 0], [0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], "output": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 6, 6, 6, 6, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 1, 6, 6, 6, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 2, 1, 2, 1, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 2, 2, 2, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [8, 8, 8, 8, 8, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 0, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 8, 8, 8], [8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 0, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 8, 8, 8, 8, 8, 0, 0, 8, 8, 8, 8], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 3, 3, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 4, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}, {"input": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 0, 2, 2, 2, 2, 2, 2, 2, 0, 2, 2, 2, 2, 0, 0, 2, 2, 2, 2, 2], [0, 0, 0, 2, 0, 0, 0, 0, 0, 2, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 2, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 4, 0, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 4, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], "output": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 2, 2, 0, 2, 2, 2, 2, 0, 0, 2, 2, 2, 2, 2], [0, 0, 0, 2, 0, 0, 0, 0, 0, 2, 1, 1, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 2, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 3, 3, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 4, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 3, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}], "test": [{"input": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 0, 0, 8, 8, 8, 8, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 4, 4, 4, 4, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 8, 8, 8, 8, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 4, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 8, 8, 8, 8, 0, 0, 0, 0, 0, 0], [0, 0, 4, 4, 4, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 4, 4, 4, 4, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [6, 6, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [6, 6, 6, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [6, 6, 6, 6, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 2, 2, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 2, 2, 2, 2, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 2, 2, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 2, 2, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}]}, "195c6913": {"train": [{"input": [[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 0], [3, 2, 2, 3, 2, 2, 3, 8, 8, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0], [3, 2, 2, 3, 2, 2, 3, 8, 8, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3], [3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3], [3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3], [2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 4, 4, 3, 3], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 4, 4, 3, 3], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]], "output": [[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 0], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 0, 2, 8, 2], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 0, 2, 0, 0], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 0, 0, 0, 8, 0, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 2, 2, 8, 2, 2, 4, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 2, 0, 0, 0, 0, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 2, 0, 0, 0, 3, 3, 3, 3], [3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 3, 3, 3, 3], [3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 3], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 3, 3, 3], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 3, 3, 3, 3, 3], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 3, 3, 3, 3, 3, 3], [2, 2, 8, 2, 2, 8, 2, 2, 8, 2, 2, 8, 2, 4, 3, 3, 3, 3, 3, 3], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 3, 3], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]]}, {"input": [[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8], [2, 4, 4, 2, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8], [2, 4, 4, 2, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2], [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2], [2, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2], [2, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2], [2, 2, 2, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2, 2, 2], [2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 8, 8, 2, 2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 9, 9, 2, 2], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 9, 9, 2, 2], [4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2, 2, 2], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2, 2, 2]], "output": [[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 8, 8, 8, 1, 8, 8, 8, 8], [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 8, 8, 8, 4, 8, 8, 8, 8], [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 9, 8, 8, 8, 8, 8, 1, 8, 8, 2, 2], [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 8, 8, 4, 1, 4, 1, 4, 1, 4, 9, 2, 2, 2], [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 8, 8, 8, 1, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 4, 8, 8, 2, 2, 2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 1, 8, 8, 2, 2, 2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 4, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2], [2, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2], [2, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2], [2, 2, 2, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [4, 1, 4, 1, 4, 1, 4, 1, 4, 1, 4, 1, 4, 1, 4, 9, 2, 2, 2, 2, 2, 2, 2, 2, 2], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2, 2, 2], [2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 8, 8, 2, 2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2], [4, 1, 4, 1, 4, 1, 4, 1, 4, 1, 4, 1, 4, 1, 4, 1, 4, 9, 2, 2, 2, 2, 2, 2, 2], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2, 2, 2]]}, {"input": [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 2, 2, 1, 2, 2, 1, 3, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 2, 2, 1, 2, 2, 1, 3, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 4], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 4, 4], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 4, 4, 4, 4, 4, 4], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 4, 4, 4, 4, 4], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 4, 4, 4, 4, 4], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 4, 4, 4, 4, 4, 4, 4], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 4, 4, 1, 4, 4, 4, 4, 4, 4, 4, 4, 4], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [1, 1, 1, 1, 1, 1, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 4, 4, 4], [1, 1, 1, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 4, 4, 4], [1, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 1, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 1, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 1, 1, 1, 1, 4, 4, 4, 4, 4, 1], [2, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 4, 4, 4, 1], [4, 4, 4, 4, 4, 4, 4, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 4, 4, 4, 4, 4, 4, 1], [4, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1], [1, 1, 1, 1, 1, 4, 4, 1, 1, 1, 1, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 1], [1, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 1, 1], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 8, 8, 1], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 4, 1, 1, 4, 1, 8, 8, 1], [2, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], "output": [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 4], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 8, 1, 4, 4, 4], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 4, 2, 3, 2, 2, 3], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 2, 4, 4, 4, 4], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 3, 4, 4, 4, 4], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 4, 4, 2, 4, 4, 4, 4], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 4, 4, 1, 4, 4, 4, 4, 2, 4, 4, 4, 4], [1, 1, 1, 1, 1, 1, 1, 1, 8, 1, 1, 1, 4, 4, 4, 4, 4, 4, 4, 4, 3, 4, 4, 4, 4], [1, 1, 1, 1, 1, 1, 4, 4, 2, 3, 2, 2, 3, 2, 2, 3, 2, 2, 3, 2, 2, 8, 4, 4, 4], [1, 1, 1, 4, 4, 4, 4, 4, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 4, 4, 4], [1, 4, 4, 4, 4, 4, 4, 4, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 8, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 2, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 1, 2, 2, 3, 2, 2], [4, 4, 4, 4, 4, 4, 4, 4, 2, 4, 4, 4, 4, 1, 1, 1, 1, 1, 1, 4, 3, 4, 4, 4, 1], [2, 2, 3, 2, 2, 3, 2, 2, 3, 8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 4, 4, 4, 1], [4, 4, 4, 4, 4, 4, 4, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 4, 4, 2, 4, 4, 4, 1], [4, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 4, 4, 4, 4, 4, 4, 3, 4, 4, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 4, 4, 4, 4, 4, 4, 4, 2, 4, 4, 1, 1], [1, 1, 1, 1, 1, 4, 4, 1, 1, 8, 1, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 4, 1, 1, 1], [1, 4, 4, 4, 4, 4, 4, 4, 4, 2, 2, 3, 2, 2, 3, 2, 2, 3, 2, 2, 3, 8, 1, 1, 1], [4, 4, 4, 4, 4, 4, 4, 4, 4, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 1, 1], [4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 4, 4, 4, 4, 4, 4, 1, 4, 1, 1, 4, 1, 1, 1, 1], [2, 2, 3, 2, 2, 3, 2, 2, 3, 2, 8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]}], "test": [{"input": [[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 1, 1, 4, 2, 2, 4, 8, 8, 4, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 1, 1, 4, 2, 2, 4, 8, 8, 4, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4], [4, 4, 4, 4, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4], [4, 4, 7, 7, 7, 7, 7, 4, 4, 4, 4, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4], [7, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4], [7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4, 4, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4], [1, 7, 7, 4, 4, 4, 4, 4, 4, 4, 4, 4, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4], [7, 7, 7, 4, 4, 4, 4, 4, 4, 4, 4, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4], [7, 7, 4, 4, 4, 4, 4, 4, 4, 4, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4], [7, 4, 4, 4, 4, 4, 4, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4], [7, 4, 4, 4, 4, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4, 4], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4, 4], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 6, 6, 4, 4, 4], [1, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 6, 6, 4, 4, 4], [7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [7, 7, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]]}, {"input": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8, 8, 8], [8, 3, 3, 8, 3, 3, 8, 3, 3, 8, 6, 6, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8, 8, 8], [8, 3, 3, 8, 3, 3, 8, 3, 3, 8, 6, 6, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9], [3, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 8], [9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 8], [9, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8, 8], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 7, 7, 8, 8, 8, 8], [3, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 7, 7, 8, 8, 8, 8], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]]}]}, "21897d95": {"train": [{"input": [[6, 6, 6, 6, 6, 7, 7, 7, 4, 4, 4, 4], [6, 6, 6, 6, 6, 7, 7, 7, 4, 4, 4, 4], [6, 6, 6, 1, 6, 7, 7, 7, 4, 4, 4, 4], [6, 6, 6, 3, 1, 7, 7, 7, 4, 9, 9, 9], [6, 6, 6, 1, 6, 7, 7, 7, 4, 4, 4, 9], [6, 6, 6, 6, 6, 7, 7, 7, 4, 4, 4, 9], [6, 6, 6, 6, 6, 7, 1, 7, 4, 4, 4, 4], [6, 6, 6, 6, 6, 7, 1, 1, 4, 4, 4, 4], [6, 6, 6, 6, 6, 7, 1, 7, 4, 4, 4, 4], [6, 6, 6, 6, 6, 7, 7, 7, 4, 4, 4, 4], [7, 7, 1, 7, 7, 7, 7, 7, 7, 7, 7, 7], [7, 1, 0, 1, 7, 7, 7, 7, 7, 1, 1, 1], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 1, 7], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]], "output": [[7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 3, 3, 3, 7, 7, 7], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 3, 3, 3, 7, 7, 7], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 3, 3, 3, 7, 7, 7], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 3, 3, 3, 7, 7, 7], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 7, 7, 7], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 7, 7, 7], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 7, 7, 7], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 7, 7, 7], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 7, 7, 7], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 7, 7, 7], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 7, 7, 7], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 7, 7, 7]]}, {"input": [[7, 7, 7, 7, 7, 7, 1, 7, 3, 3], [7, 7, 7, 7, 7, 7, 1, 1, 3, 3], [7, 7, 7, 7, 7, 7, 1, 7, 3, 3], [8, 8, 8, 3, 1, 7, 7, 7, 3, 1], [8, 8, 8, 1, 1, 7, 7, 7, 1, 6], [8, 1, 8, 3, 1, 7, 7, 7, 3, 1], [8, 1, 1, 3, 3, 1, 9, 1, 3, 3], [8, 1, 8, 3, 3, 7, 1, 7, 3, 3], [8, 8, 8, 3, 3, 5, 5, 5, 3, 3], [8, 8, 8, 3, 3, 5, 5, 5, 3, 3]], "output": [[6, 6, 6, 6, 6, 6, 6, 6, 7, 7], [6, 6, 6, 6, 6, 6, 6, 6, 7, 7], [6, 6, 6, 6, 6, 6, 6, 6, 7, 7], [3, 3, 3, 8, 8, 6, 6, 6, 7, 7], [3, 3, 3, 8, 8, 6, 6, 6, 7, 7], [3, 3, 3, 8, 8, 6, 6, 6, 7, 7], [3, 3, 3, 8, 8, 6, 6, 6, 7, 7], [3, 3, 3, 8, 8, 6, 6, 6, 7, 7], [3, 3, 3, 8, 8, 9, 9, 9, 7, 7], [3, 3, 3, 8, 8, 9, 9, 9, 7, 7]]}, {"input": [[6, 6, 6, 3, 1, 3, 3, 3, 3], [6, 6, 6, 1, 1, 3, 3, 3, 3], [6, 6, 6, 3, 1, 3, 3, 3, 3], [6, 6, 6, 4, 4, 4, 4, 1, 4], [6, 6, 6, 4, 4, 4, 1, 1, 1], [6, 6, 6, 4, 4, 4, 4, 4, 4], [6, 6, 6, 6, 1, 6, 4, 4, 4], [6, 6, 6, 6, 1, 1, 4, 4, 4], [6, 6, 6, 6, 1, 6, 4, 4, 4], [5, 5, 5, 4, 4, 4, 4, 4, 4], [9, 9, 9, 8, 8, 8, 8, 8, 8], [9, 5, 5, 8, 8, 8, 1, 7, 1], [9, 5, 5, 8, 8, 8, 8, 1, 8], [5, 5, 5, 4, 4, 4, 4, 4, 4]], "output": [[5, 5, 5, 5, 5, 3, 3, 3, 3, 3, 3, 3, 3, 3], [5, 5, 5, 5, 5, 3, 3, 3, 3, 3, 3, 3, 3, 3], [5, 5, 5, 5, 5, 3, 3, 3, 3, 3, 3, 3, 3, 3], [7, 8, 8, 8, 6, 3, 3, 3, 6, 6, 6, 4, 4, 4], [7, 8, 8, 8, 6, 3, 3, 3, 6, 6, 6, 4, 4, 4], [7, 8, 8, 8, 6, 3, 3, 3, 6, 6, 6, 4, 4, 4], [7, 8, 8, 8, 6, 6, 6, 6, 6, 6, 6, 4, 4, 4], [7, 8, 8, 8, 6, 6, 6, 6, 6, 6, 6, 4, 4, 4], [7, 8, 8, 8, 6, 6, 6, 6, 6, 6, 6, 4, 4, 4]]}, {"input": [[2, 2, 2, 2, 2, 1, 2, 3, 3, 3], [2, 2, 2, 2, 2, 1, 1, 3, 3, 3], [2, 2, 2, 2, 2, 1, 2, 3, 3, 3], [6, 6, 6, 6, 1, 6, 6, 3, 1, 3], [6, 6, 6, 1, 1, 1, 6, 1, 1, 3], [1, 1, 1, 6, 6, 6, 6, 3, 1, 3], [6, 1, 6, 6, 6, 6, 6, 3, 3, 3], [4, 4, 4, 4, 4, 4, 4, 3, 3, 3], [4, 4, 4, 4, 4, 4, 4, 3, 3, 3], [4, 4, 4, 4, 4, 4, 4, 3, 3, 3]], "output": [[6, 6, 6, 6, 6, 6, 6, 2, 2, 2], [6, 6, 6, 6, 6, 6, 6, 2, 2, 2], [6, 6, 6, 6, 6, 6, 6, 2, 2, 2], [3, 3, 3, 3, 3, 3, 3, 2, 2, 2], [3, 3, 3, 3, 3, 3, 3, 2, 2, 2], [3, 3, 3, 3, 3, 3, 3, 2, 2, 2], [3, 3, 3, 3, 3, 3, 3, 2, 2, 2], [6, 6, 6, 6, 6, 6, 6, 2, 2, 2], [6, 6, 6, 6, 6, 6, 6, 2, 2, 2], [6, 6, 6, 6, 6, 6, 6, 2, 2, 2]]}], "test": [{"input": [[9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 7, 7, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 3, 3, 3, 1, 3, 3, 3, 3, 3, 3, 9, 9, 9, 7, 7, 7, 7, 9, 9], [9, 9, 9, 9, 3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 1, 1, 3, 3, 3, 3, 3, 9, 9, 9, 7, 7, 7, 7, 9, 9], [9, 9, 9, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 9, 9, 7, 7, 7, 7, 9, 9], [9, 9, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 9, 9, 9, 1, 9, 9, 9, 9], [9, 9, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 9, 9, 1, 1, 1, 9, 9, 4], [9, 3, 3, 3, 3, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 9, 9, 9, 9, 9, 9, 4, 4], [9, 3, 3, 3, 0, 1, 8, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 9, 9, 9, 9, 9, 4, 4, 4], [9, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 1, 2, 2, 0, 0, 0, 0, 0, 9, 9, 9, 9, 9, 4, 4, 4], [9, 3, 3, 0, 0, 0, 0, 0, 0, 0, 2, 2, 2, 1, 1, 1, 2, 2, 2, 2, 0, 0, 9, 9, 9, 9, 4, 4, 4, 4], [9, 3, 3, 0, 0, 0, 0, 0, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 0, 9, 9, 9, 9, 4, 4, 4, 4], [9, 3, 3, 0, 0, 0, 0, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 9, 9, 9, 9, 4, 4, 4, 4], [9, 3, 3, 0, 0, 0, 0, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 9, 9, 4, 4, 4, 4, 4], [6, 6, 6, 0, 0, 0, 0, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 9, 9, 4, 4, 4, 4, 4], [6, 6, 6, 0, 0, 0, 0, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 9, 9, 4, 4, 4, 4, 4], [6, 6, 6, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 2, 2, 2, 2, 2, 2, 9, 9, 4, 4, 4, 4, 4], [6, 6, 6, 6, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 9, 4, 4, 4, 4, 4], [6, 6, 6, 6, 6, 6, 0, 1, 0, 0, 0, 0, 0, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 9, 4, 4, 4, 4, 4], [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 0, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 9, 4, 4, 4, 4, 4], [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 1, 6, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 9, 9, 4, 4, 4, 4], [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 9, 9, 9, 4, 4, 4, 4], [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 1, 6, 9, 2, 2, 2, 2, 2, 2, 2, 2, 9, 9, 9, 8, 8, 4, 4], [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 9, 2, 2, 2, 2, 2, 2, 2, 9, 9, 9, 8, 8, 8, 4, 4], [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 9, 9, 9, 2, 2, 2, 2, 9, 9, 9, 9, 8, 8, 8, 8, 4, 4], [6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8, 1, 8], [6, 6, 6, 6, 6, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 1, 9, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 0, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 1, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]]}, {"input": [[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 1, 1, 1, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 1, 5, 5], [3, 3, 1, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 6, 6, 6, 6, 6], [3, 1, 1, 1, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 6, 6, 6, 6, 6], [3, 3, 3, 3, 3, 3, 8, 8, 8, 9, 9, 9, 8, 8, 8, 8, 8, 8, 6, 6, 6, 6, 6, 6], [3, 3, 3, 3, 3, 3, 8, 8, 8, 8, 8, 9, 8, 8, 8, 8, 8, 8, 6, 6, 6, 6, 6, 6], [3, 3, 3, 3, 3, 3, 8, 8, 8, 8, 8, 9, 8, 8, 8, 8, 8, 8, 6, 6, 6, 6, 6, 6], [3, 3, 3, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 6, 6, 6, 6, 6], [3, 3, 3, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 6, 6, 6, 6, 6], [3, 3, 3, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 6, 6, 6, 6, 6], [3, 3, 3, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 6, 6, 6, 6, 6], [3, 3, 3, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 6, 6, 6, 6, 6], [3, 3, 3, 3, 3, 3, 8, 1, 7, 1, 8, 8, 8, 8, 8, 8, 8, 8, 6, 6, 6, 6, 6, 6], [3, 3, 3, 3, 3, 3, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 6, 6, 6, 6, 6], [3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 1, 4, 4, 4, 4, 4, 6, 6, 6, 6, 6, 6], [3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 1, 1, 1, 4, 4, 4, 4, 6, 6, 6, 6, 6, 6], [3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 6, 6, 1, 1, 1, 6], [3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 6, 6, 6, 1, 6, 6], [2, 2, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [2, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]]}]}, "221dfab4": {"train": [{"input": [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 8, 8, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 8, 8, 8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 1], [1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 8, 8, 8, 8, 8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 8, 8, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 8, 8, 8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 8, 8, 1, 1, 1, 1, 1], [1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 8, 8, 8, 1, 1, 1, 1], [1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 8, 8, 8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 8, 8, 8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 8, 8, 8, 8, 8, 1, 1, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8, 8, 1, 1, 1], [1, 1, 8, 8, 8, 8, 8, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1], [1, 1, 1, 1, 8, 8, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 8, 8, 8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 4, 4, 4, 1, 1, 1, 1, 1, 1, 1]], "output": [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 3, 3, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8, 4, 4, 4, 4, 8, 8, 8, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 8, 8, 8, 1, 1, 1, 1, 8, 8, 8, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 4, 4, 4, 8, 8, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 3, 3, 3, 1, 1, 1, 1, 3, 3, 3, 3, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 4, 4, 4, 4, 8, 8, 1, 1, 1, 1, 1], [1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 8, 8, 8, 8, 8, 1, 1, 1, 4, 4, 4, 4, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 8, 8, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 8, 8, 1, 1, 1, 1, 8, 8, 8, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 4, 4, 4, 8, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 8, 8, 8, 1, 1, 1, 1, 4, 4, 4, 4, 1, 1, 1, 1, 1, 1, 1], [1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 8, 8, 1, 1, 1, 1, 1], [1, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 1, 1, 1], [1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 8, 8, 8, 1, 4, 4, 4, 4, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 8, 8, 8, 1, 1, 1, 1, 1, 1, 4, 4, 4, 4, 1, 1, 1, 1, 1, 1, 1], [1, 8, 8, 8, 8, 8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8, 1, 1, 1], [1, 1, 3, 3, 3, 3, 3, 1, 1, 1, 1, 3, 3, 3, 3, 3, 3, 3, 3, 1, 1, 1], [1, 1, 1, 1, 8, 8, 1, 1, 8, 8, 8, 1, 1, 1, 1, 8, 8, 8, 8, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 8, 8, 4, 4, 4, 4, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 4, 4, 4, 1, 1, 1, 1, 1, 1, 1]]}, {"input": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 2, 2, 2, 8, 2, 2, 2, 8, 8, 8, 8, 8, 2, 2, 2, 2, 8, 8], [8, 8, 8, 8, 2, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 2, 2, 2, 2, 2, 8, 8], [8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 8, 8], [8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8], [8, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 2, 2, 2, 8, 8, 8, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8], [8, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2, 2, 2, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [2, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [2, 8, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2, 2], [8, 8, 8, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2, 2, 8, 8], [8, 8, 8, 8, 2, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 2, 2, 2, 2, 8, 2, 2], [8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 8], [8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 2, 2, 2, 8, 8], [8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 4, 4, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]], "output": [[8, 8, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 4, 4, 4, 2, 2, 8, 2, 2, 2, 8, 8, 8, 8, 8, 2, 2, 2, 2, 8, 8], [8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 2, 2, 2, 2, 2, 8, 8], [8, 8, 4, 4, 4, 8, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 8, 8], [8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8], [8, 8, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 4, 4, 4, 8, 2, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 2, 2, 2, 8, 8, 8, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8], [8, 2, 4, 4, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 2, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 3, 3, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2, 2, 2, 8, 8], [8, 8, 4, 4, 4, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8], [8, 8, 4, 4, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [3, 3, 3, 3, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [2, 8, 8, 8, 8, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2, 2], [8, 8, 4, 4, 4, 2, 2, 2, 2, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2, 2, 8, 8], [8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 2, 2, 2, 2, 8, 2, 2], [8, 8, 4, 4, 4, 8, 8, 8, 8, 2, 2, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 8], [8, 8, 3, 3, 3, 8, 3, 3, 3, 3, 3, 8, 8, 8, 8, 8, 8, 3, 3, 3, 8, 8], [8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 8, 8, 8], [8, 8, 4, 4, 4, 8, 8, 8, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 4, 4, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]]}], "test": [{"input": [[8, 8, 8, 8, 8, 8, 8, 4, 4, 4, 4, 4, 4, 4, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 6, 6, 6, 6, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 6, 6, 6, 6, 6, 6, 6, 6, 8, 8, 8, 8, 6, 6, 6, 6, 6, 8, 8, 8, 8, 8], [8, 8, 8, 8, 6, 6, 6, 6, 6, 6, 8, 8, 8, 6, 6, 6, 6, 6, 6, 6, 6, 6, 8, 8, 8], [8, 8, 8, 8, 8, 6, 6, 8, 8, 8, 8, 8, 8, 8, 8, 6, 6, 6, 6, 6, 6, 6, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 6, 6, 6, 6, 8, 6, 6, 6, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 6, 6, 6, 6, 6, 6, 6, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 6, 8, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 6, 6, 6, 6, 6, 6, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 6, 6, 6, 6, 6, 6, 6, 6, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 6, 6, 6, 6, 6, 6, 6, 8, 8], [8, 8, 8, 8, 6, 6, 6, 6, 6, 6, 6, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 6, 8, 8, 8], [8, 8, 8, 6, 6, 6, 6, 6, 6, 6, 6, 6, 8, 8, 8, 8, 8, 8, 6, 6, 6, 6, 8, 8, 8], [8, 8, 6, 6, 6, 6, 6, 6, 6, 6, 8, 8, 8, 8, 8, 8, 8, 6, 6, 6, 6, 6, 6, 6, 6], [8, 8, 8, 8, 6, 6, 6, 6, 6, 6, 8, 8, 8, 8, 8, 8, 8, 8, 6, 6, 6, 6, 6, 6, 6], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 6, 6, 6, 6, 6], [8, 8, 8, 8, 8, 6, 6, 8, 8, 8, 6, 6, 6, 6, 6, 6, 6, 8, 8, 6, 6, 6, 6, 6, 8], [8, 8, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 6, 6, 6, 6, 6, 6, 6, 6, 6, 8, 8, 8, 8, 8, 8, 8, 8]]}, {"input": [[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 2, 2], [2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 2], [2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 2], [2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1, 2], [4, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 1, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 2], [4, 2, 2, 2, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 2, 2, 2, 1, 1, 1, 2], [4, 2, 2, 1, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 2, 2, 2, 1, 1, 1, 2], [4, 2, 2, 1, 1, 1, 1, 2, 2, 2, 2, 1, 1, 1, 2, 2, 2, 2, 1, 1, 1, 1, 2, 2, 2, 2, 1, 1, 2, 2], [4, 2, 2, 1, 1, 1, 1, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 1, 1, 1, 2, 2, 2, 2, 2, 1, 1, 2, 2], [2, 2, 2, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2], [2, 2, 2, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2], [2, 2, 2, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2], [2, 2, 2, 1, 1, 2, 2, 2, 2, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 2, 2, 2, 2], [2, 1, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 2, 2, 2, 2], [2, 2, 1, 1, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, 2, 1, 2, 2, 2, 2, 2, 1, 1, 1, 2, 2, 2, 2], [2, 2, 1, 1, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 1, 1, 2, 2, 2, 2, 1, 1, 1, 1, 2, 2, 2, 2], [2, 2, 2, 2, 2, 1, 1, 2, 1, 1, 1, 1, 2, 2, 2, 2, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 2, 2, 2, 2, 1, 1, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2], [2, 2, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2], [2, 2, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2], [2, 2, 2, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 2, 2, 2], [2, 2, 2, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 2, 2, 2], [2, 2, 2, 1, 1, 1, 1, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 1, 1, 1, 2, 2, 2, 2], [2, 2, 1, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2], [2, 2, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 2, 2, 2, 2], [2, 2, 2, 1, 1, 1, 2, 2, 2, 2, 2, 2, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]]}]}, "269e22fb": {"train": [{"input": [[7, 7, 8, 8, 8, 8, 8, 8, 8, 8], [7, 7, 7, 7, 7, 7, 8, 8, 8, 8], [7, 7, 7, 7, 7, 7, 7, 8, 8, 8], [7, 7, 8, 8, 8, 7, 7, 7, 8, 8], [8, 8, 8, 8, 8, 8, 7, 7, 7, 8], [8, 8, 8, 8, 8, 8, 8, 7, 7, 7], [8, 8, 8, 8, 8, 8, 8, 8, 7, 7], [8, 8, 8, 8, 8, 8, 8, 8, 8, 7]], "output": [[7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 7, 7, 8, 8, 8], [7, 7, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 7, 7, 8, 8, 8], [7, 7, 7, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 7, 7, 7, 8, 8], [7, 7, 8, 8, 8, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 7, 7, 7, 8, 8], [8, 8, 8, 8, 8, 8, 7, 7, 7, 8, 8, 8, 8, 8, 7, 7, 7, 7, 7, 8], [8, 8, 8, 8, 8, 8, 8, 7, 7, 7, 8, 8, 8, 8, 7, 8, 8, 7, 7, 8], [8, 8, 8, 8, 8, 8, 8, 8, 7, 7, 8, 8, 7, 7, 7, 8, 8, 7, 7, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 7, 8, 8, 7, 8, 7, 8, 8, 7, 7, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 7, 7, 7, 7, 8, 7, 8, 8, 7, 7, 8], [8, 7, 7, 7, 7, 7, 7, 7, 7, 7, 8, 7, 7, 8, 7, 8, 8, 7, 7, 8], [8, 7, 8, 8, 8, 8, 8, 8, 8, 7, 7, 7, 7, 8, 7, 8, 8, 7, 7, 8], [8, 7, 7, 7, 7, 7, 7, 7, 7, 7, 8, 8, 7, 8, 7, 8, 8, 7, 7, 8], [8, 7, 8, 7, 8, 8, 8, 8, 8, 7, 8, 8, 7, 7, 7, 8, 8, 7, 7, 8], [7, 7, 7, 8, 7, 7, 7, 7, 7, 7, 8, 8, 8, 8, 7, 8, 8, 7, 7, 8], [8, 7, 8, 7, 7, 8, 8, 8, 8, 7, 8, 8, 8, 8, 7, 7, 7, 7, 7, 8], [7, 7, 7, 8, 7, 8, 8, 8, 8, 7, 8, 8, 8, 7, 7, 8, 7, 7, 8, 8], [8, 7, 8, 7, 7, 8, 8, 8, 8, 7, 8, 8, 8, 7, 8, 8, 8, 7, 7, 8], [7, 7, 7, 8, 7, 8, 8, 8, 8, 7, 8, 8, 8, 7, 7, 8, 8, 8, 7, 7], [8, 7, 8, 7, 7, 8, 8, 8, 7, 8, 7, 8, 8, 8, 7, 8, 7, 7, 7, 8], [7, 7, 7, 8, 8, 8, 8, 7, 8, 8, 8, 7, 8, 8, 7, 7, 7, 8, 8, 8]]}, {"input": [[8, 8, 7, 8, 8, 8, 7, 8], [8, 8, 8, 7, 8, 7, 8, 8], [7, 8, 8, 8, 7, 8, 8, 8], [7, 8, 8, 8, 7, 8, 8, 8], [7, 8, 8, 8, 7, 8, 8, 8], [8, 8, 8, 8, 7, 8, 8, 8], [8, 8, 8, 8, 7, 7, 7, 7], [7, 7, 8, 8, 7, 8, 8, 8], [8, 7, 8, 8, 7, 7, 7, 7], [8, 7, 7, 7, 7, 8, 8, 8], [8, 7, 7, 8, 7, 7, 7, 7], [8, 7, 7, 7, 7, 8, 8, 8], [8, 7, 8, 8, 7, 8, 8, 8], [7, 7, 8, 8, 7, 7, 8, 8], [8, 8, 8, 8, 7, 7, 7, 8], [8, 8, 8, 8, 8, 7, 7, 7]], "output": [[8, 8, 8, 7, 7, 7, 8, 8, 7, 8, 8, 8, 7, 8, 8, 8, 8, 7, 7, 7], [8, 7, 7, 7, 8, 7, 8, 8, 8, 7, 8, 7, 8, 8, 8, 7, 7, 8, 7, 8], [7, 7, 8, 8, 8, 7, 7, 8, 8, 8, 7, 8, 8, 8, 8, 7, 8, 7, 7, 7], [8, 7, 7, 8, 8, 8, 7, 8, 8, 8, 7, 8, 8, 8, 8, 7, 7, 8, 7, 8], [8, 8, 7, 7, 8, 7, 7, 8, 8, 8, 7, 8, 8, 8, 8, 7, 8, 7, 7, 7], [8, 7, 7, 7, 7, 7, 8, 8, 8, 8, 7, 8, 8, 8, 8, 7, 7, 8, 7, 8], [8, 7, 7, 8, 8, 7, 8, 8, 8, 8, 7, 7, 7, 7, 7, 7, 8, 7, 7, 7], [8, 7, 7, 8, 8, 7, 7, 7, 8, 8, 7, 8, 8, 8, 8, 8, 7, 8, 7, 8], [8, 7, 7, 8, 8, 7, 8, 7, 8, 8, 7, 7, 7, 7, 7, 7, 7, 7, 7, 8], [8, 7, 7, 8, 8, 7, 8, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 7, 8], [8, 7, 7, 8, 8, 7, 8, 7, 7, 8, 7, 7, 7, 7, 7, 7, 7, 7, 7, 8], [8, 7, 7, 8, 8, 7, 8, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 7, 7, 8, 8, 7, 8, 7, 8, 8, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 7, 7, 8, 8, 7, 7, 7, 8, 8, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8], [8, 7, 7, 8, 8, 7, 8, 8, 8, 8, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8], [8, 7, 7, 7, 7, 7, 8, 8, 8, 8, 8, 7, 7, 7, 8, 8, 8, 8, 8, 8], [8, 8, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 7, 7, 7, 8, 8, 8, 7, 7], [8, 8, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 7, 7, 7, 7, 7, 7, 7], [8, 8, 8, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 7, 7, 7, 7, 7, 7], [8, 8, 8, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 7, 7]]}, {"input": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 3, 3, 3, 3, 3, 3], [0, 0, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 0, 0, 0, 0, 0], [3, 3, 3, 3, 3, 0, 0, 0, 0, 0], [0, 0, 0, 0, 3, 3, 3, 3, 3, 3], [0, 0, 0, 0, 0, 0, 3, 0, 0, 0], [0, 0, 0, 0, 0, 0, 3, 3, 3, 3], [0, 0, 0, 0, 0, 0, 0, 0, 3, 3], [0, 0, 0, 0, 0, 0, 0, 0, 3, 0]], "output": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0], [0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 0, 3, 3, 3, 0], [0, 0, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 0, 3, 0], [3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 0, 0, 3, 3], [3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 3], [0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 0, 3, 3, 3], [0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 3, 0, 0, 3, 3, 3, 0, 0], [0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 3], [0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 3, 0, 0, 0, 0, 0, 0, 0, 3, 0], [0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 0, 0], [0, 0, 0, 0, 3, 3, 3, 0, 0, 3, 0, 3, 0, 3, 0, 0, 0, 0, 3, 0], [0, 0, 0, 3, 3, 3, 0, 0, 0, 3, 0, 3, 0, 3, 0, 0, 0, 0, 0, 3], [0, 0, 3, 3, 3, 0, 0, 0, 0, 3, 0, 3, 0, 3, 0, 0, 0, 0, 0, 0], [0, 3, 3, 3, 0, 0, 0, 0, 0, 3, 0, 3, 0, 3, 0, 0, 0, 0, 0, 0], [0, 3, 3, 0, 0, 0, 0, 0, 0, 3, 0, 3, 0, 3, 3, 3, 3, 3, 3, 0], [0, 3, 3, 0, 0, 0, 0, 0, 0, 3, 0, 3, 3, 0, 3, 0, 3, 0, 3, 0], [0, 3, 3, 0, 0, 0, 0, 0, 0, 3, 0, 3, 0, 3, 0, 3, 0, 3, 0, 3], [3, 3, 3, 3, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 3, 0, 3, 0, 3]]}, {"input": [[8, 8, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8], [8, 8, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8], [8, 7, 7, 7, 7, 7, 8, 8, 8, 8, 8, 7], [8, 7, 7, 8, 8, 7, 8, 8, 8, 8, 7, 7], [8, 7, 7, 8, 8, 7, 7, 7, 8, 8, 7, 7], [8, 7, 7, 8, 8, 7, 8, 7, 8, 8, 7, 8], [8, 7, 7, 8, 8, 7, 8, 7, 7, 7, 7, 8], [8, 7, 7, 8, 8, 7, 8, 7, 7, 8, 7, 7], [8, 7, 7, 8, 8, 7, 8, 7, 7, 7, 7, 8], [8, 7, 7, 8, 8, 7, 8, 7, 8, 8, 7, 7], [8, 7, 7, 8, 8, 7, 7, 7, 8, 8, 7, 8], [8, 7, 7, 8, 8, 7, 8, 8, 8, 8, 7, 7], [8, 7, 7, 7, 7, 7, 8, 8, 8, 8, 7, 8]], "output": [[8, 8, 8, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 7, 7], [8, 8, 8, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 7, 7, 7, 7, 7, 7], [8, 8, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 7, 7, 7, 7, 7, 7, 7], [8, 8, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 7, 7, 7, 8, 8, 8, 7, 7], [8, 7, 7, 7, 7, 7, 8, 8, 8, 8, 8, 7, 7, 7, 8, 8, 8, 8, 8, 8], [8, 7, 7, 8, 8, 7, 8, 8, 8, 8, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8], [8, 7, 7, 8, 8, 7, 7, 7, 8, 8, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8], [8, 7, 7, 8, 8, 7, 8, 7, 8, 8, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 7, 7, 8, 8, 7, 8, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 7, 7, 8, 8, 7, 8, 7, 7, 8, 7, 7, 7, 7, 7, 7, 7, 7, 7, 8], [8, 7, 7, 8, 8, 7, 8, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 7, 8], [8, 7, 7, 8, 8, 7, 8, 7, 8, 8, 7, 7, 7, 7, 7, 7, 7, 7, 7, 8], [8, 7, 7, 8, 8, 7, 7, 7, 8, 8, 7, 8, 8, 8, 8, 8, 7, 8, 7, 8], [8, 7, 7, 8, 8, 7, 8, 8, 8, 8, 7, 7, 7, 7, 7, 7, 8, 7, 7, 7], [8, 7, 7, 7, 7, 7, 8, 8, 8, 8, 7, 8, 8, 8, 8, 7, 7, 8, 7, 8], [8, 8, 7, 7, 8, 7, 7, 8, 8, 8, 7, 8, 8, 8, 8, 7, 8, 7, 7, 7], [8, 7, 7, 8, 8, 8, 7, 8, 8, 8, 7, 8, 8, 8, 8, 7, 7, 8, 7, 8], [7, 7, 8, 8, 8, 7, 7, 8, 8, 8, 7, 8, 8, 8, 8, 7, 8, 7, 7, 7], [8, 7, 7, 7, 8, 7, 8, 8, 8, 7, 8, 7, 8, 8, 8, 7, 7, 8, 7, 8], [8, 8, 8, 7, 7, 7, 8, 8, 7, 8, 8, 8, 7, 8, 8, 8, 8, 7, 7, 7]]}, {"input": [[8, 7, 8, 7, 8, 8, 8, 8, 8, 7], [8, 7, 8, 7, 8, 8, 8, 8, 8, 8], [8, 7, 8, 7, 8, 8, 8, 8, 8, 8], [8, 7, 8, 7, 7, 7, 7, 7, 7, 8], [8, 7, 7, 8, 7, 8, 7, 8, 7, 8], [8, 7, 8, 7, 8, 7, 8, 7, 8, 7], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [8, 8, 8, 7, 8, 7, 8, 7, 8, 7]], "output": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 7, 8, 8], [8, 8, 8, 8, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 8, 7, 7, 7, 8], [8, 8, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 8, 7, 8], [7, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 7, 7, 8, 8, 7, 7], [7, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 7, 8, 8, 8, 8, 7], [8, 8, 8, 8, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 8, 7, 7, 7], [8, 8, 8, 8, 8, 8, 7, 8, 8, 8, 8, 8, 7, 8, 8, 7, 7, 7, 8, 8], [8, 8, 8, 8, 8, 8, 7, 7, 7, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 7], [8, 8, 8, 8, 8, 8, 8, 8, 7, 8, 7, 8, 8, 8, 8, 8, 8, 8, 7, 8], [8, 8, 8, 8, 8, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 8, 8], [8, 8, 8, 8, 7, 7, 7, 8, 8, 7, 8, 7, 8, 7, 8, 8, 8, 8, 7, 8], [8, 8, 8, 7, 7, 7, 8, 8, 8, 7, 8, 7, 8, 7, 8, 8, 8, 8, 8, 7], [8, 8, 7, 7, 7, 8, 8, 8, 8, 7, 8, 7, 8, 7, 8, 8, 8, 8, 8, 8], [8, 7, 7, 7, 8, 8, 8, 8, 8, 7, 8, 7, 8, 7, 8, 8, 8, 8, 8, 8], [8, 7, 7, 8, 8, 8, 8, 8, 8, 7, 8, 7, 8, 7, 7, 7, 7, 7, 7, 8], [8, 7, 7, 8, 8, 8, 8, 8, 8, 7, 8, 7, 7, 8, 7, 8, 7, 8, 7, 8], [8, 7, 7, 8, 8, 8, 8, 8, 8, 7, 8, 7, 8, 7, 8, 7, 8, 7, 8, 7], [7, 7, 7, 7, 8, 8, 8, 8, 8, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 7, 8, 7, 8, 7, 8, 7]]}], "test": [{"input": [[8, 7, 8, 7, 8, 7, 8, 7, 7, 7], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 7, 8, 7, 8, 7, 8, 7, 8, 7], [7, 8, 7, 8, 7, 8, 7, 8, 8, 7], [7, 8, 8, 8, 8, 8, 8, 7, 8, 7], [7, 7, 7, 7, 7, 7, 8, 7, 8, 7], [7, 7, 7, 7, 7, 7, 8, 7, 8, 7], [8, 7, 7, 7, 7, 7, 8, 7, 8, 7], [7, 8, 7, 7, 7, 7, 8, 7, 8, 7], [7, 7, 8, 8, 8, 8, 8, 8, 8, 8]]}, {"input": [[6, 6, 6, 6, 6, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 6, 6, 6, 6], [6, 6, 6, 5, 6, 6, 6, 6, 6, 6], [5, 5, 5, 5, 6, 6, 6, 6, 6, 6], [5, 5, 6, 6, 6, 6, 6, 6, 6, 6], [6, 5, 6, 6, 6, 6, 6, 6, 6, 6], [5, 5, 5, 5, 5, 6, 6, 6, 6, 6], [5, 6, 6, 5, 5, 5, 6, 6, 6, 6], [5, 6, 6, 6, 5, 5, 5, 6, 6, 6], [5, 6, 6, 6, 6, 5, 5, 5, 6, 6], [5, 6, 6, 6, 6, 6, 5, 5, 5, 6]]}]}, "2b83f449": {"train": [{"input": [[3, 8, 8, 8, 8, 8, 8, 0, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3], [0, 0, 0, 7, 7, 7, 0, 0, 0, 7, 7, 7, 0, 0, 0, 0, 0, 0, 0], [3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3], [0, 0, 0, 0, 0, 7, 7, 7, 0, 0, 0, 0, 7, 7, 7, 0, 0, 0, 0], [3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3], [0, 0, 7, 7, 7, 0, 0, 7, 7, 7, 0, 0, 0, 0, 7, 7, 7, 0, 0], [3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3], [0, 0, 0, 0, 7, 7, 7, 0, 0, 0, 7, 7, 7, 0, 0, 0, 0, 0, 0], [3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 8, 8, 3], [0, 0, 0, 0, 0, 0, 0, 7, 7, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0], [3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3], [0, 0, 0, 0, 7, 7, 7, 0, 0, 0, 0, 0, 7, 7, 7, 0, 0, 0, 0], [8, 8, 0, 8, 8, 8, 8, 8, 0, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]], "output": [[8, 8, 8, 8, 6, 8, 8, 0, 8, 8, 6, 8, 8, 8, 8, 8, 8, 8, 8], [0, 0, 0, 8, 6, 8, 0, 0, 0, 8, 6, 8, 0, 0, 0, 0, 0, 0, 0], [3, 3, 8, 8, 6, 8, 6, 8, 8, 8, 6, 8, 8, 6, 8, 8, 8, 8, 8], [0, 0, 0, 0, 0, 8, 6, 8, 0, 0, 0, 0, 8, 6, 8, 0, 0, 0, 0], [8, 8, 8, 6, 8, 8, 6, 8, 6, 8, 8, 8, 8, 6, 8, 6, 8, 8, 8], [0, 0, 8, 6, 8, 0, 0, 8, 6, 8, 0, 0, 0, 0, 8, 6, 8, 0, 0], [3, 3, 8, 6, 8, 6, 8, 8, 6, 8, 8, 6, 8, 8, 8, 6, 8, 3, 3], [0, 0, 0, 0, 8, 6, 8, 0, 0, 0, 8, 6, 8, 0, 0, 0, 0, 0, 0], [3, 8, 8, 8, 8, 6, 8, 8, 6, 8, 8, 6, 8, 8, 3, 0, 8, 8, 3], [0, 0, 0, 0, 0, 0, 0, 8, 6, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0], [8, 8, 8, 8, 8, 6, 8, 8, 6, 8, 8, 8, 8, 6, 8, 8, 8, 8, 8], [0, 0, 0, 0, 8, 6, 8, 0, 0, 0, 0, 0, 8, 6, 8, 0, 0, 0, 0], [8, 8, 0, 3, 8, 6, 8, 8, 0, 3, 8, 8, 8, 6, 8, 8, 8, 8, 3]]}, {"input": [[3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3], [0, 0, 0, 7, 7, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 7, 7, 0, 0], [3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3], [0, 0, 0, 0, 0, 7, 7, 7, 0, 0, 0, 0, 0, 0, 0, 0], [3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3], [0, 0, 7, 7, 7, 0, 0, 0, 0, 0, 7, 7, 7, 0, 0, 0], [3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3], [0, 0, 0, 0, 0, 0, 7, 7, 7, 0, 0, 0, 0, 0, 0, 0], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]], "output": [[8, 8, 8, 8, 6, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [0, 0, 0, 8, 6, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [3, 3, 8, 8, 6, 8, 8, 8, 8, 8, 8, 8, 6, 8, 8, 8], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 6, 8, 0, 0], [8, 8, 8, 8, 8, 8, 6, 8, 8, 8, 8, 8, 6, 8, 3, 3], [0, 0, 0, 0, 0, 8, 6, 8, 0, 0, 0, 0, 0, 0, 0, 0], [8, 8, 8, 6, 8, 8, 6, 8, 8, 8, 8, 6, 8, 8, 8, 8], [0, 0, 8, 6, 8, 0, 0, 0, 0, 0, 8, 6, 8, 0, 0, 0], [3, 3, 8, 6, 8, 8, 8, 6, 8, 8, 8, 6, 8, 8, 3, 3], [0, 0, 0, 0, 0, 0, 8, 6, 8, 0, 0, 0, 0, 0, 0, 0], [3, 8, 8, 8, 8, 8, 8, 6, 8, 8, 8, 8, 8, 8, 8, 3]]}], "test": [{"input": [[3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3], [0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 7, 7, 0, 0, 0, 0, 0], [3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 7, 7, 0, 0], [3, 8, 8, 8, 8, 8, 8, 0, 8, 8, 8, 8, 8, 8, 8, 8, 3], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3], [0, 0, 0, 0, 0, 0, 7, 7, 7, 0, 0, 0, 0, 0, 0, 0, 0], [3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3], [0, 0, 0, 7, 7, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 8, 8, 8, 8, 8, 3], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3], [0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 7, 7, 0, 0, 0, 0, 0], [3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3], [0, 0, 0, 0, 0, 0, 7, 7, 7, 0, 0, 0, 0, 0, 0, 0, 0], [3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3], [0, 0, 7, 7, 7, 0, 0, 0, 0, 0, 0, 7, 7, 7, 0, 0, 0], [8, 8, 8, 8, 8, 8, 8, 0, 8, 8, 8, 8, 8, 8, 8, 8, 8]]}]}, "2c181942": {"train": [{"input": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 6, 6, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 3, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 3, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 4, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 4, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 4, 4, 4, 8, 8, 8, 8], [8, 8, 8, 8, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 3, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 3, 8, 8, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]], "output": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 3, 3, 8, 8, 6, 6, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 3, 3, 3, 8, 8, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 3, 3, 3, 8, 8, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 3, 3, 8, 8, 1, 1, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]]}, {"input": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 8, 8, 3, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 1, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 1, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 4, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8], [8, 8, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]], "output": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 3, 8, 8, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 1, 1, 8, 8, 3, 3, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 1, 1, 1, 1, 8, 8, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 1, 1, 1, 1, 8, 8, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 1, 1, 8, 8, 4, 4, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]]}, {"input": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 8, 8, 8, 8], [8, 8, 8, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 8, 8, 8, 8], [8, 8, 2, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 7, 8, 8, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 8, 8, 8], [8, 8, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]], "output": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 1, 1, 8, 8, 7, 7, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 8, 8, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 8, 8, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 3, 3, 8, 8, 7, 7, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]]}], "test": [{"input": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 8, 8, 8, 8, 8, 8], [8, 8, 8, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 8, 8, 8, 8, 8, 8], [8, 8, 8, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 2, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 6, 6, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 8, 8, 4, 8, 8], [8, 8, 8, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 4, 4, 4, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 4, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 6, 6, 6, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 6, 6, 6, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]]}]}, "31f7f899": {"train": [{"input": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 5, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 5, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8], [8, 5, 8, 8, 8, 8, 2, 7, 8, 8, 8, 8, 8], [8, 5, 8, 8, 8, 8, 2, 7, 1, 8, 8, 8, 8], [6, 5, 6, 6, 6, 6, 2, 7, 1, 6, 6, 6, 6], [8, 5, 8, 8, 8, 8, 2, 7, 1, 8, 8, 8, 8], [8, 5, 8, 8, 8, 8, 2, 7, 8, 8, 8, 8, 8], [8, 5, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8], [8, 5, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]], "output": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 7, 1, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 2, 7, 1, 8, 8, 8, 8], [8, 5, 8, 8, 8, 8, 2, 7, 1, 8, 8, 8, 8], [6, 5, 6, 6, 6, 6, 2, 7, 1, 6, 6, 6, 6], [8, 5, 8, 8, 8, 8, 2, 7, 1, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 2, 7, 1, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 7, 1, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]]}, {"input": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 7, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 7, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 4, 8, 7, 8, 5, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 4, 8, 7, 8, 5, 8, 1, 8, 8, 8, 9, 8, 8], [6, 6, 4, 6, 7, 6, 5, 6, 1, 6, 6, 6, 9, 6, 6], [8, 8, 4, 8, 7, 8, 5, 8, 1, 8, 8, 8, 9, 8, 8], [8, 8, 4, 8, 7, 8, 5, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 7, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 7, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]], "output": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 7, 8, 5, 8, 1, 8, 8, 8, 9, 8, 8], [8, 8, 4, 8, 7, 8, 5, 8, 1, 8, 8, 8, 9, 8, 8], [6, 6, 4, 6, 7, 6, 5, 6, 1, 6, 6, 6, 9, 6, 6], [8, 8, 4, 8, 7, 8, 5, 8, 1, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 7, 8, 5, 8, 1, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]]}, {"input": [[8, 8, 8, 8, 8, 8, 8], [8, 8, 1, 8, 8, 8, 8], [4, 8, 1, 8, 1, 8, 7], [4, 6, 1, 6, 1, 6, 7], [4, 8, 1, 8, 1, 8, 7], [8, 8, 1, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8]], "output": [[8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 7], [4, 8, 1, 8, 1, 8, 7], [4, 6, 1, 6, 1, 6, 7], [4, 8, 1, 8, 1, 8, 7], [8, 8, 8, 8, 8, 8, 7], [8, 8, 8, 8, 8, 8, 8]]}], "test": [{"input": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [9, 8, 5, 5, 5, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [9, 8, 5, 5, 5, 8, 8, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8], [9, 8, 5, 5, 5, 8, 8, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8], [9, 6, 5, 5, 5, 6, 6, 2, 2, 2, 6, 6, 6, 6, 6, 6, 7], [9, 8, 5, 5, 5, 8, 8, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8], [9, 8, 5, 5, 5, 8, 8, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8], [9, 8, 5, 5, 5, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]]}]}, "332f06d7": {"train": [{"input": [[3, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 1, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 1, 1, 3, 1, 1, 1, 1, 1, 1, 3, 3], [3, 1, 1, 3, 1, 1, 1, 1, 1, 1, 3, 3], [3, 1, 1, 3, 1, 1, 3, 1, 1, 1, 3, 3], [3, 1, 1, 3, 1, 1, 3, 1, 1, 1, 1, 3], [3, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 3], [3, 1, 1, 1, 1, 1, 3, 3, 1, 1, 3, 3], [3, 1, 1, 1, 3, 3, 3, 3, 0, 0, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 0, 0, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]], "output": [[3, 0, 0, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 0, 0, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 1, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 1, 1, 3, 1, 1, 1, 1, 1, 1, 3, 3], [3, 1, 1, 3, 1, 1, 1, 1, 1, 1, 3, 3], [3, 1, 1, 3, 1, 1, 3, 1, 1, 1, 3, 3], [3, 1, 1, 3, 1, 1, 3, 1, 1, 1, 1, 3], [3, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 3], [3, 1, 1, 1, 1, 1, 3, 3, 1, 1, 3, 3], [3, 1, 1, 1, 3, 3, 3, 3, 1, 1, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 1, 1, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]]}, {"input": [[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 1, 1, 1, 1, 1, 0, 0, 3, 3], [3, 3, 3, 3, 3, 1, 1, 1, 1, 1, 0, 0, 3, 3], [3, 3, 3, 3, 3, 1, 1, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 1, 1, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 1, 1, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 1, 1, 1, 1, 1, 3, 3, 3, 3], [3, 3, 3, 3, 3, 1, 1, 1, 1, 1, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 3, 3, 3, 3], [3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 3, 3], [3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 3, 3], [3, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]], "output": [[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 1, 1, 1, 1, 1, 1, 1, 3, 3], [3, 3, 3, 3, 3, 1, 1, 1, 1, 1, 1, 1, 3, 3], [3, 3, 3, 3, 3, 1, 1, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 1, 1, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 1, 1, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 1, 1, 1, 0, 0, 3, 3, 3, 3], [3, 3, 3, 3, 3, 1, 1, 1, 0, 0, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 3, 3, 3, 3], [3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 3, 3], [3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 3, 3], [3, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]]}, {"input": [[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3], [2, 2, 2, 1, 1, 1, 1, 3, 3, 1, 1, 1, 1, 1, 1, 3], [2, 2, 2, 1, 1, 3, 3, 3, 3, 3, 1, 1, 1, 1, 1, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 1, 3], [3, 3, 3, 1, 1, 1, 1, 1, 1, 1, 3, 3, 3, 1, 1, 3], [3, 3, 3, 1, 1, 1, 1, 1, 1, 1, 3, 3, 3, 1, 1, 3], [3, 3, 3, 1, 1, 1, 1, 1, 1, 1, 3, 3, 3, 1, 1, 3], [3, 3, 1, 1, 1, 1, 3, 1, 1, 1, 3, 3, 3, 1, 1, 3], [3, 3, 1, 1, 1, 1, 3, 1, 1, 3, 3, 3, 1, 1, 1, 3], [3, 3, 1, 1, 1, 1, 3, 1, 1, 3, 3, 3, 1, 1, 1, 3], [3, 3, 1, 1, 1, 3, 3, 1, 1, 3, 3, 3, 1, 1, 1, 3], [3, 0, 0, 0, 1, 3, 3, 1, 1, 1, 1, 1, 1, 1, 1, 3], [3, 0, 0, 0, 1, 3, 3, 1, 1, 1, 1, 1, 1, 1, 1, 3], [3, 0, 0, 0, 1, 3, 3, 1, 1, 1, 1, 1, 1, 1, 1, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]], "output": [[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3], [2, 2, 2, 1, 1, 1, 1, 3, 3, 1, 1, 1, 1, 1, 1, 3], [2, 2, 2, 1, 1, 3, 3, 3, 3, 3, 1, 1, 1, 1, 1, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 1, 3], [3, 3, 3, 1, 1, 1, 1, 1, 1, 1, 3, 3, 3, 1, 1, 3], [3, 3, 3, 1, 1, 1, 1, 0, 0, 0, 3, 3, 3, 1, 1, 3], [3, 3, 3, 1, 1, 1, 1, 0, 0, 0, 3, 3, 3, 1, 1, 3], [3, 3, 1, 1, 1, 1, 3, 0, 0, 0, 3, 3, 3, 1, 1, 3], [3, 3, 1, 1, 1, 1, 3, 1, 1, 3, 3, 3, 1, 1, 1, 3], [3, 3, 1, 1, 1, 1, 3, 1, 1, 3, 3, 3, 1, 1, 1, 3], [3, 3, 1, 1, 1, 3, 3, 1, 1, 3, 3, 3, 1, 1, 1, 3], [3, 1, 1, 1, 1, 3, 3, 1, 1, 1, 1, 1, 1, 1, 1, 3], [3, 1, 1, 1, 1, 3, 3, 1, 1, 1, 1, 1, 1, 1, 1, 3], [3, 1, 1, 1, 1, 3, 3, 1, 1, 1, 1, 1, 1, 1, 1, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]]}, {"input": [[3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 0, 1, 1, 3, 3, 3, 3, 3, 3], [3, 3, 3, 1, 3, 1, 1, 1, 3, 3], [3, 3, 3, 1, 3, 1, 3, 1, 3, 3], [3, 3, 3, 1, 1, 1, 3, 1, 3, 3], [3, 3, 3, 3, 3, 3, 3, 1, 3, 3], [3, 3, 3, 3, 1, 1, 1, 1, 3, 3], [3, 3, 3, 3, 1, 3, 3, 3, 3, 3], [3, 3, 3, 3, 1, 3, 3, 3, 3, 3], [3, 3, 3, 3, 2, 3, 3, 3, 3, 3]], "output": [[3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 1, 1, 1, 3, 3, 3, 3, 3, 3], [3, 3, 3, 1, 3, 1, 1, 1, 3, 3], [3, 3, 3, 1, 3, 1, 3, 1, 3, 3], [3, 3, 3, 1, 1, 1, 3, 1, 3, 3], [3, 3, 3, 3, 3, 3, 3, 1, 3, 3], [3, 3, 3, 3, 1, 1, 1, 1, 3, 3], [3, 3, 3, 3, 1, 3, 3, 3, 3, 3], [3, 3, 3, 3, 1, 3, 3, 3, 3, 3], [3, 3, 3, 3, 0, 3, 3, 3, 3, 3]]}], "test": [{"input": [[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 1, 1, 1, 1, 1, 1, 1, 3, 3, 3, 3, 3, 3, 1, 1, 1, 1, 1, 3], [3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3], [3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3], [3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3], [3, 1, 1, 1, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 1, 1, 1, 1, 3], [3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 1, 1, 1, 1, 3], [3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 3, 1, 1, 1, 3], [3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 3, 3, 1, 1, 3], [3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 3, 3, 1, 1, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 1, 1, 1, 3, 3, 3, 1, 1, 1, 3], [3, 0, 0, 0, 0, 1, 1, 1, 3, 1, 1, 1, 1, 3, 3, 1, 1, 1, 1, 3], [3, 0, 0, 0, 0, 1, 1, 1, 3, 1, 1, 1, 1, 3, 3, 1, 1, 1, 1, 3], [3, 0, 0, 0, 0, 1, 1, 1, 3, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 3], [3, 0, 0, 0, 0, 1, 1, 1, 3, 1, 1, 1, 1, 1, 3, 1, 2, 2, 2, 2], [3, 3, 3, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 2, 2, 2, 2], [3, 3, 3, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 1, 2, 2, 2, 2], [3, 3, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 1, 2, 2, 2, 2], [3, 3, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]]}]}, "35ab12c3": {"train": [{"input": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 8, 0, 0, 8], [0, 0, 0, 0, 0, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 3, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8], [0, 0, 0, 0, 0, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 8, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 4, 0, 0, 0, 0, 0, 0, 4, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], "output": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 7, 6, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 7, 6, 0, 6, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 7, 6, 0, 0, 0, 6, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 7, 6, 0, 0, 0, 0, 0, 6, 7, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 6, 0, 0, 0, 0, 0, 0, 0, 6, 7, 0, 0, 1, 8, 8, 8, 8], [0, 0, 0, 0, 0, 3, 3, 0, 0, 0, 0, 0, 6, 7, 0, 1, 8, 0, 1, 8], [0, 0, 0, 0, 3, 0, 0, 3, 0, 0, 0, 0, 0, 6, 0, 1, 8, 0, 1, 8], [0, 0, 0, 3, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 1, 8, 0, 1, 8], [0, 0, 3, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 1, 8, 0, 1, 8], [0, 0, 0, 3, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 1, 8, 0, 1, 8], [0, 0, 0, 0, 3, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 1, 8, 0, 1, 8], [0, 0, 0, 0, 0, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 1, 8, 1, 8, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 8, 8, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 4, 4, 4, 4, 4, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}, {"input": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 0, 6, 0, 0, 0, 0], [0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 6, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 6, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], "output": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 1, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 1, 0, 8, 0, 0, 4, 4, 4, 0, 0, 0, 0, 0], [0, 0, 0, 0, 1, 0, 8, 0, 0, 0, 4, 6, 6, 6, 0, 0, 0, 0], [0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 4, 6, 0, 0, 6, 0, 0, 0], [0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 4, 6, 0, 0, 0, 6, 0, 0], [0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 4, 6, 0, 0, 0, 0, 6, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 6, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 6, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 6, 0, 0, 0, 0, 0, 0], [0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 6, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}, {"input": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 7, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0, 0, 0], [0, 0, 0, 0, 0, 0, 4, 2, 0, 2, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 7, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], "output": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 0], [0, 8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0], [0, 8, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 1, 0], [0, 8, 1, 0, 7, 7, 7, 7, 7, 7, 7, 0, 8, 1, 0], [0, 8, 1, 0, 7, 0, 0, 0, 0, 0, 0, 7, 8, 1, 0], [0, 8, 1, 0, 7, 0, 4, 2, 2, 2, 7, 0, 8, 1, 0], [0, 8, 1, 0, 7, 0, 0, 0, 0, 7, 0, 0, 8, 1, 0], [0, 8, 1, 0, 7, 0, 0, 0, 7, 0, 0, 0, 8, 1, 0], [0, 8, 1, 0, 7, 0, 0, 7, 0, 0, 0, 0, 8, 1, 0], [0, 8, 1, 0, 7, 7, 7, 0, 0, 0, 0, 0, 8, 1, 0], [0, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 0], [0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}], "test": [{"input": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 6, 0, 0, 7, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 3, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 6, 0, 0, 0, 1, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}]}, "3a25b0d8": {"train": [{"input": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 1, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 1, 1, 8, 8, 8, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 1, 1, 1, 8, 8, 8, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 1, 8, 1, 1, 1, 1, 1, 1, 1, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 1, 8, 8, 1, 8, 8, 8, 1, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 1, 8, 1, 1, 1, 1, 1, 1, 1, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 1, 1, 8, 8, 1, 8, 8, 1, 1, 8, 8, 1, 1, 1, 1, 1, 8, 8, 8, 8], [8, 8, 8, 1, 1, 1, 1, 1, 1, 1, 8, 8, 1, 1, 7, 7, 7, 1, 1, 8, 8, 8], [8, 8, 8, 8, 1, 8, 8, 8, 1, 8, 8, 1, 1, 1, 1, 7, 1, 1, 1, 1, 8, 8], [8, 8, 8, 8, 1, 8, 8, 8, 1, 8, 1, 3, 1, 1, 4, 1, 4, 1, 1, 3, 1, 8], [8, 8, 8, 8, 1, 1, 1, 1, 1, 8, 1, 3, 3, 1, 4, 4, 4, 1, 3, 3, 1, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 6, 6, 1, 6, 6, 1, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 1, 1, 1, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 9, 9, 9, 1, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]], "output": [[8, 8, 8, 1, 1, 1, 1, 1, 8, 8, 8], [8, 8, 1, 1, 7, 7, 7, 1, 1, 8, 8], [8, 1, 1, 1, 7, 7, 7, 1, 1, 1, 8], [1, 3, 1, 1, 1, 1, 1, 1, 1, 3, 1], [1, 3, 3, 1, 4, 4, 4, 1, 3, 3, 1], [1, 3, 1, 1, 1, 1, 1, 1, 1, 3, 1], [8, 1, 1, 6, 6, 1, 6, 6, 1, 1, 8], [8, 8, 1, 1, 1, 1, 1, 1, 1, 8, 8], [8, 8, 8, 1, 9, 9, 9, 1, 8, 8, 8], [8, 8, 8, 1, 9, 9, 9, 1, 8, 8, 8], [8, 8, 8, 1, 1, 1, 1, 1, 8, 8, 8]]}, {"input": [[1, 1, 2, 2, 1, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 2, 4, 4, 2, 4, 4, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 2, 2, 3, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 2, 3, 3, 3, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 2, 3, 2, 3, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 2, 2, 2, 8, 2, 2, 2, 1, 1, 1, 1, 2, 2, 1, 2, 2, 1, 1, 1, 1, 1], [1, 1, 1, 2, 8, 2, 1, 1, 1, 1, 1, 2, 1, 1, 2, 1, 1, 2, 1, 1, 1, 1], [1, 1, 1, 2, 8, 2, 1, 1, 1, 1, 1, 2, 1, 1, 2, 1, 1, 2, 1, 1, 1, 1], [1, 1, 2, 2, 8, 2, 2, 1, 1, 1, 1, 1, 2, 2, 1, 2, 2, 1, 1, 1, 1, 1], [1, 1, 2, 8, 8, 8, 2, 1, 1, 1, 1, 1, 2, 1, 1, 1, 2, 1, 1, 1, 1, 1], [1, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1, 2, 1, 1, 1, 2, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 2, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 2, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 1, 2, 2, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 2, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 1, 1, 1, 2, 2, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], "output": [[1, 2, 2, 1, 2, 2, 1], [2, 4, 4, 2, 4, 4, 2], [2, 4, 4, 2, 4, 4, 2], [1, 2, 2, 3, 2, 2, 1], [1, 2, 3, 3, 3, 2, 1], [1, 2, 3, 3, 3, 2, 1], [2, 2, 2, 2, 2, 2, 2], [1, 1, 2, 8, 2, 1, 1], [1, 1, 2, 8, 2, 1, 1], [1, 2, 2, 8, 2, 2, 1], [1, 2, 8, 8, 8, 2, 1], [2, 2, 8, 8, 8, 2, 2], [2, 2, 2, 2, 2, 2, 2]]}], "test": [{"input": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 4, 4, 4, 4, 4, 4, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 4, 3, 3, 3, 3, 4, 8, 8, 8, 8, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 4, 3, 3, 3, 3, 4, 8, 8, 8, 8, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 4, 4, 4, 3, 3, 4, 8, 8, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 4, 4, 4, 4, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 4, 5, 5, 4, 2, 4, 5, 5, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 4, 5, 5, 4, 2, 4, 5, 5, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 4, 4, 4, 2, 2, 2, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 4, 1, 4, 7, 4, 2, 4, 7, 4, 1, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 4, 4, 4, 7, 4, 4, 4, 7, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 4, 7, 7, 7, 4, 7, 7, 7, 4, 0, 0, 0, 0, 0, 0, 0, 4, 4, 0, 0, 0, 4, 4, 0, 0, 0, 0], [0, 4, 4, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 4, 0, 0, 0, 0, 4, 0, 0, 4, 4, 4, 0, 0, 4, 0, 0, 0], [0, 4, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 0, 0, 0, 4, 0, 0, 0, 0, 4, 0, 0, 0, 0, 4, 0, 0], [0, 4, 7, 7, 7, 7, 4, 4, 4, 7, 7, 7, 7, 4, 0, 0, 0, 4, 0, 0, 0, 0, 4, 0, 0, 0, 0, 4, 0, 0], [0, 4, 4, 4, 4, 4, 6, 6, 6, 4, 4, 4, 4, 4, 0, 0, 0, 4, 4, 4, 0, 0, 4, 0, 0, 4, 4, 4, 0, 0], [0, 0, 0, 0, 0, 0, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 4, 4, 4, 4, 4, 4, 4, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 0, 0, 0, 4, 4, 4, 0, 0, 0, 4, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 0, 0, 0, 4, 0, 4, 0, 0, 0, 4, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 4, 4, 0, 0, 0, 4, 4, 4, 4, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 0, 4, 4, 4, 0, 4, 4, 4, 0, 4, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 4, 0, 4, 4, 4, 0, 4, 4, 4, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 0, 0, 0, 4, 0, 0, 0, 4, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 0, 0, 0, 0, 4, 0, 0, 0, 0, 4, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 0, 0, 0, 4, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 4, 4, 4, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}, {"input": [[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 2, 2, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 8, 2, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 8, 8, 8, 2, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 1, 1, 2, 7, 7, 7, 2, 9, 9, 2, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 2, 2, 2, 7, 7, 7, 2, 2, 2, 2, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 7, 7, 2, 7, 7, 2, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 2, 2, 2, 2, 2, 2, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 2, 2, 2, 4, 4, 4, 4, 4, 4, 4, 2, 8, 8, 8, 2, 8, 8, 8, 2, 4, 4, 4, 4, 4], [4, 4, 4, 2, 2, 4, 2, 2, 4, 4, 4, 4, 4, 4, 2, 8, 8, 8, 2, 8, 8, 8, 2, 4, 4, 4, 4, 4], [4, 2, 2, 2, 4, 4, 4, 2, 2, 2, 4, 4, 4, 2, 5, 2, 2, 2, 3, 2, 2, 2, 5, 2, 4, 4, 4, 4], [2, 4, 4, 2, 2, 4, 2, 2, 4, 4, 2, 4, 4, 2, 2, 4, 2, 3, 3, 3, 2, 4, 2, 2, 4, 4, 4, 4], [2, 4, 4, 2, 4, 2, 4, 2, 4, 4, 2, 4, 4, 4, 4, 4, 4, 2, 2, 2, 4, 4, 4, 4, 4, 4, 4, 4], [2, 2, 2, 2, 4, 4, 4, 2, 2, 2, 2, 4, 4, 4, 4, 4, 4, 2, 4, 2, 4, 4, 4, 4, 4, 4, 4, 4], [4, 2, 2, 4, 4, 4, 4, 4, 2, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 2, 2, 2, 2, 2, 2, 2, 2, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 2, 4, 4, 4, 2, 4, 4, 4, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 2, 4, 4, 4, 2, 4, 4, 4, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [2, 4, 2, 2, 2, 4, 2, 2, 2, 4, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [2, 2, 2, 2, 4, 4, 4, 2, 2, 2, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 2, 2, 4, 2, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 2, 2, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]]}]}, "3dc255db": {"train": [{"input": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0], [6, 0, 0, 6, 4, 4, 0, 0, 0, 0, 0, 0], [0, 0, 6, 0, 4, 4, 4, 0, 0, 0, 0, 0], [0, 0, 4, 4, 4, 4, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 0, 0], [0, 0, 0, 0, 3, 3, 7, 0, 0, 0, 0, 0], [0, 0, 0, 0, 3, 3, 0, 7, 0, 0, 0, 0], [0, 0, 0, 3, 3, 3, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 3, 3, 3, 0, 0, 0]], "output": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0], [6, 0, 0, 0, 4, 4, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 4, 4, 4, 6, 6, 0, 0, 0], [0, 0, 4, 4, 4, 4, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 0, 0], [0, 0, 0, 0, 3, 3, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 3, 3, 0, 0, 0, 0, 0, 0], [0, 7, 7, 3, 3, 3, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 3, 3, 3, 0, 0, 0]]}, {"input": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 3, 0, 0, 0, 0, 6, 0, 0], [0, 0, 3, 0, 0, 0, 6, 0, 6, 0], [0, 3, 9, 3, 0, 0, 6, 9, 6, 0], [3, 9, 0, 0, 3, 0, 0, 0, 0, 0], [0, 0, 9, 0, 3, 0, 0, 0, 0, 0], [0, 0, 0, 0, 3, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], "output": [[0, 0, 9, 0, 0, 0, 0, 0, 0, 0], [0, 0, 9, 0, 0, 0, 0, 0, 0, 0], [0, 0, 9, 0, 0, 0, 0, 9, 0, 0], [0, 0, 3, 0, 0, 0, 0, 6, 0, 0], [0, 0, 3, 0, 0, 0, 6, 0, 6, 0], [0, 3, 0, 3, 0, 0, 6, 0, 6, 0], [3, 0, 0, 0, 3, 0, 0, 0, 0, 0], [0, 0, 0, 0, 3, 0, 0, 0, 0, 0], [0, 0, 0, 0, 3, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}, {"input": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 6, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 6, 6, 6, 0, 0, 0, 0, 0], [0, 0, 0, 6, 6, 0, 0, 6, 0, 0, 0, 0], [0, 0, 6, 0, 0, 0, 0, 0, 6, 0, 0, 0], [0, 0, 6, 7, 7, 7, 7, 0, 6, 0, 0, 0], [0, 0, 6, 0, 0, 0, 7, 0, 6, 0, 0, 0], [0, 0, 6, 7, 0, 7, 0, 0, 6, 0, 0, 0], [0, 0, 6, 0, 0, 0, 0, 0, 6, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], "output": [[0, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 6, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 6, 6, 6, 0, 0, 0, 0, 0], [0, 0, 0, 6, 6, 0, 0, 6, 0, 0, 0, 0], [0, 0, 6, 0, 0, 0, 0, 0, 6, 0, 0, 0], [0, 0, 6, 0, 0, 0, 0, 0, 6, 0, 0, 0], [0, 0, 6, 0, 0, 0, 0, 0, 6, 0, 0, 0], [0, 0, 6, 0, 0, 0, 0, 0, 6, 0, 0, 0], [0, 0, 6, 0, 0, 0, 0, 0, 6, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}], "test": [{"input": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 2, 2, 2, 2, 2, 0, 0, 0], [0, 0, 0, 0, 0, 0, 4, 4, 4, 0, 2, 0, 0], [0, 0, 0, 0, 0, 2, 2, 2, 2, 2, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 9, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 9, 9, 0, 0, 0, 0, 0, 8, 0, 0, 0], [0, 9, 0, 0, 9, 0, 0, 0, 0, 0, 8, 0, 0], [0, 9, 7, 7, 9, 0, 0, 2, 0, 0, 2, 8, 0], [0, 9, 0, 7, 9, 0, 0, 0, 0, 2, 8, 0, 0], [0, 9, 7, 0, 9, 0, 0, 0, 0, 8, 0, 0, 0]]}]}, "446ef5d2": {"train": [{"input": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 7, 7, 7], [8, 8, 8, 8, 8, 8, 8, 8, 8, 7, 7, 7, 7, 7, 7, 8, 8, 2, 2, 2], [8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 0, 0, 7, 0, 7, 8, 8, 7, 7, 7], [8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 7, 0, 0, 0, 7, 8, 8, 7, 7, 7], [8, 7, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 7, 0, 0, 0, 0, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 7, 0, 7, 0, 0, 8, 8, 8, 8, 8, 8, 8, 8, 8, 7, 7, 7, 8, 8], [8, 7, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 7, 8, 8], [8, 7, 2, 7, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 7, 2, 7, 8, 8], [8, 7, 2, 2, 2, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 7, 7, 7, 4, 8], [8, 7, 7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 4, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]], "output": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 8, 8], [8, 8, 8, 8, 8, 8, 8, 7, 0, 0, 0, 0, 0, 0, 0, 7, 0, 7, 8, 8], [8, 8, 8, 8, 8, 8, 8, 7, 0, 7, 0, 0, 0, 7, 0, 0, 0, 7, 8, 8], [8, 8, 8, 8, 8, 8, 8, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 8, 8], [8, 8, 8, 8, 8, 8, 8, 7, 2, 7, 2, 2, 2, 2, 2, 2, 2, 7, 8, 8], [8, 8, 8, 8, 8, 8, 8, 7, 2, 2, 2, 7, 7, 7, 7, 7, 2, 7, 8, 8], [8, 8, 8, 8, 8, 8, 8, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]]}, {"input": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 4, 4, 8, 8, 8, 9, 9, 3, 8], [8, 4, 3, 3, 3, 8, 3, 3, 3, 8], [8, 8, 3, 9, 9, 8, 3, 3, 3, 8], [8, 8, 3, 9, 9, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 3, 3, 3, 8, 8, 3, 9, 9, 8], [8, 9, 9, 3, 8, 8, 3, 9, 3, 8], [8, 9, 9, 3, 8, 8, 3, 3, 3, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8]], "output": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 3, 3, 3, 3, 3, 3, 8, 8], [8, 8, 3, 9, 9, 9, 9, 3, 8, 8], [8, 8, 3, 9, 9, 9, 9, 3, 8, 8], [8, 8, 3, 9, 9, 9, 9, 3, 8, 8], [8, 8, 3, 9, 3, 3, 3, 3, 8, 8], [8, 8, 3, 3, 3, 3, 3, 3, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8]]}], "test": [{"input": [[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 1, 1, 4, 1, 1, 1], [4, 1, 3, 1, 1, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 3, 3, 3, 4, 6, 6, 1], [4, 1, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 3, 1, 1, 4, 1, 6, 1], [4, 1, 1, 1, 1, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 3, 1, 1, 4, 1, 6, 1], [4, 1, 1, 1, 1, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 6, 6, 1], [4, 1, 1, 6, 6, 4, 1, 1, 1, 1, 3, 1, 4, 4, 4, 4, 1, 1, 7, 1, 1, 1, 4, 1, 1, 1], [4, 1, 1, 6, 1, 4, 3, 3, 3, 3, 3, 1, 4, 4, 4, 4, 1, 1, 7, 1, 1, 1, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 1, 1, 1, 1, 1, 1, 4, 4, 4, 4, 7, 7, 7, 1, 1, 1, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 1, 1, 1, 1, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 1, 1, 1, 1, 4, 4, 4, 4], [1, 1, 1, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [6, 6, 6, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [1, 1, 1, 4, 4, 4, 4, 1, 1, 1, 1, 1, 1, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 1, 1, 4, 4, 4, 4, 3, 3, 3, 3, 3, 1, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 6, 6, 4, 4, 4, 4, 1, 1, 1, 1, 3, 1, 4, 4, 4, 4, 4, 4, 4, 1, 1, 1, 1, 1, 1], [4, 1, 1, 4, 4, 4, 4, 1, 1, 1, 1, 3, 1, 4, 1, 1, 6, 1, 1, 4, 7, 7, 7, 1, 1, 1], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 6, 6, 6, 4, 1, 1, 7, 1, 1, 1], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 1, 1, 1, 4, 1, 1, 7, 1, 1, 1], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 1, 1, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 1, 7, 1, 1, 4, 4, 4, 4, 4, 4, 4, 4, 1, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 1, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4, 4, 1, 7, 1, 1, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 1, 1, 1, 1, 4, 4, 4, 4, 4, 4, 4, 4, 1, 7, 1, 1, 4, 4, 4, 4, 4, 4, 4, 4], [4, 2, 1, 1, 1, 1, 4, 4, 4, 4, 4, 4, 4, 4, 1, 7, 1, 1, 4, 4, 4, 4, 4, 4, 4, 4], [4, 2, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]]}, {"input": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8, 8, 4, 4, 8], [8, 8, 8, 8, 8, 8, 3, 3, 9, 9, 8, 8, 8, 8, 8, 9, 9, 9, 4, 8], [9, 9, 9, 8, 8, 8, 9, 3, 9, 9, 8, 8, 8, 8, 8, 9, 3, 9, 8, 8], [9, 3, 3, 8, 8, 8, 3, 3, 3, 3, 8, 8, 8, 8, 8, 3, 3, 9, 8, 8], [9, 3, 9, 8, 8, 8, 9, 3, 3, 3, 8, 8, 8, 8, 8, 3, 3, 9, 8, 8], [9, 3, 3, 8, 8, 8, 3, 3, 9, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [9, 3, 9, 8, 8, 8, 9, 9, 9, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [9, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 8, 8, 8, 8], [9, 9, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 9, 3, 3, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 9, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 9, 8, 8, 8, 8], [8, 8, 8, 9, 9, 9, 8, 8, 8, 8, 8, 8, 8, 9, 3, 3, 8, 8, 8, 8], [8, 8, 8, 9, 3, 9, 8, 8, 8, 8, 8, 8, 8, 9, 9, 9, 8, 8, 8, 8], [8, 8, 8, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 9, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 9, 3, 3, 8, 8, 8, 8, 8, 3, 3, 9, 8, 8, 8, 8, 8, 8], [8, 8, 8, 3, 3, 3, 8, 8, 8, 8, 8, 3, 3, 9, 8, 8, 8, 8, 8, 8], [8, 8, 8, 9, 9, 9, 8, 8, 8, 8, 8, 9, 9, 9, 8, 8, 8, 8, 8, 8]]}]}, "4c3d4a41": {"train": [{"input": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 3, 0, 2, 0, 5, 0, 5, 0, 5], [0, 5, 0, 0, 0, 0, 0, 5, 0, 5, 0, 3, 0, 2, 0, 4, 0, 7, 0, 5], [0, 5, 0, 5, 0, 0, 0, 5, 0, 5, 0, 3, 0, 2, 0, 4, 0, 7, 0, 5], [0, 5, 0, 5, 0, 5, 0, 5, 0, 0, 0, 0, 0, 2, 0, 4, 0, 0, 0, 5], [0, 5, 5, 5, 5, 5, 5, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5]], "output": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 3, 0, 2, 0, 4, 0, 7, 0, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 5, 0, 2, 0, 4, 0, 5, 0, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 5, 0, 5, 0, 4, 0, 5, 0, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 5, 0, 5, 0, 5, 0, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 5, 5, 5, 0, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5]]}, {"input": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 5, 0, 5, 0, 5, 0, 5, 0, 5], [0, 0, 0, 0, 0, 0, 0, 5, 0, 5, 0, 4, 0, 5, 0, 5, 0, 8, 0, 5], [0, 5, 0, 0, 0, 0, 0, 5, 0, 5, 0, 4, 0, 6, 0, 7, 0, 8, 0, 5], [0, 5, 0, 5, 0, 5, 0, 5, 0, 0, 0, 0, 0, 6, 0, 7, 0, 0, 0, 5], [0, 5, 5, 5, 5, 5, 5, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5]], "output": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 4, 0, 5, 0, 5, 0, 8, 0, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 4, 0, 6, 0, 7, 0, 5, 0, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 5, 0, 6, 0, 7, 0, 5, 0, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 5, 0, 5, 0, 5, 0, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 5, 5, 5, 0, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5]]}], "test": [{"input": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 5, 0, 5, 0, 5, 0, 8, 0, 5], [0, 0, 0, 5, 0, 5, 0, 0, 0, 5, 0, 5, 0, 5, 0, 5, 0, 8, 0, 5], [0, 5, 0, 5, 0, 5, 0, 0, 0, 5, 0, 9, 0, 6, 0, 3, 0, 8, 0, 5], [0, 5, 0, 5, 0, 5, 0, 5, 0, 0, 0, 0, 0, 6, 0, 3, 0, 0, 0, 5], [0, 5, 5, 5, 5, 5, 5, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5]]}, {"input": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 3, 0, 2, 0, 5, 0, 5, 0, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 3, 0, 2, 0, 4, 0, 7, 0, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 3, 0, 2, 0, 4, 0, 7, 0, 5], [0, 5, 0, 5, 0, 5, 0, 5, 0, 0, 0, 0, 0, 2, 0, 4, 0, 0, 0, 5], [0, 5, 5, 5, 5, 5, 5, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5]]}]}, "4c7dc4dd": {"train": [{"input": [[1, 2, 3, 4, 1, 1, 3, 4, 1, 2, 3, 4, 1, 2, 3, 4, 1, 2, 3, 4, 1, 2, 3, 4, 1, 2, 3, 4, 1, 2], [4, 1, 2, 3, 4, 1, 2, 3, 4, 1, 2, 3, 4, 1, 2, 3, 4, 1, 2, 3, 4, 1, 2, 3, 4, 1, 2, 3, 4, 1], [3, 4, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [2, 3, 1, 4, 4, 4, 4, 4, 4, 4, 4, 1, 2, 3, 4, 1, 2, 3, 4, 1, 2, 3, 4, 1, 1, 3, 4, 1, 1, 3], [1, 2, 1, 4, 1, 0, 0, 0, 0, 4, 3, 4, 1, 2, 3, 4, 1, 2, 3, 4, 4, 4, 4, 4, 4, 4, 3, 4, 1, 2], [4, 1, 1, 4, 6, 1, 1, 1, 6, 4, 2, 3, 4, 1, 2, 3, 4, 1, 2, 4, 1, 0, 0, 0, 0, 4, 2, 3, 1, 1], [3, 4, 1, 4, 0, 0, 0, 0, 1, 4, 1, 2, 3, 4, 1, 2, 3, 4, 1, 4, 6, 0, 0, 0, 6, 4, 1, 2, 1, 4], [1, 3, 1, 4, 0, 0, 0, 0, 1, 4, 4, 1, 2, 3, 1, 1, 2, 3, 1, 4, 0, 0, 0, 0, 0, 4, 4, 1, 1, 3], [1, 2, 1, 4, 0, 0, 0, 0, 1, 4, 3, 4, 1, 2, 3, 4, 1, 2, 3, 4, 0, 0, 0, 0, 0, 4, 3, 4, 1, 2], [4, 1, 1, 4, 4, 4, 4, 4, 4, 4, 2, 1, 4, 1, 2, 3, 4, 1, 2, 4, 0, 0, 0, 0, 1, 4, 2, 3, 1, 1], [3, 4, 1, 2, 3, 4, 1, 2, 3, 4, 1, 1, 3, 4, 1, 2, 3, 4, 1, 4, 4, 4, 4, 4, 4, 4, 1, 2, 1, 4], [1, 3, 1, 1, 2, 3, 4, 1, 2, 3, 4, 1, 1, 3, 4, 1, 2, 3, 4, 1, 2, 3, 4, 1, 2, 3, 4, 1, 1, 3], [1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2], [4, 1, 2, 3, 4, 1, 2, 3, 4, 1, 2, 3, 4, 1, 2, 3, 4, 1, 2, 3, 4, 1, 2, 3, 4, 1, 2, 3, 4, 1], [3, 4, 1, 2, 3, 4, 1, 2, 3, 4, 1, 2, 3, 4, 1, 2, 1, 4, 1, 2, 3, 4, 1, 1, 1, 4, 1, 2, 3, 4], [2, 3, 4, 1, 2, 3, 4, 1, 2, 3, 4, 1, 2, 3, 4, 1, 2, 3, 4, 1, 2, 1, 4, 1, 2, 1, 4, 1, 1, 3], [1, 2, 3, 4, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 2, 3, 4, 1, 2], [4, 1, 2, 3, 4, 2, 2, 3, 4, 1, 2, 3, 4, 1, 2, 3, 4, 4, 4, 4, 4, 4, 4, 2, 4, 1, 2, 3, 4, 1], [3, 4, 1, 2, 3, 2, 4, 4, 4, 4, 4, 4, 4, 4, 1, 2, 4, 0, 0, 0, 0, 0, 4, 2, 3, 4, 1, 2, 3, 4], [2, 3, 4, 1, 2, 2, 4, 0, 0, 0, 0, 0, 4, 3, 4, 1, 4, 0, 0, 0, 0, 0, 4, 2, 2, 1, 4, 1, 2, 3], [1, 2, 3, 4, 1, 2, 4, 6, 0, 0, 0, 2, 4, 2, 3, 4, 4, 0, 0, 0, 0, 0, 4, 2, 1, 2, 3, 4, 1, 2], [4, 1, 2, 3, 4, 2, 4, 0, 0, 0, 0, 0, 4, 1, 2, 3, 4, 0, 0, 0, 0, 0, 4, 2, 4, 1, 2, 3, 4, 1], [3, 4, 1, 2, 3, 2, 4, 0, 0, 0, 0, 0, 4, 1, 1, 2, 4, 0, 0, 0, 0, 0, 4, 2, 3, 4, 1, 2, 3, 4], [2, 3, 4, 1, 2, 2, 4, 2, 0, 0, 0, 0, 4, 3, 4, 1, 4, 4, 4, 4, 4, 4, 4, 2, 2, 3, 4, 1, 2, 3], [1, 2, 3, 1, 1, 2, 4, 4, 4, 4, 4, 4, 4, 2, 3, 4, 1, 2, 3, 4, 1, 2, 3, 2, 1, 2, 3, 4, 1, 2], [4, 1, 2, 3, 4, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 2, 3, 4, 1], [3, 4, 1, 2, 3, 4, 1, 2, 3, 4, 1, 2, 3, 4, 1, 1, 3, 4, 1, 2, 3, 4, 1, 2, 3, 4, 1, 2, 3, 4], [2, 3, 4, 1, 2, 3, 4, 1, 2, 3, 4, 1, 2, 3, 4, 1, 2, 3, 4, 1, 2, 1, 4, 1, 2, 3, 4, 1, 2, 3], [1, 2, 3, 4, 1, 2, 3, 1, 1, 2, 3, 4, 1, 2, 3, 4, 1, 2, 3, 4, 1, 2, 3, 4, 1, 2, 3, 4, 1, 2], [4, 1, 2, 3, 4, 1, 2, 3, 4, 1, 2, 3, 4, 1, 2, 3, 4, 1, 2, 3, 4, 1, 2, 3, 4, 1, 2, 3, 4, 1]], "output": [[0, 0, 0, 0, 0], [6, 2, 2, 2, 2], [2, 0, 0, 0, 0], [2, 0, 0, 0, 0], [2, 0, 0, 0, 0]]}, {"input": [[4, 1, 2, 3, 0, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 3, 0, 1, 2, 4, 4, 1, 2, 3, 4, 1, 2, 4, 0, 1], [1, 2, 4, 0, 1, 4, 3, 0, 1, 2, 3, 4, 1, 2, 4, 0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2], [2, 3, 0, 1, 2, 4, 0, 8, 8, 8, 8, 8, 8, 3, 4, 1, 2, 3, 0, 4, 2, 3, 0, 1, 4, 3, 0, 1, 2, 3], [3, 0, 1, 2, 3, 4, 1, 8, 0, 4, 0, 0, 8, 0, 4, 2, 3, 0, 1, 2, 3, 0, 1, 2, 3, 4, 1, 2, 3, 0], [0, 1, 2, 3, 0, 4, 2, 8, 4, 4, 0, 0, 8, 1, 4, 3, 0, 1, 2, 3, 4, 4, 2, 3, 0, 1, 2, 3, 0, 1], [1, 2, 3, 0, 4, 4, 4, 8, 0, 4, 4, 4, 8, 2, 4, 0, 1, 2, 3, 4, 1, 2, 3, 0, 1, 2, 3, 0, 1, 2], [2, 3, 0, 1, 2, 4, 0, 8, 0, 4, 0, 0, 8, 3, 4, 1, 2, 3, 4, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3], [3, 0, 1, 2, 3, 4, 1, 8, 8, 8, 8, 8, 8, 4, 4, 2, 3, 0, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 0], [0, 1, 2, 3, 0, 4, 2, 3, 0, 1, 2, 3, 0, 1, 4, 3, 0, 1, 2, 2, 0, 1, 2, 3, 0, 1, 2, 3, 2, 1], [1, 2, 3, 0, 4, 4, 4, 0, 1, 2, 3, 0, 1, 2, 4, 0, 4, 2, 3, 2, 1, 8, 8, 8, 8, 8, 8, 0, 2, 2], [2, 4, 0, 1, 4, 4, 0, 4, 2, 3, 0, 1, 2, 4, 4, 1, 2, 3, 0, 2, 2, 8, 0, 0, 0, 0, 8, 1, 2, 3], [3, 0, 1, 2, 3, 4, 1, 2, 3, 0, 4, 2, 3, 0, 4, 4, 3, 0, 1, 2, 4, 8, 0, 0, 0, 0, 8, 2, 2, 0], [4, 1, 2, 3, 0, 4, 2, 3, 0, 1, 2, 3, 4, 1, 4, 4, 0, 1, 2, 2, 0, 8, 0, 0, 0, 0, 8, 3, 2, 1], [1, 2, 3, 0, 1, 4, 3, 0, 1, 2, 3, 0, 1, 2, 4, 4, 1, 2, 3, 2, 1, 8, 0, 0, 0, 0, 8, 0, 2, 2], [2, 3, 0, 1, 2, 4, 4, 1, 2, 3, 0, 4, 2, 3, 4, 1, 2, 3, 0, 2, 2, 8, 8, 8, 8, 8, 8, 1, 2, 3], [3, 0, 1, 2, 3, 4, 4, 2, 3, 4, 1, 2, 3, 0, 4, 2, 3, 0, 4, 2, 3, 0, 1, 2, 3, 0, 1, 2, 2, 0], [0, 4, 2, 3, 0, 4, 2, 3, 0, 1, 2, 3, 0, 1, 4, 3, 0, 1, 2, 2, 0, 1, 2, 3, 0, 1, 2, 3, 2, 1], [1, 2, 3, 0, 1, 4, 3, 4, 1, 2, 3, 0, 1, 2, 4, 0, 1, 2, 3, 2, 4, 2, 3, 0, 1, 2, 3, 0, 2, 2], [2, 3, 0, 1, 2, 4, 0, 1, 2, 3, 0, 1, 2, 3, 4, 1, 4, 3, 4, 2, 2, 4, 0, 1, 2, 3, 0, 1, 2, 3], [3, 0, 1, 2, 3, 4, 1, 2, 3, 0, 4, 2, 3, 0, 4, 4, 3, 0, 1, 2, 3, 0, 1, 2, 3, 0, 4, 2, 2, 0], [0, 1, 2, 3, 0, 4, 2, 3, 0, 1, 2, 3, 0, 1, 4, 3, 0, 1, 2, 2, 0, 1, 2, 3, 0, 1, 2, 3, 2, 1], [1, 2, 3, 0, 4, 4, 4, 0, 1, 2, 3, 0, 1, 2, 4, 0, 1, 2, 3, 2, 1, 8, 8, 8, 8, 8, 8, 0, 2, 2], [2, 3, 0, 1, 2, 4, 0, 8, 8, 8, 8, 8, 8, 3, 4, 1, 2, 3, 0, 2, 2, 8, 0, 0, 2, 0, 8, 1, 2, 3], [3, 0, 1, 2, 4, 4, 1, 8, 4, 0, 4, 4, 8, 0, 4, 4, 3, 0, 1, 2, 3, 8, 2, 2, 2, 0, 8, 2, 2, 0], [0, 1, 2, 3, 0, 4, 4, 8, 0, 0, 4, 4, 8, 1, 4, 3, 0, 1, 2, 2, 0, 8, 0, 0, 0, 0, 8, 3, 2, 1], [1, 2, 3, 0, 1, 4, 4, 8, 4, 0, 0, 0, 8, 2, 4, 0, 1, 2, 3, 2, 1, 8, 0, 2, 2, 0, 8, 0, 2, 2], [2, 3, 0, 1, 4, 4, 0, 8, 4, 0, 4, 4, 8, 4, 4, 1, 2, 3, 4, 2, 2, 8, 8, 8, 8, 8, 8, 1, 2, 3], [3, 0, 1, 2, 3, 4, 1, 8, 8, 8, 8, 8, 8, 0, 4, 2, 3, 0, 1, 2, 3, 0, 0, 0, 0, 0, 0, 4, 2, 0], [0, 1, 2, 3, 0, 4, 2, 3, 0, 1, 2, 3, 0, 1, 4, 3, 0, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1], [1, 2, 3, 0, 1, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 0, 1, 2, 3, 0, 4, 2, 3, 0, 4, 2, 3, 0, 1, 2]], "output": [[2, 2, 0, 2], [0, 0, 0, 2], [2, 2, 2, 2], [2, 0, 0, 2]]}], "test": [{"input": [[3, 4, 6, 6, 3, 4, 6, 6, 3, 4, 5, 6, 3, 6, 5, 6, 6, 4, 5, 6, 6, 4, 5, 6, 3, 6, 5, 6, 3, 6], [4, 5, 6, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 6, 3, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 4, 5], [5, 6, 3, 3, 5, 6, 3, 6, 6, 6, 3, 6, 5, 3, 3, 6, 6, 6, 6, 4, 6, 6, 3, 4, 5, 6, 3, 6, 6, 6], [6, 3, 4, 3, 6, 5, 5, 5, 5, 5, 5, 5, 6, 3, 4, 5, 6, 3, 6, 5, 5, 5, 5, 5, 5, 5, 4, 6, 6, 3], [3, 4, 5, 3, 3, 5, 0, 0, 0, 0, 0, 5, 6, 3, 6, 6, 6, 4, 5, 5, 0, 0, 0, 0, 0, 5, 5, 6, 3, 4], [6, 5, 6, 3, 4, 5, 0, 0, 1, 0, 0, 5, 4, 3, 6, 3, 6, 5, 6, 5, 0, 0, 0, 0, 0, 5, 6, 6, 4, 5], [5, 6, 3, 3, 6, 5, 0, 1, 8, 1, 0, 5, 6, 3, 6, 4, 6, 6, 3, 5, 0, 0, 0, 0, 0, 5, 3, 6, 5, 6], [6, 3, 4, 3, 6, 5, 0, 0, 1, 0, 0, 5, 6, 3, 4, 5, 6, 3, 4, 5, 0, 0, 0, 0, 0, 5, 4, 6, 6, 6], [3, 4, 5, 3, 3, 5, 0, 0, 0, 0, 0, 5, 3, 3, 6, 6, 6, 4, 5, 5, 0, 0, 0, 0, 0, 5, 5, 6, 3, 4], [4, 6, 6, 3, 4, 5, 5, 5, 5, 5, 5, 5, 4, 3, 6, 3, 6, 5, 6, 5, 5, 5, 5, 5, 5, 5, 6, 6, 4, 5], [5, 6, 3, 3, 6, 6, 3, 4, 5, 6, 3, 4, 5, 3, 3, 4, 6, 6, 3, 4, 6, 6, 3, 4, 6, 6, 6, 6, 5, 6], [6, 3, 4, 3, 6, 3, 4, 5, 6, 3, 4, 5, 6, 3, 6, 5, 6, 3, 4, 5, 6, 3, 4, 6, 6, 3, 4, 6, 6, 3], [3, 4, 5, 3, 6, 6, 6, 6, 3, 4, 5, 6, 3, 3, 6, 6, 6, 4, 5, 6, 3, 4, 6, 6, 3, 4, 5, 6, 3, 4], [4, 5, 6, 3, 4, 5, 6, 3, 4, 5, 6, 3, 4, 3, 6, 3, 6, 5, 6, 3, 4, 5, 6, 3, 4, 5, 6, 6, 4, 5], [5, 6, 3, 3, 6, 6, 3, 6, 5, 6, 3, 4, 6, 3, 3, 4, 6, 6, 3, 4, 5, 6, 3, 4, 5, 6, 3, 6, 6, 6], [6, 6, 6, 3, 6, 6, 6, 5, 6, 3, 4, 6, 6, 3, 4, 5, 6, 3, 6, 5, 6, 6, 4, 5, 6, 3, 4, 6, 6, 3], [6, 4, 6, 3, 3, 6, 5, 6, 3, 4, 5, 6, 3, 3, 5, 6, 6, 4, 5, 6, 3, 6, 5, 6, 3, 4, 5, 6, 3, 4], [4, 5, 6, 3, 4, 5, 6, 6, 6, 5, 6, 3, 4, 3, 6, 6, 6, 5, 6, 6, 4, 6, 6, 6, 4, 5, 6, 6, 4, 5], [5, 6, 3, 3, 5, 6, 6, 4, 6, 6, 3, 4, 5, 3, 3, 4, 6, 6, 3, 4, 6, 6, 6, 4, 5, 6, 3, 6, 5, 6], [6, 3, 4, 3, 6, 3, 4, 5, 6, 3, 4, 5, 6, 3, 4, 5, 6, 6, 4, 5, 6, 3, 6, 5, 6, 3, 4, 6, 6, 3], [3, 4, 5, 3, 3, 5, 5, 5, 5, 5, 5, 5, 3, 3, 5, 6, 6, 4, 5, 5, 5, 5, 5, 5, 5, 5, 5, 6, 6, 4], [4, 5, 6, 3, 6, 5, 8, 1, 8, 1, 8, 5, 6, 3, 6, 3, 6, 5, 6, 5, 0, 0, 0, 0, 0, 5, 6, 6, 4, 6], [6, 6, 3, 3, 5, 5, 1, 8, 0, 8, 1, 5, 5, 3, 6, 4, 6, 6, 3, 5, 0, 0, 4, 0, 0, 5, 3, 6, 5, 6], [6, 3, 4, 3, 6, 5, 8, 0, 0, 0, 8, 5, 6, 3, 4, 5, 6, 3, 6, 5, 0, 4, 2, 4, 0, 5, 4, 6, 6, 3], [6, 4, 5, 3, 3, 5, 1, 8, 0, 8, 1, 5, 3, 3, 6, 6, 6, 6, 5, 5, 0, 0, 4, 0, 0, 5, 5, 6, 3, 4], [4, 5, 6, 3, 4, 5, 8, 1, 8, 1, 8, 5, 4, 3, 6, 3, 6, 5, 6, 5, 0, 0, 0, 0, 0, 5, 6, 6, 4, 5], [5, 6, 3, 3, 5, 5, 5, 5, 5, 5, 5, 5, 5, 3, 3, 4, 6, 6, 3, 5, 5, 5, 5, 5, 5, 5, 3, 6, 5, 6], [6, 3, 6, 3, 6, 3, 4, 6, 6, 3, 4, 5, 6, 3, 4, 5, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 3], [3, 4, 5, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 6, 6, 3, 4, 5, 6, 3, 4, 5, 6, 3, 6, 5, 6, 3, 6], [4, 6, 6, 3, 4, 5, 6, 3, 6, 5, 6, 3, 4, 5, 6, 3, 4, 5, 6, 3, 6, 5, 6, 3, 4, 5, 6, 3, 4, 5]]}, {"input": [[2, 3, 4, 5, 6, 2, 3, 4, 5, 6, 2, 3, 8, 8, 6, 2, 3, 4, 5, 6, 2, 3, 4, 5, 6, 2, 3, 4, 5, 6], [6, 8, 3, 4, 5, 6, 2, 3, 4, 5, 6, 2, 3, 4, 5, 6, 8, 3, 4, 5, 6, 2, 3, 4, 5, 8, 2, 3, 8, 5], [5, 6, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 8, 4], [8, 5, 8, 1, 3, 8, 5, 6, 2, 3, 4, 5, 6, 2, 8, 8, 5, 6, 2, 3, 4, 5, 6, 2, 3, 4, 5, 1, 8, 3], [3, 4, 8, 1, 2, 3, 3, 3, 3, 3, 3, 3, 3, 6, 2, 3, 4, 5, 6, 3, 3, 3, 3, 3, 3, 3, 3, 1, 6, 2], [2, 3, 4, 1, 6, 3, 0, 8, 3, 3, 8, 0, 3, 5, 6, 2, 3, 4, 5, 3, 0, 8, 0, 0, 0, 0, 3, 1, 8, 6], [8, 2, 3, 1, 8, 3, 8, 3, 3, 3, 3, 8, 3, 4, 5, 6, 2, 3, 4, 3, 8, 3, 0, 0, 0, 0, 3, 1, 4, 5], [8, 6, 2, 1, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 8, 6, 8, 8, 3, 0, 0, 0, 0, 0, 0, 3, 1, 8, 4], [4, 5, 8, 1, 8, 3, 3, 3, 3, 3, 3, 3, 3, 2, 3, 4, 8, 6, 2, 3, 0, 0, 0, 0, 0, 0, 3, 1, 2, 3], [3, 4, 5, 1, 2, 3, 8, 3, 3, 3, 3, 8, 3, 6, 2, 3, 4, 5, 6, 3, 0, 0, 0, 0, 0, 0, 3, 1, 6, 2], [8, 3, 4, 1, 8, 3, 0, 8, 3, 3, 8, 0, 3, 5, 6, 2, 3, 4, 5, 3, 0, 0, 0, 0, 0, 0, 3, 1, 5, 6], [8, 8, 3, 1, 8, 3, 3, 3, 3, 3, 3, 3, 3, 4, 5, 6, 2, 3, 4, 3, 3, 3, 3, 3, 3, 3, 3, 1, 4, 8], [5, 6, 2, 1, 4, 5, 6, 2, 3, 4, 5, 6, 2, 3, 4, 5, 8, 2, 3, 4, 5, 6, 2, 3, 4, 5, 6, 1, 3, 4], [4, 5, 6, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 8], [8, 4, 8, 6, 2, 3, 4, 5, 6, 2, 3, 4, 5, 6, 2, 3, 4, 5, 6, 2, 3, 4, 5, 6, 2, 3, 4, 5, 6, 8], [2, 3, 4, 8, 6, 8, 8, 8, 5, 6, 2, 3, 4, 5, 6, 2, 3, 4, 5, 6, 2, 3, 8, 8, 6, 2, 8, 4, 8, 8], [6, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 3, 8, 5], [5, 4, 2, 3, 4, 5, 6, 2, 3, 4, 5, 6, 2, 3, 4, 8, 6, 6, 6, 6, 6, 6, 6, 6, 8, 4, 6, 8, 3, 4], [4, 4, 6, 6, 6, 6, 6, 6, 6, 6, 6, 5, 8, 2, 3, 4, 6, 0, 0, 0, 0, 8, 0, 6, 3, 4, 5, 6, 2, 3], [3, 4, 8, 6, 0, 0, 0, 0, 0, 0, 6, 4, 5, 6, 2, 3, 6, 0, 0, 0, 0, 6, 8, 6, 2, 4, 4, 5, 8, 2], [2, 4, 4, 6, 0, 0, 0, 0, 0, 0, 6, 3, 4, 5, 6, 2, 6, 0, 0, 0, 0, 0, 0, 6, 6, 4, 3, 4, 5, 6], [6, 4, 3, 6, 0, 0, 0, 0, 0, 0, 6, 2, 3, 4, 8, 6, 6, 0, 0, 0, 0, 0, 0, 6, 5, 4, 2, 8, 4, 5], [5, 4, 8, 6, 0, 0, 0, 0, 0, 0, 6, 6, 2, 3, 4, 8, 6, 0, 0, 0, 0, 0, 0, 6, 4, 4, 6, 2, 3, 4], [4, 4, 6, 6, 0, 0, 0, 0, 0, 0, 6, 5, 6, 2, 3, 8, 6, 0, 0, 0, 0, 0, 0, 6, 3, 4, 5, 6, 2, 3], [8, 4, 5, 6, 0, 0, 0, 0, 0, 0, 6, 4, 5, 6, 2, 8, 6, 6, 6, 6, 6, 6, 6, 6, 2, 4, 4, 8, 8, 2], [8, 4, 4, 6, 6, 6, 6, 6, 6, 6, 6, 3, 4, 8, 6, 2, 3, 4, 8, 6, 2, 3, 4, 5, 6, 4, 3, 4, 5, 6], [6, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 3, 4, 8], [5, 6, 2, 8, 4, 5, 6, 2, 3, 4, 5, 6, 2, 3, 4, 5, 6, 2, 8, 8, 8, 6, 2, 3, 4, 5, 6, 2, 3, 4], [4, 8, 6, 2, 3, 8, 5, 6, 2, 3, 8, 5, 6, 2, 3, 4, 5, 8, 2, 3, 4, 5, 6, 8, 3, 4, 5, 6, 2, 3], [3, 4, 5, 6, 8, 3, 4, 5, 6, 8, 3, 8, 5, 8, 2, 3, 4, 8, 6, 2, 8, 4, 5, 6, 2, 3, 4, 5, 6, 8]]}]}, "4e34c42c": {"train": [{"input": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 1, 1, 8, 8, 2, 8, 2, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 4, 1, 1, 1, 2, 2, 2, 3, 2, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 1, 1, 8, 8, 2, 8, 2, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 1, 1, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 1, 4, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 1, 1, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 3, 2, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]], "output": [[6, 8, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8], [6, 1, 1, 1, 8, 8, 2, 8, 2, 3, 3, 3], [6, 1, 4, 1, 1, 1, 2, 2, 2, 3, 2, 3], [6, 1, 1, 1, 8, 8, 2, 8, 2, 3, 3, 3], [6, 8, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8]]}, {"input": [[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 1, 1, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 8, 1, 1, 1, 3, 3, 1, 4, 4, 4], [4, 1, 1, 1, 4, 4, 4, 4, 4, 4, 8, 4, 4, 1, 4, 4, 1, 4, 4, 4], [4, 1, 2, 1, 4, 4, 4, 4, 4, 4, 8, 1, 1, 1, 3, 3, 1, 4, 4, 4], [4, 1, 1, 1, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 1, 1, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 1, 1, 1, 4, 4, 4, 4, 4, 4, 4, 5, 3, 5, 4, 4], [4, 4, 4, 4, 4, 1, 2, 1, 3, 3, 3, 3, 3, 3, 3, 3, 5, 3, 4, 4], [4, 4, 4, 4, 4, 1, 1, 1, 4, 4, 4, 4, 4, 4, 4, 5, 3, 5, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 6, 6, 6, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 5, 3, 5, 6, 6, 8, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 3, 5, 3, 6, 6, 8, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 5, 3, 5, 6, 6, 8, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 6, 6, 6, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]], "output": [[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 6, 6, 6, 4, 4, 4, 4, 1, 1, 1, 1], [1, 1, 1, 4, 4, 4, 4, 4, 4, 4, 5, 3, 5, 6, 6, 8, 1, 1, 1, 3, 3, 1], [1, 2, 1, 3, 3, 3, 3, 3, 3, 3, 3, 5, 3, 6, 6, 8, 4, 4, 1, 4, 4, 1], [1, 1, 1, 4, 4, 4, 4, 4, 4, 4, 5, 3, 5, 6, 6, 8, 1, 1, 1, 3, 3, 1], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 6, 6, 6, 4, 4, 4, 4, 1, 1, 1, 1]]}], "test": [{"input": [[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 6, 6, 1, 1, 1, 1, 3, 3, 3, 3, 3, 3, 3, 3, 6, 6, 3, 3], [3, 3, 8, 6, 1, 3, 3, 8, 8, 8, 3, 3, 3, 9, 9, 9, 6, 8, 3, 3], [3, 3, 6, 6, 1, 1, 1, 1, 7, 8, 3, 3, 3, 9, 3, 3, 6, 6, 3, 3], [3, 3, 3, 3, 3, 3, 3, 8, 8, 8, 3, 3, 3, 9, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 1, 1, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 7, 1, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 4, 4, 4, 3, 3, 3, 3, 3, 1, 7, 8, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 4, 3, 3, 3, 3, 3, 3, 8, 8, 8, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 4, 3, 3, 3, 3, 3, 3, 3, 7, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 1, 1, 1, 3, 3, 3, 3, 3, 3, 7, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 1, 6, 1, 3, 3, 3, 3, 3, 3, 7, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 1, 1, 3, 3, 3, 1, 1, 1, 3, 3], [3, 3, 3, 3, 2, 3, 2, 3, 3, 1, 6, 1, 3, 3, 3, 1, 7, 1, 3, 3], [3, 3, 3, 3, 4, 4, 4, 3, 3, 3, 8, 3, 3, 3, 3, 3, 8, 3, 3, 3], [3, 3, 3, 4, 4, 1, 4, 4, 3, 3, 8, 3, 3, 3, 3, 3, 8, 3, 3, 3], [3, 3, 3, 3, 4, 4, 4, 3, 3, 3, 8, 3, 3, 3, 3, 3, 8, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 3, 2, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]]}, {"input": [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 2, 4, 2, 1, 1, 1, 1, 3, 2, 2, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 2, 2, 1, 1, 1], [1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 2, 4, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 2, 2, 2, 1, 1, 1, 1, 1, 4, 4, 4, 1, 6, 1, 6, 1, 8, 8, 8], [1, 1, 3, 1, 1, 1, 1, 1, 1, 4, 7, 4, 8, 8, 8, 8, 8, 8, 7, 8], [9, 9, 3, 9, 9, 1, 1, 1, 1, 4, 4, 4, 8, 1, 1, 1, 8, 8, 8, 8], [1, 9, 9, 9, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 6, 1, 6, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 8, 8, 8, 3, 3, 3, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 8, 7, 8, 3, 3, 1, 1, 1, 1, 1, 1], [1, 1, 1, 6, 4, 4, 4, 1, 1, 8, 8, 8, 3, 3, 3, 1, 1, 1, 1, 1], [1, 6, 6, 6, 4, 7, 4, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 6, 4, 4, 4, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]}]}, "581f7754": {"train": [{"input": [[1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1], [1, 8, 8, 8, 1, 1, 1, 1], [1, 8, 4, 8, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 8, 1], [1, 1, 1, 1, 8, 8, 4, 1], [1, 1, 1, 1, 1, 1, 8, 1], [1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1], [1, 8, 1, 8, 1, 1, 1, 1], [1, 8, 4, 8, 1, 1, 1, 1], [1, 8, 1, 8, 1, 1, 1, 1], [1, 8, 8, 8, 1, 1, 1, 1], [1, 1, 1, 1, 1, 4, 1, 1]], "output": [[1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 8, 8, 8, 1], [1, 1, 1, 1, 8, 4, 8, 1], [1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 8, 1, 1], [1, 1, 1, 8, 8, 4, 1, 1], [1, 1, 1, 1, 1, 8, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 8, 1, 8, 1], [1, 1, 1, 1, 8, 4, 8, 1], [1, 1, 1, 1, 8, 1, 8, 1], [1, 1, 1, 1, 8, 8, 8, 1], [1, 1, 1, 1, 1, 4, 1, 1]]}, {"input": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 4, 8, 8, 8, 8, 8, 8, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 4, 8, 8, 8, 8, 8, 8, 3, 1, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8], [1, 8, 4, 8, 3, 3, 3, 8, 8, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 4, 8, 3, 8, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 8, 8, 8], [8, 8, 6, 8, 3, 8, 3, 8, 8, 8, 8, 8, 8, 8, 3, 1, 3, 3, 8, 8, 8], [8, 8, 8, 8, 3, 1, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 8, 8, 8], [6, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]], "output": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 3, 8, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 3, 8, 3, 8, 3, 3, 3, 8, 8, 8, 8, 8, 8, 3, 8, 8, 8], [1, 8, 4, 8, 3, 1, 3, 8, 3, 1, 3, 8, 8, 8, 3, 1, 3, 3, 8, 8, 8], [8, 8, 4, 8, 8, 8, 8, 8, 3, 3, 3, 8, 8, 8, 8, 8, 8, 3, 8, 8, 8], [8, 8, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [6, 8, 6, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]]}, {"input": [[3, 3, 3, 3, 2, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 1, 1, 1, 3, 3, 3, 3, 3, 3, 3], [3, 3, 1, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 2, 3, 3, 3, 3, 3, 3, 3, 3], [3, 1, 1, 1, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 1, 1, 1, 1, 1, 3, 3, 3], [3, 3, 3, 1, 3, 3, 3, 1, 3, 3, 3], [3, 3, 3, 1, 1, 1, 2, 1, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 1, 1, 1, 3, 3, 3, 3], [3, 3, 3, 1, 1, 2, 1, 1, 3, 3, 3], [3, 3, 3, 3, 3, 1, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]], "output": [[3, 3, 3, 3, 2, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 1, 1, 1, 3, 3, 3, 3, 3], [3, 3, 3, 3, 1, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 2, 3, 3, 3, 3, 3, 3], [3, 3, 3, 1, 1, 1, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 1, 1, 1, 1, 1, 3, 3, 3, 3, 3], [3, 1, 3, 3, 3, 1, 3, 3, 3, 3, 3], [3, 1, 1, 1, 2, 1, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 1, 1, 1, 3, 3, 3, 3, 3], [3, 3, 1, 1, 2, 1, 1, 3, 3, 3, 3], [3, 3, 3, 3, 1, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]]}], "test": [{"input": [[8, 8, 8, 6, 6, 6, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 6, 4, 6, 8, 8, 8, 8, 8, 8, 8, 4, 6, 4, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 4, 4, 8, 8, 8, 8], [6, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 4, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 4, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 4, 4, 4, 8, 8, 8, 8, 4, 8, 8, 8, 8, 8, 8, 3, 3, 3], [8, 2, 8, 4, 8, 4, 8, 8, 4, 4, 4, 8, 8, 8, 8, 8, 8, 3, 8, 3], [8, 8, 8, 4, 4, 4, 8, 8, 4, 4, 6, 8, 8, 8, 8, 8, 8, 3, 8, 3], [8, 8, 8, 8, 6, 8, 8, 8, 8, 8, 4, 8, 8, 8, 8, 8, 8, 3, 6, 3], [4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]]}, {"input": [[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 3, 1, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 1, 1, 1, 4, 4, 4, 4, 4, 4, 3, 1, 4, 4, 4, 4, 4, 4, 4, 4], [4, 1, 1, 1, 4, 1, 4, 4, 4, 4, 4, 4, 4, 1, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 1, 4, 1, 1, 1, 4, 4, 4, 4, 4, 1, 4, 4, 4, 4, 4, 4, 4, 2], [4, 4, 4, 1, 2, 1, 4, 4, 4, 4, 4, 4, 4, 1, 4, 4, 4, 1, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 4, 1, 1, 1, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 2, 1, 4, 8, 4, 1, 4, 1, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 4, 1, 4, 4, 4, 2, 4, 1, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 1, 4, 4, 4, 4, 4, 4, 4, 4, 4, 8], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]]}]}, "58490d8a": {"train": [{"input": [[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 8, 3, 3, 3], [3, 3, 3, 8, 8, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 3, 8, 3, 3], [3, 3, 3, 8, 3, 8, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 3, 3, 3], [3, 3, 3, 3, 8, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 8, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 8, 3, 8, 3, 3, 3, 3, 3, 3, 3], [3, 3, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 4, 3, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3], [0, 8, 0, 0, 0, 0, 0, 3, 3, 3, 3, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [0, 4, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 2, 3, 3, 3, 3], [0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 2, 3, 3, 3, 3], [0, 2, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]], "output": [[0, 0, 0, 0, 0, 0, 0], [0, 8, 0, 8, 0, 8, 0], [0, 0, 0, 0, 0, 0, 0], [0, 4, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0], [0, 2, 0, 2, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0]]}, {"input": [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 2, 2, 2, 1, 1, 1, 3, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 1, 1, 3, 3, 3, 1, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 3, 1, 1, 1, 1, 1, 1, 3, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [3, 3, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 8, 1, 8, 1, 1, 1, 1, 1, 1, 1, 1, 8, 1, 8, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 8, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 8, 1, 8, 1, 1, 1, 1, 1, 1, 1, 1, 8, 1, 8, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 3, 1, 1], [1, 1, 1, 1, 1, 4, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1], [1, 1, 1, 1, 4, 4, 4, 1, 1, 1, 1, 3, 3, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 1, 1, 1]], "output": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 2, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 8, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 3, 0, 3, 0, 3, 0, 3, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}, {"input": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 0, 0, 0, 0, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 1, 0, 0, 0, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 0, 0, 0, 0, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 4, 0, 0, 0, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 0, 0, 0, 0, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 4, 8, 8, 8, 8, 8, 8], [8, 8, 8, 2, 2, 8, 8, 8, 8, 8, 8, 8, 4, 4, 4, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 8, 4, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]], "output": [[0, 0, 0, 0, 0], [0, 1, 0, 1, 0], [0, 0, 0, 0, 0], [0, 4, 0, 0, 0], [0, 0, 0, 0, 0]]}], "test": [{"input": [[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 8, 8, 4, 4, 4, 4, 4, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 8, 8, 4, 4, 4, 4, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4, 8, 8, 4, 4], [4, 6, 4, 4, 4, 4, 4, 4, 4, 4, 4, 3, 4, 4, 4, 4, 3, 4, 4, 4, 4, 8, 8, 4, 4], [4, 6, 6, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4], [4, 6, 4, 4, 4, 4, 4, 2, 4, 4, 4, 4, 4, 4, 4, 4, 3, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 2, 4, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 1, 4], [4, 4, 4, 4, 4, 4, 4, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 4, 1, 4], [4, 4, 3, 4, 4, 4, 4, 4, 4, 4, 1, 1, 1, 4, 4, 4, 4, 8, 8, 4, 4, 4, 4, 4, 4], [4, 3, 3, 3, 4, 4, 4, 4, 4, 4, 1, 4, 1, 4, 4, 4, 4, 8, 8, 4, 4, 4, 4, 4, 4], [4, 4, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 4, 2, 4, 4], [4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 4, 4, 4], [4, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 4, 4, 4, 4, 4, 4, 8, 8, 4, 4, 4], [4, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 4, 4, 4, 4, 4, 4, 8, 8, 4, 4, 4], [4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 4, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4], [4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 4, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4]]}]}, "5961cc34": {"train": [{"input": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 4, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8]], "output": [[8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8]]}, {"input": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 3, 3, 8, 8, 8, 8, 8, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 8, 8, 8, 8], [8, 1, 1, 1, 1, 8, 8, 8, 1, 1, 1, 3, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 8, 8, 8], [8, 1, 1, 1, 1, 8, 8, 8, 1, 1, 1, 3, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 8, 8, 8], [8, 8, 1, 1, 8, 8, 8, 8, 8, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]], "output": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]]}, {"input": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 8, 8, 8, 8], [8, 8, 8, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 8, 8, 8], [8, 8, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 3, 1, 1, 1, 1, 1, 8, 8], [8, 8, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 3, 1, 1, 1, 1, 1, 8, 8], [8, 8, 8, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 8, 8, 8], [8, 8, 8, 8, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 3, 1, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 3, 1, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8]], "output": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 8, 8, 8, 8], [8, 8, 8, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 8, 8, 8], [8, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 8, 8], [8, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 8, 8], [8, 8, 8, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 8, 8, 8], [8, 8, 8, 8, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 8, 8, 8, 8], [8, 8, 8, 8, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8], [8, 8, 8, 8, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8], [8, 8, 8, 8, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8], [8, 8, 8, 8, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8], [8, 8, 8, 8, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8], [8, 8, 8, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8], [2, 2, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8], [2, 2, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8], [8, 8, 8, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8], [8, 8, 8, 8, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8]]}, {"input": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 1, 1, 1, 1, 1, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 1, 1, 1, 1, 1, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 1, 1, 1, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]], "output": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [8, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [8, 8, 8, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]]}], "test": [{"input": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 1, 1, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8], [8, 1, 1, 1, 1, 1, 1, 1, 3, 8, 8, 8, 8, 8, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8], [8, 1, 1, 1, 1, 1, 1, 1, 3, 8, 8, 8, 8, 8, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8], [8, 1, 1, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]]}]}, "62593bfd": {"train": [{"input": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 4, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 4, 0, 0, 0, 2, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 4, 0, 0, 0, 2, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 2, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], "output": [[0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0], [0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0], [0, 0, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0], [0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [4, 4, 0, 0, 0, 2, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 4, 0, 0, 0, 2, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 4, 0, 0, 0, 2, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 4, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}, {"input": [[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 3, 3, 3, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 3, 5, 3, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 1, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 3, 3, 3, 5, 5, 5, 5, 5], [5, 5, 5, 5, 1, 1, 1, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 3, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 7, 5, 5, 9, 9, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 7, 7, 5, 5, 9, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 2, 2, 2, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 4, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 2, 5, 2, 2, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 4, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 2, 2, 2, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 4, 4, 4, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5]], "output": [[5, 5, 5, 5, 2, 2, 2, 5, 5, 5, 5, 5, 5, 7, 5, 5, 9, 9, 5, 5, 5, 5, 3, 3, 3, 5, 5, 5, 5, 5], [5, 5, 5, 5, 2, 5, 2, 2, 5, 5, 5, 5, 7, 7, 5, 5, 9, 5, 5, 5, 5, 5, 3, 5, 3, 5, 5, 5, 5, 5], [5, 5, 5, 5, 2, 2, 2, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 3, 3, 3, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 3, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 4, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 1, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 4, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 1, 1, 1, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 4, 4, 4, 5, 5, 5, 5, 5, 5, 5]]}], "test": [{"input": [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 5, 1, 1, 1, 1, 1], [1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 5, 5, 5, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 5, 1, 5, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 5, 5, 5, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 5, 1, 1, 1, 5, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 3, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 3, 3, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 3, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 3, 1, 1, 1, 1], [1, 1, 1, 1, 1, 3, 3, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1], [1, 1, 1, 1, 3, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 4, 4, 4, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]}, {"input": [[9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 3, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 3, 9, 3, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 4, 4, 4, 9, 9, 9, 9, 9, 3, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 1, 9, 9, 9, 9, 9, 9, 9, 9, 9, 4, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 1, 1, 1, 1, 9, 9, 9, 9, 9, 9, 9, 9, 4, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 1, 9, 9, 9, 9, 7, 7, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 1, 9, 9, 9, 9, 7, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 6, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 6, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 6, 6, 6, 6, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 6, 9, 9, 9, 9, 9], [9, 9, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9]]}]}, "6ffbe589": {"train": [{"input": [[0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 0, 0], [0, 0, 0, 0, 0, 0, 3, 3, 8, 8, 0, 8, 8, 8, 0, 8, 0, 3, 0, 0], [0, 0, 0, 0, 0, 0, 0, 3, 8, 0, 8, 8, 0, 8, 8, 8, 8, 3, 0, 0], [0, 0, 0, 0, 0, 0, 0, 3, 0, 8, 0, 0, 6, 0, 0, 8, 0, 3, 0, 0], [0, 0, 0, 0, 0, 0, 0, 3, 8, 8, 6, 6, 6, 6, 0, 8, 8, 3, 3, 0], [0, 0, 0, 0, 0, 0, 0, 3, 8, 0, 0, 6, 0, 6, 0, 0, 8, 3, 0, 0], [0, 0, 0, 0, 0, 0, 0, 3, 8, 8, 0, 6, 6, 6, 6, 8, 8, 3, 3, 0], [0, 0, 0, 0, 0, 0, 0, 3, 0, 8, 0, 0, 6, 0, 0, 8, 0, 3, 0, 0], [0, 0, 0, 0, 0, 0, 3, 3, 8, 8, 8, 8, 0, 8, 8, 8, 8, 3, 0, 0], [0, 0, 0, 0, 0, 0, 0, 3, 0, 8, 0, 8, 8, 8, 0, 8, 0, 3, 0, 0], [0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 3, 0, 0, 0, 6, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], "output": [[0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 3, 0, 0], [0, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 0], [0, 3, 0, 8, 0, 8, 8, 8, 0, 8, 0, 3, 3], [0, 3, 8, 8, 8, 8, 0, 8, 8, 8, 8, 3, 0], [0, 3, 0, 8, 0, 0, 6, 0, 0, 8, 0, 3, 3], [3, 3, 8, 8, 6, 6, 6, 6, 0, 8, 8, 3, 0], [3, 3, 8, 0, 0, 6, 0, 6, 0, 0, 8, 3, 0], [0, 3, 8, 8, 0, 6, 6, 6, 6, 8, 8, 3, 0], [0, 3, 0, 8, 0, 0, 6, 0, 0, 8, 0, 3, 0], [0, 3, 8, 8, 8, 8, 0, 8, 8, 0, 8, 3, 0], [0, 3, 0, 8, 0, 8, 8, 8, 0, 8, 8, 3, 0], [0, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 0], [0, 0, 0, 0, 0, 3, 0, 3, 0, 0, 0, 0, 0]]}, {"input": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 5, 0, 5, 0, 5, 5, 5, 5, 0, 5, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 5, 0, 0, 3, 3, 3, 3, 0, 0, 5, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 3, 3, 0, 4, 0, 0, 3, 0, 5, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 5, 0, 3, 0, 4, 4, 0, 3, 3, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 5, 0, 3, 0, 4, 4, 4, 3, 0, 5, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 5, 0, 3, 0, 0, 0, 0, 3, 0, 5, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 5, 0, 0, 3, 3, 3, 3, 0, 0, 5, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 5, 0, 5, 5, 0, 5, 5, 5, 0, 5, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], "output": [[5, 0, 5, 0, 5, 5, 5, 5, 0, 5], [0, 0, 0, 0, 0, 0, 3, 0, 0, 0], [5, 0, 0, 3, 3, 3, 3, 0, 0, 5], [0, 0, 3, 0, 0, 0, 0, 3, 3, 5], [5, 3, 3, 0, 4, 4, 4, 3, 0, 0], [5, 0, 3, 0, 4, 4, 0, 3, 0, 5], [5, 0, 3, 0, 4, 0, 0, 3, 0, 5], [5, 0, 0, 3, 3, 3, 3, 0, 0, 5], [0, 0, 0, 0, 0, 3, 0, 0, 0, 0], [5, 0, 5, 5, 0, 5, 5, 5, 0, 5]]}, {"input": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 4, 0, 4, 0, 4, 0, 4, 0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 4, 0, 4, 0, 4, 0, 4, 0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 4, 0, 0, 0, 0, 1, 0, 0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 4, 0, 1, 0, 0, 1, 1, 0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 4, 0, 0, 1, 2, 2, 0, 1, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 4, 1, 0, 2, 2, 1, 0, 0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 4, 0, 0, 1, 1, 0, 0, 0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 4, 0, 0, 1, 1, 0, 0, 0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 4, 0, 4, 0, 4, 0, 4, 0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 4, 0, 4, 0, 4, 0, 4, 0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], "output": [[0, 4, 0, 4, 0, 4, 0, 4, 0, 4], [4, 0, 4, 0, 4, 0, 4, 0, 4, 0], [0, 4, 0, 1, 1, 0, 0, 0, 0, 4], [4, 0, 0, 1, 0, 1, 0, 0, 4, 0], [0, 4, 1, 0, 2, 2, 0, 1, 0, 4], [4, 0, 0, 0, 2, 2, 1, 1, 4, 0], [0, 4, 0, 1, 1, 0, 1, 0, 0, 4], [4, 0, 0, 0, 0, 1, 0, 0, 4, 0], [0, 4, 0, 4, 0, 4, 0, 4, 0, 4], [4, 0, 4, 0, 4, 0, 4, 0, 4, 0]]}], "test": [{"input": [[0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 5, 0, 0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 3, 3, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 5, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 5, 5, 0, 3, 3, 3, 3, 3, 3, 0, 0, 5, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 3, 3, 0, 5, 5], [0, 0, 0, 0, 0, 0, 0, 5, 5, 0, 3, 3, 0, 4, 0, 3, 3, 0, 5, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 3, 3, 0, 5, 5], [0, 0, 0, 0, 0, 0, 0, 5, 5, 0, 3, 3, 3, 3, 3, 3, 0, 0, 5, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 5, 0, 5, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 5, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}]}, "71e489b6": {"train": [{"input": [[1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], "output": [[1, 1, 7, 0, 7, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 7, 7, 7, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 7, 7], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 7, 0], [1, 1, 1, 1, 1, 1, 1, 7, 7, 7, 1, 1, 1, 1, 1, 7, 7], [1, 1, 1, 1, 1, 1, 1, 7, 0, 7, 1, 1, 1, 1, 1, 1, 1], [0, 0, 0, 0, 0, 0, 0, 7, 7, 7, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 7, 7, 7, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 7, 7, 0, 7, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 7, 0, 0, 7, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 7, 7, 7, 7, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 7, 7, 0], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 7, 0, 7, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 7, 7, 7, 1]]}, {"input": [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1], [1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1], [1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1], [1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1], [1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1], [1, 1, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0], [1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1], [1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], "output": [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 7, 0], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 7, 7], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1], [1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1], [1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 7, 7, 7], [1, 1, 0, 0, 0, 0, 1, 7, 7, 7, 1, 1, 0, 7, 0, 7], [1, 1, 0, 0, 0, 0, 1, 7, 0, 7, 1, 1, 0, 7, 7, 7], [1, 1, 0, 0, 0, 0, 1, 7, 7, 7, 1, 1, 0, 1, 7, 0], [1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 7, 7], [1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 7, 7, 7, 7, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 7, 0, 0, 7, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 7, 7, 7, 7, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]}, {"input": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1], [0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0], [0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0], [0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0], [0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0], [0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0], [0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0], [0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0], [0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], "output": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0], [0, 0, 0, 0, 1, 1, 1, 1, 1, 7, 7, 7, 1, 1, 0, 0], [0, 0, 0, 0, 1, 1, 1, 1, 1, 7, 0, 7, 7, 1, 0, 0], [0, 0, 0, 0, 1, 1, 1, 1, 1, 7, 7, 0, 7, 1, 0, 0], [0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 7, 7, 7, 1, 0, 0], [0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0], [0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0], [0, 0, 0, 0, 1, 1, 7, 7, 7, 1, 1, 1, 1, 1, 0, 0], [0, 0, 0, 0, 1, 1, 7, 0, 7, 1, 1, 1, 1, 1, 0, 0], [0, 0, 0, 0, 0, 0, 7, 7, 7, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}], "test": [{"input": [[1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1], [1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0], [1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1], [1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0], [1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0], [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0], [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]]}, {"input": [[0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [0, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1], [0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0], [0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1], [0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1], [0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1], [0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1], [0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1], [0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1], [0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1], [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1], [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1, 1, 1], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1, 1]]}]}, "7666fa5d": {"train": [{"input": [[8, 8, 8, 8, 8, 8, 8, 8, 4, 8, 8, 8, 4, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 4, 8, 8, 8, 4, 8, 8, 8, 4], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 8, 8, 8, 4, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 8, 8, 8, 4, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 4, 8, 8, 8, 8, 4, 8, 8, 8, 8, 8, 4, 8], [8, 8, 4, 8, 8, 8, 8, 4, 8, 8, 8, 8, 8, 4, 8, 8], [8, 8, 8, 8, 8, 8, 4, 8, 8, 8, 8, 8, 4, 8, 8, 8], [8, 8, 8, 8, 8, 4, 8, 8, 8, 8, 8, 4, 8, 8, 8, 8], [8, 8, 8, 8, 4, 8, 8, 8, 8, 8, 4, 8, 8, 8, 8, 8], [8, 8, 8, 4, 8, 8, 8, 8, 8, 4, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 4, 8, 8, 8, 8, 8, 8, 8]], "output": [[8, 8, 8, 8, 8, 8, 8, 8, 4, 8, 8, 8, 4, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 4, 2, 2, 8, 4, 2, 2, 8, 4], [8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 4, 2, 2, 2, 4, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 8, 2, 2, 4, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 4, 2, 2, 8, 8, 4, 8, 8, 8, 8, 8, 4, 8], [8, 8, 4, 2, 2, 2, 2, 4, 2, 2, 8, 8, 8, 4, 8, 8], [8, 8, 8, 2, 2, 2, 4, 2, 2, 2, 2, 8, 4, 8, 8, 8], [8, 8, 8, 8, 2, 4, 2, 2, 2, 2, 2, 4, 8, 8, 8, 8], [8, 8, 8, 8, 4, 8, 2, 2, 2, 2, 4, 8, 8, 8, 8, 8], [8, 8, 8, 4, 8, 8, 8, 2, 2, 4, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 4, 8, 8, 8, 8, 8, 8, 8]]}, {"input": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 8, 8, 8, 8, 8], [8, 8, 8, 8, 3, 8, 8, 8, 8, 3, 8, 8, 8, 8, 8, 8], [8, 8, 8, 3, 8, 8, 8, 8, 3, 8, 8, 8, 8, 8, 8, 8], [8, 8, 3, 8, 8, 8, 8, 3, 8, 8, 8, 8, 8, 8, 8, 8], [8, 3, 8, 8, 8, 8, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8], [3, 8, 8, 8, 8, 8, 8, 8, 8, 3, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 3, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 3, 8, 8, 8, 8, 8, 8, 8, 3], [8, 8, 8, 8, 8, 8, 3, 8, 8, 8, 8, 8, 8, 8, 3, 8], [8, 8, 8, 8, 8, 3, 8, 8, 8, 8, 8, 8, 8, 3, 8, 8], [8, 8, 8, 8, 3, 8, 8, 8, 8, 8, 8, 8, 3, 8, 8, 8], [8, 8, 8, 3, 8, 8, 8, 8, 8, 8, 8, 3, 8, 8, 8, 8], [8, 8, 3, 8, 8, 8, 8, 8, 8, 8, 3, 8, 8, 8, 8, 8], [8, 3, 8, 8, 8, 8, 8, 8, 8, 3, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 3, 8, 8, 8, 8, 8, 8, 8]], "output": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 8, 8, 8, 8, 8], [8, 8, 8, 8, 3, 8, 8, 8, 8, 3, 8, 8, 8, 8, 8, 8], [8, 8, 8, 3, 2, 2, 8, 8, 3, 2, 2, 8, 8, 8, 8, 8], [8, 8, 3, 2, 2, 2, 2, 3, 2, 2, 2, 2, 8, 8, 8, 8], [8, 3, 2, 2, 2, 2, 3, 2, 2, 2, 2, 2, 2, 8, 8, 8], [3, 2, 2, 2, 2, 2, 2, 2, 2, 3, 2, 2, 2, 2, 8, 8], [8, 2, 2, 2, 2, 2, 2, 2, 3, 2, 2, 2, 2, 2, 2, 8], [8, 8, 2, 2, 2, 2, 2, 3, 2, 2, 2, 2, 2, 2, 2, 3], [8, 8, 8, 2, 2, 2, 3, 2, 2, 2, 2, 2, 2, 2, 3, 8], [8, 8, 8, 8, 2, 3, 2, 2, 2, 2, 2, 2, 2, 3, 8, 8], [8, 8, 8, 8, 3, 2, 2, 2, 2, 2, 2, 2, 3, 8, 8, 8], [8, 8, 8, 3, 8, 2, 2, 2, 2, 2, 2, 3, 8, 8, 8, 8], [8, 8, 3, 8, 8, 8, 2, 2, 2, 2, 3, 8, 8, 8, 8, 8], [8, 3, 8, 8, 8, 8, 8, 2, 2, 3, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 3, 8, 8, 8, 8, 8, 8, 8]]}], "test": [{"input": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 9, 8, 8, 8, 8, 9, 8, 8, 8, 8, 8, 8], [8, 8, 8, 9, 8, 8, 8, 8, 9, 8, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 8, 9, 8, 8, 8, 8, 9, 8, 8, 8], [8, 8, 8, 8, 8, 8, 9, 8, 8, 8, 8, 9, 8, 8, 8, 9], [8, 8, 8, 8, 8, 9, 8, 8, 8, 8, 9, 8, 8, 8, 9, 8], [8, 8, 8, 8, 9, 8, 8, 8, 8, 9, 8, 8, 8, 9, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8, 8, 9, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 9, 8, 8, 8, 9, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 9, 8, 8, 8, 9, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 9, 8, 8, 8, 9, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 9, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 9, 8, 8, 8, 8, 8, 8, 8, 8]]}]}, "7b3084d4": {"train": [{"input": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 9, 0, 9, 0, 0, 0, 0, 0, 0, 0, 7, 7, 5, 0, 0, 0, 0], [0, 0, 0, 9, 9, 9, 9, 0, 0, 0, 0, 0, 0, 0, 7, 7, 0, 0, 0, 0], [0, 0, 0, 9, 9, 9, 0, 0, 0, 0, 0, 0, 0, 7, 0, 7, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 4, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3]], "output": [[5, 7, 7, 9, 9, 9], [7, 7, 9, 9, 9, 9], [7, 4, 7, 9, 3, 9], [4, 4, 4, 3, 3, 3], [4, 4, 4, 4, 4, 3], [4, 4, 4, 3, 3, 3]]}, {"input": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 7, 7, 7, 7, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 7, 7, 7, 7, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 7, 7, 7, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 7, 7, 7, 7, 7, 0, 0, 0, 0, 0, 0, 0, 2, 2, 2, 2, 2], [0, 0, 0, 5, 7, 7, 7, 7, 0, 0, 0, 0, 0, 0, 0, 2, 2, 2, 2, 2], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 2, 2], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 0, 2, 2], [0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 0, 2, 2], [3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0], [3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0], [3, 3, 3, 3, 3, 0, 0, 0, 6, 6, 6, 6, 6, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 6, 6, 6, 6, 6, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 6, 6, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 6, 6, 6, 6, 6, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 6, 6, 6, 6, 6, 0, 0, 0, 0, 0, 0, 0]], "output": [[5, 7, 7, 7, 7, 2, 2, 2, 2, 2], [7, 7, 7, 7, 7, 2, 2, 2, 2, 2], [7, 7, 7, 7, 7, 7, 2, 2, 2, 2], [7, 7, 7, 7, 7, 2, 2, 6, 2, 2], [7, 7, 3, 7, 7, 2, 2, 6, 2, 2], [3, 3, 3, 3, 3, 6, 6, 6, 6, 6], [3, 3, 3, 3, 3, 6, 6, 6, 6, 6], [3, 3, 3, 3, 3, 3, 3, 6, 6, 6], [3, 3, 3, 3, 3, 6, 6, 6, 6, 6], [3, 3, 3, 3, 3, 6, 6, 6, 6, 6]]}, {"input": [[2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 6, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 6, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 8, 8, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0], [0, 0, 0, 5, 7, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], "output": [[5, 7, 7, 8], [7, 8, 8, 8], [6, 6, 8, 2], [6, 6, 2, 2]]}], "test": [{"input": [[0, 2, 2, 2, 0, 2, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 2, 2, 2, 2, 2, 2, 2, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0], [0, 2, 2, 2, 2, 2, 2, 2, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0], [0, 0, 0, 0, 2, 2, 2, 2, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0], [0, 2, 2, 2, 2, 2, 2, 2, 0, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 0], [0, 2, 2, 2, 2, 2, 2, 2, 0, 0, 0, 0, 3, 3, 3, 0, 3, 3, 3, 0], [0, 2, 2, 2, 2, 2, 2, 2, 0, 0, 0, 0, 3, 3, 3, 0, 3, 3, 3, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 0, 3, 3, 3, 0], [0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [7, 7, 7, 7, 7, 7, 7, 0, 0, 8, 8, 8, 8, 8, 8, 8, 0, 0, 0, 0], [7, 7, 7, 7, 7, 7, 7, 0, 0, 8, 8, 8, 8, 8, 8, 8, 0, 0, 0, 0], [7, 7, 7, 7, 7, 7, 7, 0, 0, 8, 8, 8, 8, 8, 8, 8, 0, 0, 0, 0], [0, 0, 7, 7, 7, 7, 7, 0, 0, 8, 8, 8, 8, 8, 8, 8, 8, 0, 0, 0], [7, 7, 7, 7, 7, 7, 7, 0, 0, 8, 8, 8, 8, 8, 8, 8, 0, 0, 0, 0], [7, 7, 7, 7, 7, 7, 7, 0, 0, 8, 8, 8, 8, 8, 8, 8, 0, 0, 0, 0], [7, 7, 7, 7, 7, 7, 5, 0, 0, 8, 8, 8, 8, 8, 8, 8, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0]]}]}, "7b5033c1": {"train": [{"input": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 3, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 4, 4, 4, 4, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 6, 6, 6, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]], "output": [[1], [1], [1], [1], [1], [3], [3], [3], [3], [2], [2], [2], [2], [2], [4], [4], [4], [4], [4], [6], [6], [6]]}, {"input": [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 8, 8, 8, 8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 3, 3, 3, 3, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], "output": [[8], [8], [8], [8], [8], [8], [8], [2], [2], [2], [2], [2], [2], [2], [2], [3], [3], [3], [3], [3], [3], [3], [3], [3]]}], "test": [{"input": [[2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [2, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 8, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 8, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 8, 4, 4, 4, 2, 2, 3, 3, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2, 2, 4, 2, 2, 1, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2, 2, 4, 4, 1, 1, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]]}]}, "7ed72f31": {"train": [{"input": [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 3, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 4, 1, 2, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 4, 2, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 2, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 2, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 2, 5, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 5, 5, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 5, 1, 1, 1, 1, 8, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 8, 8, 1, 8, 8, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], "output": [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 3, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 3, 3, 1, 1, 1, 4, 4, 1, 2, 1, 4, 4], [1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 1, 4, 4, 2, 4, 4, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 2, 4, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 2, 4, 1, 1], [1, 1, 1, 1, 5, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 5, 5, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 5, 2, 5, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 5, 5, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 5, 1, 1, 1, 1, 8, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 8, 8, 1, 8, 8, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 8, 8, 1, 8, 8, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 8, 1, 1, 1, 1]]}, {"input": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 2, 4, 4, 4, 8, 8, 8, 1, 1, 2, 8, 8, 8, 8], [8, 8, 8, 8, 2, 4, 8, 4, 8, 8, 8, 8, 1, 2, 8, 8, 8, 8], [8, 8, 8, 8, 2, 4, 4, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 2, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 8, 3, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 8, 3, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]], "output": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 4, 4, 4, 2, 4, 4, 4, 8, 8, 8, 1, 1, 2, 1, 1, 8, 8], [8, 4, 8, 4, 2, 4, 8, 4, 8, 8, 8, 8, 1, 2, 1, 8, 8, 8], [8, 4, 4, 4, 2, 4, 4, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 4, 2, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 8, 3, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 8, 3, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 8, 3, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 8, 3, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]]}], "test": [{"input": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 4, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 4, 4, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 4, 4, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 3, 3, 3, 3, 3, 3, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 8, 3, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 8, 8, 8, 8, 8, 8]]}, {"input": [[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 1, 1, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 3, 2, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 4, 4, 4, 3, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 4, 3, 3, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 8, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 8, 8, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 8, 3, 8, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 3, 3, 3, 3, 3, 3, 3, 2, 8, 8, 8, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 8, 3, 3, 3, 3, 3, 3, 3, 3, 2, 3, 3, 8, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 8, 8, 8, 3, 3, 3, 3, 3, 3, 3, 3, 2, 8, 8, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 2, 3, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 6, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 2, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 6, 3, 6, 3, 3, 3, 3, 3], [3, 3, 3, 3, 2, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 6, 6, 3, 3, 3, 3, 3], [3, 3, 3, 3, 2, 1, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]]}]}, "80a900e0": {"train": [{"input": [[1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0], [0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1], [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0], [0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1], [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0], [0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1], [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0], [0, 1, 0, 1, 0, 1, 0, 8, 0, 1, 0, 1, 0, 1], [1, 0, 1, 0, 1, 0, 3, 0, 8, 0, 1, 0, 1, 0], [0, 1, 0, 1, 0, 3, 0, 3, 0, 8, 0, 1, 0, 1], [1, 0, 1, 0, 3, 0, 1, 0, 3, 0, 1, 0, 1, 0], [0, 1, 0, 8, 0, 3, 0, 3, 0, 1, 0, 1, 0, 1], [1, 0, 1, 0, 8, 0, 3, 0, 1, 0, 1, 0, 1, 0], [0, 1, 0, 1, 0, 8, 0, 1, 0, 1, 0, 1, 0, 1], [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0], [0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1], [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0], [0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1], [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0], [0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1]], "output": [[1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0], [0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 8], [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 8, 0], [0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 8, 0, 1], [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 8, 0, 1, 0], [0, 1, 0, 1, 0, 1, 0, 1, 0, 8, 0, 1, 0, 8], [1, 0, 1, 0, 1, 0, 1, 0, 8, 0, 1, 0, 8, 0], [0, 1, 0, 1, 0, 1, 0, 8, 0, 1, 0, 8, 0, 1], [1, 0, 1, 0, 1, 0, 3, 0, 8, 0, 8, 0, 1, 0], [0, 1, 0, 1, 0, 3, 0, 3, 0, 8, 0, 1, 0, 1], [1, 0, 1, 0, 3, 0, 1, 0, 3, 0, 1, 0, 1, 0], [0, 1, 0, 8, 0, 3, 0, 3, 0, 1, 0, 1, 0, 1], [1, 0, 8, 0, 8, 0, 3, 0, 1, 0, 1, 0, 1, 0], [0, 8, 0, 1, 0, 8, 0, 1, 0, 1, 0, 1, 0, 1], [8, 0, 1, 0, 8, 0, 1, 0, 1, 0, 1, 0, 1, 0], [0, 1, 0, 8, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1], [1, 0, 8, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0], [0, 8, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1], [8, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0], [0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1]]}, {"input": [[1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0], [0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1], [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0], [0, 1, 0, 1, 0, 1, 0, 1, 0, 2, 0, 4, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1], [1, 0, 1, 0, 1, 0, 1, 0, 2, 0, 3, 0, 4, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0], [0, 1, 0, 1, 0, 1, 0, 2, 0, 3, 0, 3, 0, 4, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1], [1, 0, 1, 0, 1, 0, 1, 0, 3, 0, 1, 0, 3, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0], [0, 1, 0, 1, 0, 1, 0, 4, 0, 3, 0, 3, 0, 2, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1], [1, 0, 1, 0, 1, 0, 1, 0, 4, 0, 3, 0, 2, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0], [0, 1, 0, 1, 0, 1, 0, 1, 0, 4, 0, 2, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1], [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0], [0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1], [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0], [0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1], [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0], [0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1], [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0], [0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1], [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0], [0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1], [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0], [0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1], [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0], [0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1]], "output": [[1, 0, 2, 0, 1, 0, 2, 0, 1, 0, 1, 0, 1, 0, 4, 0, 1, 0, 4, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0], [0, 1, 0, 2, 0, 1, 0, 2, 0, 1, 0, 1, 0, 4, 0, 1, 0, 4, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1], [1, 0, 1, 0, 2, 0, 1, 0, 2, 0, 1, 0, 4, 0, 1, 0, 4, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0], [0, 1, 0, 1, 0, 2, 0, 1, 0, 2, 0, 4, 0, 1, 0, 4, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1], [1, 0, 1, 0, 1, 0, 2, 0, 2, 0, 3, 0, 4, 0, 4, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0], [0, 1, 0, 1, 0, 1, 0, 2, 0, 3, 0, 3, 0, 4, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1], [1, 0, 1, 0, 1, 0, 1, 0, 3, 0, 1, 0, 3, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0], [0, 1, 0, 1, 0, 1, 0, 4, 0, 3, 0, 3, 0, 2, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1], [1, 0, 1, 0, 1, 0, 4, 0, 4, 0, 3, 0, 2, 0, 2, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0], [0, 1, 0, 1, 0, 4, 0, 1, 0, 4, 0, 2, 0, 1, 0, 2, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1], [1, 0, 1, 0, 4, 0, 1, 0, 4, 0, 1, 0, 2, 0, 1, 0, 2, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0], [0, 1, 0, 4, 0, 1, 0, 4, 0, 1, 0, 1, 0, 2, 0, 1, 0, 2, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1], [1, 0, 4, 0, 1, 0, 4, 0, 1, 0, 1, 0, 1, 0, 2, 0, 1, 0, 2, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0], [0, 4, 0, 1, 0, 4, 0, 1, 0, 1, 0, 1, 0, 1, 0, 2, 0, 1, 0, 2, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1], [4, 0, 1, 0, 4, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 2, 0, 1, 0, 2, 0, 1, 0, 1, 0, 1, 0, 1, 0], [0, 1, 0, 4, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 2, 0, 1, 0, 2, 0, 1, 0, 1, 0, 1, 0, 1], [1, 0, 4, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 2, 0, 1, 0, 2, 0, 1, 0, 1, 0, 1, 0], [0, 4, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 2, 0, 1, 0, 2, 0, 1, 0, 1, 0, 1], [4, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 2, 0, 1, 0, 2, 0, 1, 0, 1, 0], [0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 2, 0, 1, 0, 2, 0, 1, 0, 1], [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 2, 0, 1, 0, 2, 0, 1, 0], [0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 2, 0, 1, 0, 2, 0, 1], [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 2, 0, 1, 0, 2, 0], [0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 2, 0, 1, 0, 2]]}], "test": [{"input": [[1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1], [0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0], [1, 0, 1, 0, 1, 0, 2, 0, 4, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1], [0, 1, 0, 1, 0, 2, 0, 6, 0, 4, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0], [1, 0, 1, 0, 2, 0, 6, 0, 6, 0, 4, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1], [0, 1, 0, 1, 0, 6, 0, 1, 0, 6, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0], [1, 0, 1, 0, 9, 0, 6, 0, 6, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1], [0, 1, 0, 1, 0, 9, 0, 6, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0], [1, 0, 1, 0, 1, 0, 9, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1], [0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0], [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1], [0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 6, 0, 1, 0, 1, 0, 1, 0], [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 3, 0, 6, 0, 1, 0, 1, 0, 1], [0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 3, 0, 3, 0, 6, 0, 1, 0, 1, 0], [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 3, 0, 1, 0, 3, 0, 1, 0, 1, 0, 1], [0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 3, 0, 3, 0, 7, 0, 1, 0, 1, 0], [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 3, 0, 7, 0, 1, 0, 1, 0, 1], [0, 1, 0, 1, 0, 2, 0, 1, 0, 1, 0, 1, 0, 7, 0, 1, 0, 1, 0, 1, 0], [1, 0, 1, 0, 2, 0, 2, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1], [0, 1, 0, 2, 0, 1, 0, 2, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0], [1, 0, 6, 0, 2, 0, 2, 0, 6, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1], [0, 1, 0, 6, 0, 2, 0, 6, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0]]}]}, "88e364bc": {"train": [{"input": [[7, 7, 7, 7, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 5], [7, 1, 1, 2, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 2, 1, 1, 5], [7, 7, 7, 7, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 5], [7, 1, 1, 2, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 2, 1, 1, 5], [7, 7, 7, 7, 7, 0, 0, 0, 7, 7, 7, 7, 7, 0, 0, 5, 5, 5, 5, 5], [0, 0, 0, 0, 0, 0, 0, 7, 7, 0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 7, 7, 0, 4, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0], [0, 7, 7, 7, 7, 7, 7, 0, 0, 0, 7, 7, 7, 0, 0, 5, 5, 5, 5, 0], [0, 7, 0, 0, 0, 0, 0, 0, 0, 7, 7, 0, 0, 5, 5, 5, 0, 0, 5, 0], [0, 7, 0, 0, 0, 0, 0, 0, 0, 7, 5, 5, 5, 5, 0, 0, 0, 0, 5, 0], [0, 7, 7, 0, 4, 0, 0, 0, 0, 7, 5, 0, 0, 0, 0, 0, 0, 5, 5, 0], [0, 0, 7, 7, 7, 7, 0, 0, 0, 7, 5, 0, 0, 0, 0, 0, 0, 5, 0, 0], [0, 0, 0, 0, 0, 7, 7, 7, 7, 7, 5, 0, 0, 0, 0, 4, 0, 5, 0, 0], [0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 0, 0, 0, 0, 0, 0, 5, 0, 0], [0, 5, 5, 5, 5, 5, 5, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 0], [0, 5, 0, 0, 4, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0], [0, 5, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 4, 0, 0, 0, 0, 5, 0], [0, 5, 5, 0, 0, 0, 5, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 0], [0, 0, 5, 0, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 0, 0], [0, 0, 5, 5, 5, 0, 0, 0, 0, 0, 0, 0, 5, 5, 0, 0, 0, 0, 0, 0]], "output": [[7, 7, 7, 7, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 5], [7, 1, 1, 2, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 2, 1, 1, 5], [7, 7, 7, 7, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 5], [7, 1, 1, 2, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 2, 1, 1, 5], [7, 7, 7, 7, 7, 0, 0, 0, 7, 7, 7, 7, 7, 0, 0, 5, 5, 5, 5, 5], [0, 0, 0, 0, 0, 0, 0, 7, 7, 0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 7, 7, 0, 0, 0, 4, 7, 0, 0, 0, 0, 0, 0, 0], [0, 7, 7, 7, 7, 7, 7, 0, 0, 0, 7, 7, 7, 0, 0, 5, 5, 5, 5, 0], [0, 7, 0, 0, 0, 0, 0, 0, 0, 7, 7, 0, 0, 5, 5, 5, 0, 0, 5, 0], [0, 7, 0, 0, 0, 0, 0, 0, 0, 7, 5, 5, 5, 5, 0, 0, 0, 0, 5, 0], [0, 7, 7, 0, 0, 0, 0, 0, 4, 7, 5, 0, 0, 0, 0, 0, 0, 5, 5, 0], [0, 0, 7, 7, 7, 7, 0, 0, 0, 7, 5, 0, 0, 0, 0, 0, 0, 5, 0, 0], [0, 0, 0, 0, 0, 7, 7, 7, 7, 7, 5, 4, 0, 0, 0, 0, 0, 5, 0, 0], [0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 0, 0, 0, 0, 0, 0, 5, 0, 0], [0, 5, 5, 5, 5, 5, 5, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 0], [0, 5, 4, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0], [0, 5, 0, 0, 0, 0, 5, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0], [0, 5, 5, 0, 0, 0, 5, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 0], [0, 0, 5, 0, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 0, 0], [0, 0, 5, 5, 5, 0, 0, 0, 0, 0, 0, 0, 5, 5, 0, 0, 0, 0, 0, 0]]}, {"input": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 5, 5, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 1, 5, 5, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 1, 5, 5, 1, 5, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 1, 5, 5, 2, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 2, 5, 5, 5], [0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 0, 0, 0, 5, 5, 5, 5, 5, 5, 5], [0, 0, 0, 0, 0, 5, 5, 0, 0, 5, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 5, 5, 5, 5, 0, 0, 0, 0, 5, 5, 5, 0, 0, 0, 0, 0, 0, 0], [0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 0, 0, 0, 0], [0, 0, 5, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0], [0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0], [0, 0, 5, 0, 0, 0, 5, 5, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0], [0, 0, 5, 0, 0, 0, 5, 5, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0], [0, 5, 5, 0, 0, 0, 5, 5, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0], [0, 5, 0, 0, 0, 0, 5, 5, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0], [0, 5, 0, 0, 0, 0, 5, 5, 5, 0, 0, 4, 0, 0, 5, 5, 0, 0, 0, 0], [0, 5, 5, 0, 5, 5, 5, 0, 5, 0, 0, 0, 0, 5, 5, 0, 0, 0, 0, 0], [0, 0, 5, 5, 5, 0, 0, 0, 5, 0, 0, 0, 5, 5, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 5, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], "output": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 5, 5, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 1, 5, 5, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 1, 5, 5, 1, 5, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 1, 5, 5, 2, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 2, 5, 5, 5], [0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 0, 0, 0, 5, 5, 5, 5, 5, 5, 5], [0, 0, 0, 0, 0, 5, 5, 0, 0, 5, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 5, 5, 5, 5, 0, 0, 0, 0, 5, 5, 5, 0, 0, 0, 0, 0, 0, 0], [0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 0, 0, 0, 0], [0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0], [0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0], [0, 0, 5, 0, 0, 4, 5, 5, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0], [0, 0, 5, 0, 0, 0, 5, 5, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0], [0, 5, 5, 0, 0, 0, 5, 5, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0], [0, 5, 0, 0, 0, 0, 5, 5, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0], [0, 5, 0, 0, 0, 0, 5, 5, 5, 0, 0, 0, 0, 0, 5, 5, 0, 0, 0, 0], [0, 5, 5, 0, 5, 5, 5, 0, 5, 0, 0, 0, 4, 5, 5, 0, 0, 0, 0, 0], [0, 0, 5, 5, 5, 0, 0, 0, 5, 0, 0, 0, 5, 5, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 5, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}, {"input": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 1, 5, 1, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 1, 5, 1, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 2, 5, 2, 5], [0, 0, 0, 0, 5, 5, 5, 5, 5, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 5], [0, 0, 0, 0, 5, 0, 0, 0, 5, 5, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 5, 4, 0, 0, 0, 0, 5, 5, 5, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0], [0, 0, 5, 5, 5, 0, 0, 0, 0, 0, 0, 0, 5, 5, 0, 0, 0, 0, 0, 0], [0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 0, 0, 0, 0, 0], [0, 0, 5, 0, 0, 5, 5, 5, 5, 0, 0, 0, 0, 5, 5, 0, 0, 0, 0, 0], [0, 0, 5, 5, 5, 5, 0, 0, 5, 5, 5, 5, 5, 5, 5, 5, 5, 0, 0, 0], [0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0], [0, 0, 5, 5, 0, 0, 4, 0, 0, 0, 4, 0, 0, 0, 0, 0, 5, 0, 0, 0], [0, 0, 0, 5, 5, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 0, 5, 0, 0, 0], [0, 0, 0, 0, 5, 0, 0, 5, 5, 0, 0, 0, 5, 5, 5, 5, 5, 0, 0, 0], [0, 0, 0, 0, 5, 0, 0, 5, 5, 5, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 5, 0, 0, 5, 0, 5, 5, 0, 5, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 5, 0, 0, 5, 0, 0, 5, 5, 5, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 5, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], "output": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 1, 5, 1, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 1, 5, 1, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 2, 5, 2, 5], [0, 0, 0, 0, 5, 5, 5, 5, 5, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 5], [0, 0, 0, 0, 5, 0, 0, 0, 5, 5, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 5, 5, 5, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0], [0, 0, 5, 5, 5, 0, 0, 0, 0, 0, 0, 0, 5, 5, 0, 0, 0, 0, 0, 0], [0, 0, 5, 0, 0, 4, 0, 0, 0, 0, 0, 0, 0, 5, 5, 0, 0, 0, 0, 0], [0, 0, 5, 0, 0, 5, 5, 5, 5, 0, 0, 0, 0, 5, 5, 0, 0, 0, 0, 0], [0, 0, 5, 5, 5, 5, 0, 0, 5, 5, 5, 5, 5, 5, 5, 5, 5, 0, 0, 0], [0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0], [0, 0, 5, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0], [0, 0, 0, 5, 5, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 0, 5, 0, 0, 0], [0, 0, 0, 0, 5, 0, 0, 5, 5, 0, 0, 0, 5, 5, 5, 5, 5, 0, 0, 0], [0, 0, 0, 0, 5, 0, 0, 5, 5, 5, 4, 0, 5, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 5, 0, 0, 5, 0, 5, 5, 0, 5, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 5, 0, 4, 5, 0, 0, 5, 5, 5, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 5, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}], "test": [{"input": [[5, 5, 5, 5, 5, 5, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [5, 5, 5, 1, 5, 5, 5, 0, 0, 5, 5, 5, 0, 0, 0, 0, 0, 0, 0, 0], [5, 1, 5, 5, 1, 5, 5, 0, 0, 5, 0, 5, 5, 5, 0, 0, 0, 0, 0, 0], [5, 5, 1, 5, 5, 2, 5, 0, 0, 5, 0, 4, 0, 5, 0, 0, 5, 5, 0, 0], [5, 5, 5, 2, 5, 5, 5, 0, 0, 5, 0, 0, 0, 5, 5, 5, 5, 5, 5, 0], [5, 5, 5, 5, 5, 5, 5, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 4, 0, 0, 0, 0, 0, 0, 5, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 0, 0, 0, 0, 0, 0, 0, 5, 0], [0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 5, 0, 0, 0, 0, 0, 0, 0, 5, 0], [0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 5, 5, 5, 0, 0, 5, 5, 5, 5, 0], [0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1, 1, 5, 5, 5, 5, 0, 0, 0, 0], [0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 1, 1, 0, 4, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 1, 0], [1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 1, 0], [1, 2, 1, 2, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0], [1, 5, 1, 5, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 4, 0, 1, 0], [1, 5, 1, 5, 1, 0, 0, 0, 0, 1, 1, 4, 1, 1, 1, 1, 1, 1, 1, 0], [1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0]]}, {"input": [[6, 6, 6, 6, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [6, 2, 6, 2, 6, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 6, 6, 6, 6], [6, 1, 6, 1, 6, 0, 0, 3, 0, 0, 0, 0, 0, 3, 0, 0, 6, 0, 0, 6], [6, 1, 6, 1, 6, 0, 3, 3, 0, 0, 0, 0, 0, 3, 6, 6, 6, 6, 0, 6], [6, 6, 6, 6, 6, 0, 3, 0, 0, 0, 0, 0, 3, 3, 6, 0, 0, 0, 0, 6], [0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 3, 0, 6, 0, 4, 0, 0, 6], [0, 0, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 3, 3, 6, 6, 6, 0, 0, 6], [0, 0, 3, 0, 0, 0, 0, 0, 0, 4, 0, 0, 0, 3, 3, 0, 6, 0, 0, 6], [0, 0, 3, 3, 0, 0, 0, 0, 0, 0, 0, 4, 0, 3, 3, 0, 6, 0, 4, 6], [0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 0, 0, 6, 6, 0, 6], [0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 6, 6, 6], [0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 3, 3, 0, 4, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 3, 3, 0, 0, 0, 0, 0, 0, 3, 0, 0, 3, 3, 3, 3, 3, 3, 3], [0, 0, 3, 0, 0, 0, 0, 0, 0, 3, 3, 0, 0, 3, 3, 3, 2, 3, 3, 3], [0, 0, 3, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 3, 2, 3, 3, 1, 3, 3], [0, 0, 3, 3, 0, 4, 0, 0, 0, 3, 0, 0, 0, 3, 3, 1, 3, 3, 1, 3], [0, 0, 0, 3, 3, 3, 0, 0, 3, 3, 0, 0, 0, 3, 3, 3, 1, 3, 3, 3], [0, 0, 0, 0, 0, 3, 3, 3, 3, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3]]}]}, "89565ca0": {"train": [{"input": [[0, 0, 0, 0, 0, 0, 0, 5, 0, 5, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0], [0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 5, 5, 0, 0, 5, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0], [5, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 5, 3, 3, 5, 5, 0], [0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0], [0, 0, 3, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0], [0, 0, 3, 5, 1, 0, 0, 5, 0, 1, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0], [0, 0, 5, 0, 1, 0, 0, 0, 5, 1, 0, 0, 0, 3, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 3, 0, 0, 0], [0, 0, 5, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 3, 0, 0, 0], [0, 0, 3, 0, 5, 0, 0, 0, 0, 1, 0, 0, 0, 3, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0], [0, 0, 3, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 3, 0, 5, 0, 5, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0], [0, 5, 3, 5, 0, 0, 0, 0, 0, 0, 5, 0, 5, 3, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 3, 0, 0, 0], [0, 0, 3, 5, 0, 0, 0, 0, 5, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 5, 2, 2, 2, 0], [0, 0, 3, 0, 0, 0, 0, 0, 5, 0, 2, 0, 0, 3, 0, 2, 0, 0, 0, 0, 2, 0, 0, 0, 5, 0, 2, 0], [5, 0, 3, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 5, 0, 2, 5, 0, 0, 0, 2, 0, 0, 0, 3, 0, 2, 0], [0, 0, 3, 3, 3, 3, 3, 3, 3, 5, 2, 3, 3, 3, 3, 2, 3, 3, 3, 3, 2, 3, 3, 3, 3, 0, 2, 0], [0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 5, 0, 0, 0, 0, 2, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 2, 0], [0, 5, 5, 0, 0, 0, 0, 0, 0, 0, 2, 2, 2, 2, 2, 5, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 5, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 2, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 2, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 5, 0, 0, 0, 0, 2, 5, 0, 0, 0, 0, 2, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 2, 2, 5, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 0], [0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], "output": [[1, 5, 5, 5], [3, 3, 5, 5], [2, 2, 2, 2]]}, {"input": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 4, 4, 4, 4, 5, 4, 5, 4, 4, 4, 4, 4, 4, 4, 4, 0], [0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 4, 0, 0, 0, 0, 4, 0, 0, 0, 0, 4, 0, 0, 0, 4, 0], [0, 1, 0, 5, 2, 2, 2, 2, 2, 2, 2, 2, 4, 2, 2, 2, 0, 4, 0, 5, 0, 0, 4, 0, 5, 5, 4, 0], [0, 1, 0, 0, 2, 0, 0, 1, 0, 0, 2, 0, 4, 0, 0, 2, 0, 4, 0, 0, 0, 0, 4, 0, 0, 5, 5, 0], [0, 1, 1, 1, 2, 1, 1, 1, 0, 0, 2, 5, 4, 0, 0, 2, 0, 4, 0, 5, 5, 0, 4, 0, 0, 0, 4, 0], [0, 1, 0, 0, 2, 0, 0, 1, 0, 0, 2, 0, 4, 0, 0, 2, 0, 4, 0, 0, 0, 5, 4, 0, 0, 0, 4, 5], [0, 1, 0, 0, 5, 0, 0, 1, 0, 0, 2, 0, 4, 0, 0, 2, 0, 4, 0, 0, 0, 0, 4, 0, 0, 0, 4, 0], [0, 5, 0, 0, 2, 0, 0, 1, 0, 0, 2, 0, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 0], [0, 1, 0, 0, 2, 0, 0, 1, 0, 0, 2, 0, 0, 0, 0, 2, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 1, 1, 1, 2, 1, 1, 1, 0, 0, 2, 2, 2, 2, 2, 5, 0, 8, 5, 8, 8, 8, 8, 8, 8, 0, 0, 0], [0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 2, 0, 8, 0, 0, 0, 5, 0, 0, 8, 0, 0, 5], [0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 2, 0, 8, 0, 0, 5, 0, 0, 0, 8, 0, 0, 0], [0, 5, 0, 0, 2, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 5, 0, 8, 0, 5, 5, 5, 0, 0, 8, 0, 0, 0], [0, 0, 0, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 5, 8, 0, 0, 5, 0, 0, 0, 5, 0, 5, 5], [0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 5, 0, 0, 0, 0, 8, 0, 0, 5, 0, 0, 0, 8, 0, 0, 0], [0, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 0, 0, 8, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0], [0, 3, 0, 3, 0, 0, 0, 3, 0, 0, 0, 3, 0, 0, 3, 0, 0, 8, 8, 8, 8, 8, 8, 8, 8, 0, 0, 0], [0, 3, 0, 3, 0, 0, 0, 3, 5, 0, 5, 3, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 3, 0, 3, 0, 0, 0, 3, 0, 0, 0, 3, 0, 0, 3, 0, 0, 0, 5, 0, 0, 0, 5, 0, 0, 0, 0, 0], [0, 3, 3, 3, 3, 3, 3, 3, 5, 3, 3, 5, 3, 3, 3, 0, 0, 0, 5, 5, 0, 0, 0, 0, 5, 5, 0, 0], [0, 0, 0, 0, 5, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], "output": [[8, 5, 5, 5], [1, 1, 5, 5], [2, 2, 2, 5], [4, 4, 4, 5], [3, 3, 3, 3]]}, {"input": [[0, 0, 0, 0, 0, 0, 0, 0, 7, 0, 7, 0, 0, 0, 0, 7, 0, 0, 0, 0], [0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 8, 8, 8, 8, 0, 0], [0, 1, 0, 0, 1, 0, 0, 0, 0, 7, 7, 0, 0, 0, 8, 0, 0, 8, 0, 0], [0, 1, 0, 0, 7, 7, 0, 0, 0, 1, 0, 0, 0, 0, 8, 0, 0, 8, 0, 0], [0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 8, 8, 8, 8, 0, 0], [0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 1, 0, 0, 1, 7, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 0, 0, 0, 0], [0, 1, 0, 0, 7, 2, 0, 0, 0, 1, 0, 2, 0, 0, 0, 7, 0, 0, 0, 0], [0, 1, 1, 1, 1, 2, 1, 1, 1, 1, 0, 2, 0, 0, 0, 2, 0, 0, 0, 7], [0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 2, 0, 7, 0, 2, 0, 0, 0, 0], [0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 2, 0, 0, 0, 2, 0, 7, 0, 0], [0, 7, 0, 0, 0, 2, 0, 0, 0, 0, 0, 7, 0, 0, 0, 2, 0, 0, 0, 0], [0, 0, 0, 0, 0, 2, 0, 0, 0, 7, 0, 2, 0, 0, 0, 2, 0, 0, 0, 0], [0, 7, 0, 0, 0, 7, 2, 2, 2, 2, 2, 7, 2, 2, 2, 2, 0, 0, 0, 0], [0, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0], [0, 0, 0, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [0, 0, 0, 4, 0, 2, 0, 0, 0, 4, 0, 0, 4, 0, 0, 2, 4, 0, 0, 4], [0, 0, 0, 4, 0, 2, 0, 0, 0, 4, 0, 0, 4, 0, 0, 2, 4, 0, 0, 4], [0, 0, 0, 4, 0, 2, 2, 2, 7, 4, 2, 7, 4, 2, 7, 2, 4, 0, 0, 4], [7, 0, 0, 4, 0, 0, 0, 0, 0, 4, 0, 0, 4, 0, 0, 0, 4, 0, 0, 4], [0, 0, 0, 7, 0, 0, 0, 0, 0, 4, 0, 0, 4, 0, 0, 0, 4, 7, 0, 4], [0, 0, 0, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]], "output": [[8, 7, 7, 7], [1, 1, 7, 7], [2, 2, 2, 7], [4, 4, 4, 4]]}], "test": [{"input": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 2, 2, 2, 2, 2, 0, 0, 9, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0], [0, 2, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 9, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0], [0, 2, 2, 2, 2, 2, 0, 0, 0, 0, 0, 8, 8, 8, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 1, 0, 0], [0, 2, 0, 0, 0, 2, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 1, 0, 8, 0, 0, 0, 0, 0, 0, 8, 0, 1, 0, 0], [0, 9, 0, 0, 0, 2, 0, 0, 9, 0, 0, 8, 0, 0, 0, 0, 1, 0, 8, 0, 0, 0, 0, 0, 0, 8, 0, 1, 0, 0], [0, 2, 2, 2, 2, 2, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 1, 0, 8, 0, 0, 0, 9, 0, 0, 8, 0, 1, 9, 0], [0, 2, 0, 0, 0, 2, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 9, 0, 8, 0, 0, 0, 0, 0, 0, 8, 0, 1, 0, 0], [0, 2, 0, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 0, 1, 0, 8, 0, 0, 0, 0, 0, 0, 8, 0, 1, 0, 0], [0, 2, 2, 3, 2, 2, 3, 0, 0, 3, 0, 8, 0, 0, 3, 0, 1, 0, 8, 0, 0, 9, 0, 0, 0, 8, 0, 1, 0, 0], [0, 0, 0, 3, 0, 0, 3, 0, 0, 3, 0, 8, 0, 0, 3, 0, 1, 0, 8, 0, 0, 0, 0, 0, 0, 8, 0, 1, 9, 0], [0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 0, 1, 0, 8, 0, 0, 0, 0, 0, 0, 8, 0, 1, 0, 0], [0, 9, 0, 3, 0, 0, 3, 0, 0, 3, 0, 8, 0, 0, 3, 0, 1, 0, 8, 0, 0, 0, 0, 0, 0, 8, 0, 1, 0, 0], [0, 0, 0, 3, 0, 0, 3, 0, 0, 3, 0, 8, 8, 8, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 1, 0, 0], [0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 9, 3, 3, 3, 3, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0], [0, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 9, 4, 4, 4, 4, 4, 0, 1, 0, 0], [0, 4, 0, 0, 0, 4, 0, 0, 0, 0, 0, 0, 4, 0, 0, 9, 1, 0, 0, 0, 0, 0, 0, 0, 0, 4, 0, 1, 0, 0], [0, 9, 0, 0, 0, 4, 0, 9, 0, 0, 0, 0, 4, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 4, 0, 1, 0, 9], [0, 4, 0, 0, 0, 4, 0, 0, 0, 0, 0, 0, 4, 0, 0, 0, 1, 0, 0, 0, 0, 9, 0, 0, 0, 4, 0, 1, 0, 0], [0, 4, 4, 4, 4, 4, 4, 4, 4, 9, 4, 4, 4, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 4, 0, 1, 0, 0], [0, 4, 0, 0, 0, 4, 0, 0, 0, 0, 0, 9, 4, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 4, 0, 1, 0, 0], [0, 4, 0, 0, 0, 4, 0, 0, 0, 0, 0, 0, 4, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 1, 9, 0, 0], [0, 4, 9, 0, 0, 4, 0, 0, 0, 0, 0, 0, 4, 0, 9, 0, 0, 0, 9, 0, 0, 0, 0, 0, 0, 4, 0, 0, 0, 0], [0, 4, 0, 0, 0, 4, 0, 0, 0, 0, 0, 0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 0, 0, 0, 0], [0, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 0, 0, 9, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 9, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 9, 9, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [9, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 9, 0, 0, 0, 0, 0, 0, 0, 0]]}]}, "8b7bacbf": {"train": [{"input": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 0, 0], [0, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 5, 0], [2, 0, 0, 2, 0, 0, 0, 0, 0, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 5, 0], [2, 0, 0, 2, 0, 0, 0, 0, 2, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 1, 5, 5, 1, 0], [0, 2, 2, 1, 0, 0, 0, 0, 2, 0, 0, 2, 0, 0, 0, 0, 0, 2, 1, 0, 0, 0, 1, 0], [0, 0, 0, 0, 1, 1, 1, 1, 1, 2, 2, 1, 0, 0, 0, 0, 2, 0, 2, 2, 0, 0, 1, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 2, 0, 0, 0, 0, 2, 0, 1, 0], [0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 2, 0, 0, 2, 2, 0, 1, 0], [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 2, 0, 0, 0, 1, 0], [0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0], [0, 0, 0, 0, 0, 0, 0, 3, 3, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0], [0, 0, 0, 0, 0, 0, 3, 0, 0, 3, 0, 0, 0, 2, 2, 1, 0, 0, 0, 0, 0, 0, 1, 0], [0, 0, 0, 0, 0, 0, 3, 0, 0, 3, 0, 0, 2, 0, 0, 2, 0, 0, 0, 0, 0, 0, 1, 0], [0, 0, 0, 1, 0, 0, 1, 3, 3, 1, 1, 1, 2, 0, 0, 2, 0, 0, 0, 0, 0, 0, 1, 0], [0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 2, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 1, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 1, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 7]], "output": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 0, 0], [0, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 7, 7, 5, 0], [2, 0, 0, 2, 0, 0, 0, 0, 0, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 5, 7, 7, 5, 0], [2, 0, 0, 2, 0, 0, 0, 0, 2, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 1, 5, 5, 1, 0], [0, 2, 2, 1, 0, 0, 0, 0, 2, 0, 0, 2, 0, 0, 0, 0, 0, 2, 1, 0, 0, 0, 1, 0], [0, 0, 0, 0, 1, 1, 1, 1, 1, 2, 2, 1, 0, 0, 0, 0, 2, 7, 2, 2, 0, 0, 1, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 2, 7, 7, 7, 7, 2, 0, 1, 0], [0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 2, 7, 7, 2, 2, 0, 1, 0], [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 2, 2, 0, 0, 0, 1, 0], [0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0], [0, 0, 0, 0, 0, 0, 0, 3, 3, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0], [0, 0, 0, 0, 0, 0, 3, 7, 7, 3, 0, 0, 0, 2, 2, 1, 0, 0, 0, 0, 0, 0, 1, 0], [0, 0, 0, 0, 0, 0, 3, 7, 7, 3, 0, 0, 2, 7, 7, 2, 0, 0, 0, 0, 0, 0, 1, 0], [0, 0, 0, 1, 0, 0, 1, 3, 3, 1, 1, 1, 2, 7, 7, 2, 0, 0, 0, 0, 0, 0, 1, 0], [0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 2, 7, 7, 7, 2, 0, 0, 0, 0, 0, 0, 1, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 7, 7, 2, 0, 0, 0, 0, 0, 0, 0, 1, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 7]]}, {"input": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 2, 2, 0, 0, 0, 0, 0, 0, 0, 2, 2, 0, 0], [0, 0, 0, 2, 2, 0, 2, 0, 0, 2, 0, 0, 0, 0, 0, 2, 0, 0, 2, 0], [0, 0, 2, 0, 0, 2, 2, 0, 0, 2, 0, 0, 2, 2, 0, 2, 0, 0, 2, 0], [0, 0, 2, 0, 0, 2, 1, 2, 2, 0, 0, 2, 0, 0, 2, 0, 2, 2, 1, 0], [0, 0, 0, 2, 2, 1, 1, 1, 0, 0, 0, 2, 0, 0, 2, 0, 0, 1, 0, 0], [0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 2, 2, 0, 0, 1, 0, 0, 0], [0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0], [0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0], [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 2, 2, 0], [1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 2], [4, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 2]], "output": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 2, 2, 0, 0, 0, 0, 0, 0, 0, 2, 2, 0, 0], [0, 0, 0, 2, 2, 0, 2, 4, 4, 2, 0, 0, 0, 0, 0, 2, 4, 4, 2, 0], [0, 0, 2, 4, 4, 2, 2, 4, 4, 2, 0, 0, 2, 2, 0, 2, 4, 4, 2, 0], [0, 0, 2, 4, 4, 2, 1, 2, 2, 0, 0, 2, 0, 0, 2, 0, 2, 2, 1, 0], [0, 0, 0, 2, 2, 1, 1, 1, 0, 0, 0, 2, 0, 0, 2, 0, 0, 1, 0, 0], [0, 0, 0, 1, 0, 0, 0, 0, 1, 1, 0, 0, 2, 2, 0, 0, 1, 0, 0, 0], [0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0], [0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0], [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 2, 2, 0], [1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 2], [4, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 2]]}, {"input": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2], [0, 0, 2, 2, 0, 0, 0, 0, 0, 0, 0, 2, 2, 0, 0, 0, 0, 2, 0, 0], [0, 2, 0, 0, 2, 0, 0, 0, 0, 0, 2, 0, 0, 2, 0, 0, 0, 2, 0, 0], [0, 2, 0, 0, 2, 0, 0, 0, 0, 0, 2, 0, 0, 2, 0, 0, 0, 3, 2, 2], [0, 1, 2, 2, 1, 1, 1, 1, 1, 1, 1, 2, 2, 1, 0, 0, 0, 3, 0, 0], [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 3, 0, 0], [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 3, 0], [1, 1, 0, 0, 0, 0, 0, 2, 2, 1, 1, 1, 0, 0, 0, 0, 2, 2, 3, 0], [0, 1, 0, 0, 0, 0, 2, 0, 0, 2, 0, 0, 0, 0, 0, 2, 0, 0, 2, 0], [1, 1, 0, 0, 0, 0, 2, 0, 0, 2, 0, 0, 0, 0, 0, 2, 0, 0, 2, 0], [0, 0, 0, 0, 0, 0, 1, 2, 2, 0, 0, 0, 0, 0, 0, 3, 2, 2, 0, 0], [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0], [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 3, 3, 2, 2, 3, 0, 0, 0, 0], [0, 0, 2, 2, 1, 0, 0, 0, 0, 0, 3, 0, 2, 0, 0, 2, 0, 0, 0, 0], [0, 2, 0, 0, 2, 0, 0, 0, 3, 3, 3, 0, 2, 0, 0, 2, 0, 0, 0, 0], [0, 2, 0, 0, 2, 0, 0, 0, 3, 4, 3, 0, 0, 2, 2, 0, 0, 0, 0, 0]], "output": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2], [0, 0, 2, 2, 0, 0, 0, 0, 0, 0, 0, 2, 2, 0, 0, 0, 0, 2, 4, 4], [0, 2, 0, 0, 2, 0, 0, 0, 0, 0, 2, 0, 0, 2, 0, 0, 0, 2, 4, 4], [0, 2, 0, 0, 2, 0, 0, 0, 0, 0, 2, 0, 0, 2, 0, 0, 0, 3, 2, 2], [0, 1, 2, 2, 1, 1, 1, 1, 1, 1, 1, 2, 2, 1, 0, 0, 0, 3, 0, 0], [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 3, 0, 0], [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 3, 0], [1, 1, 0, 0, 0, 0, 0, 2, 2, 1, 1, 1, 0, 0, 0, 0, 2, 2, 3, 0], [0, 1, 0, 0, 0, 0, 2, 0, 0, 2, 0, 0, 0, 0, 0, 2, 4, 4, 2, 0], [1, 1, 0, 0, 0, 0, 2, 0, 0, 2, 0, 0, 0, 0, 0, 2, 4, 4, 2, 0], [0, 0, 0, 0, 0, 0, 1, 2, 2, 0, 0, 0, 0, 0, 0, 3, 2, 2, 0, 0], [0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0], [0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 3, 3, 2, 2, 3, 0, 0, 0, 0], [0, 0, 2, 2, 1, 0, 0, 0, 0, 0, 3, 0, 2, 4, 4, 2, 0, 0, 0, 0], [0, 2, 0, 0, 2, 0, 0, 0, 3, 3, 3, 0, 2, 4, 4, 2, 0, 0, 0, 0], [0, 2, 0, 0, 2, 0, 0, 0, 3, 4, 3, 0, 0, 2, 2, 0, 0, 0, 0, 0]]}, {"input": [[0, 0, 2, 2, 0, 0, 0, 0], [0, 2, 0, 0, 2, 0, 0, 0], [0, 2, 0, 0, 2, 0, 0, 0], [0, 0, 2, 2, 0, 1, 0, 0], [0, 0, 0, 0, 0, 0, 1, 0], [0, 0, 0, 0, 0, 1, 0, 0], [0, 0, 0, 0, 0, 0, 1, 1], [0, 0, 0, 0, 0, 0, 1, 4]], "output": [[0, 0, 2, 2, 0, 0, 0, 0], [0, 2, 4, 4, 2, 0, 0, 0], [0, 2, 4, 4, 2, 0, 0, 0], [0, 0, 2, 2, 0, 1, 0, 0], [0, 0, 0, 0, 0, 0, 1, 0], [0, 0, 0, 0, 0, 1, 0, 0], [0, 0, 0, 0, 0, 0, 1, 1], [0, 0, 0, 0, 0, 0, 1, 4]]}], "test": [{"input": [[6, 1, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 3, 8], [1, 1, 1, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 3, 3], [5, 5, 5, 1, 9, 9, 1, 1, 1, 1, 1, 1, 1, 9, 9, 9, 9, 9, 9, 9, 5, 5, 5, 5, 5, 5, 3, 3, 5, 5], [5, 5, 5, 9, 5, 5, 9, 5, 5, 5, 5, 5, 9, 5, 5, 5, 5, 5, 5, 5, 9, 5, 5, 5, 5, 5, 3, 5, 5, 5], [5, 5, 5, 9, 5, 5, 9, 5, 5, 5, 5, 5, 9, 5, 5, 5, 5, 5, 5, 5, 9, 5, 5, 5, 5, 3, 3, 5, 5, 5], [5, 5, 5, 5, 9, 9, 5, 5, 5, 5, 5, 5, 5, 9, 9, 9, 9, 9, 9, 9, 1, 1, 1, 1, 5, 3, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 1, 3, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 3, 1, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 3, 5, 5, 1, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 9, 9, 3, 5, 5, 1, 5, 5], [5, 5, 5, 5, 9, 9, 9, 9, 9, 9, 9, 5, 5, 5, 5, 5, 9, 3, 3, 5, 5, 9, 5, 5, 9, 5, 5, 1, 5, 5], [5, 5, 5, 9, 5, 5, 5, 5, 5, 5, 5, 9, 5, 5, 5, 9, 9, 9, 3, 3, 3, 9, 5, 5, 9, 5, 5, 1, 5, 5], [5, 5, 5, 9, 5, 5, 5, 5, 5, 5, 5, 9, 5, 5, 5, 9, 5, 9, 5, 5, 3, 3, 9, 9, 5, 5, 5, 1, 1, 5], [5, 5, 3, 3, 9, 9, 9, 9, 9, 9, 9, 5, 5, 5, 5, 9, 5, 9, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 1, 5], [5, 0, 3, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 9, 5, 9, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 1, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 9, 5, 9, 5, 5, 5, 5, 5, 5, 5, 5, 5, 1, 1, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 9, 5, 9, 5, 5, 5, 5, 5, 5, 5, 5, 5, 1, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 9, 5, 9, 5, 5, 5, 5, 5, 5, 5, 5, 5, 1, 5, 5], [5, 5, 0, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 9, 5, 9, 5, 5, 5, 5, 5, 5, 5, 5, 1, 1, 5, 5], [5, 5, 3, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 9, 5, 9, 5, 5, 5, 5, 5, 5, 5, 5, 1, 5, 5, 5], [5, 5, 3, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 3, 3, 3, 9, 5, 5, 5, 5, 5, 5, 5, 5, 1, 1, 5, 5, 5], [5, 5, 3, 3, 5, 5, 5, 5, 5, 5, 3, 3, 3, 3, 5, 5, 5, 5, 5, 5, 9, 9, 9, 1, 1, 1, 5, 5, 5, 5], [5, 5, 5, 3, 9, 9, 3, 3, 3, 3, 3, 5, 5, 5, 5, 5, 5, 5, 5, 9, 5, 5, 9, 1, 1, 5, 5, 5, 5, 5], [5, 5, 5, 9, 5, 5, 9, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 9, 5, 5, 5, 9, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 9, 5, 5, 9, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 9, 5, 5, 5, 9, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 9, 9, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 9, 5, 5, 5, 9, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 9, 5, 5, 5, 9, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 9, 9, 9, 9, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5]]}, {"input": [[0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 8, 8, 0], [0, 0, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 8, 0, 0, 8], [0, 0, 0, 0, 2, 2, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 8, 8, 0, 0, 8, 0, 0, 8], [0, 0, 0, 2, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 8, 0, 0, 8, 8, 0], [0, 0, 0, 2, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 8, 0, 0, 0, 0, 0], [0, 0, 0, 1, 2, 2, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 8, 1, 0, 0, 0, 0, 0], [0, 0, 1, 1, 0, 0, 1, 1, 6, 6, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 0], [0, 0, 1, 0, 0, 0, 0, 6, 0, 0, 0, 6, 0, 0, 0, 8, 8, 0, 0, 0, 0, 0, 0, 0, 1, 0], [0, 0, 1, 0, 0, 0, 0, 6, 0, 0, 0, 6, 0, 0, 8, 0, 0, 8, 0, 0, 0, 5, 5, 5, 1, 0], [0, 0, 1, 0, 0, 0, 0, 6, 0, 0, 0, 6, 0, 0, 8, 0, 0, 8, 0, 0, 5, 0, 0, 0, 5, 0], [0, 0, 1, 0, 0, 0, 0, 0, 6, 6, 6, 0, 0, 9, 9, 8, 8, 9, 0, 0, 5, 0, 0, 0, 5, 0], [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 9, 0, 0, 0, 0, 9, 9, 0, 0, 5, 5, 5, 0, 0], [0, 0, 1, 0, 0, 0, 0, 0, 8, 8, 9, 9, 0, 0, 0, 0, 0, 0, 9, 0, 0, 0, 0, 0, 0, 0], [0, 0, 1, 0, 0, 0, 0, 8, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 9, 0, 0, 0, 0, 0, 0, 0], [0, 0, 1, 0, 0, 0, 0, 8, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 9, 0, 0, 0, 0, 0, 0, 0], [0, 0, 1, 0, 0, 0, 0, 0, 8, 8, 0, 0, 0, 0, 0, 0, 0, 0, 9, 0, 0, 0, 8, 8, 0, 0], [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 8, 9, 0, 0, 8, 0, 0, 8, 0], [0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 8, 0, 0, 8, 0, 0, 8, 0], [0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 8, 0, 0, 9, 8, 8, 0, 0], [0, 1, 0, 0, 0, 0, 0, 9, 9, 9, 9, 0, 0, 0, 0, 9, 8, 8, 9, 9, 9, 0, 0, 0, 0, 0], [1, 1, 0, 0, 9, 9, 9, 0, 0, 0, 9, 9, 9, 9, 9, 9, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [4, 1, 0, 0, 9, 3, 9, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}]}, "8b9c3697": {"train": [{"input": [[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 1, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 1, 1, 1, 1, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 1, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 1, 1, 4, 4, 4, 4, 4, 4, 4, 4, 2, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 1, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 1, 1, 1, 1, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 1, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 1, 1, 1, 1, 1, 1, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 4, 1, 1, 4, 1, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 4, 4, 4, 4, 1, 4, 4, 4, 4], [4, 4, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 4, 4, 4, 4, 1, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4]], "output": [[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 1, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 1, 1, 1, 1, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 1, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 1, 1, 2, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 1, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 1, 1, 1, 1, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 1, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 1, 1, 1, 1, 1, 1, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 4, 1, 1, 4, 1, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 4, 4, 4, 4, 1, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 4, 4, 4, 4, 1, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]]}, {"input": [[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 2, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 2, 3, 3, 3, 3, 3], [3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 4, 3, 4, 3, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 4, 3, 3, 3, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 4, 3, 3, 3, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 2, 3, 3, 3, 3, 3, 3, 4, 3, 3, 3, 3, 4, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 3, 3, 3, 4, 3, 3, 3], [2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 4, 4, 3, 4, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 2, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]], "output": [[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 0, 0, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 0, 0, 3, 3, 3, 3, 3], [3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 3, 3, 3, 3, 3, 0, 0, 3, 3, 3, 3, 3], [3, 3, 3, 3, 4, 3, 4, 3, 4, 3, 3, 3, 3, 3, 3, 0, 0, 3, 3, 3, 3, 3], [3, 3, 3, 3, 4, 3, 2, 3, 4, 3, 3, 3, 3, 3, 3, 0, 0, 3, 3, 3, 3, 3], [3, 3, 3, 3, 4, 3, 0, 3, 4, 3, 3, 3, 3, 3, 3, 0, 0, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 0, 3, 3, 3, 3, 3, 3, 3, 3, 0, 0, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 0, 3, 3, 3, 3, 3, 3, 3, 3, 0, 0, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 0, 3, 3, 3, 3, 3, 3, 4, 3, 2, 2, 3, 4, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 2, 2, 3, 4, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 4, 4, 3, 4, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]]}, {"input": [[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 4, 4], [4, 2, 4, 4, 4, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 8, 4, 4, 8, 4, 4, 8, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 8, 4, 4, 4, 4, 4, 8, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 8, 4, 4, 4, 4, 4, 8, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 2, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 2, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 8, 4, 8, 8, 8, 4, 8, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 8, 4, 4, 4, 4, 4, 8, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 8, 4, 4, 4, 4, 4, 8, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 2, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 2, 4, 4, 4, 4], [4, 4, 4, 4, 2, 2, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 2, 2, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]], "output": [[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 8, 4, 4, 8, 4, 4, 8, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 8, 4, 4, 2, 4, 4, 8, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 8, 4, 4, 0, 4, 4, 8, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 0, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 0, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 0, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 8, 4, 8, 8, 8, 4, 8, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 8, 4, 2, 2, 2, 4, 8, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 8, 4, 2, 2, 2, 4, 8, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 0, 0, 0, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 0, 0, 0, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 0, 0, 0, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 0, 0, 0, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]]}], "test": [{"input": [[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 3, 3, 3, 3, 3, 3, 3, 3, 2, 2, 3, 3], [3, 3, 3, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 2, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 2, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 4, 3, 4, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 3, 3, 4, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 3, 3, 3, 4, 3, 3, 3, 4, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 2, 2, 3, 3, 3, 3, 3, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 2, 3, 3, 3], [3, 3, 2, 2, 3, 3, 3, 3, 3, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 2, 3, 3, 3], [3, 3, 2, 2, 3, 3, 3, 3, 3, 4, 4, 3, 3, 3, 2, 3, 3, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 3, 3, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 3, 3, 3, 4, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 3, 3, 3, 4, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 4, 4, 3, 4, 3], [3, 3, 3, 3, 3, 3, 2, 2, 3, 3, 3, 3, 3, 3, 3, 2, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4], [3, 3, 3, 3, 3, 3, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]]}]}, "8e5c0c38": {"train": [{"input": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 1, 1, 1, 8, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 1, 1, 8, 8, 8, 8, 8, 1, 1, 8, 8, 6, 6, 6, 6, 8, 8, 8], [8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 6, 8, 6, 8, 8, 8], [8, 8, 8, 8, 1, 1, 8, 1, 1, 1, 8, 1, 8, 8, 8, 8, 6, 6, 6, 8, 8, 8], [8, 8, 8, 8, 1, 1, 1, 1, 8, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 2, 2, 2, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 8, 2, 2, 2, 8, 2, 2, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]], "output": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 1, 1, 1, 8, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 1, 1, 8, 8, 8, 8, 8, 1, 1, 8, 8, 8, 6, 6, 6, 8, 8, 8], [8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 6, 8, 6, 8, 8, 8], [8, 8, 8, 8, 8, 1, 8, 1, 1, 1, 8, 1, 8, 8, 8, 8, 6, 6, 6, 8, 8, 8], [8, 8, 8, 8, 8, 1, 1, 1, 8, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 8, 2, 2, 2, 8, 2, 2, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 8, 8, 2, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]]}, {"input": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 2, 2, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 2, 0, 0, 2, 2, 0, 0], [0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 2, 2, 0, 0, 2, 2, 0, 0], [0, 0, 0, 0, 0, 8, 8, 0, 8, 0, 0, 0, 0, 0, 0, 2, 2, 2, 2, 0, 0, 0], [0, 0, 0, 0, 0, 8, 8, 0, 8, 0, 0, 0, 0, 0, 0, 0, 2, 2, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 4, 0, 0, 0, 4, 4, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], "output": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 2, 2, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 0, 0, 2, 2, 0, 0], [0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 2, 2, 0, 0, 2, 2, 0, 0], [0, 0, 0, 0, 0, 0, 8, 0, 8, 0, 0, 0, 0, 0, 0, 2, 2, 2, 2, 0, 0, 0], [0, 0, 0, 0, 0, 0, 8, 0, 8, 0, 0, 0, 0, 0, 0, 0, 2, 2, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 0, 0, 0, 4, 4, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}], "test": [{"input": [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 6, 6, 6, 6, 1, 6, 6, 6, 6, 6, 6, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 6, 6, 1, 1, 1, 1, 1, 1, 6, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 6, 1, 1, 1, 1, 1, 1, 6, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 6, 6, 6, 6, 1, 1, 1, 6, 1, 1, 6, 6, 6, 1, 1, 1, 1, 1, 1], [1, 1, 1, 6, 6, 6, 6, 1, 6, 6, 6, 6, 1, 6, 6, 6, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 6, 6, 6, 1, 1, 6, 6, 6, 6, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 6, 1, 1, 1, 1, 6, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 6, 6, 6, 6, 6, 6, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 6, 6, 1, 1, 1, 1, 6, 6, 6, 6, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 6, 6, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 6, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 3, 3, 1], [1, 1, 1, 1, 1, 1, 4, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 3, 3, 1], [1, 1, 1, 1, 1, 4, 4, 4, 4, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1], [1, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 4, 4, 4, 4, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 4, 4, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]}, {"input": [[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 2, 4, 4, 4, 4, 4, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 2, 2, 2, 4, 4, 4, 2, 2, 2, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 2, 4, 4, 2, 2, 2, 4, 4, 2, 2, 4, 4, 4, 3, 3, 3, 3, 3, 3, 4], [4, 4, 2, 2, 4, 4, 4, 4, 4, 2, 2, 4, 4, 4, 4, 4, 3, 4, 3, 4, 4, 4], [4, 4, 4, 2, 4, 2, 2, 2, 4, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 2, 2, 2, 4, 2, 2, 2, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 1, 4, 4, 1, 1, 1, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 4, 1, 1, 1, 1, 4, 1, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 1, 4, 4, 4, 4, 4, 4, 1, 1, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 4, 4, 4, 4, 4, 4, 1, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 1, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]]}]}, "8f3a5a89": {"train": [{"input": [[8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 1, 1, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 1, 1, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 1, 1, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 1, 8, 8, 1, 1, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 1, 8, 8, 1, 1, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 1, 8, 8, 1, 1, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 1, 1, 1, 1], [8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 1, 1, 1, 1], [8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 1, 1, 1, 1], [6, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 1, 1, 1, 1]], "output": [[7, 7, 7, 7, 7, 7, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8], [7, 8, 8, 8, 8, 7, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8], [7, 8, 8, 8, 8, 7, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8], [7, 8, 8, 8, 8, 7, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8], [7, 8, 8, 8, 8, 7, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8], [7, 8, 8, 8, 8, 7, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8], [7, 8, 8, 8, 8, 7, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8], [7, 8, 8, 8, 8, 7, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8], [7, 8, 8, 8, 8, 7, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8], [7, 8, 8, 8, 8, 7, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8], [7, 8, 8, 8, 8, 7, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8], [7, 8, 8, 8, 8, 7, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8], [7, 8, 8, 8, 8, 7, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8], [7, 8, 8, 8, 8, 7, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8], [7, 8, 8, 8, 8, 7, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8], [6, 7, 7, 7, 7, 7, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8]]}, {"input": [[8, 8, 8, 8, 1, 1, 8, 8, 8, 8], [8, 8, 8, 8, 1, 1, 8, 8, 8, 8], [8, 8, 8, 8, 1, 1, 8, 8, 8, 1], [8, 8, 8, 8, 8, 8, 8, 8, 8, 1], [1, 8, 8, 8, 8, 8, 8, 8, 8, 1], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 1, 8, 8, 8], [6, 8, 8, 8, 8, 1, 1, 8, 8, 8]], "output": [[7, 7, 7, 7, 1, 1, 7, 7, 7, 7], [7, 8, 8, 7, 1, 1, 7, 8, 7, 7], [7, 8, 8, 7, 1, 1, 7, 8, 7, 1], [7, 7, 8, 7, 7, 7, 7, 8, 7, 1], [1, 7, 8, 8, 8, 8, 8, 8, 7, 1], [7, 7, 8, 8, 8, 8, 8, 8, 7, 7], [7, 8, 8, 8, 8, 8, 8, 8, 8, 7], [7, 8, 8, 8, 8, 7, 7, 7, 8, 7], [7, 8, 8, 8, 7, 7, 1, 7, 8, 7], [6, 7, 7, 7, 7, 1, 1, 7, 7, 7]]}, {"input": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8], [1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8], [1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8], [1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [1, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 1, 1], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 1, 1], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 1, 1], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 1, 1], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 1, 1], [6, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 1, 1]], "output": [[7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 1, 7, 7, 7, 7, 7, 7, 7], [7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 7, 1, 7, 8, 8, 8, 8, 8, 7], [7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 7, 1, 7, 8, 8, 8, 8, 8, 7], [1, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 7, 1, 7, 8, 8, 8, 8, 8, 7], [1, 1, 7, 7, 8, 8, 8, 8, 8, 8, 8, 7, 1, 7, 8, 8, 8, 8, 8, 7], [1, 1, 1, 7, 7, 8, 8, 8, 8, 8, 8, 7, 7, 7, 8, 8, 8, 8, 8, 7], [1, 1, 1, 1, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 7], [1, 1, 1, 1, 1, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 7], [1, 1, 1, 7, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 7], [7, 7, 7, 7, 8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 7], [7, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 7], [7, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 7], [7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 7], [7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 7, 7, 7, 7, 7, 7, 7], [7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 7, 1, 1, 1, 1, 1, 1], [7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 7, 1, 1, 1, 1, 1, 1], [7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 7, 1, 1, 1, 1, 1, 1], [7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 7, 1, 1, 1, 1, 1, 1], [7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 7, 1, 1, 1, 1, 1, 1], [6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 1, 1, 1, 1, 1, 1]]}], "test": [{"input": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 8], [8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 8], [8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 8], [8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1], [8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1], [6, 8, 1, 1, 1, 1, 8, 8, 8, 1, 1, 1]]}]}, "9385bd28": {"train": [{"input": [[0, 0, 0, 2, 2, 0, 0, 0, 0, 0], [0, 0, 0, 2, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 1, 1, 0, 0, 0, 0], [0, 0, 0, 0, 1, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 1, 0, 0, 0], [0, 0, 0, 0, 0, 1, 1, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 2, 0, 0], [0, 0, 0, 0, 0, 0, 2, 2, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0, 0, 0], [2, 3, 0, 0, 0, 0, 0, 0, 0, 0]], "output": [[0, 0, 0, 2, 2, 3, 3, 3, 0, 0], [0, 0, 0, 2, 3, 3, 3, 3, 0, 0], [0, 0, 0, 3, 1, 1, 0, 3, 0, 0], [0, 0, 0, 3, 1, 0, 0, 3, 0, 0], [0, 0, 0, 3, 0, 0, 1, 3, 0, 0], [0, 0, 0, 3, 0, 1, 1, 3, 0, 0], [0, 0, 0, 3, 3, 3, 3, 2, 0, 0], [0, 0, 0, 3, 3, 3, 2, 2, 0, 0], [1, 0, 0, 0, 0, 0, 0, 0, 0, 0], [2, 3, 0, 0, 0, 0, 0, 0, 0, 0]]}, {"input": [[0, 0, 0, 0, 0, 0, 0, 0, 4, 4], [0, 0, 0, 0, 0, 0, 0, 0, 0, 4], [0, 0, 0, 0, 0, 1, 1, 0, 0, 0], [0, 0, 0, 0, 0, 1, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 1, 0, 0], [0, 1, 3, 0, 4, 0, 1, 1, 0, 0], [0, 4, 5, 0, 4, 4, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], "output": [[0, 0, 0, 0, 5, 5, 5, 5, 4, 4], [0, 0, 0, 0, 5, 5, 5, 5, 5, 4], [0, 0, 0, 0, 5, 1, 1, 3, 5, 5], [0, 0, 0, 0, 5, 1, 3, 3, 5, 5], [0, 0, 0, 0, 5, 3, 3, 3, 5, 5], [0, 0, 0, 0, 5, 3, 3, 3, 5, 5], [0, 0, 0, 0, 5, 3, 3, 1, 5, 5], [0, 1, 3, 0, 4, 3, 1, 1, 5, 5], [0, 4, 5, 0, 4, 4, 5, 5, 5, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}, {"input": [[3, 3, 5, 5, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 5, 3, 3, 3, 3, 5, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 5, 5, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 2, 3, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 4, 3, 3, 1, 1, 3, 3, 2, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 3, 2, 2, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 3, 3, 3, 3, 3, 3, 3], [3, 3, 1, 6, 3, 3, 3, 3, 4, 4, 3, 3, 3, 3, 3, 3, 3], [3, 3, 2, 9, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 4, 0, 3, 3, 3, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 8, 9, 3, 3, 3, 1, 1, 3, 3, 3, 3, 3, 3, 3, 3]], "output": [[3, 3, 5, 5, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 5, 3, 3, 3, 3, 5, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 5, 5, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 2, 2, 9, 9, 9, 9, 9, 9, 9, 9, 3, 3, 3], [3, 3, 3, 3, 2, 9, 9, 9, 9, 9, 9, 9, 9, 9, 3, 3, 3], [3, 3, 3, 3, 9, 9, 9, 6, 6, 1, 1, 9, 9, 2, 3, 3, 3], [3, 3, 3, 3, 9, 9, 9, 6, 6, 6, 1, 9, 2, 2, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 6, 6, 6, 6, 3, 3, 3, 3, 3, 3], [3, 3, 1, 6, 3, 3, 3, 6, 6, 6, 6, 3, 3, 3, 3, 3, 3], [3, 3, 2, 9, 3, 3, 3, 6, 6, 6, 6, 3, 3, 3, 3, 3, 3], [3, 3, 4, 0, 3, 3, 3, 1, 6, 6, 6, 3, 3, 3, 3, 3, 3], [3, 3, 8, 9, 3, 3, 3, 1, 1, 6, 6, 3, 3, 3, 3, 3, 3]]}, {"input": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 0, 0, 0, 0, 4], [0, 0, 0, 0, 0, 0, 6, 6, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4], [0, 0, 0, 0, 0, 0, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 6, 0, 0, 0, 7, 7, 0, 0], [6, 7, 0, 0, 0, 0, 0, 0, 0, 6, 6, 0, 0, 0, 7, 7, 0, 0], [4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [7, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], "output": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 4, 4, 4, 4], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 4, 4, 4, 4], [0, 0, 0, 0, 0, 0, 6, 6, 7, 7, 7, 0, 4, 4, 4, 4, 4, 4], [0, 0, 0, 0, 0, 0, 6, 7, 7, 7, 7, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 7, 7, 7, 7, 7, 0, 0, 0, 0, 0, 0, 0], [3, 3, 0, 0, 0, 0, 7, 7, 7, 7, 6, 0, 0, 0, 6, 6, 0, 0], [6, 7, 0, 0, 0, 0, 7, 7, 7, 6, 6, 0, 0, 0, 6, 6, 0, 0], [4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [7, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}], "test": [{"input": [[9, 9, 9, 9, 4, 4, 9, 2, 2, 9, 9, 9, 9], [9, 9, 9, 9, 4, 9, 9, 2, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 1, 1, 9, 9, 9, 9, 9, 9, 9, 9, 9, 1], [9, 1, 9, 9, 9, 9, 9, 9, 9, 9, 9, 1, 1], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 5, 5, 9, 9, 9, 9, 9, 9, 5], [9, 9, 9, 9, 5, 9, 9, 9, 9, 9, 9, 5, 5], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 3, 3, 9, 9, 9, 9, 9, 9, 3], [9, 9, 9, 9, 3, 9, 9, 9, 9, 9, 9, 3, 3], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 2, 2, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 5, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 3, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 1, 4, 9, 9, 4, 9, 9, 2, 9, 9, 9, 9], [9, 4, 6, 9, 4, 4, 9, 2, 2, 9, 9, 9, 9]]}, {"input": [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 8, 8, 1, 1, 1, 7, 7, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 8, 8, 1, 1, 1, 7, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 6, 6, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 6, 1, 3, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 3, 1, 1, 1, 1, 6, 1, 2, 2, 1, 1], [1, 1, 3, 4, 1, 1, 1, 1, 1, 1, 3, 1, 6, 6, 1, 1, 2, 1, 1], [1, 1, 6, 8, 1, 1, 1, 1, 1, 3, 3, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 7, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 1, 1, 1, 1, 1], [1, 1, 8, 2, 1, 1, 1, 1, 7, 1, 1, 1, 1, 2, 2, 1, 1, 1, 1], [1, 1, 2, 4, 1, 1, 1, 7, 7, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]}]}, "9bbf930d": {"train": [{"input": [[6, 7, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 7, 7, 7], [6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [6, 7, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 7, 4], [6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 7, 4], [6, 7, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 7, 4], [6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4], [6, 7, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [6, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 7], [6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [6, 7, 7, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 9], [6, 7, 7, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 7, 9], [6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 9, 7, 9], [6, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 7, 7], [6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [6, 7, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 7, 7]], "output": [[6, 7, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 7, 6, 7], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 6], [6, 7, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 4, 7, 4], [6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 7, 4], [6, 7, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 7, 4], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4], [6, 7, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [6, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 7], [6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [6, 7, 7, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 9], [6, 7, 7, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 7, 9], [6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 9, 7, 9], [6, 7, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 7, 7], [6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [6, 7, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 6, 7]]}, {"input": [[6, 7, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 7, 7], [6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [6, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 7, 7], [6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [6, 7, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 7, 7], [6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [6, 7, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 7, 7], [6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [6, 7, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 7, 7], [6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [6, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 7], [6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [6, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 7], [6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [6, 7, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 7, 7], [6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [6, 7, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 7, 7], [6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [6, 7, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 7, 7]], "output": [[6, 7, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 7, 7], [6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [6, 7, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 7, 7], [6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [6, 7, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 7, 7], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 6], [6, 7, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 7, 7], [6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [6, 7, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 7, 7], [6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [6, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 7], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 6], [6, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 7], [6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [6, 7, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 7, 7], [6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [6, 7, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 7, 7], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 6], [6, 7, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 7, 7]]}, {"input": [[6, 7, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [6, 7, 7, 7, 7, 7, 7, 7, 7, 4, 7, 7, 7, 7, 7, 4], [6, 7, 4, 4, 4, 4, 4, 4, 7, 4, 7, 4, 4, 4, 7, 4], [6, 7, 7, 7, 7, 7, 7, 4, 7, 4, 7, 4, 7, 4, 7, 4], [6, 7, 2, 2, 2, 2, 2, 2, 7, 7, 7, 7, 7, 4, 7, 4], [6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [6, 7, 2, 2, 2, 2, 2, 2, 7, 5, 7, 5, 7, 9, 7, 0], [6, 7, 7, 7, 7, 7, 7, 7, 7, 5, 7, 5, 7, 9, 7, 0], [6, 7, 5, 5, 5, 5, 5, 5, 5, 5, 7, 5, 7, 9, 7, 0], [6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 5, 7, 9, 7, 0], [6, 7, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 7, 9, 7, 0], [6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [6, 7, 3, 3, 3, 3, 3, 7, 7, 7, 3, 3, 3, 3, 3, 3], [6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 3], [6, 7, 3, 3, 3, 3, 3, 7, 7, 7, 3, 3, 3, 3, 7, 3], [6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 3, 7, 3], [6, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 7]], "output": [[6, 7, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 7, 7, 7, 7, 7, 4], [6, 7, 4, 4, 4, 4, 4, 4, 7, 4, 7, 4, 4, 4, 7, 4], [6, 7, 7, 7, 7, 7, 7, 4, 7, 4, 7, 4, 7, 4, 7, 4], [6, 7, 2, 2, 2, 2, 2, 2, 7, 7, 7, 7, 7, 4, 7, 4], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 6], [6, 7, 2, 2, 2, 2, 2, 2, 7, 5, 7, 5, 7, 9, 7, 0], [6, 7, 7, 7, 7, 7, 7, 7, 6, 5, 7, 5, 7, 9, 7, 0], [6, 7, 5, 5, 5, 5, 5, 5, 5, 5, 7, 5, 7, 9, 7, 0], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 5, 7, 9, 7, 0], [6, 7, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 7, 9, 7, 0], [6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 6, 7], [6, 7, 3, 3, 3, 3, 3, 7, 7, 7, 3, 3, 3, 3, 3, 3], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 3], [6, 7, 3, 3, 3, 3, 3, 7, 7, 7, 3, 3, 3, 3, 7, 3], [6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 3, 7, 3], [6, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 7]]}], "test": [{"input": [[6, 7, 9, 9, 9, 9, 9, 9, 9, 7, 7, 7, 7, 7, 7, 7, 9, 9, 9, 9, 9], [6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 9], [6, 7, 9, 9, 9, 9, 9, 9, 9, 7, 7, 7, 7, 0, 7, 0, 9, 9, 9, 7, 9], [6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 0, 7, 0, 7, 7, 9, 7, 9], [6, 7, 0, 0, 0, 0, 0, 0, 0, 7, 7, 7, 0, 0, 7, 0, 7, 7, 9, 7, 9], [6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 0, 7, 7, 9, 7, 9], [6, 7, 0, 0, 0, 0, 0, 0, 0, 7, 7, 7, 0, 0, 0, 0, 7, 7, 9, 7, 9], [6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 0, 7, 7, 7, 7, 7, 7], [6, 7, 0, 0, 0, 0, 0, 0, 0, 7, 7, 7, 0, 7, 0, 8, 8, 8, 8, 7, 7], [6, 7, 7, 7, 7, 7, 7, 7, 1, 7, 1, 7, 0, 7, 0, 7, 7, 7, 7, 7, 7], [6, 7, 8, 8, 8, 8, 8, 7, 1, 7, 1, 7, 0, 7, 0, 8, 8, 8, 8, 7, 7], [6, 7, 7, 7, 7, 7, 7, 7, 1, 7, 1, 7, 0, 7, 0, 7, 7, 7, 7, 7, 7], [6, 7, 1, 1, 1, 1, 1, 1, 1, 7, 1, 7, 0, 7, 0, 7, 7, 7, 7, 7, 7], [6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 1, 7, 0, 7, 0, 0, 0, 7, 7, 7, 7], [6, 7, 1, 1, 1, 1, 1, 1, 1, 1, 1, 7, 0, 7, 7, 7, 7, 7, 7, 7, 7], [6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 0, 0, 0, 0, 0, 7, 7, 7, 7], [6, 7, 4, 4, 4, 4, 4, 4, 4, 4, 7, 7, 7, 4, 4, 4, 4, 7, 7, 7, 7], [6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 4, 7, 7, 7, 7], [6, 7, 4, 4, 4, 4, 4, 4, 4, 4, 2, 7, 2, 4, 4, 7, 4, 7, 7, 7, 7], [6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 2, 7, 2, 7, 4, 7, 4, 7, 9, 7, 9], [6, 7, 2, 2, 2, 2, 2, 2, 2, 2, 2, 7, 2, 7, 4, 7, 4, 9, 9, 7, 9], [6, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 2, 7, 7, 7, 7, 7, 7, 7, 9], [6, 7, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 7, 7, 7, 7, 9, 9, 9, 9]]}]}, "a47bf94d": {"train": [{"input": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 3, 3, 3, 8, 8, 8, 8, 8, 8, 0, 0, 8, 8, 0, 0, 0, 0, 0, 0, 0], [0, 0, 3, 3, 3, 0, 0, 0, 0, 0, 8, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 8, 8, 9, 9, 9, 9, 8, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 4, 0, 0, 0, 8, 0, 9, 9, 9, 9, 0, 0, 4, 0, 4, 0, 0, 0, 0], [0, 0, 4, 0, 4, 8, 5, 8, 5, 9, 9, 9, 9, 8, 8, 0, 4, 0, 0, 0, 0, 0], [0, 0, 0, 4, 0, 0, 0, 8, 0, 9, 9, 9, 9, 0, 0, 4, 0, 4, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 8, 0, 9, 9, 9, 9, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 2, 2, 2, 0, 0, 8, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 2, 2, 2, 8, 8, 8, 0, 0, 8, 8, 8, 8, 8, 0, 0, 0, 0, 0, 0, 0], [0, 0, 2, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 0, 6, 0, 0, 0, 0], [0, 0, 0, 0, 0, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 6, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 0, 6, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], "output": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 2, 0, 0, 0, 0], [0, 0, 3, 0, 3, 8, 8, 8, 8, 8, 8, 0, 0, 8, 8, 0, 2, 0, 0, 0, 0, 0], [0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 8, 0, 0, 8, 0, 2, 0, 2, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 8, 8, 9, 9, 9, 9, 8, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 4, 0, 0, 0, 8, 0, 9, 9, 9, 9, 0, 0, 4, 0, 4, 0, 0, 0, 0], [0, 0, 4, 0, 4, 8, 5, 8, 5, 9, 9, 9, 9, 8, 8, 0, 4, 0, 0, 0, 0, 0], [0, 0, 0, 4, 0, 0, 0, 8, 0, 9, 9, 9, 9, 0, 0, 4, 0, 4, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 8, 0, 9, 9, 9, 9, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 2, 0, 0, 0, 8, 0, 0, 8, 0, 0, 0, 0, 3, 0, 3, 0, 0, 0, 0], [0, 0, 2, 0, 2, 8, 8, 8, 0, 0, 8, 8, 8, 8, 8, 0, 3, 0, 0, 0, 0, 0], [0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 3, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 0, 6, 0, 0, 0, 0], [0, 0, 6, 0, 6, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 6, 0, 0, 0, 0, 0], [0, 0, 0, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 0, 6, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}, {"input": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 1, 1, 1, 0, 2, 2, 2, 0, 3, 3, 3, 0, 4, 4, 4, 0, 0, 0, 0], [0, 0, 0, 1, 1, 1, 0, 2, 2, 2, 0, 3, 3, 3, 0, 4, 4, 4, 0, 0, 0, 0], [0, 0, 0, 1, 1, 1, 0, 2, 2, 2, 0, 3, 3, 3, 0, 4, 4, 4, 0, 0, 0, 0], [0, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 0, 0], [0, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 8, 8, 8, 5, 8, 5, 8, 0, 0, 0], [0, 0, 0, 0, 8, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 8, 0, 8, 0, 0, 0], [0, 0, 0, 0, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 0, 0, 8, 0, 8, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 8, 0, 0, 0, 8, 0, 8, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 8, 0, 8, 0, 0, 0], [0, 0, 0, 0, 8, 8, 8, 5, 8, 5, 8, 5, 8, 5, 8, 8, 8, 0, 8, 0, 0, 0], [0, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 0, 0, 8, 0, 0, 0], [0, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 8, 8, 8, 0, 0, 0], [0, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], "output": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 1, 0, 0, 0, 2, 0, 0, 0, 3, 0, 0, 0, 4, 0, 0, 0, 0, 0], [0, 0, 0, 1, 0, 1, 0, 2, 0, 2, 0, 3, 0, 3, 0, 4, 0, 4, 0, 0, 0, 0], [0, 0, 0, 0, 1, 0, 0, 0, 2, 0, 0, 0, 3, 0, 0, 0, 4, 0, 0, 0, 0, 0], [0, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 0, 0], [0, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 8, 8, 8, 5, 8, 5, 8, 0, 0, 0], [0, 0, 0, 0, 8, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0, 8, 0, 8, 0, 0, 0], [0, 0, 0, 0, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 0, 0, 8, 0, 8, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 8, 0, 0, 0, 8, 0, 8, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 8, 0, 8, 0, 0, 0], [0, 0, 0, 0, 8, 8, 8, 5, 8, 5, 8, 5, 8, 5, 8, 8, 8, 0, 8, 0, 0, 0], [0, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 0, 0, 8, 0, 0, 0], [0, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 8, 8, 8, 0, 0, 0], [0, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 0, 0], [0, 0, 0, 4, 0, 4, 0, 2, 0, 2, 0, 1, 0, 1, 0, 3, 0, 3, 0, 0, 0, 0], [0, 0, 0, 0, 4, 0, 0, 0, 2, 0, 0, 0, 1, 0, 0, 0, 3, 0, 0, 0, 0, 0], [0, 0, 0, 4, 0, 4, 0, 2, 0, 2, 0, 1, 0, 1, 0, 3, 0, 3, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}, {"input": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 2, 2, 2, 0, 0, 0, 0, 0, 4, 4, 4, 0, 6, 6, 6, 0, 0, 0, 0, 0], [0, 0, 2, 2, 2, 0, 0, 0, 0, 0, 4, 4, 4, 0, 6, 6, 6, 0, 0, 0, 0, 0], [0, 0, 2, 2, 2, 0, 0, 0, 0, 0, 4, 4, 4, 0, 6, 6, 6, 0, 0, 0, 0, 0], [0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0], [0, 0, 9, 9, 9, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0], [0, 0, 9, 9, 9, 0, 0, 8, 8, 8, 5, 8, 5, 8, 0, 8, 0, 0, 0, 0, 0, 0], [0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 8, 0, 8, 9, 9, 9, 9, 0, 0, 0, 0], [0, 0, 0, 8, 8, 8, 8, 8, 0, 0, 9, 9, 0, 8, 9, 9, 9, 9, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 9, 9, 0, 8, 9, 9, 9, 9, 0, 0, 0, 0], [0, 0, 0, 8, 8, 8, 5, 8, 5, 8, 8, 8, 0, 8, 0, 8, 0, 0, 0, 0, 0, 0], [0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 0, 0, 8, 0, 8, 0, 0, 0, 0, 0, 0], [0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 8, 8, 8, 0, 8, 0, 0, 0, 0, 0, 0], [0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], "output": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 2, 0, 0, 0, 1, 0, 0, 0, 4, 0, 0, 0, 6, 0, 0, 0, 0, 0, 0], [0, 0, 2, 0, 2, 0, 1, 0, 1, 0, 4, 0, 4, 0, 6, 0, 6, 0, 0, 0, 0, 0], [0, 0, 0, 2, 0, 0, 0, 1, 0, 0, 0, 4, 0, 0, 0, 6, 0, 0, 0, 0, 0, 0], [0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0], [0, 0, 9, 9, 9, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0], [0, 0, 9, 9, 9, 0, 0, 8, 8, 8, 5, 8, 5, 8, 0, 8, 0, 0, 0, 0, 0, 0], [0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 8, 0, 8, 9, 9, 9, 9, 0, 0, 0, 0], [0, 0, 0, 8, 8, 8, 8, 8, 0, 0, 9, 9, 0, 8, 9, 9, 9, 9, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 9, 9, 0, 8, 9, 9, 9, 9, 0, 0, 0, 0], [0, 0, 0, 8, 8, 8, 5, 8, 5, 8, 8, 8, 0, 8, 0, 8, 0, 0, 0, 0, 0, 0], [0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 0, 0, 8, 0, 8, 0, 0, 0, 0, 0, 0], [0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 8, 8, 8, 0, 8, 0, 0, 0, 0, 0, 0], [0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0], [0, 0, 4, 0, 4, 0, 2, 0, 2, 0, 1, 0, 1, 0, 6, 0, 6, 0, 0, 0, 0, 0], [0, 0, 0, 4, 0, 0, 0, 2, 0, 0, 0, 1, 0, 0, 0, 6, 0, 0, 0, 0, 0, 0], [0, 0, 4, 0, 4, 0, 2, 0, 2, 0, 1, 0, 1, 0, 6, 0, 6, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}], "test": [{"input": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 2, 2, 2, 0, 0, 3, 3, 3, 0, 0, 6, 6, 6, 0, 0], [0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 2, 2, 2, 0, 0, 3, 3, 3, 0, 0, 6, 6, 6, 0, 0], [0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 2, 2, 2, 0, 0, 3, 3, 3, 0, 0, 6, 6, 6, 0, 0], [0, 0, 8, 0, 0, 0, 0, 8, 0, 0, 0, 0, 8, 0, 0, 0, 0, 8, 0, 0, 0, 0, 8, 0, 0, 0], [0, 0, 8, 0, 0, 0, 0, 8, 0, 0, 0, 0, 8, 8, 8, 8, 0, 8, 0, 0, 8, 8, 8, 0, 0, 0], [0, 0, 8, 0, 0, 0, 0, 8, 0, 9, 9, 0, 0, 0, 0, 5, 0, 5, 0, 0, 5, 0, 0, 0, 0, 0], [0, 0, 8, 0, 0, 0, 0, 8, 8, 9, 9, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 0, 0], [0, 0, 8, 0, 0, 0, 0, 0, 0, 9, 9, 0, 0, 0, 0, 5, 0, 5, 0, 0, 5, 0, 8, 0, 0, 0], [0, 0, 8, 8, 8, 8, 8, 8, 8, 9, 9, 8, 8, 0, 0, 8, 0, 8, 0, 0, 8, 0, 8, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 9, 9, 0, 8, 0, 0, 8, 0, 8, 0, 0, 8, 0, 8, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 5, 0, 8, 0, 9, 9, 9, 9, 9, 0, 0], [0, 0, 0, 0, 0, 0, 0, 8, 8, 8, 8, 5, 8, 5, 8, 8, 8, 8, 0, 9, 9, 9, 9, 9, 0, 0], [0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 8, 0, 0, 5, 0, 0, 0, 9, 9, 9, 9, 9, 0, 0], [0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 8, 0, 0, 8, 8, 8, 0, 0, 8, 0, 8, 0, 0, 0], [0, 9, 9, 9, 9, 0, 0, 8, 0, 0, 0, 0, 9, 9, 0, 0, 0, 5, 0, 0, 8, 0, 8, 0, 0, 0], [0, 9, 9, 9, 9, 8, 5, 8, 5, 8, 8, 5, 9, 9, 8, 8, 8, 8, 8, 8, 8, 0, 8, 0, 0, 0], [0, 9, 9, 9, 9, 0, 0, 8, 0, 0, 0, 0, 8, 0, 0, 0, 0, 5, 0, 0, 0, 0, 8, 0, 0, 0], [0, 0, 8, 0, 0, 0, 0, 8, 0, 0, 0, 0, 8, 0, 0, 0, 0, 8, 0, 0, 0, 0, 8, 0, 0, 0], [0, 0, 8, 0, 0, 0, 0, 8, 0, 0, 0, 0, 8, 0, 0, 0, 0, 8, 0, 0, 0, 0, 8, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 0, 4, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 0, 4, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}]}, "b5ca7ac4": {"train": [{"input": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 8, 8, 8, 8, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 8, 5, 5, 5, 8, 0, 0, 0, 2, 2, 2, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0], [0, 8, 5, 5, 5, 8, 0, 0, 0, 2, 1, 1, 1, 2, 0, 8, 8, 8, 8, 8, 0, 0], [0, 8, 5, 5, 5, 8, 0, 0, 0, 2, 1, 1, 1, 2, 0, 8, 3, 3, 3, 8, 0, 0], [0, 8, 8, 8, 8, 8, 0, 0, 0, 2, 1, 1, 1, 2, 0, 8, 3, 3, 3, 8, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 2, 2, 2, 0, 8, 3, 3, 3, 8, 0, 0], [0, 0, 0, 0, 2, 2, 2, 2, 2, 0, 0, 0, 0, 0, 0, 8, 8, 8, 8, 8, 0, 0], [0, 0, 0, 0, 2, 3, 3, 3, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 2, 3, 3, 3, 2, 0, 0, 0, 0, 2, 2, 2, 2, 2, 0, 0, 0, 0], [0, 0, 0, 0, 2, 3, 3, 3, 2, 0, 0, 0, 0, 2, 9, 9, 9, 2, 0, 0, 0, 0], [0, 0, 0, 0, 2, 2, 2, 2, 2, 0, 0, 0, 0, 2, 9, 9, 9, 2, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 9, 9, 9, 2, 0, 0, 0, 0], [2, 2, 2, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 2, 2, 2, 0, 0, 0, 0], [2, 6, 6, 6, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [2, 6, 6, 6, 2, 0, 0, 0, 0, 0, 0, 0, 8, 8, 8, 8, 8, 0, 0, 0, 0, 0], [2, 6, 6, 6, 2, 2, 2, 2, 2, 2, 0, 0, 8, 4, 4, 4, 8, 0, 0, 0, 0, 0], [2, 2, 2, 2, 2, 2, 4, 4, 4, 2, 0, 0, 8, 4, 4, 4, 8, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 2, 4, 4, 4, 2, 0, 0, 8, 4, 4, 4, 8, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 2, 4, 4, 4, 2, 0, 0, 8, 8, 8, 8, 8, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 2, 2, 2, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], "output": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [8, 8, 8, 8, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [8, 5, 5, 5, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 2, 2, 2], [8, 5, 5, 5, 8, 8, 8, 8, 8, 8, 0, 0, 0, 0, 0, 0, 0, 2, 1, 1, 1, 2], [8, 5, 5, 5, 8, 8, 3, 3, 3, 8, 0, 0, 0, 0, 0, 0, 0, 2, 1, 1, 1, 2], [8, 8, 8, 8, 8, 8, 3, 3, 3, 8, 0, 0, 0, 0, 0, 0, 0, 2, 1, 1, 1, 2], [0, 0, 0, 0, 0, 8, 3, 3, 3, 8, 0, 0, 0, 0, 0, 0, 0, 2, 2, 2, 2, 2], [0, 0, 0, 0, 0, 8, 8, 8, 8, 8, 0, 0, 2, 2, 2, 2, 2, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 3, 3, 3, 2, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 3, 3, 3, 2, 2, 2, 2, 2, 2], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 3, 3, 3, 2, 2, 9, 9, 9, 2], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 2, 2, 2, 2, 9, 9, 9, 2], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 9, 9, 9, 2], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 6, 6, 6, 2, 0, 0, 0, 0, 0], [8, 8, 8, 8, 8, 0, 0, 0, 0, 0, 0, 0, 2, 6, 6, 6, 2, 0, 0, 0, 0, 0], [8, 4, 4, 4, 8, 0, 0, 0, 0, 0, 0, 0, 2, 6, 6, 6, 2, 2, 2, 2, 2, 2], [8, 4, 4, 4, 8, 0, 0, 0, 0, 0, 0, 0, 2, 2, 2, 2, 2, 2, 4, 4, 4, 2], [8, 4, 4, 4, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 4, 4, 4, 2], [8, 8, 8, 8, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 4, 4, 4, 2], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 2, 2, 2], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}, {"input": [[4, 4, 4, 4, 4, 4, 4, 4, 2, 2, 2, 2, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 2, 5, 5, 5, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 8, 8, 8, 8, 8, 4, 2, 5, 5, 5, 2, 4, 2, 2, 2, 2, 2, 4, 4, 4], [4, 4, 8, 9, 9, 9, 8, 4, 2, 5, 5, 5, 2, 4, 2, 3, 3, 3, 2, 4, 4, 4], [4, 4, 8, 9, 9, 9, 8, 4, 2, 2, 2, 2, 2, 4, 2, 3, 3, 3, 2, 4, 4, 4], [4, 4, 8, 9, 9, 9, 8, 4, 4, 4, 4, 4, 4, 4, 2, 3, 3, 3, 2, 4, 4, 4], [4, 4, 8, 8, 8, 8, 8, 4, 4, 4, 4, 4, 4, 4, 2, 2, 2, 2, 2, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [8, 8, 8, 8, 8, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 2, 2, 2, 2, 4], [8, 3, 3, 3, 8, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 6, 6, 6, 2, 4], [8, 3, 3, 3, 8, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 6, 6, 6, 2, 4], [8, 3, 3, 3, 8, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 6, 6, 6, 2, 4], [8, 8, 8, 8, 8, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 2, 2, 2, 2, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 8, 8, 8, 8, 8, 4, 4, 4, 4, 4, 4, 4, 2, 2, 2, 2, 2, 4, 4, 4], [4, 4, 8, 1, 1, 1, 8, 4, 4, 4, 4, 4, 4, 4, 2, 1, 1, 1, 2, 4, 4, 4], [4, 4, 8, 1, 1, 1, 8, 4, 8, 8, 8, 8, 8, 4, 2, 1, 1, 1, 2, 4, 4, 4], [4, 4, 8, 1, 1, 1, 8, 4, 8, 6, 6, 6, 8, 4, 2, 1, 1, 1, 2, 4, 4, 4], [4, 4, 8, 8, 8, 8, 8, 4, 8, 6, 6, 6, 8, 4, 2, 2, 2, 2, 2, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 8, 6, 6, 6, 8, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 8, 8, 8, 8, 8, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]], "output": [[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 2, 2, 2, 2, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 5, 5, 5, 2, 4, 4, 4, 4, 4], [8, 8, 8, 8, 8, 4, 4, 4, 4, 4, 4, 4, 2, 5, 5, 5, 2, 2, 2, 2, 2, 2], [8, 9, 9, 9, 8, 4, 4, 4, 4, 4, 4, 4, 2, 5, 5, 5, 2, 2, 3, 3, 3, 2], [8, 9, 9, 9, 8, 4, 4, 4, 4, 4, 4, 4, 2, 2, 2, 2, 2, 2, 3, 3, 3, 2], [8, 9, 9, 9, 8, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 3, 3, 3, 2], [8, 8, 8, 8, 8, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 2, 2, 2, 2], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [8, 8, 8, 8, 8, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 2, 2, 2, 2], [8, 3, 3, 3, 8, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 6, 6, 6, 2], [8, 3, 3, 3, 8, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 6, 6, 6, 2], [8, 3, 3, 3, 8, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 6, 6, 6, 2], [8, 8, 8, 8, 8, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 2, 2, 2, 2], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [8, 8, 8, 8, 8, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 2, 2, 2, 2], [8, 1, 1, 1, 8, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 1, 1, 1, 2], [8, 1, 1, 1, 8, 8, 8, 8, 8, 8, 4, 4, 4, 4, 4, 4, 4, 2, 1, 1, 1, 2], [8, 1, 1, 1, 8, 8, 6, 6, 6, 8, 4, 4, 4, 4, 4, 4, 4, 2, 1, 1, 1, 2], [8, 8, 8, 8, 8, 8, 6, 6, 6, 8, 4, 4, 4, 4, 4, 4, 4, 2, 2, 2, 2, 2], [4, 4, 4, 4, 4, 8, 6, 6, 6, 8, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 8, 8, 8, 8, 8, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]]}, {"input": [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1], [1, 1, 1, 8, 8, 8, 8, 8, 1, 1, 1, 1, 2, 3, 3, 3, 2, 1, 1, 1, 1, 1], [1, 1, 1, 8, 2, 2, 2, 8, 1, 1, 1, 1, 2, 3, 3, 3, 2, 1, 1, 1, 1, 1], [1, 1, 1, 8, 2, 2, 2, 8, 1, 1, 1, 1, 2, 3, 3, 3, 2, 1, 1, 1, 1, 1], [1, 1, 1, 8, 2, 2, 2, 8, 1, 1, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1], [1, 1, 1, 8, 8, 8, 8, 8, 1, 1, 1, 1, 8, 8, 8, 8, 8, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 8, 6, 6, 6, 8, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 8, 6, 6, 6, 8, 1, 1, 1, 1, 1], [1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 8, 6, 6, 6, 8, 1, 1, 1, 1, 1], [1, 1, 2, 5, 5, 5, 2, 1, 1, 1, 1, 1, 8, 8, 8, 8, 8, 1, 1, 1, 1, 1], [1, 1, 2, 5, 5, 5, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 2, 5, 5, 5, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 4, 4, 4, 2, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8, 8, 1, 1, 2, 4, 4, 4, 2, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 8, 3, 3, 3, 8, 1, 1, 2, 4, 4, 4, 2, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 8, 3, 3, 3, 8, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 8, 3, 3, 3, 8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8, 8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], "output": [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2], [8, 8, 8, 8, 8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 3, 3, 3, 2], [8, 2, 2, 2, 8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 3, 3, 3, 2], [8, 2, 2, 2, 8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 3, 3, 3, 2], [8, 2, 2, 2, 8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 8, 6, 6, 6, 8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 8, 6, 6, 6, 8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 8, 6, 6, 6, 8, 1, 1, 2, 2, 2, 2, 2, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 8, 8, 8, 8, 8, 1, 1, 2, 5, 5, 5, 2, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 5, 5, 5, 2, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 5, 5, 5, 2, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 4, 4, 4, 2], [8, 8, 8, 8, 8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 4, 4, 4, 2], [8, 3, 3, 3, 8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 4, 4, 4, 2], [8, 3, 3, 3, 8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2], [8, 3, 3, 3, 8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [8, 8, 8, 8, 8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]}], "test": [{"input": [[9, 2, 2, 2, 2, 2, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 2, 4, 4, 4, 2, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 2, 4, 4, 4, 2, 9, 9, 9, 9, 2, 2, 2, 2, 2, 9, 9, 9, 9, 9, 9, 9], [9, 2, 4, 4, 4, 2, 9, 9, 9, 9, 2, 3, 3, 3, 2, 9, 8, 8, 8, 8, 8, 9], [9, 2, 2, 2, 2, 2, 9, 9, 9, 9, 2, 3, 3, 3, 2, 9, 8, 4, 4, 4, 8, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 2, 3, 3, 3, 2, 9, 8, 4, 4, 4, 8, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 2, 2, 2, 2, 2, 9, 8, 4, 4, 4, 8, 9], [9, 9, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 9], [9, 9, 8, 1, 1, 1, 8, 9, 9, 9, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9], [9, 9, 8, 1, 1, 1, 8, 9, 9, 9, 8, 7, 7, 7, 8, 9, 9, 9, 9, 9, 9, 9], [9, 9, 8, 1, 1, 1, 8, 9, 9, 9, 8, 7, 7, 7, 8, 9, 9, 9, 9, 9, 9, 9], [9, 9, 8, 8, 8, 8, 8, 9, 9, 9, 8, 7, 7, 7, 8, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 2, 2, 2, 2, 2, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 2, 6, 6, 6, 2, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 2, 2, 2, 2, 2, 9, 2, 6, 6, 6, 2, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 2, 3, 3, 3, 2, 9, 2, 6, 6, 6, 2, 9, 8, 8, 8, 8, 8, 9, 9], [9, 9, 9, 2, 3, 3, 3, 2, 9, 2, 2, 2, 2, 2, 9, 8, 3, 3, 3, 8, 9, 9], [9, 9, 9, 2, 3, 3, 3, 2, 9, 9, 9, 9, 9, 9, 9, 8, 3, 3, 3, 8, 9, 9], [9, 9, 9, 2, 2, 2, 2, 2, 9, 9, 9, 9, 9, 9, 9, 8, 3, 3, 3, 8, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 8, 8, 8, 8, 8, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9]]}]}, "b6f77b65": {"train": [{"input": [[9, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 5, 1, 1, 1, 1, 0, 0, 0], [0, 0, 0, 0, 5, 0, 0, 0, 1, 0, 0, 0], [0, 0, 0, 0, 5, 0, 0, 0, 1, 0, 0, 0], [0, 0, 6, 3, 3, 3, 0, 5, 5, 7, 0, 0], [0, 0, 6, 0, 0, 3, 0, 5, 0, 7, 0, 0], [0, 0, 6, 0, 0, 3, 0, 5, 0, 7, 0, 0], [0, 4, 2, 2, 2, 2, 2, 2, 2, 2, 7, 0], [0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0], [0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0], [0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0]], "output": [[9, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 5, 1, 1, 1, 1, 0, 0, 0], [0, 0, 0, 0, 5, 0, 0, 0, 1, 0, 0, 0], [0, 0, 0, 0, 5, 0, 0, 0, 1, 0, 0, 0], [0, 0, 6, 3, 3, 3, 0, 5, 5, 7, 0, 0], [0, 0, 6, 0, 0, 3, 0, 5, 0, 7, 0, 0], [0, 0, 6, 0, 0, 3, 0, 5, 0, 7, 0, 0], [0, 4, 2, 2, 2, 2, 2, 2, 2, 2, 7, 0], [0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0], [0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0], [0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0]]}, {"input": [[4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 2, 5, 5, 5, 8, 0, 0], [0, 0, 0, 0, 0, 2, 0, 0, 0, 8, 0, 0], [0, 0, 0, 0, 0, 2, 0, 0, 0, 8, 0, 0], [0, 0, 0, 0, 0, 2, 0, 0, 0, 8, 0, 0], [0, 0, 9, 4, 4, 4, 7, 0, 2, 3, 3, 0], [0, 0, 9, 0, 0, 0, 7, 0, 2, 0, 3, 0], [0, 0, 9, 0, 0, 0, 7, 0, 2, 0, 3, 0], [0, 6, 5, 5, 0, 1, 7, 7, 7, 7, 4, 0], [0, 6, 0, 5, 0, 1, 0, 0, 0, 0, 4, 0], [0, 6, 0, 5, 0, 1, 0, 0, 0, 0, 4, 0]], "output": [[4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 2, 5, 5, 5, 8, 0, 0], [0, 0, 9, 0, 0, 2, 0, 0, 0, 8, 0, 0], [0, 0, 9, 0, 0, 2, 0, 0, 0, 8, 0, 0], [0, 0, 9, 0, 0, 2, 7, 0, 2, 8, 0, 0], [0, 6, 5, 5, 0, 1, 7, 0, 2, 3, 3, 0], [0, 6, 0, 5, 0, 1, 7, 0, 2, 0, 3, 0], [0, 6, 0, 5, 0, 1, 7, 7, 7, 7, 3, 0]]}, {"input": [[6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 2, 5, 5, 5, 8, 0, 0], [0, 0, 0, 0, 0, 2, 0, 0, 0, 8, 0, 0], [0, 0, 0, 0, 0, 2, 0, 0, 0, 8, 0, 0], [0, 0, 0, 0, 0, 2, 0, 0, 0, 8, 0, 0], [0, 0, 9, 4, 4, 4, 7, 0, 2, 3, 3, 0], [0, 0, 9, 0, 0, 0, 7, 0, 2, 0, 3, 0], [0, 0, 9, 0, 0, 0, 7, 0, 2, 0, 3, 0], [0, 6, 5, 5, 0, 1, 7, 7, 7, 7, 4, 0], [0, 6, 0, 5, 0, 1, 0, 0, 0, 0, 4, 0], [0, 6, 0, 5, 0, 1, 0, 0, 0, 0, 4, 0]], "output": [[6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0], [0, 0, 0, 0, 0, 2, 5, 5, 5, 8, 0, 0], [0, 0, 0, 0, 0, 2, 0, 0, 0, 8, 0, 0], [0, 0, 0, 0, 0, 2, 7, 0, 2, 3, 3, 0], [0, 0, 0, 0, 0, 2, 7, 0, 2, 0, 3, 0], [0, 0, 9, 4, 4, 4, 7, 0, 2, 0, 3, 0], [0, 0, 9, 5, 0, 1, 7, 7, 7, 7, 4, 0], [0, 0, 9, 5, 0, 1, 0, 0, 0, 0, 4, 0], [0, 0, 5, 5, 0, 1, 0, 0, 0, 0, 4, 0]]}, {"input": [[4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 5, 1, 1, 1, 1, 0, 0, 0], [0, 0, 0, 0, 5, 0, 0, 0, 1, 0, 0, 0], [0, 0, 0, 0, 5, 0, 0, 0, 1, 0, 0, 0], [0, 0, 6, 3, 3, 3, 0, 5, 5, 7, 0, 0], [0, 0, 6, 0, 0, 3, 0, 5, 0, 7, 0, 0], [0, 0, 6, 0, 0, 3, 0, 5, 0, 7, 0, 0], [0, 4, 2, 2, 2, 2, 2, 2, 2, 2, 7, 0], [0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0], [0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0], [0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0]], "output": [[4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 5, 1, 1, 1, 1, 0, 0, 0], [0, 0, 0, 0, 5, 0, 0, 0, 1, 0, 0, 0], [0, 0, 0, 0, 5, 0, 0, 0, 1, 0, 0, 0], [0, 0, 6, 3, 3, 3, 0, 5, 5, 7, 7, 0], [0, 0, 6, 0, 0, 3, 0, 5, 0, 7, 7, 0], [0, 0, 6, 0, 0, 3, 0, 5, 0, 7, 7, 0], [0, 0, 2, 2, 2, 2, 2, 2, 2, 2, 7, 0]]}, {"input": [[7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 5, 1, 1, 1, 1, 0, 0, 0], [0, 0, 0, 0, 5, 0, 0, 0, 1, 0, 0, 0], [0, 0, 0, 0, 5, 0, 0, 0, 1, 0, 0, 0], [0, 0, 6, 3, 3, 3, 0, 5, 5, 7, 0, 0], [0, 0, 6, 0, 0, 3, 0, 5, 0, 7, 0, 0], [0, 0, 6, 0, 0, 3, 0, 5, 0, 7, 0, 0], [0, 4, 2, 2, 2, 2, 2, 2, 2, 2, 7, 0], [0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0], [0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0], [0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0]], "output": [[7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 5, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 5, 1, 1, 1, 1, 0, 0, 0], [0, 4, 6, 3, 3, 3, 0, 5, 1, 0, 0, 0], [0, 4, 6, 0, 0, 3, 0, 5, 1, 0, 0, 0], [0, 4, 6, 0, 0, 3, 0, 5, 5, 0, 0, 0], [0, 4, 2, 2, 2, 2, 2, 2, 2, 2, 0, 0]]}], "test": [{"input": [[3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 4, 3, 3, 0, 0, 0, 0, 0, 0, 0], [0, 0, 4, 0, 3, 0, 8, 7, 7, 7, 0, 0], [0, 0, 4, 0, 3, 0, 8, 0, 0, 7, 0, 0], [0, 0, 4, 0, 3, 0, 8, 0, 0, 7, 0, 0], [0, 0, 6, 5, 5, 5, 5, 5, 0, 7, 0, 0], [0, 0, 6, 0, 0, 0, 0, 5, 0, 7, 0, 0], [0, 0, 6, 0, 0, 0, 0, 5, 0, 7, 0, 0], [0, 3, 1, 1, 1, 0, 2, 2, 2, 2, 9, 0], [0, 3, 0, 0, 1, 0, 2, 0, 0, 0, 9, 0], [0, 3, 0, 0, 1, 0, 2, 0, 0, 0, 9, 0]]}, {"input": [[7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 4, 3, 3, 0, 0, 0, 0, 0, 0, 0], [0, 0, 4, 0, 3, 0, 8, 7, 7, 7, 0, 0], [0, 0, 4, 0, 3, 0, 8, 0, 0, 7, 0, 0], [0, 0, 4, 0, 3, 0, 8, 0, 0, 7, 0, 0], [0, 0, 6, 5, 5, 5, 5, 5, 0, 7, 0, 0], [0, 0, 6, 0, 0, 0, 0, 5, 0, 7, 0, 0], [0, 0, 6, 0, 0, 0, 0, 5, 0, 7, 0, 0], [0, 3, 1, 1, 1, 0, 2, 2, 2, 2, 9, 0], [0, 3, 0, 0, 1, 0, 2, 0, 0, 0, 9, 0], [0, 3, 0, 0, 1, 0, 2, 0, 0, 0, 9, 0]]}, {"input": [[2, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 4, 3, 3, 0, 0, 0, 0, 0, 0, 0], [0, 0, 4, 0, 3, 0, 8, 7, 7, 7, 0, 0], [0, 0, 4, 0, 3, 0, 8, 0, 0, 7, 0, 0], [0, 0, 4, 0, 3, 0, 8, 0, 0, 7, 0, 0], [0, 0, 6, 5, 5, 5, 5, 5, 0, 7, 0, 0], [0, 0, 6, 0, 0, 0, 0, 5, 0, 7, 0, 0], [0, 0, 6, 0, 0, 0, 0, 5, 0, 7, 0, 0], [0, 3, 1, 1, 1, 0, 2, 2, 2, 2, 9, 0], [0, 3, 0, 0, 1, 0, 2, 0, 0, 0, 9, 0], [0, 3, 0, 0, 1, 0, 2, 0, 0, 0, 9, 0]]}]}, "c4d067a0": {"train": [{"input": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 4, 8, 4, 8, 8, 8, 8, 4, 4, 4, 8, 8, 4, 4, 4, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 4, 4, 4, 8, 8, 4, 4, 4, 8, 8, 8, 8, 8, 8, 8], [8, 1, 8, 1, 8, 8, 8, 8, 4, 4, 4, 8, 8, 4, 4, 4, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 1, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]], "output": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 4, 8, 4, 8, 8, 8, 8, 4, 4, 4, 8, 8, 4, 4, 4, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 4, 4, 4, 8, 8, 4, 4, 4, 8, 8, 8, 8, 8, 8, 8], [8, 1, 8, 1, 8, 8, 8, 8, 4, 4, 4, 8, 8, 4, 4, 4, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 1, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 8, 8, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 8, 8, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 8, 8, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 8, 8, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 8, 8, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 1, 8, 8, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]]}, {"input": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 2, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 1, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 8, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8], [8, 1, 8, 1, 8, 3, 8, 8, 8, 8, 2, 2, 8, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 1, 8, 1, 8, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]], "output": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 2, 8, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 1, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 8, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8], [8, 1, 8, 1, 8, 3, 8, 8, 8, 8, 2, 2, 8, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 1, 8, 1, 8, 3, 8, 8, 8, 8, 1, 1, 8, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 8, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 8, 1, 1, 8, 3, 3, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 8, 1, 1, 8, 3, 3, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 8, 1, 1, 8, 3, 3, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 8, 1, 1, 8, 3, 3, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]]}, {"input": [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 2, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 3, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 3, 1, 4, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 1, 1, 1, 4, 4, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 1, 1, 1, 4, 4, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], "output": [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 2, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 3, 1, 3, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 3, 1, 4, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 1, 1, 1, 3, 3, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2, 1, 1, 1, 3, 3, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 1, 1, 1, 3, 3, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 1, 1, 1, 3, 3, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 1, 1, 1, 4, 4, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 1, 1, 1, 4, 4, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]}], "test": [{"input": [[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 1, 3, 1, 3, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 4, 3, 4, 3, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 1, 3, 4, 3, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 8, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 3, 2, 2, 2, 2, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 3, 2, 2, 2, 2, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 3, 2, 2, 2, 2, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 3, 2, 2, 2, 2, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]]}]}, "c7f57c3e": {"train": [{"input": [[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 2, 1, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 8, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 8, 8, 8, 4, 4, 4], [4, 4, 4, 4, 1, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 1, 2, 1, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 2, 2, 1, 1, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 2, 2, 1, 1, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 2, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 2, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]], "output": [[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 2, 1, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 3, 3, 3, 4, 4, 4], [4, 4, 4, 4, 1, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 1, 2, 1, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 8, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 8, 8, 8, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 2, 2, 1, 1, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 2, 2, 1, 1, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 8, 8, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 8, 8, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 8, 8, 8, 8, 8, 8, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 8, 8, 8, 8, 8, 8, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]]}, {"input": [[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 1, 3, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 1, 3, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 1, 1, 3, 3, 1, 1, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 1, 1, 3, 3, 1, 1, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 2, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 2, 3, 3, 3, 3, 3, 3], [3, 3, 3, 6, 3, 3, 3, 3, 1, 1, 3, 3, 1, 1, 3, 3, 3, 3], [3, 3, 1, 3, 1, 3, 3, 3, 1, 1, 3, 3, 1, 1, 3, 3, 3, 3], [3, 3, 3, 2, 3, 3, 3, 3, 3, 3, 4, 4, 3, 3, 3, 3, 3, 3], [3, 3, 1, 3, 1, 3, 3, 3, 3, 3, 4, 4, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]], "output": [[3, 3, 6, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 1, 3, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 1, 3, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 6, 6, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 6, 6, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 1, 1, 3, 3, 1, 1, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 1, 1, 3, 3, 1, 1, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 2, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 2, 2, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 1, 1, 3, 3, 1, 1, 3, 3, 3, 3], [3, 3, 1, 3, 1, 3, 3, 3, 1, 1, 3, 3, 1, 1, 3, 3, 3, 3], [3, 3, 3, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 1, 3, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 4, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]]}], "test": [{"input": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 1, 3, 3, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 6, 6, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 2, 2, 2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 2, 8, 8, 8, 8, 8, 8], [8, 8, 1, 1, 1, 3, 3, 3, 3, 3, 3, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 8, 8, 8, 8, 8, 8], [8, 8, 1, 1, 1, 3, 3, 3, 3, 3, 3, 1, 1, 1, 8, 8, 8, 8, 8, 1, 3, 3, 1, 8, 8, 8, 8, 8], [8, 8, 1, 1, 1, 3, 3, 3, 3, 3, 3, 1, 1, 1, 8, 8, 8, 8, 8, 8, 1, 1, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]]}]}, "cbebaa4b": {"train": [{"input": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 2, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 8, 8, 8, 8, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 8, 0, 0, 0], [0, 0, 0, 0, 2, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 8, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 1, 2, 0, 0, 0, 0, 0, 0, 0, 2, 0, 2, 0, 0, 0], [0, 0, 2, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 1, 0, 0, 0, 0, 1, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 4, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 5, 5, 5, 5, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0], [0, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 5, 5, 5, 5, 0, 0], [0, 3, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 2, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], "output": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 8, 8, 8, 8, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 4, 2, 1, 1, 1, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 4, 0, 0, 0, 1, 2, 5, 5, 5, 5, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 1, 0, 0, 0, 0, 5, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 5, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 2, 5, 5, 5, 5, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}, {"input": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 6, 6, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 0, 6, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 0, 6, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 2, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 0, 0, 0, 2, 0, 0, 2, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 4, 0, 0, 0, 3, 3, 3, 3, 0, 0], [0, 0, 2, 0, 2, 0, 0, 0, 0, 2, 4, 4, 4, 2, 0, 0, 3, 0, 0, 3, 0, 0], [0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 4, 4, 4, 0, 0, 0, 2, 0, 0, 3, 0, 0], [0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0], [0, 0, 1, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 3, 3, 0, 0], [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 1, 1, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 8, 8, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], "output": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 6, 6, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 6, 0, 6, 0, 0, 8, 8, 8, 8, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 6, 0, 6, 0, 0, 8, 0, 0, 8, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 2, 0, 2, 0, 0, 2, 0, 0, 2, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 3, 0, 0, 3, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 2, 2, 0, 0, 3, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 4, 4, 4, 0, 0, 3, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 1, 1, 2, 4, 4, 4, 2, 3, 3, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}], "test": [{"input": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 2, 0, 0, 0, 8, 8, 8, 8, 8, 8, 8, 2, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 8, 0, 0, 0, 8, 0, 0, 0, 0, 0, 8, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 8, 8, 0, 0, 0, 2, 0, 0, 0, 0, 0, 8, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 8, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 9, 9, 9, 9, 9, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 9, 0, 0, 0, 9, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 2, 0, 0, 0, 9, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 2, 9, 9, 9, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 2, 0, 0, 0], [0, 0, 0, 0, 0, 2, 0, 2, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 4, 4, 4, 2, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 4, 4, 4, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 4, 4, 4, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 2, 4, 4, 4, 4, 4, 0, 0, 0, 0, 2, 0, 0, 0, 0, 2, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 3, 3, 3, 2, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0], [0, 3, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 7, 7, 7, 7, 7, 7, 0, 0, 0, 0, 0, 0, 0, 0], [0, 3, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 2, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}, {"input": [[0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 8, 8, 8, 8, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 8, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 8, 0, 0, 8, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 2, 0, 0, 0], [0, 0, 0, 8, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 0, 0, 0, 6, 0, 0, 0], [0, 0, 0, 8, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 0, 0, 0, 6, 0, 0, 0], [0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 6, 6, 6, 6, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 4, 4, 4, 0, 0, 0, 0, 0, 5, 5, 5, 5, 5, 5, 0, 0], [0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 4, 4, 4, 2, 0, 0, 0, 0, 5, 0, 0, 0, 0, 5, 0, 0], [0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0, 5, 0, 0], [0, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 0, 0, 5, 5, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0, 2, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 2, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 2, 0, 0, 0, 1, 0, 0, 0, 0, 0, 2, 0, 0, 2, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 2, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 2, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 2, 0, 0, 0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}]}, "d35bdbdc": {"train": [{"input": [[0, 0, 0, 0, 0, 0, 0, 3, 3, 3], [0, 0, 5, 5, 5, 5, 5, 3, 4, 3], [0, 5, 0, 0, 0, 0, 0, 3, 3, 3], [0, 5, 4, 4, 4, 0, 0, 0, 0, 0], [5, 0, 4, 2, 4, 0, 0, 6, 6, 6], [0, 5, 4, 4, 4, 0, 5, 6, 1, 6], [0, 5, 5, 5, 5, 5, 0, 6, 6, 6], [0, 0, 1, 1, 1, 0, 0, 0, 0, 0], [0, 0, 1, 3, 1, 0, 0, 0, 0, 0], [0, 0, 1, 1, 1, 0, 0, 0, 0, 0]], "output": [[0, 0, 0, 0, 0, 0, 0, 3, 3, 3], [0, 0, 5, 5, 5, 5, 5, 3, 2, 3], [0, 5, 0, 0, 0, 0, 0, 3, 3, 3], [0, 5, 0, 0, 0, 0, 0, 0, 0, 0], [5, 0, 0, 0, 0, 0, 0, 6, 6, 6], [0, 5, 0, 0, 0, 0, 5, 6, 3, 6], [0, 5, 5, 5, 5, 5, 0, 6, 6, 6], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}, {"input": [[2, 2, 2, 0, 0, 0, 3, 3, 3, 0], [2, 6, 2, 0, 0, 0, 3, 2, 3, 0], [2, 2, 2, 0, 5, 0, 3, 3, 3, 0], [0, 0, 0, 5, 0, 5, 5, 5, 5, 0], [8, 8, 8, 0, 5, 0, 0, 1, 1, 1], [8, 3, 8, 0, 0, 5, 0, 1, 4, 1], [8, 8, 8, 0, 5, 0, 0, 1, 1, 1], [0, 5, 0, 5, 4, 4, 4, 0, 0, 0], [0, 5, 5, 0, 4, 8, 4, 0, 0, 0], [0, 0, 0, 0, 4, 4, 4, 0, 0, 0]], "output": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 5, 0, 0, 0, 0, 0], [0, 0, 0, 5, 0, 5, 5, 5, 5, 0], [8, 8, 8, 0, 5, 0, 0, 1, 1, 1], [8, 2, 8, 0, 0, 5, 0, 1, 8, 1], [8, 8, 8, 0, 5, 0, 0, 1, 1, 1], [0, 5, 0, 5, 0, 0, 0, 0, 0, 0], [0, 5, 5, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}, {"input": [[1, 1, 1, 0, 0, 0, 0, 4, 4, 4], [1, 2, 1, 0, 0, 0, 0, 4, 3, 4], [1, 1, 1, 0, 0, 5, 0, 4, 4, 4], [0, 5, 0, 0, 5, 0, 5, 0, 0, 0], [0, 5, 0, 5, 3, 3, 3, 5, 0, 0], [0, 0, 5, 0, 3, 6, 3, 0, 5, 0], [0, 0, 0, 0, 3, 3, 3, 0, 5, 0], [2, 2, 2, 0, 0, 0, 0, 7, 7, 7], [2, 9, 2, 0, 0, 0, 0, 7, 4, 7], [2, 2, 2, 0, 0, 0, 0, 7, 7, 7]], "output": [[1, 1, 1, 0, 0, 0, 0, 0, 0, 0], [1, 9, 1, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 0, 0, 5, 0, 0, 0, 0], [0, 5, 0, 0, 5, 0, 5, 0, 0, 0], [0, 5, 0, 5, 0, 0, 0, 5, 0, 0], [0, 0, 5, 0, 0, 0, 0, 0, 5, 0], [0, 0, 0, 0, 0, 0, 0, 0, 5, 0], [0, 0, 0, 0, 0, 0, 0, 7, 7, 7], [0, 0, 0, 0, 0, 0, 0, 7, 3, 7], [0, 0, 0, 0, 0, 0, 0, 7, 7, 7]]}], "test": [{"input": [[0, 0, 0, 0, 0, 0, 0, 2, 0, 0], [0, 0, 4, 0, 0, 0, 2, 3, 2, 0], [0, 4, 2, 4, 0, 0, 0, 2, 0, 0], [0, 0, 4, 0, 0, 0, 0, 5, 0, 0], [0, 0, 0, 0, 0, 0, 5, 0, 0, 0], [0, 0, 0, 0, 1, 0, 5, 0, 3, 0], [0, 7, 0, 1, 9, 1, 5, 3, 3, 3], [7, 1, 7, 0, 1, 0, 5, 0, 3, 0], [0, 7, 0, 5, 0, 5, 0, 0, 0, 0], [0, 5, 5, 0, 5, 0, 0, 0, 0, 0]]}, {"input": [[0, 0, 0, 4, 4, 4, 0, 2, 2, 2], [0, 0, 5, 4, 2, 4, 5, 2, 3, 2], [0, 0, 5, 4, 4, 4, 5, 2, 2, 2], [0, 0, 0, 5, 0, 0, 5, 5, 5, 5], [0, 0, 5, 1, 1, 1, 0, 0, 0, 5], [0, 5, 0, 1, 3, 1, 5, 5, 5, 0], [0, 0, 5, 1, 1, 1, 5, 3, 3, 3], [6, 6, 6, 5, 0, 0, 5, 3, 6, 3], [6, 9, 6, 5, 5, 5, 5, 3, 3, 3], [6, 6, 6, 0, 0, 0, 0, 0, 0, 0]]}, {"input": [[0, 0, 0, 1, 1, 1, 0, 6, 6, 6], [0, 0, 0, 1, 3, 1, 0, 6, 1, 6], [0, 0, 0, 1, 1, 1, 0, 6, 6, 6], [5, 5, 5, 5, 5, 0, 0, 0, 5, 0], [5, 2, 2, 2, 5, 0, 0, 0, 5, 0], [5, 2, 4, 2, 5, 3, 3, 3, 5, 0], [5, 2, 2, 2, 5, 3, 9, 3, 0, 5], [0, 5, 5, 0, 5, 3, 3, 3, 0, 5], [0, 7, 7, 7, 0, 5, 5, 0, 0, 5], [0, 7, 2, 7, 0, 0, 0, 5, 5, 0]]}]}, "da515329": {"train": [{"input": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 8, 0, 8, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], "output": [[0, 8, 0, 8, 8, 8, 8, 8, 8, 8, 8, 8], [0, 8, 0, 8, 0, 0, 0, 0, 0, 0, 0, 8], [0, 8, 0, 8, 0, 8, 8, 8, 8, 8, 0, 8], [0, 8, 0, 8, 0, 8, 0, 0, 0, 8, 0, 8], [0, 8, 0, 8, 8, 0, 8, 8, 0, 8, 0, 8], [0, 8, 0, 0, 0, 8, 0, 8, 0, 8, 0, 8], [0, 8, 8, 8, 8, 8, 0, 8, 0, 8, 0, 8], [0, 0, 0, 0, 0, 0, 0, 8, 0, 8, 0, 8], [8, 8, 8, 8, 8, 8, 8, 8, 0, 8, 0, 8], [0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 8], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8]]}, {"input": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 8, 8, 8, 0, 8, 8, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], "output": [[0, 8, 0, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 0, 0, 0], [8, 8, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 8, 8, 8, 0], [8, 0, 0, 8, 0, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 0, 0, 0, 8, 0], [8, 0, 8, 8, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 8, 8, 8, 0, 8, 0], [8, 0, 8, 0, 0, 8, 0, 8, 8, 8, 8, 8, 8, 8, 0, 0, 0, 0, 8, 0, 8, 0], [8, 0, 8, 0, 8, 8, 0, 8, 0, 0, 0, 0, 0, 8, 8, 8, 8, 0, 8, 0, 8, 0], [8, 0, 8, 0, 8, 0, 0, 8, 0, 8, 8, 8, 0, 0, 0, 0, 8, 0, 8, 0, 8, 0], [8, 0, 8, 0, 8, 0, 8, 8, 0, 8, 0, 8, 8, 8, 8, 0, 8, 0, 8, 0, 8, 0], [8, 0, 8, 0, 8, 0, 8, 0, 0, 8, 0, 0, 0, 0, 8, 0, 8, 0, 8, 0, 8, 0], [8, 0, 8, 0, 8, 0, 8, 8, 8, 0, 8, 8, 8, 0, 8, 0, 8, 0, 8, 0, 8, 0], [8, 0, 8, 0, 8, 0, 0, 0, 0, 8, 0, 0, 8, 0, 8, 0, 8, 0, 8, 0, 8, 0], [8, 0, 8, 0, 8, 8, 8, 8, 0, 8, 0, 8, 8, 0, 8, 0, 8, 0, 8, 0, 8, 0], [8, 0, 8, 0, 0, 0, 0, 8, 8, 8, 0, 8, 0, 0, 8, 0, 8, 0, 8, 0, 8, 0], [8, 0, 8, 8, 8, 8, 0, 0, 0, 0, 0, 8, 0, 8, 8, 0, 8, 0, 8, 0, 8, 0], [8, 0, 0, 0, 0, 8, 8, 8, 8, 8, 8, 8, 0, 8, 0, 0, 8, 0, 8, 0, 8, 0], [8, 8, 8, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 8, 8, 0, 8, 0, 8, 0], [0, 0, 0, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 8, 0, 0, 8, 0, 8, 0], [8, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 8, 8, 0, 8, 0], [0, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 8, 0, 0, 8, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 8, 8, 0], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 8, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 8]]}, {"input": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 8, 8, 0, 8, 8, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], "output": [[8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [8, 0, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0], [8, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0], [8, 0, 8, 0, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 8, 0], [8, 0, 8, 0, 8, 0, 0, 0, 0, 0, 0, 0, 8, 0, 8, 0], [8, 0, 8, 0, 8, 0, 8, 8, 8, 8, 8, 0, 8, 0, 8, 0], [8, 0, 8, 0, 8, 0, 8, 0, 0, 0, 8, 0, 8, 0, 8, 0], [8, 0, 8, 0, 8, 8, 0, 8, 8, 0, 8, 0, 8, 0, 8, 0], [8, 0, 8, 0, 0, 0, 8, 0, 8, 0, 8, 0, 8, 0, 8, 0], [8, 0, 8, 8, 8, 8, 8, 0, 8, 0, 8, 0, 8, 0, 8, 0], [8, 0, 0, 0, 0, 0, 0, 0, 8, 0, 8, 0, 8, 0, 8, 0], [8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 8, 0, 8, 0, 8, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 8, 0, 8, 0], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 8, 0, 8, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 8, 0], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 0, 8, 0]]}], "test": [{"input": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 8, 8, 8, 8, 0, 8, 8, 8, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}]}, "db0c5428": {"train": [{"input": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 6, 6, 4, 4, 6, 4, 4, 6, 6, 8, 8, 8, 8, 8, 8], [8, 8, 8, 6, 4, 6, 6, 4, 6, 6, 4, 6, 8, 8, 8, 8, 8, 8], [8, 8, 8, 4, 6, 9, 6, 9, 6, 9, 6, 4, 8, 8, 8, 8, 8, 8], [8, 8, 8, 4, 6, 6, 8, 8, 8, 6, 6, 4, 8, 8, 8, 8, 8, 8], [8, 8, 8, 6, 4, 9, 8, 8, 8, 9, 4, 6, 8, 8, 8, 8, 8, 8], [8, 8, 8, 4, 6, 6, 8, 8, 8, 6, 6, 4, 8, 8, 8, 8, 8, 8], [8, 8, 8, 4, 6, 9, 6, 9, 6, 9, 6, 4, 8, 8, 8, 8, 8, 8], [8, 8, 8, 6, 4, 6, 6, 4, 6, 6, 4, 6, 8, 8, 8, 8, 8, 8], [8, 8, 8, 6, 6, 4, 4, 6, 4, 4, 6, 6, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]], "output": [[9, 6, 4, 8, 8, 8, 6, 9, 6, 8, 8, 8, 4, 6, 9, 8, 8, 8], [6, 4, 6, 8, 8, 8, 6, 4, 6, 8, 8, 8, 6, 4, 6, 8, 8, 8], [4, 6, 6, 8, 8, 8, 4, 6, 4, 8, 8, 8, 6, 6, 4, 8, 8, 8], [8, 8, 8, 6, 6, 4, 4, 6, 4, 4, 6, 6, 8, 8, 8, 8, 8, 8], [8, 8, 8, 6, 4, 6, 6, 4, 6, 6, 4, 6, 8, 8, 8, 8, 8, 8], [8, 8, 8, 4, 6, 9, 6, 9, 6, 9, 6, 4, 8, 8, 8, 8, 8, 8], [6, 6, 4, 4, 6, 6, 9, 9, 9, 6, 6, 4, 4, 6, 6, 8, 8, 8], [9, 4, 6, 6, 4, 9, 9, 6, 9, 9, 4, 6, 6, 4, 9, 8, 8, 8], [6, 6, 4, 4, 6, 6, 9, 9, 9, 6, 6, 4, 4, 6, 6, 8, 8, 8], [8, 8, 8, 4, 6, 9, 6, 9, 6, 9, 6, 4, 8, 8, 8, 8, 8, 8], [8, 8, 8, 6, 4, 6, 6, 4, 6, 6, 4, 6, 8, 8, 8, 8, 8, 8], [8, 8, 8, 6, 6, 4, 4, 6, 4, 4, 6, 6, 8, 8, 8, 8, 8, 8], [4, 6, 6, 8, 8, 8, 4, 6, 4, 8, 8, 8, 6, 6, 4, 8, 8, 8], [6, 4, 6, 8, 8, 8, 6, 4, 6, 8, 8, 8, 6, 4, 6, 8, 8, 8], [9, 6, 4, 8, 8, 8, 6, 9, 6, 8, 8, 8, 4, 6, 9, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]]}, {"input": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8], [8, 8, 8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8], [8, 8, 8, 1, 1, 7, 1, 7, 1, 7, 1, 1, 8, 8, 8, 8], [8, 8, 8, 1, 1, 1, 8, 8, 8, 1, 1, 1, 8, 8, 8, 8], [8, 8, 8, 1, 1, 7, 8, 8, 8, 7, 1, 1, 8, 8, 8, 8], [8, 8, 8, 1, 1, 1, 8, 8, 8, 1, 1, 1, 8, 8, 8, 8], [8, 8, 8, 1, 1, 7, 1, 7, 1, 7, 1, 1, 8, 8, 8, 8], [8, 8, 8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8], [8, 8, 8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]], "output": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [7, 1, 1, 8, 8, 8, 1, 7, 1, 8, 8, 8, 1, 1, 7, 8], [1, 1, 1, 8, 8, 8, 1, 1, 1, 8, 8, 8, 1, 1, 1, 8], [1, 1, 1, 8, 8, 8, 1, 1, 1, 8, 8, 8, 1, 1, 1, 8], [8, 8, 8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8], [8, 8, 8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8], [8, 8, 8, 1, 1, 7, 1, 7, 1, 7, 1, 1, 8, 8, 8, 8], [1, 1, 1, 1, 1, 1, 7, 7, 7, 1, 1, 1, 1, 1, 1, 8], [7, 1, 1, 1, 1, 7, 7, 1, 7, 7, 1, 1, 1, 1, 7, 8], [1, 1, 1, 1, 1, 1, 7, 7, 7, 1, 1, 1, 1, 1, 1, 8], [8, 8, 8, 1, 1, 7, 1, 7, 1, 7, 1, 1, 8, 8, 8, 8], [8, 8, 8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8], [8, 8, 8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8], [1, 1, 1, 8, 8, 8, 1, 1, 1, 8, 8, 8, 1, 1, 1, 8], [1, 1, 1, 8, 8, 8, 1, 1, 1, 8, 8, 8, 1, 1, 1, 8], [7, 1, 1, 8, 8, 8, 1, 7, 1, 8, 8, 8, 1, 1, 7, 8]]}, {"input": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 5, 6, 5, 5, 5, 5, 5, 6, 5, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 6, 5, 5, 5, 5, 5, 5, 5, 6, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 5, 5, 3, 5, 2, 5, 3, 5, 5, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 5, 5, 5, 8, 8, 8, 5, 5, 5, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 5, 5, 2, 8, 8, 8, 2, 5, 5, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 5, 5, 5, 8, 8, 8, 5, 5, 5, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 5, 5, 3, 5, 2, 5, 3, 5, 5, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 6, 5, 5, 5, 5, 5, 5, 5, 6, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 5, 6, 5, 5, 5, 5, 5, 6, 5, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]], "output": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 3, 5, 5, 8, 8, 8, 5, 2, 5, 8, 8, 8, 5, 5, 3, 8], [8, 8, 8, 5, 5, 6, 8, 8, 8, 5, 5, 5, 8, 8, 8, 6, 5, 5, 8], [8, 8, 8, 5, 6, 5, 8, 8, 8, 5, 5, 5, 8, 8, 8, 5, 6, 5, 8], [8, 8, 8, 8, 8, 8, 5, 6, 5, 5, 5, 5, 5, 6, 5, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 6, 5, 5, 5, 5, 5, 5, 5, 6, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 5, 5, 3, 5, 2, 5, 3, 5, 5, 8, 8, 8, 8], [8, 8, 8, 5, 5, 5, 5, 5, 5, 3, 2, 3, 5, 5, 5, 5, 5, 5, 8], [8, 8, 8, 2, 5, 5, 5, 5, 2, 2, 5, 2, 2, 5, 5, 5, 5, 2, 8], [8, 8, 8, 5, 5, 5, 5, 5, 5, 3, 2, 3, 5, 5, 5, 5, 5, 5, 8], [8, 8, 8, 8, 8, 8, 5, 5, 3, 5, 2, 5, 3, 5, 5, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 6, 5, 5, 5, 5, 5, 5, 5, 6, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 5, 6, 5, 5, 5, 5, 5, 6, 5, 8, 8, 8, 8], [8, 8, 8, 5, 6, 5, 8, 8, 8, 5, 5, 5, 8, 8, 8, 5, 6, 5, 8], [8, 8, 8, 5, 5, 6, 8, 8, 8, 5, 5, 5, 8, 8, 8, 6, 5, 5, 8], [8, 8, 8, 3, 5, 5, 8, 8, 8, 5, 2, 5, 8, 8, 8, 5, 5, 3, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]]}], "test": [{"input": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 3, 7, 7, 7, 7, 7, 7, 7, 3, 8, 8, 8, 8], [8, 8, 8, 8, 7, 7, 4, 7, 7, 7, 4, 7, 7, 8, 8, 8, 8], [8, 8, 8, 8, 7, 4, 4, 7, 1, 7, 4, 4, 7, 8, 8, 8, 8], [8, 8, 8, 8, 7, 7, 7, 8, 8, 8, 7, 7, 7, 8, 8, 8, 8], [8, 8, 8, 8, 7, 7, 1, 8, 8, 8, 1, 7, 7, 8, 8, 8, 8], [8, 8, 8, 8, 7, 7, 7, 8, 8, 8, 7, 7, 7, 8, 8, 8, 8], [8, 8, 8, 8, 7, 4, 4, 7, 1, 7, 4, 4, 7, 8, 8, 8, 8], [8, 8, 8, 8, 7, 7, 4, 7, 7, 7, 4, 7, 7, 8, 8, 8, 8], [8, 8, 8, 8, 3, 7, 7, 7, 7, 7, 7, 7, 3, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]]}]}, "db695cfb": {"train": [{"input": [[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 6, 5, 5], [5, 1, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 6, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 6, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 1, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 6, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5]], "output": [[5, 5, 5, 5, 5, 5, 5, 5, 6, 5, 5, 5], [5, 5, 5, 5, 5, 5, 5, 6, 5, 5, 5, 5], [5, 5, 5, 5, 5, 5, 6, 5, 5, 6, 5, 5], [5, 1, 5, 5, 5, 6, 5, 5, 5, 5, 5, 6], [5, 5, 1, 5, 6, 5, 5, 5, 5, 5, 6, 5], [5, 5, 5, 6, 5, 5, 5, 5, 5, 6, 5, 5], [5, 5, 6, 5, 1, 5, 5, 5, 6, 5, 5, 5], [5, 6, 5, 5, 5, 1, 5, 6, 5, 5, 5, 5], [6, 5, 5, 5, 5, 5, 6, 5, 5, 5, 5, 5], [5, 5, 5, 5, 5, 6, 5, 1, 5, 5, 5, 5], [5, 5, 5, 5, 6, 5, 5, 5, 5, 5, 5, 5], [5, 5, 5, 6, 5, 5, 5, 5, 5, 5, 5, 5], [5, 5, 6, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 6, 5, 5, 6, 5, 5, 5, 5, 5, 5, 5], [6, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5]]}, {"input": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 6, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]], "output": [[8, 6, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 6, 8, 8, 8, 1, 8, 8, 8, 8], [8, 8, 8, 6, 8, 1, 8, 8, 8, 8, 8], [8, 8, 8, 8, 6, 8, 8, 8, 8, 8, 8], [8, 8, 8, 1, 8, 6, 8, 8, 8, 8, 8], [8, 8, 1, 8, 8, 8, 6, 8, 8, 8, 8], [8, 1, 8, 8, 8, 8, 8, 6, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 6, 8, 8]]}, {"input": [[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 1, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 6, 4, 4, 4, 4], [4, 1, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]], "output": [[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 1, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 1, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 1, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 1, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 1, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 1, 4, 4, 4, 4, 6, 4, 4, 4, 4], [4, 1, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]]}, {"input": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 6, 8, 8, 8, 8, 8, 8, 8], [8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 8, 8, 8, 8, 8, 8], [8, 6, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 8, 8]], "output": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 6, 8, 8, 8, 8, 8, 8, 8], [8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 6], [8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 8], [8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 8, 8, 6, 8, 8], [8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 8, 8, 6, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 6, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 1, 8, 6, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 8, 8, 8, 8, 8, 8], [8, 6, 8, 8, 8, 8, 8, 8, 6, 8, 1, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 6, 8, 8, 8, 1, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 6, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 6, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 1, 6, 8, 8, 8, 8, 8, 8, 8, 8, 6, 8, 8]]}, {"input": [[4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 1, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 1, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4]], "output": [[4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 1, 4, 4, 4, 4, 4, 4, 4], [4, 4, 1, 4, 4, 4, 4, 4, 4], [4, 4, 4, 1, 4, 4, 4, 4, 4], [4, 4, 4, 4, 1, 4, 4, 4, 4], [4, 4, 4, 4, 4, 1, 4, 4, 4], [4, 4, 4, 4, 4, 4, 1, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4]]}], "test": [{"input": [[3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 3, 3, 3, 3, 3, 3, 3, 1, 3, 3, 3], [3, 3, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 6, 3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 3, 3, 3, 3, 3, 3], [3, 3, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 6, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 6, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 6, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 6, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 6, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 6, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 6, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 1, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 1, 3, 3, 3, 3, 3, 3, 3, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]]}]}, "dbff022c": {"train": [{"input": [[0, 4, 4, 4, 4, 4, 0, 3, 4, 6, 9, 5], [0, 4, 0, 4, 0, 4, 0, 3, 6, 7, 2, 9], [0, 4, 4, 4, 4, 4, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0], [0, 3, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0], [0, 3, 0, 0, 3, 0, 0, 7, 7, 7, 7, 0], [0, 3, 0, 0, 3, 0, 0, 7, 0, 7, 7, 0], [0, 0, 3, 0, 3, 0, 0, 7, 0, 0, 7, 0], [0, 0, 0, 3, 0, 0, 0, 7, 7, 0, 7, 0], [0, 0, 0, 0, 0, 0, 0, 7, 7, 7, 7, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], "output": [[0, 4, 4, 4, 4, 4, 0, 3, 4, 6, 9, 5], [0, 4, 6, 4, 6, 4, 0, 3, 6, 7, 2, 9], [0, 4, 4, 4, 4, 4, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0], [0, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0], [0, 3, 3, 3, 3, 0, 0, 7, 7, 7, 7, 0], [0, 3, 3, 3, 3, 0, 0, 7, 0, 7, 7, 0], [0, 0, 3, 3, 3, 0, 0, 7, 0, 0, 7, 0], [0, 0, 0, 3, 0, 0, 0, 7, 7, 0, 7, 0], [0, 0, 0, 0, 0, 0, 0, 7, 7, 7, 7, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}, {"input": [[2, 6, 0, 0, 8, 8, 8, 8, 0, 0], [4, 5, 0, 0, 8, 0, 0, 8, 0, 0], [6, 8, 0, 0, 8, 0, 0, 8, 0, 0], [8, 1, 0, 0, 8, 8, 8, 8, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 5, 5, 5, 5, 5, 0, 4, 4, 4], [0, 5, 0, 0, 0, 5, 0, 4, 0, 4], [0, 0, 5, 5, 5, 5, 0, 4, 4, 4], [0, 0, 0, 0, 0, 0, 0, 4, 0, 4], [0, 0, 0, 0, 0, 0, 0, 4, 4, 4]], "output": [[2, 6, 0, 0, 8, 8, 8, 8, 0, 0], [4, 5, 0, 0, 8, 1, 1, 8, 0, 0], [6, 8, 0, 0, 8, 1, 1, 8, 0, 0], [8, 1, 0, 0, 8, 8, 8, 8, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 5, 5, 5, 5, 5, 0, 4, 4, 4], [0, 5, 0, 0, 0, 5, 0, 4, 5, 4], [0, 0, 5, 5, 5, 5, 0, 4, 4, 4], [0, 0, 0, 0, 0, 0, 0, 4, 5, 4], [0, 0, 0, 0, 0, 0, 0, 4, 4, 4]]}, {"input": [[0, 2, 6, 0, 4, 4, 4, 4, 4, 0, 2, 2, 2, 2, 0], [0, 7, 1, 0, 4, 0, 0, 0, 4, 0, 2, 0, 2, 2, 0], [0, 0, 0, 0, 4, 0, 0, 0, 4, 0, 2, 2, 0, 2, 0], [0, 0, 0, 0, 4, 4, 4, 4, 4, 0, 2, 2, 2, 2, 0]], "output": [[0, 2, 6, 0, 4, 4, 4, 4, 4, 0, 2, 2, 2, 2, 0], [0, 7, 1, 0, 4, 0, 0, 0, 4, 0, 2, 7, 2, 2, 0], [0, 0, 0, 0, 4, 0, 0, 0, 4, 0, 2, 2, 7, 2, 0], [0, 0, 0, 0, 4, 4, 4, 4, 4, 0, 2, 2, 2, 2, 0]]}], "test": [{"input": [[0, 1, 1, 1, 1, 1, 0, 0, 3, 3, 3, 3, 0], [0, 1, 0, 0, 0, 1, 0, 0, 3, 0, 0, 3, 0], [0, 1, 1, 0, 0, 1, 0, 0, 3, 0, 0, 3, 0], [0, 0, 1, 1, 1, 1, 0, 0, 3, 3, 3, 3, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 0, 0], [0, 0, 2, 0, 0, 0, 0, 5, 0, 0, 0, 5, 0], [0, 2, 0, 2, 0, 0, 0, 5, 0, 0, 0, 0, 5], [0, 0, 2, 0, 0, 0, 0, 0, 5, 5, 5, 5, 5], [0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 0, 0], [5, 4, 6, 7, 8, 0, 0, 0, 0, 2, 0, 2, 0], [9, 8, 3, 1, 2, 0, 0, 0, 0, 0, 2, 0, 0]]}]}, "de809cff": {"train": [{"input": [[0, 3, 3, 0, 3, 3, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0], [0, 3, 3, 3, 3, 3, 3, 0, 0, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0], [0, 3, 3, 3, 3, 3, 0, 3, 3, 3, 3, 3, 3, 0, 0, 3, 0, 0, 0, 0], [0, 3, 3, 3, 3, 3, 3, 0, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0], [0, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 0, 4, 0, 0, 0, 3, 0], [0, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0], [0, 3, 3, 0, 3, 3, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0], [0, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [0, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [0, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 0, 4, 4, 4], [0, 0, 0, 0, 0, 0, 4, 4, 4, 4, 0, 4, 4, 4, 4, 4, 4, 4, 4, 4], [0, 3, 0, 4, 0, 0, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [0, 0, 0, 0, 0, 0, 4, 4, 4, 4, 4, 4, 4, 0, 4, 4, 4, 4, 4, 4], [0, 0, 0, 0, 4, 0, 4, 4, 4, 4, 4, 4, 0, 4, 4, 4, 4, 4, 4, 4], [3, 3, 0, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 0, 4, 4], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 0, 0, 4, 4, 4, 4, 4, 4], [0, 3, 3, 3, 0, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 4, 0, 0, 0, 0, 0]], "output": [[0, 3, 4, 8, 4, 3, 4, 4, 4, 4, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0], [0, 3, 4, 4, 4, 4, 4, 8, 8, 4, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0], [0, 3, 3, 3, 3, 4, 8, 4, 4, 4, 3, 4, 4, 4, 0, 0, 0, 0, 0, 0], [0, 3, 3, 3, 3, 4, 4, 8, 4, 3, 3, 4, 8, 4, 0, 0, 0, 0, 0, 0], [0, 3, 3, 3, 3, 3, 4, 4, 4, 3, 3, 4, 4, 4, 0, 0, 0, 0, 0, 0], [0, 3, 4, 4, 4, 3, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0], [0, 3, 4, 8, 4, 3, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0], [0, 3, 4, 4, 4, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [0, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 4, 4, 3, 3, 3, 4, 4], [0, 3, 3, 3, 3, 3, 4, 4, 4, 3, 3, 3, 4, 4, 4, 3, 8, 3, 4, 4], [0, 0, 0, 0, 0, 0, 4, 4, 4, 3, 8, 3, 4, 4, 4, 3, 3, 3, 4, 4], [0, 0, 0, 0, 0, 0, 4, 4, 4, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4], [0, 0, 0, 0, 0, 0, 4, 4, 4, 4, 4, 3, 3, 8, 3, 4, 4, 4, 4, 4], [0, 4, 4, 4, 0, 0, 4, 4, 4, 4, 4, 3, 8, 3, 3, 4, 4, 4, 4, 4], [3, 4, 8, 4, 3, 3, 3, 3, 3, 3, 4, 3, 3, 3, 4, 4, 3, 3, 3, 4], [3, 4, 4, 4, 3, 3, 3, 3, 3, 3, 4, 3, 3, 3, 3, 4, 3, 8, 3, 4], [4, 4, 3, 4, 4, 4, 3, 3, 3, 3, 4, 3, 8, 8, 3, 4, 3, 3, 3, 4], [8, 4, 3, 4, 8, 4, 3, 3, 3, 3, 4, 3, 3, 3, 3, 4, 4, 4, 4, 4], [4, 4, 3, 4, 4, 4, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}, {"input": [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 2, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 2, 0], [1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 2, 0, 0, 0, 0], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 0, 1, 2, 1, 2, 1, 0, 0, 0, 0, 2, 0, 0, 0, 0], [1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [1, 1, 1, 1, 0, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [1, 1, 1, 1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 0, 2, 2, 2, 2, 2], [0, 1, 0, 0, 0, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [0, 0, 0, 0, 1, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [0, 0, 0, 0, 0, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [0, 0, 0, 0, 0, 0, 2, 2, 0, 2, 2, 2, 2, 2, 0, 2, 2, 0, 2, 2], [0, 1, 0, 0, 0, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [0, 0, 0, 0, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 0, 2, 2, 2], [0, 0, 0, 0, 0, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [0, 0, 1, 0, 0, 0, 2, 2, 2, 2, 2, 0, 2, 2, 0, 2, 2, 2, 2, 2], [0, 0, 0, 0, 0, 0, 2, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2]], "output": [[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 2, 2, 8, 2, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 2, 8, 2, 2, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 2, 8, 2, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [1, 1, 1, 2, 8, 2, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 2, 2, 2, 2], [1, 1, 1, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 8, 1, 2, 2, 2, 2], [0, 0, 0, 0, 0, 0, 2, 2, 2, 2, 2, 2, 2, 1, 1, 1, 2, 2, 2, 2], [0, 0, 0, 0, 0, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2], [0, 0, 0, 0, 0, 0, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2], [0, 0, 0, 0, 0, 0, 2, 1, 8, 1, 2, 2, 2, 1, 8, 1, 1, 8, 1, 2], [0, 0, 0, 0, 0, 0, 2, 1, 1, 1, 2, 2, 2, 1, 1, 1, 1, 1, 1, 2], [0, 0, 0, 0, 0, 0, 2, 2, 2, 2, 2, 2, 2, 2, 2, 1, 8, 1, 2, 2], [0, 0, 0, 0, 0, 0, 2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 2, 2], [0, 0, 0, 0, 0, 0, 1, 1, 1, 2, 1, 8, 1, 1, 8, 1, 2, 2, 2, 2], [0, 0, 0, 0, 0, 0, 1, 8, 1, 2, 1, 1, 1, 1, 1, 1, 2, 2, 2, 2]]}], "test": [{"input": [[0, 0, 0, 0, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6], [0, 0, 0, 0, 0, 0, 0, 0, 6, 6, 0, 6, 6, 6, 6, 6, 6, 6, 0, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6], [0, 0, 0, 0, 0, 0, 6, 0, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6], [0, 0, 1, 0, 0, 0, 0, 0, 6, 6, 1, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 0, 6, 6, 6, 6], [0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 6, 6, 6, 6, 6, 0, 6, 6, 6], [0, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 6, 6, 6, 6, 6, 6, 6, 6, 6], [0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 6, 6, 6, 6, 6, 6, 0, 6], [0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 6, 6, 6, 6, 6, 6, 6, 6, 6], [0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6], [0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 6, 6, 6, 6, 6, 6, 6, 6, 6], [0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 6, 0, 0], [0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 0, 0, 0], [0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0], [0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 6], [0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 6, 0, 0, 0, 0, 0, 0, 0], [0, 1, 1, 1, 6, 6, 6, 6, 1, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 0, 0, 0, 0], [0, 0, 0, 0, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 0, 0, 0, 0], [0, 0, 0, 0, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 0, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 0, 0, 0, 0], [0, 6, 0, 0, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 0, 6, 6, 6, 0, 6, 0, 0], [0, 0, 0, 6, 6, 6, 0, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 0, 0, 0, 6], [0, 0, 0, 0, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 0, 6, 6, 6, 6, 0, 6, 6, 6, 0, 0, 0, 0], [0, 0, 0, 0, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 0, 0, 0, 0], [1, 0, 0, 0, 0, 6, 6, 6, 6, 6, 6, 0, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 0, 0, 0], [0, 0, 0, 0, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 0, 0, 0, 0], [0, 0, 0, 0, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 0, 0, 6, 6, 6, 0, 0, 6, 0], [0, 0, 0, 0, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 6, 0, 0, 0, 0]]}]}, "e12f9a14": {"train": [{"input": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 3, 3, 8, 8, 8, 8, 8, 8, 3, 3, 8, 8, 8, 3, 8, 3, 3, 8, 8, 8], [8, 8, 8, 3, 9, 9, 3, 8, 8, 8, 8, 3, 1, 1, 3, 8, 8, 3, 4, 4, 3, 8, 8, 8], [8, 8, 8, 3, 9, 9, 3, 8, 8, 8, 8, 3, 1, 1, 3, 8, 8, 3, 4, 4, 3, 8, 8, 8], [8, 8, 8, 8, 3, 3, 8, 8, 8, 8, 8, 8, 3, 3, 8, 8, 8, 3, 8, 3, 3, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 3, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 3, 6, 6, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 3, 6, 6, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 3, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]], "output": [[8, 8, 8, 8, 8, 8, 8, 8, 9, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 4, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 9, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 4, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 9, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 4, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 9, 1, 8, 8, 8, 8, 8, 8, 8, 8, 1, 4, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 9, 1, 8, 8, 8, 8, 8, 8, 8, 8, 1, 4, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 9, 1, 8, 8, 8, 8, 8, 8, 8, 1, 4, 8, 8, 8, 8, 8], [9, 8, 8, 8, 8, 8, 8, 8, 9, 1, 8, 8, 8, 8, 8, 8, 8, 1, 4, 8, 8, 8, 8, 8], [8, 9, 8, 8, 8, 8, 8, 8, 9, 1, 8, 8, 8, 8, 8, 8, 1, 8, 4, 8, 8, 8, 8, 8], [8, 8, 9, 8, 8, 8, 8, 9, 8, 8, 1, 8, 8, 8, 8, 1, 8, 8, 4, 8, 8, 8, 8, 8], [8, 8, 8, 9, 3, 3, 9, 8, 8, 8, 8, 1, 3, 3, 1, 8, 8, 3, 4, 3, 3, 8, 8, 8], [8, 8, 8, 3, 9, 9, 3, 8, 8, 8, 8, 3, 1, 1, 3, 8, 8, 3, 4, 4, 3, 8, 8, 8], [8, 8, 8, 3, 9, 9, 3, 8, 8, 8, 8, 3, 1, 1, 3, 8, 8, 3, 4, 4, 3, 8, 8, 8], [8, 8, 8, 9, 3, 3, 9, 8, 8, 8, 8, 1, 3, 3, 1, 8, 8, 3, 4, 3, 3, 8, 8, 8], [8, 8, 9, 8, 8, 8, 8, 9, 8, 8, 1, 8, 8, 8, 8, 1, 8, 8, 4, 8, 8, 8, 8, 8], [8, 9, 8, 8, 8, 8, 8, 8, 9, 1, 8, 8, 8, 8, 8, 8, 1, 8, 4, 8, 8, 8, 8, 8], [9, 8, 8, 8, 8, 8, 8, 8, 9, 1, 8, 8, 8, 8, 8, 8, 8, 1, 4, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 9, 1, 8, 8, 8, 8, 8, 8, 8, 1, 4, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 9, 1, 8, 8, 8, 8, 8, 8, 8, 8, 1, 4, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 9, 1, 8, 8, 8, 8, 8, 8, 8, 8, 1, 4, 8, 8, 8, 8], [8, 3, 3, 3, 3, 8, 8, 8, 9, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 1, 4, 8, 8, 8], [8, 3, 6, 6, 6, 6, 6, 6, 8, 9, 1, 8, 8, 8, 8, 8, 8, 8, 8, 1, 4, 8, 8, 8], [8, 3, 6, 6, 3, 8, 8, 8, 6, 8, 9, 1, 8, 8, 8, 8, 8, 8, 8, 8, 1, 4, 8, 8], [8, 3, 3, 3, 3, 8, 8, 8, 8, 6, 8, 9, 1, 8, 8, 8, 8, 8, 8, 8, 1, 4, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 8, 9, 1, 8, 8, 8, 8, 8, 8, 8, 1, 4, 8]]}, {"input": [[8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8], [8, 3, 3, 8, 3, 8, 8, 8], [8, 3, 4, 4, 3, 8, 8, 8], [8, 3, 4, 4, 8, 8, 8, 8], [8, 8, 3, 3, 3, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8]], "output": [[8, 8, 8, 4, 8, 8, 8, 8], [8, 8, 8, 4, 8, 8, 8, 8], [8, 3, 3, 4, 3, 8, 8, 8], [8, 3, 4, 4, 3, 8, 8, 8], [8, 3, 4, 4, 4, 4, 4, 4], [8, 4, 3, 3, 3, 8, 8, 8], [4, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8]]}, {"input": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 3, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 3, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 2, 2, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 3, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 8, 3, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 4, 3, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 4, 4, 3, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 3, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 3, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 7, 7, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 3, 7, 7, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]], "output": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 4], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 4, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 4, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 2, 8, 4, 8, 8, 8], [8, 8, 8, 3, 3, 3, 3, 8, 8, 8, 8, 8, 8, 2, 8, 4, 8, 8, 8, 8], [8, 8, 8, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 8, 4, 8, 8, 8, 8, 8], [2, 2, 2, 2, 2, 2, 3, 8, 8, 8, 8, 8, 8, 4, 8, 8, 8, 8, 8, 8], [8, 8, 8, 3, 3, 3, 3, 8, 8, 8, 8, 8, 8, 4, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 4, 3, 8, 8, 8, 8, 8], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 3, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 4, 4, 3, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 3, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 3, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [7, 7, 7, 7, 7, 7, 7, 7, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 3, 7, 7, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]]}, {"input": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 3, 8, 3, 8, 8, 8, 8, 8, 8, 8, 8], [8, 3, 6, 6, 3, 8, 8, 8, 8, 8, 8, 8], [8, 3, 6, 6, 3, 8, 8, 8, 8, 8, 8, 8], [8, 8, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]], "output": [[8, 8, 6, 8, 8, 8, 6, 8, 8, 8, 8, 8], [8, 8, 6, 8, 8, 6, 8, 8, 8, 8, 8, 8], [8, 3, 6, 3, 6, 8, 8, 8, 8, 8, 8, 8], [8, 3, 6, 6, 3, 8, 8, 8, 8, 8, 8, 8], [8, 3, 6, 6, 3, 8, 8, 8, 8, 8, 8, 8], [8, 6, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8], [6, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]]}], "test": [{"input": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 8, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 2, 2, 8, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 2, 2, 8, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 8, 8, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 8, 8, 0, 0, 0, 0, 8, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 8, 3, 3, 8, 0, 0, 8, 4, 4, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 8, 3, 3, 8, 0, 0, 8, 4, 4, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 8, 0, 8, 0, 0, 0, 0, 8, 0, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 8, 8, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 7, 7, 8, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 7, 7, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 8, 0, 8, 8, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}, {"input": [[8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 3, 8, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 2, 2, 3, 8, 8, 8, 8, 8, 8, 8, 3, 4, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 3, 2, 2, 3, 8, 8, 8, 8, 8, 8, 8, 3, 4, 4, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 3, 8, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 8, 3, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 6, 6, 3, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 6, 6, 3, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 3, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 3, 3, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 7, 7, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 7, 7, 3, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 3, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [3, 3, 8, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [3, 9, 9, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [3, 9, 9, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8]]}]}, "e87109e9": {"train": [{"input": [[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 1, 0, 0, 0, 5, 5, 6, 0, 0, 0, 5, 5, 4, 0, 0, 0, 5, 5, 2, 0, 0, 0, 5], [5, 1, 0, 0, 0, 5, 5, 6, 0, 0, 0, 5, 5, 4, 0, 0, 0, 5, 5, 2, 0, 0, 0, 5], [5, 1, 0, 0, 0, 5, 5, 6, 0, 0, 0, 5, 5, 4, 0, 0, 0, 5, 5, 2, 0, 0, 0, 5], [5, 1, 0, 0, 0, 5, 5, 6, 0, 0, 0, 5, 5, 4, 0, 0, 0, 5, 5, 2, 0, 0, 0, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [1, 1, 1, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [1, 1, 1, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 3, 3], [1, 1, 1, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 3, 3], [1, 1, 1, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 3, 3], [1, 1, 1, 1, 3, 3, 3, 3, 8, 8, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 3, 3], [1, 1, 1, 1, 3, 3, 3, 3, 8, 8, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 3, 3], [1, 1, 1, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 3, 3], [1, 1, 1, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 3, 3], [1, 1, 1, 1, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3]], "output": [[3, 3, 3, 3, 3, 3, 3, 3, 8, 8, 3, 3, 3, 3, 3, 3, 8, 8, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 8, 8, 3, 3, 3, 3, 3, 3, 8, 8, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 3, 3, 3, 3, 8, 8, 3, 3, 3, 3, 3, 3, 8, 8, 3, 3, 3, 3, 3, 3], [1, 1, 1, 1, 3, 3, 3, 3, 8, 8, 3, 3, 3, 3, 3, 3, 8, 8, 3, 3, 3, 3, 3, 3], [1, 1, 1, 1, 3, 3, 3, 3, 8, 8, 3, 3, 3, 3, 3, 3, 8, 8, 4, 4, 4, 4, 3, 3], [1, 1, 1, 1, 3, 3, 3, 3, 8, 8, 3, 3, 3, 3, 3, 3, 8, 8, 4, 4, 4, 4, 3, 3], [1, 1, 1, 1, 3, 3, 3, 3, 8, 8, 3, 3, 3, 3, 3, 3, 8, 8, 4, 4, 4, 4, 3, 3], [1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 4, 4, 4, 3, 3], [1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 4, 4, 4, 4, 3, 3], [1, 1, 1, 1, 8, 8, 3, 3, 8, 8, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 3, 3], [1, 1, 1, 1, 8, 8, 3, 3, 8, 8, 3, 3, 3, 3, 3, 3, 3, 3, 4, 4, 4, 4, 3, 3], [1, 1, 1, 1, 8, 8, 3, 3, 8, 8, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 8, 8, 3, 3, 8, 8, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 8, 8, 3, 3, 8, 8, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [3, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [3, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3], [3, 3, 3, 3, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3, 3, 3, 3, 3, 3, 3]]}, {"input": [[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 0, 0, 0, 2, 5, 5, 4, 0, 0, 0, 5, 5, 0, 0, 0, 6, 5, 5, 3, 0, 0, 0, 5], [5, 0, 0, 0, 2, 5, 5, 4, 0, 0, 0, 5, 5, 0, 0, 0, 6, 5, 5, 3, 0, 0, 0, 5], [5, 0, 0, 0, 2, 5, 5, 4, 0, 0, 0, 5, 5, 0, 0, 0, 6, 5, 5, 3, 0, 0, 0, 5], [5, 0, 0, 0, 2, 5, 5, 4, 0, 0, 0, 5, 5, 0, 0, 0, 6, 5, 5, 3, 0, 0, 0, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 6, 6, 6, 6, 6, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 6, 6, 6, 6, 6, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 3, 3, 1, 1, 1, 1], [2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 3, 3, 1, 1, 1, 1], [2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 3, 3, 1, 1, 1, 1], [2, 2, 2, 2, 1, 1, 1, 1, 8, 8, 1, 1, 1, 1, 1, 1, 3, 3, 3, 3, 1, 1, 1, 1], [2, 2, 2, 2, 1, 1, 1, 1, 8, 8, 1, 1, 1, 1, 1, 1, 3, 3, 3, 3, 1, 1, 1, 1], [2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 3, 3, 3, 3, 1, 1, 1, 1], [2, 2, 2, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], "output": [[1, 1, 1, 1, 8, 8, 1, 1, 8, 8, 1, 1, 1, 6, 6, 6, 6, 6, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 8, 8, 1, 1, 8, 8, 1, 1, 1, 6, 6, 6, 6, 6, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 8, 8, 1, 1, 8, 8, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [1, 1, 1, 1, 8, 8, 1, 1, 8, 8, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [1, 1, 1, 1, 8, 8, 1, 1, 8, 8, 1, 1, 1, 1, 8, 8, 1, 1, 1, 1, 1, 1, 1, 1], [2, 2, 2, 2, 8, 8, 1, 1, 8, 8, 1, 1, 1, 1, 8, 8, 3, 3, 3, 3, 1, 1, 1, 1], [2, 2, 2, 2, 8, 8, 1, 1, 8, 8, 1, 1, 1, 1, 8, 8, 3, 3, 3, 3, 1, 1, 1, 1], [2, 2, 2, 2, 8, 8, 1, 1, 8, 8, 1, 1, 1, 1, 8, 8, 3, 3, 3, 3, 1, 1, 1, 1], [2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 3, 3, 1, 1, 1, 1], [2, 2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 3, 3, 3, 3, 1, 1, 1, 1], [2, 2, 2, 2, 1, 1, 1, 1, 8, 8, 1, 1, 1, 1, 1, 1, 3, 3, 3, 3, 1, 1, 1, 1], [2, 2, 2, 2, 1, 1, 1, 1, 8, 8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 8, 8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 8, 8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [1, 1, 1, 1, 1, 1, 1, 1, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [1, 1, 1, 1, 1, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 4, 4, 4, 4, 4, 4, 4, 4, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]]}, {"input": [[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 0, 0, 0, 4, 5, 5, 6, 0, 0, 0, 5, 5, 3, 0, 0, 0, 5, 5, 0, 0, 0, 2, 5], [5, 0, 0, 0, 4, 5, 5, 6, 0, 0, 0, 5, 5, 3, 0, 0, 0, 5, 5, 0, 0, 0, 2, 5], [5, 0, 0, 0, 4, 5, 5, 6, 0, 0, 0, 5, 5, 3, 0, 0, 0, 5, 5, 0, 0, 0, 2, 5], [5, 0, 0, 0, 4, 5, 5, 6, 0, 0, 0, 5, 5, 3, 0, 0, 0, 5, 5, 0, 0, 0, 2, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [7, 7, 7, 7, 7, 7, 7, 7, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 7, 7, 7, 7, 7, 7], [7, 7, 7, 7, 7, 7, 7, 7, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 7, 7, 7, 7, 7, 7], [7, 7, 7, 7, 7, 7, 7, 7, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 7, 7, 7, 7, 7, 7], [7, 7, 7, 7, 7, 7, 7, 7, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 7, 7, 6, 6, 6, 6], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 6, 6, 6, 6], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 6, 6, 6, 6], [7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 6, 6, 6, 6], [2, 2, 2, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 6, 6, 6, 6], [2, 2, 2, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 6, 6, 6, 6], [2, 2, 2, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 6, 6, 6, 6], [2, 2, 2, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 6, 6, 6, 6], [2, 2, 2, 7, 7, 7, 7, 7, 7, 7, 7, 7, 8, 8, 7, 7, 7, 7, 7, 7, 6, 6, 6, 6], [2, 2, 2, 7, 7, 7, 7, 7, 7, 7, 7, 7, 8, 8, 7, 7, 7, 7, 7, 7, 6, 6, 6, 6], [2, 2, 2, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 6, 6, 6, 6], [2, 2, 2, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [2, 2, 2, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [2, 2, 2, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 7, 7], [2, 2, 2, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 7, 7], [2, 2, 2, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 7, 7]], "output": [[7, 7, 7, 8, 8, 7, 7, 7, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 8, 7, 7, 7, 7], [7, 7, 7, 8, 8, 7, 7, 7, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 8, 7, 7, 7, 7], [7, 7, 7, 8, 8, 7, 7, 7, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 8, 7, 7, 7, 7], [7, 7, 7, 8, 8, 7, 7, 7, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 8, 8, 6, 6, 6, 6], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 7, 7, 7, 7, 8, 8, 6, 6, 6, 6], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 7, 7, 7, 7, 8, 8, 6, 6, 6, 6], [7, 7, 7, 8, 8, 7, 7, 7, 7, 7, 7, 7, 8, 8, 7, 7, 7, 7, 8, 8, 6, 6, 6, 6], [2, 2, 2, 8, 8, 7, 7, 7, 7, 7, 7, 7, 8, 8, 7, 7, 7, 7, 8, 8, 6, 6, 6, 6], [2, 2, 2, 8, 8, 7, 7, 7, 7, 7, 7, 7, 8, 8, 7, 7, 7, 7, 8, 8, 6, 6, 6, 6], [2, 2, 2, 8, 8, 7, 7, 7, 7, 7, 7, 7, 8, 8, 7, 7, 7, 7, 8, 8, 6, 6, 6, 6], [2, 2, 2, 8, 8, 7, 7, 7, 7, 7, 7, 7, 8, 8, 7, 7, 7, 7, 8, 8, 6, 6, 6, 6], [2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 6, 6, 6], [2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 6, 6, 6, 6], [2, 2, 2, 8, 8, 7, 7, 7, 7, 7, 7, 7, 8, 8, 7, 7, 7, 7, 7, 7, 6, 6, 6, 6], [2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [2, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 7, 7, 7, 7, 7, 7, 7, 7, 7, 7], [2, 2, 2, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 7, 7], [2, 2, 2, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 7, 7], [2, 2, 2, 7, 7, 7, 7, 7, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 7, 7]]}], "test": [{"input": [[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 1, 0, 0, 0, 5, 5, 0, 0, 0, 3, 5, 5, 2, 0, 0, 0, 5, 5, 4, 0, 0, 0, 5], [5, 1, 0, 0, 0, 5, 5, 0, 0, 0, 3, 5, 5, 2, 0, 0, 0, 5, 5, 4, 0, 0, 0, 5], [5, 1, 0, 0, 0, 5, 5, 0, 0, 0, 3, 5, 5, 2, 0, 0, 0, 5, 5, 4, 0, 0, 0, 5], [5, 1, 0, 0, 0, 5, 5, 0, 0, 0, 3, 5, 5, 2, 0, 0, 0, 5, 5, 4, 0, 0, 0, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3], [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3], [2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 2, 3, 3, 3], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 3, 3, 3], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 3, 3, 3], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 3, 3, 3], [1, 1, 1, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 3, 3, 3], [1, 1, 1, 9, 9, 9, 9, 9, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 3, 3, 3], [1, 1, 1, 9, 9, 9, 9, 9, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 3, 3, 3], [1, 1, 1, 9, 9, 9, 9, 9, 8, 8, 8, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 3, 3, 3], [1, 1, 1, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 3, 3, 3], [1, 1, 1, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 3, 3, 3], [1, 1, 1, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 3, 3, 3], [1, 1, 1, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [1, 1, 1, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [1, 1, 1, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 4, 4, 4, 4, 4, 4, 4, 4, 4, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 4, 4, 4, 4, 4, 4, 4, 4, 4, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 4, 4, 4, 4, 4, 4, 4, 4, 4, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9], [9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9]]}]}, "edb79dae": {"train": [{"input": [[1, 1, 4, 4, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [1, 1, 1, 4, 8, 8, 1, 8, 1, 8, 8, 3, 8, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [1, 1, 4, 4, 8, 8, 1, 1, 1, 8, 8, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 1, 8, 8, 8, 3, 3, 3, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [3, 3, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [3, 3, 3, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [3, 3, 2, 2, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8], [8, 8, 8, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 8, 8], [8, 8, 8, 5, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 5, 8, 8], [8, 8, 8, 5, 8, 3, 3, 3, 8, 1, 1, 1, 8, 3, 3, 3, 8, 1, 1, 1, 8, 5, 8, 8], [8, 8, 8, 5, 8, 3, 3, 3, 8, 1, 1, 1, 8, 3, 3, 3, 8, 1, 1, 1, 8, 5, 8, 8], [8, 8, 8, 5, 8, 3, 3, 3, 8, 1, 1, 1, 8, 3, 3, 3, 8, 1, 1, 1, 8, 5, 8, 8], [8, 8, 8, 5, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 5, 8, 8], [8, 8, 8, 5, 8, 3, 3, 3, 8, 1, 1, 1, 8, 8, 8, 8, 8, 3, 3, 3, 8, 5, 8, 8], [8, 8, 8, 5, 8, 3, 3, 3, 8, 1, 1, 1, 8, 8, 8, 8, 8, 3, 3, 3, 8, 5, 8, 8], [8, 8, 8, 5, 8, 3, 3, 3, 8, 1, 1, 1, 8, 8, 8, 8, 8, 3, 3, 3, 8, 5, 8, 8], [8, 8, 8, 5, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 5, 8, 8], [8, 8, 8, 5, 8, 8, 8, 8, 8, 3, 3, 3, 8, 1, 1, 1, 8, 1, 1, 1, 8, 5, 8, 8], [8, 8, 8, 5, 8, 8, 8, 8, 8, 3, 3, 3, 8, 1, 1, 1, 8, 1, 1, 1, 8, 5, 8, 8], [8, 8, 8, 5, 8, 8, 8, 8, 8, 3, 3, 3, 8, 1, 1, 1, 8, 1, 1, 1, 8, 5, 8, 8], [8, 8, 8, 5, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 5, 8, 8], [8, 8, 8, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 8, 8]], "output": [[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 5], [5, 8, 2, 8, 2, 8, 4, 8, 4, 8, 2, 8, 2, 8, 4, 8, 4, 8, 5], [5, 8, 2, 2, 2, 8, 4, 4, 4, 8, 2, 2, 2, 8, 4, 4, 4, 8, 5], [5, 8, 2, 2, 2, 8, 8, 4, 8, 8, 2, 2, 2, 8, 8, 4, 8, 8, 5], [5, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 5], [5, 8, 2, 8, 2, 8, 4, 8, 4, 8, 8, 8, 8, 8, 2, 8, 2, 8, 5], [5, 8, 2, 2, 2, 8, 4, 4, 4, 8, 8, 8, 8, 8, 2, 2, 2, 8, 5], [5, 8, 2, 2, 2, 8, 8, 4, 8, 8, 8, 8, 8, 8, 2, 2, 2, 8, 5], [5, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 5], [5, 8, 8, 8, 8, 8, 2, 8, 2, 8, 4, 8, 4, 8, 4, 8, 4, 8, 5], [5, 8, 8, 8, 8, 8, 2, 2, 2, 8, 4, 4, 4, 8, 4, 4, 4, 8, 5], [5, 8, 8, 8, 8, 8, 2, 2, 2, 8, 8, 4, 8, 8, 8, 4, 8, 8, 5], [5, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 8, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5]]}, {"input": [[2, 2, 4, 4, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [2, 2, 2, 4, 1, 1, 2, 2, 2, 2, 1, 3, 3, 3, 3, 1, 1, 6, 6, 6, 6, 1, 1], [2, 2, 4, 4, 1, 1, 2, 2, 2, 2, 1, 1, 3, 3, 1, 1, 1, 6, 6, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 2, 1, 2, 2, 1, 3, 3, 3, 3, 1, 1, 6, 6, 1, 1, 1, 1], [3, 3, 8, 8, 1, 1, 2, 1, 2, 2, 1, 3, 1, 1, 3, 1, 1, 6, 6, 6, 6, 1, 1], [3, 3, 3, 8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [3, 3, 8, 8, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [6, 6, 7, 7, 1, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [6, 6, 6, 7, 1, 5, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 5], [6, 6, 7, 7, 1, 5, 1, 2, 2, 2, 2, 1, 6, 6, 6, 6, 1, 3, 3, 3, 3, 1, 5], [1, 1, 1, 1, 1, 5, 1, 2, 2, 2, 2, 1, 6, 6, 6, 6, 1, 3, 3, 3, 3, 1, 5], [1, 1, 1, 1, 1, 5, 1, 2, 2, 2, 2, 1, 6, 6, 6, 6, 1, 3, 3, 3, 3, 1, 5], [1, 1, 1, 1, 1, 5, 1, 2, 2, 2, 2, 1, 6, 6, 6, 6, 1, 3, 3, 3, 3, 1, 5], [1, 1, 1, 1, 1, 5, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 5], [1, 1, 1, 1, 1, 5, 1, 3, 3, 3, 3, 1, 2, 2, 2, 2, 1, 6, 6, 6, 6, 1, 5], [1, 1, 1, 1, 1, 5, 1, 3, 3, 3, 3, 1, 2, 2, 2, 2, 1, 6, 6, 6, 6, 1, 5], [1, 1, 1, 1, 1, 5, 1, 3, 3, 3, 3, 1, 2, 2, 2, 2, 1, 6, 6, 6, 6, 1, 5], [1, 1, 1, 1, 1, 5, 1, 3, 3, 3, 3, 1, 2, 2, 2, 2, 1, 6, 6, 6, 6, 1, 5], [1, 1, 1, 1, 1, 5, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 5], [1, 1, 1, 1, 1, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], "output": [[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [5, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 5], [5, 1, 4, 4, 4, 4, 1, 7, 7, 7, 7, 1, 8, 8, 8, 8, 1, 5], [5, 1, 4, 4, 4, 4, 1, 7, 7, 1, 1, 1, 1, 8, 8, 1, 1, 5], [5, 1, 4, 1, 4, 4, 1, 7, 7, 1, 1, 1, 8, 8, 8, 8, 1, 5], [5, 1, 4, 1, 4, 4, 1, 7, 7, 7, 7, 1, 8, 1, 1, 8, 1, 5], [5, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 5], [5, 1, 8, 8, 8, 8, 1, 4, 4, 4, 4, 1, 7, 7, 7, 7, 1, 5], [5, 1, 1, 8, 8, 1, 1, 4, 4, 4, 4, 1, 7, 7, 1, 1, 1, 5], [5, 1, 8, 8, 8, 8, 1, 4, 1, 4, 4, 1, 7, 7, 1, 1, 1, 5], [5, 1, 8, 1, 1, 8, 1, 4, 1, 4, 4, 1, 7, 7, 7, 7, 1, 5], [5, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 5], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5]]}], "test": [{"input": [[3, 3, 9, 9, 4, 9, 9, 2, 2, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 2, 2, 1, 1, 4], [3, 3, 3, 9, 4, 9, 9, 9, 2, 4, 4, 3, 3, 4, 4, 3, 4, 4, 8, 8, 4, 4, 8, 4, 4, 2, 2, 2, 1, 4], [3, 3, 9, 9, 4, 9, 9, 2, 2, 4, 4, 3, 4, 4, 4, 3, 4, 4, 8, 8, 4, 4, 8, 4, 4, 2, 2, 1, 1, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 3, 4, 3, 3, 3, 4, 4, 8, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [1, 1, 6, 6, 4, 8, 8, 7, 7, 4, 4, 3, 3, 3, 3, 3, 4, 4, 8, 8, 4, 4, 8, 4, 4, 4, 4, 4, 4, 4], [1, 1, 1, 6, 4, 8, 8, 8, 7, 4, 4, 3, 3, 4, 4, 3, 4, 4, 8, 8, 8, 8, 8, 4, 4, 4, 4, 4, 4, 4], [1, 1, 6, 6, 4, 8, 8, 7, 7, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 9, 9, 9, 9, 9, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 9, 4, 4, 4, 4, 4, 4], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 4, 4, 9, 9, 9, 9, 9, 4, 4], [5, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 5, 4, 4, 4, 4, 9, 9, 4, 4, 4], [5, 4, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 9, 9, 9, 9, 9, 4, 5, 4, 4, 9, 9, 9, 9, 9, 4, 4], [5, 4, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 9, 9, 9, 9, 9, 4, 5, 4, 4, 4, 4, 4, 4, 4, 4, 4], [5, 4, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 9, 9, 9, 9, 9, 4, 5, 4, 4, 2, 2, 2, 2, 2, 4, 4], [5, 4, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 9, 9, 9, 9, 9, 4, 5, 4, 4, 2, 2, 2, 2, 2, 4, 4], [5, 4, 3, 3, 3, 3, 3, 4, 4, 4, 4, 4, 4, 4, 9, 9, 9, 9, 9, 4, 5, 4, 4, 2, 4, 4, 4, 2, 4, 4], [5, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 5, 4, 4, 2, 4, 4, 4, 2, 4, 4], [5, 4, 3, 3, 3, 3, 3, 4, 2, 2, 2, 2, 2, 4, 1, 1, 1, 1, 1, 4, 5, 4, 4, 2, 4, 2, 2, 2, 4, 4], [5, 4, 3, 3, 3, 3, 3, 4, 2, 2, 2, 2, 2, 4, 1, 1, 1, 1, 1, 4, 5, 4, 4, 4, 4, 4, 4, 4, 4, 4], [5, 4, 3, 3, 3, 3, 3, 4, 2, 2, 2, 2, 2, 4, 1, 1, 1, 1, 1, 4, 5, 4, 4, 4, 4, 4, 4, 4, 4, 4], [5, 4, 3, 3, 3, 3, 3, 4, 2, 2, 2, 2, 2, 4, 1, 1, 1, 1, 1, 4, 5, 4, 4, 1, 4, 4, 4, 4, 4, 4], [5, 4, 3, 3, 3, 3, 3, 4, 2, 2, 2, 2, 2, 4, 1, 1, 1, 1, 1, 4, 5, 4, 4, 1, 1, 1, 1, 1, 4, 4], [5, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 5, 4, 4, 1, 1, 4, 4, 1, 4, 4], [5, 4, 4, 4, 4, 4, 4, 4, 8, 8, 8, 8, 8, 4, 2, 2, 2, 2, 2, 4, 5, 4, 4, 1, 1, 1, 1, 1, 4, 4], [5, 4, 4, 4, 4, 4, 4, 4, 8, 8, 8, 8, 8, 4, 2, 2, 2, 2, 2, 4, 5, 4, 4, 1, 4, 4, 4, 4, 4, 4], [5, 4, 4, 4, 4, 4, 4, 4, 8, 8, 8, 8, 8, 4, 2, 2, 2, 2, 2, 4, 5, 4, 4, 4, 4, 4, 4, 4, 4, 4], [5, 4, 4, 4, 4, 4, 4, 4, 8, 8, 8, 8, 8, 4, 2, 2, 2, 2, 2, 4, 5, 4, 4, 4, 4, 4, 4, 4, 4, 4], [5, 4, 4, 4, 4, 4, 4, 4, 8, 8, 8, 8, 8, 4, 2, 2, 2, 2, 2, 4, 5, 4, 4, 4, 4, 4, 4, 4, 4, 4], [5, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 5, 4, 4, 4, 4, 4, 4, 4, 4, 4], [5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 4, 4, 4, 4, 4, 4, 4, 4, 4]]}]}, "eee78d87": {"train": [{"input": [[7, 7, 7, 7, 7, 7], [7, 7, 7, 7, 7, 7], [7, 7, 7, 7, 7, 7], [7, 7, 7, 7, 3, 7], [7, 7, 7, 3, 3, 3], [7, 7, 7, 7, 3, 7]], "output": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 7, 7, 0, 7, 7, 0, 7, 7, 0, 7, 7, 0, 7, 7, 0], [0, 7, 7, 0, 7, 7, 0, 7, 7, 0, 7, 7, 0, 7, 7, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 7, 7, 0, 7, 7, 0, 7, 7, 0, 7, 7, 0, 7, 7, 0], [0, 7, 7, 0, 7, 7, 9, 7, 7, 9, 7, 7, 0, 7, 7, 0], [0, 0, 0, 0, 0, 9, 9, 9, 9, 9, 9, 0, 0, 0, 0, 0], [0, 7, 7, 0, 7, 7, 9, 7, 7, 9, 7, 7, 0, 7, 7, 0], [0, 7, 7, 0, 7, 7, 9, 7, 7, 9, 7, 7, 0, 7, 7, 0], [0, 0, 0, 0, 0, 9, 9, 9, 9, 9, 9, 0, 0, 0, 0, 0], [0, 7, 7, 0, 7, 7, 9, 7, 7, 9, 7, 7, 0, 7, 7, 0], [0, 7, 7, 0, 7, 7, 0, 7, 7, 0, 7, 7, 0, 7, 7, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 7, 7, 0, 7, 7, 0, 7, 7, 0, 7, 7, 0, 7, 7, 0], [0, 7, 7, 0, 7, 7, 0, 7, 7, 0, 7, 7, 0, 7, 7, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}, {"input": [[7, 7, 7, 7, 7, 7], [7, 1, 7, 1, 7, 7], [7, 1, 1, 1, 7, 7], [7, 1, 7, 1, 7, 7], [7, 7, 7, 7, 7, 7], [7, 7, 7, 7, 7, 7]], "output": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [7, 0, 0, 7, 0, 0, 7, 0, 0, 7, 0, 0, 7, 0, 0, 7], [7, 0, 0, 7, 0, 0, 7, 0, 0, 7, 0, 0, 7, 0, 0, 7], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [7, 0, 0, 7, 0, 0, 7, 0, 0, 7, 0, 0, 7, 0, 0, 7], [7, 0, 0, 7, 0, 9, 7, 9, 9, 7, 9, 0, 7, 0, 0, 7], [0, 0, 0, 0, 0, 9, 9, 9, 9, 9, 9, 0, 0, 0, 0, 0], [7, 0, 0, 7, 0, 9, 7, 9, 9, 7, 9, 0, 7, 0, 0, 7], [7, 0, 0, 7, 0, 9, 7, 9, 9, 7, 9, 0, 7, 0, 0, 7], [0, 0, 0, 0, 0, 9, 9, 9, 9, 9, 9, 0, 0, 0, 0, 0], [7, 0, 0, 7, 0, 9, 7, 9, 9, 7, 9, 0, 7, 0, 0, 7], [7, 0, 0, 7, 0, 0, 7, 0, 0, 7, 0, 0, 7, 0, 0, 7], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [7, 0, 0, 7, 0, 0, 7, 0, 0, 7, 0, 0, 7, 0, 0, 7], [7, 0, 0, 7, 0, 0, 7, 0, 0, 7, 0, 0, 7, 0, 0, 7], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}, {"input": [[7, 7, 7, 7, 7, 7], [7, 6, 7, 6, 7, 7], [7, 7, 6, 7, 7, 7], [7, 6, 7, 6, 7, 7], [7, 7, 7, 7, 7, 7], [7, 7, 7, 7, 7, 7]], "output": [[0, 7, 7, 0, 7, 7, 0, 7, 7, 0, 7, 7, 0, 7, 7, 0], [7, 0, 0, 7, 0, 0, 7, 0, 0, 7, 0, 0, 7, 0, 0, 7], [7, 0, 0, 7, 0, 0, 7, 0, 0, 7, 0, 0, 7, 0, 0, 7], [0, 7, 7, 0, 7, 7, 0, 7, 7, 0, 7, 7, 0, 7, 7, 0], [7, 0, 0, 7, 0, 0, 7, 0, 0, 7, 0, 0, 7, 0, 0, 7], [7, 0, 0, 7, 0, 9, 7, 9, 9, 7, 9, 0, 7, 0, 0, 7], [0, 7, 7, 0, 7, 7, 9, 7, 7, 9, 7, 7, 0, 7, 7, 0], [7, 0, 0, 7, 0, 9, 7, 9, 9, 7, 9, 0, 7, 0, 0, 7], [7, 0, 0, 7, 0, 9, 7, 9, 9, 7, 9, 0, 7, 0, 0, 7], [0, 7, 7, 0, 7, 7, 9, 7, 7, 9, 7, 7, 0, 7, 7, 0], [7, 0, 0, 7, 0, 9, 7, 9, 9, 7, 9, 0, 7, 0, 0, 7], [7, 0, 0, 7, 0, 0, 7, 0, 0, 7, 0, 0, 7, 0, 0, 7], [0, 7, 7, 0, 7, 7, 0, 7, 7, 0, 7, 7, 0, 7, 7, 0], [7, 0, 0, 7, 0, 0, 7, 0, 0, 7, 0, 0, 7, 0, 0, 7], [7, 0, 0, 7, 0, 0, 7, 0, 0, 7, 0, 0, 7, 0, 0, 7], [0, 7, 7, 0, 7, 7, 0, 7, 7, 0, 7, 7, 0, 7, 7, 0]]}], "test": [{"input": [[7, 7, 2, 7, 2, 7], [7, 7, 7, 7, 7, 7], [7, 7, 2, 7, 2, 7], [7, 7, 7, 7, 7, 7], [7, 7, 7, 7, 7, 7], [7, 7, 7, 7, 7, 7]]}]}, "f560132c": {"train": [{"input": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 2, 2, 2, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 2, 1, 5, 2, 0, 0, 0, 0, 0, 0, 0, 7, 7, 7, 7, 0, 0, 0, 0], [0, 2, 8, 9, 0, 0, 0, 0, 0, 0, 0, 0, 7, 7, 7, 7, 0, 0, 0, 0], [0, 2, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 7, 7, 7, 0, 0, 0, 0], [0, 2, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 4, 4, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 4, 4, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 4, 4, 4, 4, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0], [0, 0, 0, 0, 4, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0, 0], [0, 0, 0, 0, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 0, 0, 0], [0, 0, 0, 0, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], "output": [[1, 1, 1, 1, 1, 5, 5, 5], [1, 1, 1, 1, 9, 5, 5, 5], [1, 1, 1, 9, 9, 5, 5, 5], [1, 1, 9, 9, 9, 5, 5, 5], [1, 9, 9, 9, 9, 9, 9, 9], [8, 8, 8, 9, 9, 9, 9, 9], [8, 8, 8, 9, 9, 9, 9, 9], [8, 8, 8, 8, 8, 9, 9, 9]]}, {"input": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 5, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 5, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 0, 0, 0], [0, 0, 0, 0, 6, 6, 6, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 0, 0, 0], [0, 0, 0, 0, 6, 6, 6, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 0, 0, 0], [0, 0, 0, 0, 6, 2, 4, 6, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 6, 8, 3, 6, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 6, 6, 6, 6, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 6, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 6, 6, 6, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 6, 6, 6, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 0], [0, 9, 9, 9, 9, 9, 9, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0], [0, 9, 9, 9, 9, 9, 9, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0], [0, 9, 9, 9, 9, 9, 9, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0], [0, 9, 9, 9, 9, 9, 9, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0], [0, 9, 9, 0, 0, 9, 9, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], "output": [[2, 2, 2, 4, 4, 4, 4, 4, 4, 4], [2, 2, 2, 4, 4, 4, 4, 4, 4, 4], [2, 2, 2, 2, 2, 4, 4, 4, 4, 4], [2, 2, 2, 2, 2, 4, 4, 4, 4, 4], [2, 2, 2, 2, 2, 3, 3, 3, 3, 3], [8, 8, 8, 2, 2, 3, 3, 3, 3, 3], [8, 8, 2, 2, 2, 2, 3, 3, 3, 3], [8, 8, 2, 2, 2, 2, 3, 3, 3, 3], [8, 8, 8, 8, 8, 3, 3, 3, 3, 3], [8, 8, 8, 8, 8, 3, 3, 3, 3, 3]]}], "test": [{"input": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 4, 4, 4, 4], [0, 1, 4, 3, 5, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 4, 4, 4, 4, 4, 4], [0, 1, 7, 0, 6, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 4, 4, 4, 4, 4, 4], [0, 1, 2, 9, 8, 0, 0, 0, 0, 0, 0, 7, 7, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4], [0, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 7, 7, 7, 7, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 7, 7, 7, 7, 7, 0, 0, 0, 0, 0, 0, 0, 0, 6, 6, 0, 0, 0, 0], [0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 7, 7, 7, 7, 7, 0, 0, 0, 0, 0, 0, 0, 0, 6, 6, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 7, 7, 7, 7, 0, 0, 0, 0, 0, 6, 6, 6, 6, 6, 6, 6, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 7, 0, 0, 0, 0, 0, 6, 6, 6, 6, 6, 6, 6, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 7, 0, 0, 0, 0, 0, 6, 6, 6, 6, 6, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 6, 6, 6, 6, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 5, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 0, 0, 5, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 9, 9, 9, 9, 9, 0, 0, 0, 0, 0, 5, 5, 0, 0, 0, 3, 3, 3, 0, 0, 3, 3, 0, 0, 0, 0, 0], [0, 0, 0, 9, 9, 9, 9, 9, 0, 0, 0, 0, 0, 5, 5, 0, 0, 0, 3, 3, 3, 0, 0, 3, 3, 3, 3, 0, 0, 0], [0, 9, 9, 9, 9, 9, 9, 9, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 3, 3, 3, 3, 3, 3, 3, 3, 0, 0, 0], [0, 9, 9, 9, 9, 9, 9, 9, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 9, 9, 9, 9, 9, 9, 9, 0, 0, 0, 8, 8, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 9, 9, 9, 9, 9, 9, 9, 0, 0, 0, 8, 8, 8, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 9, 9, 9, 9, 9, 9, 9, 0, 0, 0, 8, 8, 8, 0, 0, 0, 0, 2, 2, 0, 0, 2, 2, 2, 0, 0, 0, 0, 0], [0, 0, 0, 0, 9, 9, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 0, 0, 2, 2, 2, 0, 0, 0, 0, 0], [0, 0, 0, 0, 9, 9, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 2, 2, 2, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 2, 2, 2, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}, {"input": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 9, 9, 9, 9, 9, 9, 9, 9, 9, 9, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 9, 3, 6, 9, 9, 9, 9, 9, 9, 9, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 9, 4, 8, 9, 0, 0, 0, 0, 9, 9, 0, 0, 0, 0, 0, 0, 5, 5, 5, 0, 0, 0, 0], [0, 0, 0, 9, 9, 9, 9, 0, 0, 0, 0, 9, 9, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 0, 0, 0], [0, 0, 0, 9, 9, 0, 0, 0, 0, 0, 0, 9, 9, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 5, 0, 0], [0, 0, 0, 9, 9, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 5, 5, 5, 5, 0], [0, 0, 0, 9, 9, 9, 9, 9, 9, 9, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 0, 0, 0, 0, 0], [0, 0, 0, 9, 9, 9, 9, 9, 9, 9, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 0, 0, 5, 5, 5, 5, 5, 5], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 0, 0, 5, 5, 5, 5, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 0, 0, 5, 5, 5, 5, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 5, 5, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 7, 7, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 7, 7, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 2, 0, 0, 0, 0, 0, 0], [0, 0, 0, 7, 7, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 2, 2, 0, 0, 0, 0, 0], [0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 2, 2, 2, 0, 0, 0, 0], [0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 2, 2, 2, 2, 2, 2, 0, 0, 0], [0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}]}, "f931b4a8": {"train": [{"input": [[1, 1, 1, 1, 3, 3, 3, 3], [1, 1, 1, 1, 0, 0, 0, 0], [1, 1, 1, 1, 0, 0, 0, 0], [1, 1, 1, 1, 0, 0, 0, 0], [5, 5, 5, 5, 6, 0, 6, 0], [5, 5, 5, 5, 0, 6, 0, 6], [5, 5, 5, 5, 6, 0, 6, 0], [5, 5, 5, 5, 0, 6, 0, 6]], "output": [[6, 5, 6, 5], [5, 6, 5, 6], [6, 5, 6, 5], [5, 6, 5, 6], [6, 5, 6, 5], [5, 6, 5, 6], [6, 5, 6, 5], [5, 6, 5, 6], [6, 5, 6, 5], [5, 6, 5, 6], [6, 5, 6, 5], [5, 6, 5, 6], [6, 5, 6, 5], [5, 6, 5, 6], [6, 5, 6, 5], [5, 6, 5, 6]]}, {"input": [[8, 8, 8, 8, 6, 6, 6, 6], [8, 8, 8, 8, 6, 6, 6, 6], [8, 8, 8, 8, 6, 6, 6, 6], [8, 8, 8, 8, 6, 6, 6, 6], [1, 1, 1, 1, 4, 4, 4, 4], [5, 5, 5, 5, 4, 0, 0, 4], [1, 1, 1, 1, 4, 0, 0, 4], [5, 5, 5, 5, 4, 4, 4, 4]], "output": [[4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 1, 1, 4, 4, 1, 1, 4, 4, 1, 1, 4, 4, 1, 1, 4], [4, 1, 1, 4, 4, 1, 1, 4, 4, 1, 1, 4, 4, 1, 1, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 5, 5, 4, 4, 5, 5, 4, 4, 5, 5, 4, 4, 5, 5, 4], [4, 5, 5, 4, 4, 5, 5, 4, 4, 5, 5, 4, 4, 5, 5, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 1, 1, 4, 4, 1, 1, 4, 4, 1, 1, 4, 4, 1, 1, 4], [4, 1, 1, 4, 4, 1, 1, 4, 4, 1, 1, 4, 4, 1, 1, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4], [4, 5, 5, 4, 4, 5, 5, 4, 4, 5, 5, 4, 4, 5, 5, 4], [4, 5, 5, 4, 4, 5, 5, 4, 4, 5, 5, 4, 4, 5, 5, 4], [4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4, 4]]}, {"input": [[6, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 4, 4, 4], [6, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 4, 4, 4, 4, 4], [6, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [6, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [6, 6, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 1, 1, 1, 8, 0, 8, 0, 0, 6, 0, 6], [1, 1, 1, 1, 1, 1, 1, 1, 0, 8, 0, 0, 0, 0, 6, 0], [1, 1, 1, 1, 1, 1, 1, 1, 8, 0, 8, 0, 0, 6, 0, 6], [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0], [1, 1, 1, 1, 1, 1, 1, 1, 6, 0, 6, 0, 0, 8, 0, 8], [1, 1, 1, 1, 1, 1, 1, 1, 0, 6, 0, 0, 0, 0, 8, 0], [1, 1, 1, 1, 1, 1, 1, 1, 6, 0, 6, 0, 0, 8, 0, 8]], "output": [[8, 1, 8, 1, 1, 6, 1, 6, 8, 1], [1, 8, 1, 1, 1, 1, 6, 1, 1, 8], [8, 1, 8, 1, 1, 6, 1, 6, 8, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [1, 1, 1, 1, 1, 1, 1, 1, 1, 1], [6, 1, 6, 1, 1, 8, 1, 8, 6, 1], [1, 6, 1, 1, 1, 1, 8, 1, 1, 6], [6, 1, 6, 1, 1, 8, 1, 8, 6, 1], [8, 1, 8, 1, 1, 6, 1, 6, 8, 1], [1, 8, 1, 1, 1, 1, 6, 1, 1, 8]]}, {"input": [[4, 4, 4, 4, 4, 4, 4, 4], [0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0], [2, 5, 2, 5, 0, 0, 0, 0], [2, 5, 2, 5, 0, 0, 0, 0], [2, 5, 2, 5, 0, 0, 0, 0], [2, 5, 2, 5, 0, 0, 0, 0]], "output": [[2, 5, 2, 5], [2, 5, 2, 5], [2, 5, 2, 5], [2, 5, 2, 5]]}, {"input": [[8, 0, 8, 0, 4, 0, 4, 0], [0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 8, 0, 4, 0, 0, 4], [0, 0, 0, 0, 0, 0, 0, 0], [6, 6, 6, 6, 1, 1, 1, 1], [6, 6, 6, 6, 0, 0, 0, 0], [6, 6, 6, 6, 1, 1, 1, 1], [6, 6, 6, 6, 0, 0, 0, 0]], "output": [[1, 1, 1, 1], [6, 6, 6, 6], [1, 1, 1, 1]]}], "test": [{"input": [[2, 0, 0, 5, 1, 0, 1, 0], [0, 3, 0, 5, 0, 1, 3, 1], [5, 5, 5, 5, 1, 0, 1, 0], [0, 0, 5, 2, 0, 0, 3, 0], [2, 2, 2, 2, 0, 3, 3, 3], [2, 2, 2, 2, 3, 2, 3, 0], [2, 2, 2, 2, 3, 0, 0, 5], [2, 2, 2, 2, 3, 3, 1, 0]]}, {"input": [[2, 2, 5, 5], [2, 2, 5, 5], [7, 3, 7, 0], [6, 8, 0, 7]]}]}, "faa9f03d": {"train": [{"input": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0], [1, 1, 1, 1, 0, 0, 0, 0, 0, 3, 0, 0], [0, 0, 0, 1, 0, 0, 0, 0, 0, 3, 0, 0], [2, 3, 3, 3, 2, 0, 0, 0, 0, 3, 0, 0], [0, 0, 0, 1, 3, 0, 0, 0, 0, 3, 0, 0], [0, 0, 0, 1, 3, 1, 1, 0, 0, 3, 0, 0], [0, 0, 0, 0, 3, 0, 1, 0, 0, 3, 0, 0], [0, 0, 0, 0, 3, 0, 1, 0, 0, 3, 0, 0], [0, 0, 0, 0, 2, 3, 3, 3, 0, 2, 0, 0], [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1]], "output": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 3, 0, 0], [1, 1, 1, 1, 0, 0, 0, 0, 0, 3, 0, 0], [0, 0, 0, 1, 0, 0, 0, 0, 0, 3, 0, 0], [3, 3, 3, 1, 3, 0, 0, 0, 0, 3, 0, 0], [0, 0, 0, 1, 3, 0, 0, 0, 0, 3, 0, 0], [0, 0, 0, 1, 1, 1, 1, 0, 0, 3, 0, 0], [0, 0, 0, 0, 3, 0, 1, 0, 0, 3, 0, 0], [0, 0, 0, 0, 3, 0, 1, 0, 0, 3, 0, 0], [0, 0, 0, 0, 3, 3, 1, 3, 3, 3, 0, 0], [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1]]}, {"input": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0], [7, 7, 7, 7, 0, 0, 0, 0, 0, 0, 7, 0], [3, 3, 3, 3, 3, 0, 3, 3, 3, 4, 7, 0], [0, 0, 0, 7, 0, 0, 0, 0, 3, 0, 7, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0], [0, 0, 0, 7, 7, 7, 7, 0, 3, 0, 0, 0], [0, 0, 0, 0, 0, 0, 7, 0, 3, 0, 7, 0], [0, 0, 7, 7, 7, 7, 7, 0, 3, 0, 7, 0], [0, 0, 7, 0, 0, 0, 0, 0, 3, 0, 7, 0], [3, 3, 0, 3, 3, 3, 3, 3, 3, 0, 7, 0], [0, 0, 7, 7, 7, 7, 7, 7, 7, 7, 7, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], "output": [[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 7, 0], [7, 7, 7, 7, 0, 0, 0, 0, 0, 0, 7, 0], [0, 0, 0, 7, 0, 0, 0, 0, 3, 3, 3, 3], [0, 0, 0, 7, 0, 0, 0, 0, 3, 0, 7, 0], [0, 0, 0, 7, 0, 0, 0, 0, 3, 0, 7, 0], [0, 0, 0, 7, 7, 7, 7, 0, 3, 0, 7, 0], [0, 0, 0, 0, 0, 0, 7, 0, 3, 0, 7, 0], [0, 0, 7, 7, 7, 7, 7, 0, 3, 0, 7, 0], [0, 0, 7, 0, 0, 0, 0, 0, 3, 0, 7, 0], [3, 3, 3, 3, 3, 3, 3, 3, 3, 0, 7, 0], [0, 0, 7, 7, 7, 7, 7, 7, 7, 7, 7, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}, {"input": [[0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 6, 0], [0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 6, 0], [6, 1, 6, 6, 6, 6, 0, 6, 0, 6, 6, 0], [0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 4, 0], [0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0], [0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], "output": [[0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0], [0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0], [6, 1, 6, 6, 6, 6, 1, 6, 6, 6, 6, 0], [0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 6, 0], [0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 6, 0], [0, 1, 1, 1, 1, 1, 1, 0, 0, 0, 6, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 0]]}], "test": [{"input": [[0, 0, 0, 6, 0, 0, 0, 3, 0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0], [2, 3, 3, 3, 3, 3, 3, 2, 0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0], [3, 0, 0, 6, 0, 0, 0, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0], [3, 0, 0, 6, 0, 0, 0, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0], [3, 0, 0, 6, 0, 2, 7, 7, 7, 7, 7, 2, 0, 0, 0, 0, 0, 0, 0, 0], [3, 0, 0, 6, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [3, 0, 0, 6, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 6, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [3, 0, 0, 6, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [3, 0, 0, 6, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [3, 0, 0, 6, 6, 0, 6, 6, 6, 6, 6, 6, 6, 0, 0, 0, 0, 0, 0, 0], [3, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 6, 0, 0, 0, 0, 0, 0, 0], [3, 0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [2, 3, 3, 2, 0, 7, 0, 0, 0, 0, 0, 0, 6, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 3, 0, 7, 0, 0, 0, 0, 0, 0, 6, 6, 6, 6, 6, 6, 0, 0], [0, 0, 0, 3, 0, 7, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 0, 0], [0, 0, 0, 3, 0, 2, 7, 7, 0, 0, 0, 0, 7, 0, 7, 0, 7, 6, 7, 7], [0, 0, 0, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 0, 0], [3, 3, 3, 2, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 6, 0, 0]]}, {"input": [[0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 7, 0], [0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 7, 0], [0, 0, 0, 0, 7, 0, 0, 0, 0, 0, 7, 0], [4, 2, 7, 7, 2, 0, 0, 2, 7, 7, 2, 0], [0, 7, 0, 0, 0, 0, 0, 7, 0, 0, 4, 0], [0, 7, 0, 0, 0, 0, 0, 7, 0, 0, 0, 0], [0, 7, 0, 0, 0, 0, 0, 7, 0, 0, 0, 0], [6, 6, 6, 6, 6, 6, 6, 0, 6, 6, 6, 6], [0, 7, 0, 0, 0, 0, 0, 7, 0, 0, 0, 0], [0, 2, 7, 7, 7, 7, 7, 2, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]]}]}}

In [ ]:
import glob, os, subprocess, sys
assert not glob.glob("/kaggle/input/**/*solution*.json", recursive=True), "answer files attached"
wh = sorted({os.path.dirname(p) for p in glob.glob("/kaggle/input/**/vllm-0.19.0*.whl", recursive=True)})
assert len(wh) == 1, wh
r = subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--no-warn-conflicts", "--disable-pip-version-check",
                    "--find-links", wh[0], "--target", "/tmp/vllm_env", "vllm==0.19.0"], capture_output=True, text=True)
print(r.returncode, r.stderr[-1500:])
os.makedirs("/tmp/vllm_env/sklearn", exist_ok=True)   # numpy ABI clash in the NVARC image (see build_hybrid.py)
open("/tmp/vllm_env/sklearn/__init__.py", "w").write('__version__ = "0.0.0"\n')
open("/tmp/vllm_env/sklearn/metrics.py", "w").write('def roc_curve(*a, **k):\n    raise RuntimeError("shim")\n')
cfgs = [p for p in glob.glob("/kaggle/input/**/config.json", recursive=True) if "qwen3-8-27b-fp8" in p.lower()]
assert cfgs, "model not attached"
MODEL = os.path.dirname(min(cfgs, key=len)); print("model", MODEL, "| candidates", cfgs)

In [ ]:
!cd /kaggle/working/ab && MODEL="{MODEL}" VENV=/tmp/vllm_env TP=4 SEQS=16 KVFP8=1 MTP=1 CTX=131072 MEM=0.90 LOGDIR=/kaggle/working/server_logs bash a100/serve.sh

In [ ]:
import subprocess, sys, time
CONDITIONS = [('explore', 'medium', 32768), ('explore', 'xhigh', 32768), ('explore', 'medium', 65536), ('explore', 'xhigh', 65536)]
cutoff = NOTEBOOK_START + 10.5 * 3600
for cohort, effort, budget in CONDITIONS:
    if time.time() > cutoff:
        print("cutoff reached; skipped", cohort, effort, budget); continue
    out = f"/kaggle/working/runs/{cohort}_{effort}_{budget // 1024}k"
    t0 = time.time()
    r = subprocess.run([sys.executable, "a100/qwen_ab.py", "--cohort", cohort, "--effort", effort, "--budget", str(budget),
                        "--concurrency", "16", "--servers", "/kaggle/working/server_logs/servers.json",
                        "--out", out, "--stop-launch-at", str(cutoff)], cwd="/kaggle/working/ab",
                       capture_output=True, text=True)
    print(f"=== {out} exit {r.returncode} in {(time.time() - t0) / 3600:.2f} h")
    print("\n".join(r.stdout.splitlines()[-6:]), r.stderr[-1500:])

In [ ]:
!grep -E "SpecDecoding|Avg prompt throughput" /kaggle/working/server_logs/vllm_8000.log | tail -3
!du -sh /kaggle/working/runs/*; ls /kaggle/working/runs